# Nutrelle — etapa 0 da quinta execução: a caixa da refeição em cada foto (seção 14.3)

Notebook gerado por `eval/treino/kaggle/monta_pacote.py --kernel r5a`; **não edite à mão**. Baixa as fotos dos pratos de treino_v2 e de todas as bases externas (inclusive as partes de registro), roda o detector OWLv2 base (`google/owlv2-base-patch16-ensemble`, Apache 2.0) nas duas GPUs, com a reserva por elipse do OpenCV, e grava por base `caixas_r5_<base>.csv` (caixa, nota, método, janela do guia, sha256 de cada foto), as folhas de contato de 40 fotos (`folha_r5_<base>_<k>.jpg`, com a regra; `folha_r5_<base>_so_prato_<k>.jpg`, diagnóstico) e `resumo_r5.json`. Nenhum rótulo é lido.

**Versão do repositório: só bases liberadas** (Nutrition5k, Flavoria, LeFood-Set, GT Mensa). A completa é montada fora do repositório com `python3 monta_pacote.py push <pasta> --kernel r5a`.

Kaggle: *GPU T4 x2* e *Internet → On*.

In [ ]:
# Parâmetros
import json, os, subprocess, sys
ORCAMENTO_H = 1.5
SAIDA = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content/nutrelle_deteccao'
EXTRA = []           # ex.: ['--dominios', 'flavoria,mensa']; ['--espelho', '<pasta>']
SAIDA = os.environ.get('NUTRELLE_SAIDA') or SAIDA
EXTRA = json.loads(os.environ.get('NUTRELLE_EXTRA') or 'null') or EXTRA
for mod, pacote in (('transformers', 'transformers'), ('cv2', 'opencv-python-headless'), ('huggingface_hub', 'huggingface_hub')):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pacote], check=False)
os.makedirs('/tmp/nutrelle_codigo/listas', exist_ok=True)

In [ ]:
# deteccao_r5.py (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/deteccao_r5.py', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;HbYhzFhzjh(bq-BQwKy4>SE$m+hy=@aif<Hp(x6uSQedqEo0ysxn_N%4%ns$4kyqM%f{Q8hcU>P?Zi%CUFlOyx%q6
<KX*wM+ojI<~Y~xyJd>0y@RxmSTd(@rP|(<Z&AW9kOu<*`m$&O2jikfTq)qU_QHJ33#ExTKx0|5N}g)zl%2yPeM6db1rHNsI-mrdPrT-;d!y(=YrJcz+y*!w
Go?}tH3*U4f$lf5pLAQ&F~+woghcn(c!afk)qCLDb8w3g5a?TRrFTITtwYOmI7kOm^dyqD1h%wpB}ziG?iX+}!<z|ShRx$F9oG}Yu_U<*D&dLB3Ob{-qPIlX
kjKO*m<)%M4M+?iLhwhA39*{Vhe%n+K%W54*|oj}3Hbl}_5;D7PtG^<?QZ0q8UD5L;6}XUhq0QV1+6y6qpCr@)s}DBpi)F+C*RJo?xK~guj%O6aw{daAw%UF
1cjkme4KiN-~90;+?nuL5dYh~hCj1-h&5^GK6@#>lpOn_&(EF*yky2J7UG8!0ujcBaL4lm@|yCWaSY7?1!83Qsg`QdzH5#&A=qTw4C@!&7ZJ8wi~~-0m9PQ(
VaGx~*j!d<dn{Rby9i|DN|z7w`R<d$u!!y=EZys)#m{}<<olU!m89dixB$UdihD?XOLv`w6I#JYs;=n-`0IG7NiR~4Uxy)oGwkhx=(~@~@^BKf41M>hFYMBY
7#)^x3H>nVNl^4vFffg9WA_0vOCwz+o+|IZL@ckrM&GCW_VoR!b2x~;4fhYpF+~&-t{BS}c{QL?-|yUDZ#^ZS&X$mcI{A&paXwn@jNn5u{0h@7FVd7YK6GtN
u_?u^ikF)v>$&Xp<!+np(m`JQ-xb&uA-VU;;P_~3xUU|x*`BP!^VA3<6^gr7fhilp5112e^=FH#xicXTo@Ak~vX1d7;^3ry=b&`B^a5X@e1M6bo{G-)xE+h$
g_NH5S*ZxzkQOX6ZZ~&v1E*g!DEVuYKCEnP1Y(w2^vB77H!w)ga(fK}?`r?zl@X>5ARl>^@OUs9VqC(XbAW&I;j9yAji0MSE#M*<{Tlm^npSw=8W2WsEvElL
3y{EZj2!R1X+pmJ%xzbsC|8LUtHV!U_S(sd@z-{6Z!Gr!^Macl!GW6a_Hef+#vKU|UHGAxi3*{Edm}Ns%5;M2A=VFxnpQo?-GGZd*=-)I(dqHBPSGzq`FhAx
7_yXe9EAhZ`6JZj(c!rc9~{aH19TLCajB?l#lLnXO@yVK`u0<4Lz_H6GRR^lE4Wnx)kMVilUrE3kaBs=7Mo_v6a{Q&zT|nc<SZ~Usu^d~%S&9azTjrvcX+I;
E8klnQ$sMTNBR8yH>JS6DPc?~y|UzvO?WlAq;`TB2!myy9-W{aZuPUMU2o?`>C_GvAtWa|z6!<J{WU9|(Bzu&9Jpb}p=%ivlf0lsI}`G{dS~xqDYF!(IAH^)
V(>WOa58eB3G!B#HRC5XR~w0~JV;FhP=h=p!_+F|XC5ylV@e{xUx1y%K-OSzSW8$zzKLq$L3Roro~U{Z64BUuAL-dIA9_>g)>zvVTY|NkkUffJ0&BK1R$HFF
hRF?Tq0Ds%n}wDF`7ov2&uOR6uV$>QY&&U$aetI7&4{t-_5xXy0R^7;kt}1BmlmvUd~BBp;YX)PeV-_DWxaHL-?}PB%&kprWE4C7ari>d34cvjxuGoHxhBkd
OwfQ(8r;W(U}st=ZYwL#aU}*!h3ssnB2A7IRsx?lJ0BXLTQ07M58Irjizu;mWCe}h=^1q#Qetma-&smnzCx(D_Mg3<Dsf#eXn1-CeJi}v2}0&@+vO5GjL&RA
+Dik=sV%Cj5ObK<eC8t~{@I1;tYWrY=+{{vyUWZW%Y{n1ZgdPH=HfhpEX5j1KLY}2GaCx4{%QW8y?93GbCBe`^B2G>?_h}hZte=NDI5UT1WL8zr3tyyDfDB5
i*~0uS*)H4{}!?m_De-8VTO}h+X`nl<zknB8->nRa>0m7koOPFmG)iQGOR}K4I+T{0}~LYbTyvKb<nkhh_|DT5x8~L)~6W91xf=5uwP9C@dRiYhgp#$Q+RDq
*#U~<1pgIF#)SnGXu#8Ir0QK1ywfI!o3gm=DQ?fl8&X7cJ~CXuZn}NDg5?`kL2ncA>Ae8j^5z#ykKEs*XJD0&vE0kGa!|Sg1FDczah?EJ;q-XJGv~MrIdYV`
Kv@b`?!m#65!>AUiLbSGtZn}ML|=t2YD&__apg^fy^mco-^Ro*wtQ1>x`W$qxO7-<ig!`Yxy5=WIXbytVblGta?5O&AL41O>JBI<e|Gb>(@)l4c`!IrwfVk>
VoChtD#GNCKu|=3Kl<7NIhpvsFgwmaAv6@(4xD;iC-b@vdY7sxLsagY(G?Z1>b3Ba#ew4bh-a#a>VQw3HleG<F0l1$I??FlabFk23Qv7Edf^&?NZy45kM0&_
anSPMGayik^{nHF+@x0d_Fh+JG?Jon$lMCTLH_B6IY$pC<ZH9h>Lx`K=~Heg%;6RKj(V+l2k-I-^A-FlJV&vijo77+oM|BGeeFiwn!piur8is(*m7G{RexXf
Sz!t{?1B~h+XUvplgE1{Cv6*d%O8Swt_j+3HVDQeh*6p<q?btobbn=fE*l0v)8Y26S8kcT;x27HQJCcQhTs0sCR8;(htO=5^NNJjzr(2<0oa|C{SQNPAV|qm
n=ggA4_F5G5H3@e3=wnDI^koCt?2&EoRef|6lH8cb`c=aVF_Rn^e({4m%aB*3OP029Mk8VGkcn&6ouw-;x9<=%WdtLA5~#+Y})~4_O;A&m65oMsii4E$(W*9
HSPuNp2TeNB<QXDEVWAM&HXs6R=wVi{M9T)tiBN#y}3#oFw-gCq)u}4%t+*<_WVA6jowMh_3Q7?|B80rckD+t1$b!x(I@yFjEH6SwQJFxOmPYjF)6Hlti}-D
88FVEfQ6ar`b_QA@q5OxAn)clLjZ{d(6Z+sT3g>WIPlQcw2q3ReqRm|GulYPgV>P9aAWa`?mHbvv5S#K4xaQRJ#6X~2NUGeKS+<AGMODa!z@4&#$F0nkC`30
%qh~~hPodXn@Zmjk_)i}GT9%slw(Zg*pF{+&HilMgp@djeRAja<_Xb^Rt^=m&pxoFI@dy-Yxj-EW1k&|JJrpWxhcDtMw=Rmx=;G)mez4R6BAQL4V{|Qh2~@$
a>y&}Ghkk>NtYr7ws1K^XQj2q?-OF7s{HVy+EvLLgE^pJF(|YL(UC48#Ejg3HrA9D=1lmU;)$+QH%FQ%uX4dtjHhDuk;?@;EZ*uP!u1LP^4Yj6`<6$pJV&03
ty4h2G`^=95D6kM@&ClB$A$A~A~s09sM(ywroya0w59`X-l#lRc|@k9K&FIXXut*4{yPKMyR70p0_h@_X~?%%XAAzaR}_5BZ_L+p4zeyLECsGYSZhO49DC?(
`~j0X%{}B*crSw?N&{WQC*eBRUcSCJTyHTXWJ#W~s?Wh2yvP_20?kkXD8*eblU)h&$JnBO-7tbmY|1vJ%<EQ6;wf^$uW^KEMP)@)^FL?qF;<(aM34fJL+iO>
$3lpw`c+*jZ5-1%`D;3fBCZT({OL3SO5-z9u#tE5ut@p%p99(IC&ESsy3~;&i0fq*5Y*^c&5NqBPPPZh)<|xaAEv5us?9?2@>XM2P4nu;j0y3fNF-wS#-tBE
ZW7*5l%BVMW>qmhx~^kdJ(byJuU>W9XPk^18m2;*)&X59MF*S8cDM++(iZ3|Q-D7}F|=v^Ui6@SG!14Wp`^2J*?b}MptV#%=^U4qfB~I8C)yVpK=DGoWKx{H
`o$Q8=TviM8O{Oe9&!lp>k`5zZ!0uiPF!h2M=^BaxLyyBc^N7{6ph2v&!(t)IGp=(@ySUPw*!?oReurls`W1IPM!6LWNN!0(`vb0m!%I)H&-e5hv^S|4jhxx
)-vbaCduf&wN#NZEC@}G_^~?+ri!4!DzQI8&?#%&hgBZ@6bpDu)a)mu_+)5tKL=PReD)9{i{d?C$qwJm$&}S20G%xWh`<cTs<Oa7t9WugA7y5;$qn)@u+juU
mbZVC6KQ~8k`!>6_0Kl|5x_nEz>wD5OzgVvxnM^hwhY+G%jAr)UD!$}KP>{J(9(y(clHDFVsgrd*|`Om^&s6Ps!J?jeKE|!h!~rrfyox$@ybviAbt=j{^bA9
>$xfrlA=bu)453#raoO>bxG#0%JYsbiYncT7BR*;qIVy`hGyNJs_;PuMWZN4Hmr0Nb_#0rZ3)M0Y9m`?nFLvT0Q@qCk48v*H^GcpTXN|sLIT$NNj%9Nj~*#|
5Ui<|`eBV~6t<)IjBtz=bi6xKYyUv3T-0eC5Ymt<NcIg=jcBobD0NKKHi0j5GWuVa80{g)Worp|u>SF9i93n#w=uM+$^f>8Af{KbiPU>vL#tH|V`UOpo%fAn
$)>cFDE(Qc?X?sd4U&YG2I-VF2v2LtOlm7o#7Rm1$9+?5g_Nz_7+%(aj1o?u+@fe#6o9Qq1sX@beLO6?(Krx7J~)N)kFjkFlQ5%M1@6NPOzn`(PJ^AkFt+8H
VZ$C->w8s;*ZF@W%gPQYg@SA%rUsJ(nKnh}pQLx<cT^Jxoz4}<1P6&d;FDS`EzRYhqYn))Xtx(t!=qH@mq!eoHF%7oF+xW}XBrof4IFv0M%zFVmHRgfh7cB{
=;}MNI86Cb*ETu8xx44Ho3j*SehXt{#F%9>3MF(D=lF;_RA<=4G^mlI85O<tkKj%)Pz$c>+igX`bUSD&n7Hnq4WO|Z`gonNio)aHN69u3Az-fbZ+_&eI#7k-
n@~gMZ9HdI9ZZ8aoEfA#srBzH*)(r?YBlbMI*xY+WY%tLC8;yMg8ZFPd&siqo)a*@JL*^_pyP1;QDyB_K9j9u>U4zOcFj<Zq!OZbqFc6$TZ;53g*0my-GYOv
4=ck@v7^FL5oOVBa8P==*??r|Ol#dD8fp~$3;;~)n5}U}m$0q(hNo|?LbODw%wvsV(?C0JLFpM7fE}wp7<rBZ2`*KAEEG~hlpNTXFBx52JePnM=nAB+Fci~u
BWzf}ONe066^O*$)3CqK3t<?RhCIf?2!392AuzysrHGVdVpTQwn1WmAW^Hs?P{iZuv-HFrh5WGw<CtBeey5paD?lN@g~lGLf6`I+Cj_?H5=4^N5<-Fw3Az73
G=_~%F4WmhZaQ6IB~Z2J83$w-@~y~<wX@t^ad<GGG=E60rFaOy!BxZn2U$53_)t!}ZKatqYnMOG6@bOWONb|hZX4fFpJ<g6XRDeV<!v-n!{(f34U0t9s@1aY
>vD#xR54}N_}NAbaVnG%!Hh&cu8t2KXuY@1WziDb*Di|#O!2%PWKgyM9VTf;g-9w5?#jd<i|C-?Y(;pxp{~+#3Dxf%xdSycufNR9tj9c831G0{vyuy3oE|I>
as-ex49L^977f`9Wk&fT3hYntL~qSLW&WCzb&=_ccxrxt)00IJ7GumEmQt$e`FVu$(P%LwxE#~HE=#B1J2E@df(njBT%}YeHsd6gt!~9Kd2i@`JN91;>n7BX
Z=}ap4b{%_(jiwA%3qpz5n7}mTT#sO)-Wx9_L8F1-PF%&v=~m?$3nBo{MjSvm+DY3LPPz9I1_Gv%!v2m6e6n19HEeQB~UY4=M2Z%z}N9|)aVU4?h^52sCyDX
mWWNu6^1#IAoPE~%Gz6-aL-!F)x+TsM0pJg0cX)4l~#rzZe=kOpHK`#^an-1k7WXt;U)LQC1^`QQ(q?T3G97Yi+mEsIyr4#^2>Jt#KiMSOP1_#DhV``YUF=i
pazF&)pSODI@1zhjL&+@M!{&Eb;H-Rq>+XXOtkVM%q$|o<y4<0^nvO&<xHjDK|bx-6p3NbH{>o94%7{5fCrRtio_Zf5~4pOMTPcnMJ1-F(O05c77C>-%V>pH
fJO%qHKr<t(G<78g#SE?Zwb3jqmEvt3vYN>3yBH^I{?YMXIGDlw%6F?D`5&vJ12&T&Lt67#@@is(FJC{>FyJ#%UwNjVjRF&X|97A(|SqB6b*9VmUk1$ijmo4
ND8ud3KNOzxQMEX1B_CJ5RQSV8~V$HAHEMTIywTXoj(|1kX#?dY4K+7LL%KN`SSSD8I5k(WDG9AD_dqSC80-Bn5jRH#XQi+1gQRYlFAdBZ}=#t>p$R>4$Bma
-rNDM)*;@ym=Ed(cK$o{pYVt=pxGKHNlO5srE9Ai(g(4XV%&mPP{9cV6$I2(55a2+_*kYWfZ(8z8N*~DcTr1%fq>t^SmbtE{`Yhu)<q8=s5Z=4I(WhE#tqv=
p0~$5|M-WZB~GY)+~{%g;0g{Fi;rkvhFsI=(v`NwjMhS^cGp>~*3)M3Z`^JdM2%M;%T+lSps8TS`Sf_}iHV8_sLSVw3vChh>c!bjQzP|s>c9Sf#N~qGBQ&$)
7(S$TeAP{`bMNF`eryWC%l5;nmj*)&0r3~BgOtvymE4rmYxU>7`qmjZ=xeq=sW@ez5cHX{GGxnAvN2ix;_tcZ97@@S4G+UO#&J|Ydlf<sSk>gjg6xZmb}M>X
{QC%SGkwWWT;pT+6e076??BABet_HnXH85tPX1S6Y_@)`sFklm%9!}CUv`+bND?mz(B*`!@X-P}e{`b3QZhg2;UzqzOc;WYCUGbbwQ^%uzh#P?ZX~8hy{k!z
C^9KwWNu}tFOb%c6X+eNZ671gOJDz&-2Hil>a88LwVETm_yoGKi-9S*Cse<B6g;%5$$iUzH0_V0!sS}HLN(8SrfR2k)uZqWxl|rCPwZ{^beZ_TtzUe~KS1Ch
<^nSFxd6Ba_!gKrEOHeDTKWkoLkvJiS2VtZ{KL28(50>=Kc{Ofvn|ZoPVDUuwU_OB-iQGVyyM7WE1pFH<GiJOib{rkthWM94_jkd72kuDPRw~&{8e|#(AR$g
;HuaP_jE{=fPILB5~_>M9=$0TV5hU+Tzu6xCTzRws+RF7rZtH$GLAa<qSRk*7HO;^bfJjE6+@;gCWOw$XCJgdP?mQ_ZhYj2>&Eq<jO}=^braR4-rzJZ!bJkN
VKff|SrU`bBb(asStbw*x{5o@Z+3E`5t_{g&ylUB?3DgMiLW1$cC(P>d<xn8+W2u=H%_GF6B1vIU}87#vCnZY?h#2X!}BH%@THC5B!fo}Wd&z*jGu9<)(=$9
VwR>`46HZW+fcd9$@I!i+x>=}p*N3M`ldf2xrlguryq#N0fV(Fe|jTJ&V@vySDcU40l&a_J7jX1tTB%n<SQhu!Y4=$XAoRUi{Q;DNryKc0!>B3YwXLsYVRVP
K~D&&2Qd}UFs-J<gA}&^w?i(4R4p4;P{+%EWXvE-!XChR7o7B3Du;KJIi37`8}TN+=?1ZivFM|Fp4Rt{uGA#@S|F^j)9o7D&dCK5hU=f8BvDc_bi|{*m2)=G
ZRshoQ9r4|8F5wJo%~!UXnOVgk;sg}rEflZS1Kx^?^sQ<<}%sAsVh|D^t0V~^_Q1nZHd;(8Bd7q7dBz`F-o;lQzJ-O>30=r3qIO9J*+4dp-~IUPvbaOi4vCP
{a%e90B`!*zOAxX#G~RtWVw;1FmV7V!Gr#~Dds$@z+Ul5SwIQElE|aqHMoXvJXPrN%=jfpc13;y3<qUA{n~UOIBAr&Cm72Mw&8m;ivZ<q`mS)TC0KleH}Aob
5JiOr?PCl()n9SMI48tDyy%;al;$)x7?hc{_g|`*5l4u>vOC?J^y4D&nDMR$z=&RkXX@uky;7Nu{!HK_!^0W@wDG0<C?;?U{BIm~xTL8zF#KZ@;)kwP$bn`2
<F1(3c)_tp^NgC_l~|NH*Rq+m1#}tZuW9N@6qH4|APOIDXVC5Iw|kzUQwcM4?0~YjzVfw8Q8oBZGppOj+a+JerYGS<Z89E+4#Tg+RL`DJxHD-ZXSN7*FW0}Z
<Zqz?Yq%}7=_Xm=`szmZHh!oq>|bI#)@<-efpch0qY!rYyfI-oSPN5=SQhZiK;7c)8hdi}nI0*OwMYlR#RAWL<4D1#GNOza4;3H?x}ZsxIIaK<O9w#0?L8Rn
2smBSh&012jFy37K|~dgc~MlZ2_~Md9!!a%wz`({?&Y#@kbnD=9lK#bXQ(V$p2=Vbo9kGzb_2Rs1=bIjUJC{GnA<IDG1a`g__nvKqO=&!6aWn>e4*M5H0&Uv
&M$!577s@YvDui%q^<Y1fbJgnOMy8bL>7OARG~(SQH#iHZ6xD)qm#ZM?6@XCT^OfQJ{p__slu#{>J1-y`hnL*Dvxc=>;*`iS$6tA;cj;hx-FcV>XA>bVgU0$
l!z4vpxkuz@}B=h+C-+$kXUHfA&GvA&hjP`z?$jC|M5SYpQcQakJtTl`>DN5=#!3l#ef6X&>&H-4OZr7*>n<ZKsxYSFl?vHCE?J<gu~!kWi}W1FSI(t=_lW;
ov5+mAauB7Tp|J*uAJXnYhuCF?$JTECeF+2ctPS4X#1h$gyq!v?_C5|Avf?a3u3l6GWN`1Wq;iIJU<eZV0NUIF_F{~7tR;WjU5f){Vi<I0IEHADO06z`op~?
4{cj|_o2|I^Na2-XsQ!k(15}`WXV=wwdi*%)+OgVp6nukzb^RCLCR5bcTH=G+8iC-%pYEdPp-iEhgN$J5x>E%m6u@jbvLHsP_g*7_F@OnsaJw1kF|;okVQvd
j#-^h*4sfKL*bmv`fWJir9x0avRf-S#*M6{AAYLQGQp;wL)GEknPlAa4*yBQ^j~1U&aD0va$%L@ZwX=+AIyun)Eo%1KE3D$Z!*|sf6(`@V2Ki6LoBm8J1)B+
TLQLe9sv%>tsd>anc?y=$nI38JK8f499#w!$_$B_4vd6?BvG_A#%8$f(U;P`vJTgCc8uK+AI=%IraJTf%iUsg&QpN3UDi!1gm%js@;}eOgK|ns1u`nptGOV;
=c~NlJs|m%J#gko!#|bj2+=n8AM-g_fSoEEU&?lUsdf^7C|yD{`};%2>y(RCQ58KAH69S~+jT)EUCS87&lam?OnssagLU|Nb`{TEHB$|nS{{iWA=zubbwf3$
Vl(`@VTQM}&*e*NU8u`v0FJGkHd>jlWdG1A;-A64$c=JZY77TdLQZGT%s6Mk;3=|&T1s$IN;vS9meXuiJtXD%+v`@Q1|OdTpWM0bhM0k#!2d!g$g3sd)RQRC
QJ<3)()gH3vbZ}7{$um$nQ;Ewc;P}64J=8&GEet&Dz0h1jc|1PjO0YCd)7$yl|y#jmGB}wGa>D850X6|$9xX(FI58=CL?7qsCoq}9#F5N&fB5%7moRNaa5iL
eIg)DQ78jrz8O{ZH4Do@ucD>R?+{zRCR4(zkLC8Z)$>6gdh~WytWcfgno3w+*DaO`raW&sO|+<A^Rm`$xfqeICH%)4_+qKff_Yjh<-jUVkr0WlUhwOmD2bh!
4Kt69+xu!+N?t@<TZx#l=MZ^5-AxDxd2EdX$BEVPoK$H*a$^i{ev2o%yEX{*4$+?KuA0C&U~~;73DS1k@E1O-<dmH0epFF28`y(UTQ@;M+8!YjM+}^ha}iyC
eqTOLC}SRw$t7*&%~d)TkAJUEU^-h}zAOMhi_?6!9<-H^qFoZ!r4y5JoVplR#ll$ii(HG6$FbWiLwDkU%;_S*W~UIyL0=(EY@8R6IQzL}#c!Di(Rv3E9ilpn
Q$9)ZJ;JNC$4@EzSX-Ec7iY{3!mB^I?n~GixtJ0F<?dobYBaJiuiF=M<QqAM*c5#+*zX|3-lE_h{v8e2Xgl**9!4F*f=^Io-y{^j&_+gYurSK1<jnTDFwXCw
U!!F#=z)dT8?*sDpoiKVdLQ2nT%<iW6o>7Mv&JFnW89`vWy*yRRIp2@4E&@_V5f(VA(2p>t)auN;6@~V8GH>M*C|0Di-fkJ_O;oj%V3j;oOqbyf3&r4EYI7m
-9I8bqF0pxS2d72c&@Oi-P{R+G8ZxOUF8eCz}>t`jf;J{@swJNQMJ8Rd}{;gV2Gw%YV73`B0hBoEiH5iTk;$Ecu%BB49LclkcK+jizPR|_B0D+enSp1mF&=K
Mt+k@K4|y|TYWJ>J8==X^A9q*Y>~(>UbP7o4^(AG%{?af@4XGwInqGAve1@KMfIr{>riFedyr|eH%{)fisX*L_7|M;qMaGYt_z;<gFBXa)P!k*w*Txc_T9yj
-c|z5rt5*qziD?lDewIwK%eC?+94+J<gNloGOrInlr!V~tVPxn-D^iD`>R&CT-md?!4h*J+laZ8OA$94l2+U?Rra&cv<~;L8R$uqmKuLI1wWAymreQGe@qRH
DdS_sQl3q6Z0c53&fM<_6;s{7bBe^bHMsb+;C*~YLbT3Mc2=mgeQ^37wfOb_MC$|Ii3PwP#A$g9;A$ObD}8w%^um|lSkI>uA|F~$VM_oe3k;Y7;7y)me1ym-
%6`;jK?@BwDrnjTbNzdo)x6*5cUDGwJkQ1p`sXp*oLs+&tAn?4oIAr#C5mFV>ov`v6uW5zDhN?N*Cf;t(Nn1jSvBC1mCfE+waHYHpsbu~yMy@Ey-i8IRL&mO
XxR)Ic?*c6N_tP~x721xwx(Tt)EHpe&2{0fL@r)R8agJ%VEpg`-&Z6W-w`d{r3OklRi6GyQM+Znnx?df(hCl&lWo4==Qo^G2-%@vBEh(TE&^9}NQPeo8!3Bb
JmitD^7-a(TwozY<RCBvd0MAi*hIo6rN8ft(w}yeTZm(WM9{7R4_5=OF~hj^J=e4<W*Z^UqQ<>aFtcc#gjeI@<Ur4-_=THJV$q!FPCx8Uv38nJ9D?kI?_k5n
5x*LzaX{3yvjfiG3l_$m46DeglF7$zNUj1rI3>8fAxL14$<1X29H`Dn`B^@uiD>h#3qC4GEj*TVh~yWzE|QI2PWK<{>y`ag`W@wuIvk?nFkTZvMK%obX~!+P
?YP$|_#Fj$pvvIC>_yM^6{cH`e!s`$J&0f)=f%L?watrUr}h{thjdw(j7_8#lAjGM=c8`6%}>Jm5`=^Vw(siwtB$)X>uobagSCzA05c4_w&ubi2X*8_eOe>1
>r(w+^KFj1-l7@4D-x&@uZ01-ISPH5CQUyvxNUUALnlku@QTu##^+kJQa4FQ5{C@5P5xa_R0i?W-1NrgY(fn|A^yI<2qHDrS*tsw(P@*-gDGY7#3}D%r$J5d
RZsq&C$4EO@N*W4$lF0Eh`AW>tEmh<9Kb)b`MF7S6<v9O@Nq_S`9awu+b?}=%^m%skkV>a@SU-NaH|bHdxRc%pn1#B=9;#8Q44hEuF&fQ9ah_aP`Z8ga;-;m
=Rza)z2zL_`0?Jo7(IF1_{p(!pf&?x1G(4|H><CH8l1e&H`58BG9oFH0BEY@fsAqlwIA1}pBbIlb4xp77CHqR<eJ#3V&}F68q*n<$cO?K{_Sq@lhmCDk<zPJ
Wcj4%0D$DOkmW#Qkle!?Z>qp;J{=0?_Bk6oqQEXzUsVk1a&p;);m^3A>qWvS<ajaERK#E0#p5g^y+!qjDC`NplRJNvXopCkc3koLBvsmC<cIaJ{UjNPsw?9B
d(5^r_+(o7AJ%IvPinRVunvDCIodv1Y{_phz=SRr<kPCsvy<;z{10Gjz+04f7Aoz^R#|dHpEV+jui>VZ(trwroAySqXtpP9Q1Km*lxxsAY(%{a54&($m{)Vd
;}n7W_o5x7;x`E!x=c+U7NudP#r<tshs@-GHTqRTfYH@Bn(J^PqBaw@6`06+{1*2W{^!c}^r$kRi*S8(6MDM|d5iwNFG3Qau+X2TFP{47vDDlp0auT><7L&D
CK7N&Q0Jv>&IRQ!{>}mZ!RGy?PAcnLPxZ2ocNJWC!(@LmAfEQz3)a|wyAk?q=7hVUFDZVSlLh+=kngzIs^FxDWg5yQ&~|9pHw-`wLaU7lp}Axzt+c_wq4fD!
mfUr)f@vnCu&59H8-=S<_+_ct<ozk&6U%PiCef@GWd^kb>B0r<lBZy{ZALg>@nrCmkJK(R2z3h)3S{LCn4=!Ds8{u1h%TCRNC^cUDDYHwC64#=*)^_+TXw{i
J#F04I}qEsG?Kfct6l`@mpxvGfchvBKl8#8AoR{;qSDE`1&k}&fayr<o*-zTj>QH9-%kNK!rll4y<NBur`Si13l;kSpMxukNE~O}GJG9zaao;q>%%*QTW%-0
|2ObU_RA9SY+1Y&;G=B2A7uh6xH(o?O=}&&HUoaSo?EyH<L#y0rP&OI{R@_=q?hZxHtzye$0WtpxPFT^FHAhBSQ4Ph8*+jj&QrnFOwr1$z4sS#MYp9i;;6Mt
m`-ChFs1M=lHL4qtmoiaz3jSR2THB5MmZeb(}6gw`q^^TDp~sai0BKJk;ZJIHj0M@OJzDS?|lTvZNbw0z=o(F!jha1oDOe;`C@J_s1F{MqCPEp1E(5Q&-!CU
%&Y^6n2B<Nbr`%RtYJHzN~O{tljm!W?Yb;E2WIdPivLv4#{Z~!d%JEX6iBn^FGXDR&yWGNAyY9t7A1u)DnqPTkRUssr|o$jVbmJi&-$5is=J0T$a{6NIV%9t
=pzG@y#>!k^Sd=Wy=La_umgH&&&ci8La9BSmfwu<+Y<G(LFINoy+={VeIcetl0IjJ{piz)aGGVZg9B_~LSX)=nA3hlz%CD1-LWxyl%+1!6;s+*^4WI(tSwD~
)!lSaF`^dO{;=p0MgjD%ba)md5=(tGnCEO;Q?d`q`4Re0KC@xr_AS3;t&uk5_n;!Rd`YOa5H@Eg9C`~Y&C<x$dywmobeDONbH%}~FOU1oFMf*-BbIgQx8{x_
sp7ES=!0vujw*a%r5q;%tie9BN)V^ijs=2D1E)G|^AY1CLY8#2f?ShL1F7F=R@rGNe)@}Rl0FGo_dYzX&e~oHR&@{-Y?I`2iy<gnfC||3&X-Ip^96gjY>bYc
+D`^UWA>-lMO!*^GjnHx{rJE<5->NjeW@&6wrlxL8=v#ed0BkQzbMW91M_^i4=Jt_`mR&*+4ob<G~*RmUQ~2auY)(nd+J;8o5+Xgt+|(^89Qe!_0qMt9Gx@3
Yy`#PWJ)xy8xn6ECC{g!9Kbs?js)nn6>4jRsI#!fuFnW!iEs)?Qt_?`++Hq;ee(aPL2DL0>Z5_)5MuUIV3y@=KZ4*jF7VN|{5(9)^?%(*`7U-wJNrFnna;=u
EUK$DF;AO9doUeSD1|0KCGBOgW!E++fqvS?sxlCP8oC@+JwQrMGM7FY&|>UJy^$Pg3+DRI@N)R$Zj}JQI3HUt8`mlef)qByZgSin@F)QE#+4onnyG9d<ryHY
>~M*MvtmZ6EPvykIc0S*2>7K*s--7dvdI^7^}+o|yiG_v6h_H<dG4(Wo?gF`p4zTvY)|r1%OlnVL(mLuPDl(^+34(lJN{6MH4HKnz(6!d&^ny>CuO@o{Qd8e
Uh$hWiSl>p+&*?ab@XgGdnpF3y^gO)N54V+G<wv_`^0ntD@E4FM2dz;lL`Jp{?sbj%04niIMI)1Ul-T?fjeOj-1yp<q@zRCg_pTD8E|UUWyFNqxZ4wz9G`z-
dZ6t*1=TJ4#7fAECc)$d`~ev%ecYNDwAg<n{Fs``wKgT1g(dBNLzOSA6{R7TZZEYj?~AhS2kpu`tbS3{TdzrtvDMK=VU%j+5&8V&H0-v`k;l<-V1Cugeb%=|
#zNkOV)5c_s-pPLmtaLO&}m|9z6>M!T#bYzt%;+dnfc%I1gi}}kFwxW?M2dbAj=&HN8UNijW}+^lbimPu`<;4+Px2?&C&G)y37gX`LB($s2efB^Tl{~DS3}y
@2miejv(w^Q%kUChnuZV-+tqy#J$ho5aPJviPLvK+W252KS;Aak3+h<3r$@W8do9sZ%p3FtvMBNYrjAhRC=d0!CA*ofTyIa$?;_-<b{q)7!yf1wizTeAv0-N
=*tR>ZK!V*Nf<GFMMn+&FSm$qT}~LuQ<3OW_RE_;sIfFRaLiZ}n~2HjKZn)I&Q);L*Gc-VC&RBgSKK`1;T2=R{JXt&CL1ojiB;IOZFs<WZpNQD#JbL~nvs7H
g@s2Pu+&DAlF5gziO!rv{EfuE`rQ;pJjMV1M`QH!RRY9b=8dW$Rxoc5OSNoGl|BvRxs40q@99$58<3^tCRUF}3D5ZWyX^ay#>-11YfY(|a81O0PVuaBs>VmP
$zez=Glu^3A=sb=>AC&iEQ2~pBL_<@YAemcmA|Und0mh5I)$oOWT`dB^5fs#*gK{z*WqDjZP5h1f*p`RHWW}hNI*SS5D0tfPYz)vjyzG3wW3IV6?w&4@#Jr|
6-mfRc)%w9JM!_15+w5T$Y55&!wvqs3^Rg933DqjGVa%#OrIrxq${QejVV07`s9Krq&9Nx!_C0gIK)KU`e0(G><35qr5IAIpZi@Q`&ZZAqa29?Z2xYqUjvS5
N$wEdsuhXH4>0f}uQ6s^1@09d9J|&F!f)0zxYM06*r|H-g;If?VV1X)Eggk*e(Sb)KDh~ejvXE`V14_uv79k)pG#F|SApH51wN0TjJpPdUn5QyHSI2}Cntg^
QV6OUSIIKsQa7!ihdHATq7br}{|<>ti$P3{UJ_ipu(&>#`?=Vjy7NfE?nCa7>Ahp;jS66@!z?n-7k+<ZcUqD(I<By`rPEsB+#Wj#j%Z3FlV`tZs<UNN-v#;3
W)i=DNYwvn!B${C3>YUM0iG&dbJFj=@&jT7ZQ!?u`z%N2P$ae|A}b+?otTd+UvPCEi>)>Y>jk)!gFha1No&#u2Ac>O_jN@$RI}NJXzamyRrW1l9{yJkt1-ZO
V<%;PU{JV}0oJ;<Sdz!%46<+YYHBZikdDgA<xa>@Uu<s2aDC4M#lCC=m`2fiW+CZ)aKl4j!&!DNf12Fj-Rgw;a=_S~g>Jqf?+Ful$^%qQw@wYEu-56;1HUkw
V_ZFSr?rr8wPzS?3&Bn2WA7~*RZldvWsafdxH<E(?Mdr;^TA`>AII_v7ji)_C`v)@*NHGmaHj<U413}O#+WrlEC60W1Ui$*{M>Twuyn&TOa{~P5EpWhzZ8JB
88yAEC!TZ7^7zB={1RBz&%4h92*02%FuV_AEs0>0<8*5aI$)`sUnUylqnsmF>>I9$vF#~1fDvaal|F}>U#~2ITiEG!zwq!db^^s6(R(|=r>Ny=MzlMp5E@vj
myrNPac+}eI%0X#!@(ID*5flPjXLp6jwcEuqy1)^3w)?-)*uR9OiaRn@nB$+#l`SbXo%kS=n$OOAyCX9+mApt^p8Jr%heTlx(;9pcuG&&b@=6eAX$2wZs)Da
jI%s+Oeea+rg*gHW+KQyT5=g5(g~<p#0nwclhq~Mz=c~*DsEoyh&;QJqq9bf_Zm@;yhlyhDUw%%4fa0D)R}O`W~Am7%%bG@EnNt=kSisHl5EBYYX#?C|6TE1
d!4^CD{hrfVvF)*cSpU)oiKnA#EGxXH3A!bXvO#)iQL&lcH~ht$$^p4X~e`XgKn^3K)8L!4@nn{<3UYu4%>(TguUUZ9n?PW{<ix}b>&=M)#ril?ozrNtjNOA
IyD9u;GO{Uuz32I1sT})aG7{iYvqEH3F6pjamiOi8nXZk<KlS29^w;!k9_^-YOZ9gcAu)6NRTpRD|5>><s(FSX>J@V3sh3+&l56iYUT~p1bkJJeiR*xdjRG$
@@s5s=;=f6E(yW@JXFp(98gHT8qRVLBhKN8DG9bEyXO1en3X3`^1szMU|+`BvG?jKFSeZUfZ2`>?l|_i=4#HItM$zF%!*Ld(Nq8dGd6e-UbKtH#iN=(aYW;4
krln9j3KF%K?AQ^Bdyb={n8=^ferDRYPucU*r0-I6;WUJCt>?~an+$VCwVGw4WZixmZnU_Iq1upxYLW+MnphODME4TCxN(^NoQTTzxNuI1B;069&jP5jHnn(
9px{}O>yyx$i!Ov?QLaP`WT;~i><F*GKC3e)ToM6dAmcM<OiveS{XN?0BC{4hX6I$tgVCg^^20gJ@gZTyM=?nqVY@9n;-dCq+g8WiJM`Vb?RqYw+;@v9HA6Q
?4TYYH!~nkExV3~;R~f#haQy!V9YD?*^1D*n$(+#Pj^9P4|bht|C~|LmiHFzwpwfm?91=)ArZz+9yt8mQOHOq2lmPdl&jIDm+4a1lQ^GIA_gnjs`#|iEpc7t
Xv$lt;O@jFS(&$$toO(<L7GV-fFs8oZDYKgg^Qwz&xewpB-y4S*_$ly=7D7nyF+f14Cb@z+{0K7_gA{!V&uno^53%wE9Uh)(jvvGE9$<6I+f<V`v<o8k1yd$
9n!oc7SoRC>{W@#{sJ{rx*@)5)%K>;xP^9?8NBppjUbECQStaN|4fCvB#jNB#~w}V<X3a;LS#L*J557SbbXx<&$=2%r9p#>vXk>?*<iv1F-kx(tQBzRlAXXi
FaBB7DWxl*j_Bw2@YvZ;6UhXxYDe3PsO2q*V`a$Ewc?(6sAYjxerW~(({)8X3<#}6=FOd_?o^8{kofqo;MxoVWF1)uwY0_b+VE~t9=QYB<>E}KYWG#?-v@<k
R!*b=wte8`CQ3Tspt-Q*roB&kczCQFn%s>_w1>bNR0n<h(S!K@Yr{=&n8QS`FO>bT=Ev}^s2`(sAs2zMJ>asi>2WsqYIjI%)J$jiFZF5Itoq{*y{Lx1(@Kzk
Y(4IRGrj@u&4{~cB*iD)JA^&-(I!v`ONHJgvKVxyTta#&)T<@`<$!y^%`_mDF8zi&_PS&@Pt6^3EHZv_d25d%yysdaMcQ<SMjmsR*>g9u6g0@*s4BA`pjnNt
Ui5S1B7pLq(U7?9vJN`1+iv@sjSSWT1Bj0?<*g*KTwbQrh4*vfS)&v@<(eQan9pM}iM1Pw*10)D#s?2;Kq;&&sH;#X{-?1L^`v!~I^uFD-b<duETl`Vlel7D
>>$J0YDxOz<J?o|n~HjYdPrddGb^723!QqMrRcPsVA7f+I;3B}bXg7=u<R{pQCzC!gj__IN{kD>s`38*D#6+gdIrJJ$cp`fVwX{>pmiWMX8Wb0?8R;580@$A
ULgm|2Tx<?lz(8YA{FNy>fhspcrp-JQhpEf$K4tnuTn0FgI*z?;}t1wdsIX*kEbU%ek(ke)~OreS)bO5UsQf?oc{>9KGAKMV<=q=qren9-GKiH-)MMMrX~`p
$pzsVGAihZjs$jY>{a|G;+M}k+nch_wD9Evi#vTK;~brqFy4l;)P6tD6BMd!VM}f7ue#343%RyfV0{3&>_)H~Uy47X3Wz4e)D{&9d;_JV4U}VdC7g&;0;9G_
vg0s1AFhqxUA}VNyG1tg_LBH^HNy+Hn`S&ViyNvuQcNYZ{YnSVy|_r{VoS26t1d{kyeKG^E2mJUV0P;Rh6G?blH61y`HNdS@@)qJs5xD|`O-@6(h@uOJ9rOJ
iXPM80Qc-z<>(?{lu9L5h$o@Nmz38!B86;w_YVeYZ-S(#>}%q*38FJ~O2>>EF?w<NkyJ=OiocQ?@L!M#lsIFnqooqofappGb1~bxi#F8|22=hHTde_(`(wdL
t!ZIfr#bncl7HNTQ<%hMu#&1?NhK4DP2g?A#%OO1UzZ#yFWzpm7r~)YzlOY*C{QLyEq76JUdaWl-P5Z-B>8Y&4+|PUQ?w;i#A>XEQ3GL`3uIe|c@Bs0-zv$T
w+Yg8uXPxxPW5CNd+1|Liy#8*c)a+Z1c`ql1i-2Ks>HWl9&pcMp-OckLvs*6MMDMfn2Yelt9dgkd1GYA%gt_7J$u5yVLQGnNv`kh+(hqI`B!qy^}QBLNr^ps
1KgkmGkdM{{<vXRN)hlAGEGvYU_05q(}t@NM5yt{eU|L7k|<<ap7nFrP?S>>cikNQzxuhU6tg3cD`Mt&&Z0J!LlrA#pxgb|iP}H>yOVc0Go!HK9z3{yRtx`v
tB6v3L^&HI8`Ai=H;$eHl|PKWAZUKHsHF0|)Ft8H(z*0%<_EYYC1RL=F#;712km`#o47>xewG|F8VwqwvI!bxe{Jtk^0K;3)B~kA7q<hre2v~yPWycqp9Q6`
?}2b7it_XuDHm}nkYk9|oB3XzbOO>YJqzEyF<Mj6001VNImeD=B^?g)_>+lD#PUf)P%h%BTUaH6sLSQG3`=hr+h!I>+lwuTlCGOO6$^ejHYAxAHMlKqr=(<9
Dv5}I690gwJsMc7WS2P<SZmYNrdbhP>F@{GRO}&OjH&r@4ve)v;kj@_#yhwCxZ>}Q)h|!}?cl2^NsE%a0NKCeJzOyZ{Vsv={&f78yPk3AWF#-g+{ssaTgO&#
L0xv<V@-6jItnWXWv!s%Ps4QHzI2kbtj%FLd3iN$i#bZ4B=p~q4^6nUzlju16Un~6Vu=_>X4RDKjAF~<hZE?E10|UZocYG2R-FdpZrSw!{|~ce-)qL?T_h92
^*EyB5v9kJJ$Tcp3;8d(?8LW&nX!EPI!5o8C$9|FMio~YtM~esiY~<blSL;}E{tII4A9MIq()W=Opc))W5GM9y@{EvzwM2Y#xHI2Ae*1f-p1YYci5$?umAu6
ZFs%EoZTv300G)>z|jH#i`b&0vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/deteccao_r5.py')

In [ ]:
# treino_gpu.py (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/treino_gpu.py', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<IIBZ{#^hOh(bq-BQwKy4>SE$m+hy=@aif<Hp(x6uSRyx75<tfLmlr9v8=rA^vK9U_wKSxGTYDxj-jWATV6)EKC(Hm
YNGRX;bmU+;6s%e2My;AdQ+%8O$p?ZCbjDzN$>!xfgl28Q=I#TvJTAmr^77~D@Zu{uA8j}=Ah_;`+(Wh7Pf7bceFsb$RGZ$;$m)?GKB?9<~?!B1vArY#MT#@
mIAho^Z=^{$g_MwVIU6R+-vlNl8?BI_hHwb%$T%+eNqIhigtVv!`j{?O7Q+YhT=3|SaqgO8$j!W`>k|m9zhNv0uDp5Xr3Q^6;J_^66184ImAO>Ppp50eUOUl
VD&>9$J+8n9FytF5)CM~Sc#tdYxtF`hWZz|<d_JaC{_Y4R^}Ij(EC{*VE_;CXsX}VhcYbuwO8~%Y#jlw@lc?7#bpik-7eRYpnBU+;&;cT?6a+Jgm6*e!rBs;
mKwl`A@b7S_1st7`0v2=*3SJ@q!Ng+Dz436NEB|wU%m8E96_)<P-l^1)8Z?<H?6y_C|Pe<q{q@wqXQX!dJ#C~>+^DJ`dry)O_26bgb{w+1LNuP?4__ckTBNH
8ZM%F+=GRyv@U9Xriv?F0cc*hfgBhv0A&Xfo0dg?gCwhr_Nv2Cvv2xm=Kg~d9_%egGaM2gHrbpRp<p_^7`FByodDM#@>~_1{YUjb^lk%zA9{jw^h9RP*#)bp
rqu7<)>u@JT`MoDkSEXn+zcToA|eb4jJ7QgSU)TBk`U1mEvwZ1d1Mo_5dBC793jB&#7;t7sK70M5}=QTo5t;EWX_fVIw*rY2fC=vCunV^cn(e~faHkK9yb>I
-ID}rWTL!B3<fDBL}uT+Am2>PTAH~+P5RN;$qv?)0KjM&1k&KG3`=wOe3?b46*LwXfw_$!Vv!Z7;tV|2_ZS5IwTQz68?1dAI~=u$OR7y;MGV+<P82%<so7rv
-GxjvyNMxuznP@-EK#>sRosEsNu)Cb1xW%JhNWLu8%%20`kE<9yU7d8@{j{|D6Hq}(Sxywk?hubJq9N4gaTO3$1JX%$W^`NwyMy(s4w1|3h|Jjie6d4YB~Q2
of6stzib(J%>po^qmT#yz3z>W;)88Gf{nX}+&Ym+aeZ+^Y?!J%A~&m$b6|@`AExp#Y%ozq@OA2TY%RD~p1ZZHQ6C>a|BD$+pI7iK5y#?tG9efCY4~UAm^w?C
OGEcuiTXKYQwMR3oZyhdL5qLcA%K2niCI?4&)(T|pJhn6IO}y!0v>kldb($B;WkYdtN}v~tEAWpqbx=NNUYM=>y_!`HMYg#tL7FsfKJGq9TkJH1e5+gtwYkp
jB!kU=|OcOdtkrqqD4H-7p5(EjZpXT8}cI4Pql@8Z@^u1d^XQV7C?Yc!3#W>99B+#I1A2|Xb||1>LcZ+tyg?kNlqg$-4WLW2R>3T$;d_T8HR&?CVMY*nq^RB
9yS6dF*U;T0wwZ%7OvJeg)K_p9ilWw$RyGtU91@qI@yf$P2civ_bRp1ys7pRn>B@~7yShI4Q&?SUCX<DVwkStv)|rJBr?eS4bd=He@kXdP`5)ax7&I_Fw-iB
2e{{U_hvt`P{yBka#iD3@BJIeaP8g`>8gEpxSl*+5bB%-u<>JkfGN!F=ZW12Echb0!#=a69`qoE5I=rHr8KO}`TOXY;*%}e=b47B*N0mlSXAZZFH{e=cd~}X
iAW=wv$>OC1EHtIqXxG*;?FbQ8uLSHZ;!tRXa3b*KAmyB_R_J>J`~jFp}$TN>ziKC;Dl$!ayk{@&-oWm%JoTAuOz3Pd2m+YksbbT%oa6Qt0XCJy1V3&OtYd#
n-R0w1kM{WRJ!S|LL8KftvrA8E$E<sjjMV<N8Ye>GKMM{)7{QoFeL?t87cO<sYGTba<<wZH2IMl%p<p7^ai(hQxey)wuGTd+DS&SU(2vYV&;<h@3yi&$^(jo
;K<E9ZE!>yioi(FNzhsYy|9ScmyQA12T6`Vgi0h1?o}T~@JT$k6_+;-$TFHtOK;i(>@pWz+*2z=Pv-1}+-R~rcK}LyVPSP1`rYkS<!}%dGRF(65<E>n#^0xr
q^$WO91}M%uGWupdWcuD=m!o;D;6l04;1!#w53s;#yv{!X$#7G*gm@eKLjrZfi@sviNY>kKGQ0EX&`b=1WwLGQ)zG))!P9xm_%oQvBJSy>``!3V2GQ}De~ns
>O4~Hfw?uXr4mzU2!d_H$*NQuyVAhh*jJkj0G-n~5MhQ!9M}0jgfs6Vdy1Fv9byO@<t1j3)H6pwaxaCC-+MS}6S)8KURSZC9{Brq3oH)=l!?6c9ajp-G4GX|
0n}J+0yAZGnkM_(Be&M%(6`i^Qw*6UGUT8Ev*ql(Sc50Bm}apCPfb?E%rNgPc%?b9TI7udl~+O~%ltQ%_)RQFW~x|>u6p>FUb|Vo%&@V<XC&Bk5(^*O<p!9o
qg5)q<YbL5g7jb|is7$th$%LF77QguoV#U4x6&pgXD2+F&Oo&VBSI{<%^onm8j}cCc=|Iyx<oqfOi;=4@uvlCw;x$K7~al{Xmy`bHemR5Zq+182Y+qt>x8N3
hI^hgk>=YA!uWIEw|O!wlAV9)FT4<$r5sn%&Otwt%o;FK)qqWrZOAtpB3}LlfuJPdw!m@vdq__VP+9MZ#F?|`OPt~83=+tk%|4(+sU#GkWk5}1>!)<m$8(7m
f$}tPq0E}CCs5=H4d^<7@J?nWIu;>5l<cIL9RSg{{a%}Fk6ouKs-X=!_wcHhwADSa&aq4X>t;b#i5cWt!J9jm@2i^DWFqS1D!^uuY8zQ=auvHTVaPa$S}yh(
W~AXBDLQRItIayu6FIn7ms#emS9@jOc|r;zm5Z>%@mxWSEMW?l!}T@V2!{|ExV@e3D4aG9&8u<&+@ak|(x<Q98xR`7d(jF<7g&&H2BtrAs+$0{{C)DABifSa
e9`SVYzg2Y#6=oSii#YFZ;41DB%MeyUE7!ccJsM*i4A<2W6Vf(G<4z}Ob3YeCxbf9w69tv`LN2y@Xf!+bqbne{i=+baOMq`Kmt8N5hzH1;{6kk;EN%a;g1{|
a#~RSl<bc{%c58GEEAA+XQ9)m5?<Szwmdd~8DLG_LaaUq$$80SkQtsl0e`Dy&z&}n6>Ma50P$(0J)}7%Y3QekKn`OGx-gYaM3?VKhen{h{trnQsv?<*FdwIH
=rVa3X$xf?@O}n5GfjS=RS%(k%WS+C6~5(1yC|$AD6~oG0A%*J&VegTF4}$y0b^x`?W%8cKX@~pZFEPD?nL~z`)oXf;wty&=<xbXj?=>&tCrubyxEcIC|Bfd
GbJ`<1|FPmT5j4MAT1=&X3{V66OPf{o*T=Lce=l=SBlrN7ixzX2J+vdCNah@XbESIoa6>Ds*I(b<x6_upyX~1PL^%D&YK%=wkgWCu%&y~x7|t!%j^px5u>9~
y>O-syynWAaY5O_$||3v8EOolx2|!vkk3?$`!CuuTpX)m=b<UCJ|-T-5srw`9v)p?g;BAHh5T&B3rxc|v{qA;ZuLLhjX;EML{TafT^%-g0SQNT%)MDn+qWyY
LSBcLd%r>$$t6K|>6O#*WEHOkT&E6Qng+Rsqy!F`(l7Yy`#rc8Lq4?@?U8z=QcGjVd5wT`aWEdbAi$41M2Dxo{pM2I5#hB8prX*N1BX<C+adadwzjO>?WPs8
KBxIq1tfW3r9>!BuuV<;^)}&63~ej0-Jdmd_=%lA>_m)#{JI;2WQ!w-T_2{xAC+y-Uk~%m;UY-^8m84L?risP!UEE^vBewoHaMeZY<zYpFCE#ac>x$0*{>PX
^IdPNiP>-jSJM?&*G;Gt<E3ItT=}Sc)ysZgXo01uVKlGs*;L@;(?W<sbpza3@aKAyT+-8|0yFi=+>EY|5bzE!@=X`l_y@UP)RS6Y<s>7$SSI3vvYFc$m2Erl
mpQv}xPPRtmhk5@F7_gp>!Bg8t{6hO9X##810QO9w(_+i+0W!aE=ds(=EAVr$B&K9-fC{wO_mgfiR#=t3*$*8l{X%YTeU#q$=piS)|v0k`q8Mn9b#=qvi1kk
qoP5f<DU<q8d@uwoW&SiPk=Pk%C4t1^HX|`8q0NaZyJ>Q)E~@+1eO(pu-=TWq<nH=$Klv^73u3F++-5^5n!KRS${iK-UW!r#wVShx>&j!;YSuEz9GEa+nq1f
ycp@;N+pWSvKKlH*dTbI#fmJ4<-X#x3WaF;1=yMpi9r?j<YAfbKR68F-lvp9VqZ5gQ-CX`SLGU{)dsAZS5j;f!7h2=|4p&Op;43DJ$`(}%LQX&A!(6d_ua1u
BEr_fW+t@8wNB^7?kw4TW%oS!;i2_D_kv-VrQUIs^HMk$>pV&JB6fTbD^U=S0_(etFmqtMZLG)4{fTYF+9VyQB&$diJaBJSd7Z{KVot9<-nwLy#`qbpU7MBA
+yLb`KYL>-;hHQpcI>fk{1N+o(|bu3(RtYw@YF`d#0KE)TpB?LYj|uLjQ}X=57}i)w^m~&@j5PVQ>Nsrlc^+1Q%#u53$i9weKc`{$#CFOHs26+3wf3up6ndW
O%R#~IL&qtbdg|2hqDEa>y^2IGs9t*pV7=?fm-D%yu~2Te-^WN0zkN5ZQ{~XK=&*1BH4+G3PiTyz((IezkWU^4QDzFH;4%-9t3O3;-|71IJNn%-4$}XFp4t&
*1@3M@3*?3eCyiBHs1Al29LnJ#$uOvpEVMuk)w}pO{+78vh|_tJpqCALAO^3%q<Ucl|;HWbkS4%TLi4xFU>7Mcr!O~(Lv#_lzQ1!r9V=hF2L8|JYAs8n0&hf
mT?kQ!!QNu%0i0#M?jD6L`?~xlaL$o0Zk?QZk!Gy@&r~>VgFnNF)IX*4#OCVOZiclrmY-BqMRM~Q*3<Q+lBw4<U9o(Dj2XZYdYsWk_yB2vheyYUQzN;>$zS*
-7`;UvQ!{PR+A1Az<Wiy*|nV|8f*5#k356H{o|;VHt?L;5!Q%)QM7v;s>VdHIB=?d&7PxO>xPjq_vP@eP<gcoLF64dL4(^kEAG-VgxXLL|7!x!(=r47L!)6U
sGu{6cT9|>tWoi*qp#MzXwY9>7~XQBS2({7kKG8`K%S(1;Kl?ltd~L#gKm4+8}ejG9n?7`ofI!*oGIP=zw#4M9x^+)42j3<XN4W2N64gaw?sX9N#f)4QiQ&q
@`z|loe#Oe!K|f&zuH(em<X(;ea@;!!x%w6oYS^d&5MI*31#uLPPzN@HJiTxz0%N@F6Pc`^3j{-H18Xc)7SW}>L(nLQTqJIdObm-k<Q;$c_dJJfqHkWB|Wk@
<UD2nx9J?g_7oVtjzE$VO<A(^OJ#|k`NLk~n~4Y$$UWZ<<!lU#03Sg(@kx}wcm!5gWC6yMqNg*yxi*0xD8FzDzrodF8Pr*MY$qUc-XTI%Bfv(CzOYVYnbEFS
&A5%2Hh8|{rrHKzW7Xi&jYzbsJ&I~>hYlDHvZ&6+)_rrD4TgMj1#|J6tSr_lZcG_k1kR}IVLdg)+Xa)Q^;wk*MJLQU$ihX7ej-7?>+FSH&x%%esO{&{ZvzH-
35YXueno*Vv*b|7aIrVQn^5yF0XuosRwPX$yg7Q_vn=!YKd(44FGaN$|A;6<szlFMZyMh8<0d(Y^idX;PK)p0V$ZY_qxl_wa#*?R88!F{mO`7X0nX>4oKPiJ
;j)!V1cW@NJsV5FN0YEwpV}Ti{Y{7U3r;seu3?nltx-=_yG?S=^K*^FX)|tb%S1$tdR{w3dw_M|gz$uUc5fG}*U8AV11XjMMF#HBea%dXL<mg`$xY^htFXfX
G#8fA=g^=3-{_msgOq*MZbe{|qC87fU@yh?$<Wq+S!iKDYQl?^-iG)N{-nqRAE{f$pjVVt94kL;F@pqcJJ0U%$LTS)X)mHpO8)Dm1EURT@YH%836)_=t_@sE
ZgIiaR?}$(Dd@6LwRNau#5_%}djJp~_y~H<0^vV5sWJ=m1i8{#S-~5*6hW%pZPm(r96sK;j!J2-9#$b+N;PZ=qI^&)C=k3%-5j2*DI8sN?Z81i1q9x5sqKm~
UOrIJ?zc@eSB=fIySni1Npz>kOCX9=t&tWr33RMh+gbAA7j~f<#i|IAc_eNGES#7~3ykwxU={Ww+yA7~?|2~hj8kGQJw$N#%ZNS5k}eRxI~2?_T^Ac8e{}LD
WRH3yk_lEYp3;fNvvO#c;&8VB)O<J<2p$ozhH^h{$e%vqeA2(6D4TDCXJ1N;Y?HTxyy@Xk6^}OyRpkCNMfWS9tR}^^wkWE>5q1<Vz=JYgl8M_8{vQ2l)3E?k
<dedu;;7uk<a@pdfjeYS{FvKV{unpYlZIa2Omqu+q50Ot=9;e8<u}W)QXZ))sqS>uJWem|+WUmLe}HSiYbk!%#m+~E1&nRRe;%?_=|5MIK<CJuu~LEmcu(HY
16lfOcQ|v6t6HKVeb4>b9HGGRF6X9S_nkN-eT{+&;pvAVe`p!3t#y4G&U=-?JHDG%NU)Ue4UY6}H4v!yq*bFosp9a_A$hH0;qcBy3!pY(nF;(B49sqn8q3Rw
BQ6{?abe(vCl!z6pS(27mOsUFKrWI;$#Y^4${tX-+whds_~?Xl{ZSXQL(8_n--B&)m|U+t_q0rZN-ZU7Mybv)fDD$B*L|0uG<tWPsQk=P&YoHhC)hY-nxyNz
k(lSAi4G`$2>&+gF933|mqj9@^;^6#9yvn@gnet;AJr$12Is<yon4x}<=4#goY*iJ+W9gJ0#cfYclqnvZJ<hu1O*@8k%69+p^3?`HLzq>ctt6P1gSF-4o%%{
l?Kefr3?hN^8r+*%*|)KB73K>yVzCUY=f6ED<sk7an3DmG{84C?&x)-=A*c{{5_nq#%oqM*1;?c=-7RLP$loaLebc;9pt#T!DZ*wc9uFD)ZKQ>btEeQ^ju88
4&Hd;rE*H;yohd;k$A}u3r51&q%5T|oJ<c~j6WP;A7tAY#1klYKqh~wMfPe@upRvsGM?`7F%$&E^QdLRM1zfn^SaqN&R<I3ILkE1F)mobrBB{Le4gfgqa0;<
PoYSMSR1$#$`7;yG+0n<&$VF<D6c5L2c+-N1%`K6Mo8mI2=b<R_K<trGD96St*cu}+b%{So5z6dCq~$owrLl*84JU$l4I{8h}gQK&XDXCskOE~*j2vGQf6mg
m5B0GT^l{s6j^}A-zg^yp10_n+Sx`e<93KVI5=ElIhBEbXB!}0Y&U(m$H}-Xkt~GWoc5d)OIQ@CkC71-EpHoR99u8d-{Rz&-%b4`HXv`cp*$DP+V0GhBv>9S
t*Q!vEEZ`W2^@&o;v9c?D>Sxiq5b)(xm8M5nN+OY3m7vOU_@Ec#m)DrnqM6Uysvtiu3%v(u=66nku%#e;wHZL-<6r-NQlEsQ7b6^GSFm1j-_1v4kU<Q_$XB1
e@iN(wwqs6`b`H=o2eIGVH=Yi6_U+;e4UwsXO|%B?`^LAo)dPrc6Lr33)w@MwOe+nj(1mVxhg#*8I5d=27b8-*JCt0F%2fA4GCzCL<N)=^s@63L%KN?-pLGu
_>^bAN9F1>guZadr;^LJe}&2i6zJm}ZNu>DYS>p(%qhnaV#|vwCNK+tfkkA}5p*ky_0nFOvP>t_o{Sj6M2&=Apex;eT!4lTzzC(jhg^KX@VOL4^9OpVU7Cx?
DUt|`M5iNR8|JW9>3DrE^%@{yMqD@VffAy0ZsP78WJ9;udroC1?!orVozX3m&BN^+L6BA9Sr!2XKayuJGyvJ)Y6fA*lM@?FUpG-BOlbORMnQJ<$%yX)a5mCt
S^Re`jPp3onPsRYj>D9czG5bNsKUsk?+ex9OWiusC46;r7Nj4T^4^3~K*`|Bfmy$9aohK~59_+W7a5;e-nHjXG@GaCgOb7UgxR)T*sOhSDqONGbHqi^32+of
eU0Jr{FO=SLg*WO(lo;;W~9sZiW2yrHBf@GkH;R6Ph>;E6)~~?uL(A5U{jAB=7FoC?e*yFC0LQrq+TO4ED`qO?!~A$aeezFP<($8-tVSxh_s_F<Ck6)<$DR=
l~!p|R>xbrH5yb=$w~;coJ)M<2j2)^s1jiFsC3vi-w`#Yd(V1G&bT)cmrxwjnj4=N`{C}eQCP!Hmq8X%!FIh7(((KCNT&;FFSBSnsto32J6W@GBPiexXg}?D
A4ArZz|5|P-19UJOU*7iXtn=1-uvd!mCg5s)?nuRRY)pcKKZml*g%4&9s3$kDKtF2Pu`d3Aq_`3p^on=O5|xSo1yYnwVG~-oZR+D?cBe&QJb~i0|6PsNYn!=
$v#^gnqv~jKTC5nT!R!CmKjF`p0x5HTeFyf&Sz`@#qOX=Kw5F%bd@66GN}j=0-({r00!MWA#Z8#r;1nM?irFu9}5~mY~3og1kFj~zky%$9awM5J>X3FjT~l9
E?xQ6an+@Y!x;j`l5|+&?mCLVlHO4~tff6|He3h5ia!Eaa}>Br+yl)4XGGAVn<J!m_j7}hX|O3~Ue>p{=(i;#gT|;0jWfqwtM#7>Vx$ChxFOy@C=dJQe86+3
iu6xeCoCUERByxFyM&NOnpU(xxZ3V23Y~nfZf6T3qnZ_EK+!>%+x=OBipm^MtJCdZ96RmeTEHWxp#!cyM<kLwOlopa)YWq7Q#0Vsd<fKD6)K_?#n*M`mcyvn
n1Ouj)mQfqQXlI526XyHYum`V5P)a5>K&z|`5rj~xpNfFUGv^HLgVJGcjTKyb?<4kH4Rx$A<^y-(=yxdUUfPWeRgvmp}OuSph*MzlK0FF6x}#`X+zS!m*F47
@t#h#6-cKV;JjY`_AV%v&;Xmh;VIP7_Wfpu0rm2BuTfR=?c1aH<?G4lmqDB;^c{ufJ3loaZ6bED5<stL0O$IuWLX9b>lCyIM>Zu{3*%<YHP)4fP%)0W+!q&l
^g;!4`ig=pE2G7(;oN!=qOJ&kkslW{dCQ>+TMaP_Tk_~Zc~E2<dZd=rek2zs67^v}n1T8MePP0QT2^C1pZPs3F5)OI3z!4Cg4J$4_!YW40Xt9f8`fL7Xn-=I
V3gnOT#Xa1(H|im5U~hrj4`ZBb`FS-m%D>!yF!0y%Sjd2MpMZk5Y2++G(mdrixR5W_46owGS&gf)ds8M&vICR&w#+6a?n;iK7Fh!;#3?G3#=y%Cd`ah6^#L(
z<!|&#AxsteW$u~q1DRRfp8np1(QA^_F;4NRz0&S(4^nM2n|)v>E)frNlIKXQiCF(6x!RLm}7aIWG<mB=-h%7uCQnM;TG0~w`p*Ll;j~4CQjS6#v3>$9!349
eJCFCf`(=87=UZvG?l_p@Z(zt{uZ)gNK>v4>nay@0wd7n_1c$ek>_G4{0N;vl9<N0C=w+D(aOOIMo@F<dkrH<%hM~<fO(AK(J@`v>m*Z>q*Fm+CV{V#U9TI!
du-Y}bwg57!l0I}jKrjno;~3{o{{$1Q_{qpLMaNJwk*u^PjKp(_c{kWiyA9)E-O_wF6?N>zaN#HuC*!qrBljpqLYnuj`KO+G^i<}>JaS6x;8+bx>Oh)1IN}g
2t`QatZ@*0lVX^W!#*~?$J~erz^_DmD~Mp93sdbk*qY0ODc^4n^Iau90|9O?pK(Z;j!72RknM3JOCT-r?7?S^n07w166)_c0Ne>9J<=qo+HTUi%efQJ>GuFh
oGA053F&^T@3Wa&<ieGb-fAMj;ug%bcNw=@2--@$I;S^@b+>_$g7UC&^2L0u*M$($KEg@_6v2knOY52|ZUaa=)(lSlGQ5vF3>T1S>gwh{iD=BtP&~^=_~6|U
!D3%?(E_;x=f5m(n9999$8xY5Bh#}_q%b6B(&EJ2b@1|GQ6xlT-eaPPpZfA*g}I}yy7!g&Prj#SIPZ=;uK8CNd>9u8&{|IueF_d1wBNXQ<wBrhZ<)x8S|Gf3
pwNX903?8eFkBytKI1CyoCr3oVP)9{4I%33&yW1N*QaVL%qsHO`x!!x+R%j_)ys;2Nv!<#b-j}7LslJ`x5jPF+fAVt@a+c$q~A_{hezJ=I45GiOz|}Zje)Df
lP8oaZgF}|ze*_8se8_j#iHK(f6)M~hRX}p-KCCQuJX}ZTfs~&xDA*n>-Rhk=mfQVb1Ii)ZB6b`z)(FFYwMCL2q@ZAu&Uxn>YMC?aN-ZCe{>NWzPTzfKDe})
NviJ;#4K^{MJ@lU`ie0!`cUD_(<2V=!rt^Q`=Zo2=51x~oCKL<L|O1(=-nCr79)BgG!$Z`748t8(!?z%RjAyiAD-av?%gpzCFoY2UVl;tIY8&A6d%{@KKcj`
TwPK0$jr#}y52Vl%lvr1Q9?tM5%l@@Ovs)|@`?IY&Lr1)E<C$o&ma9YiT1eqJhp~!j$}K|Lsl9w73A5QQ0sJbOqzh*UtfE<N4F~l+{;ot{DxdjQF2+6c@<z!
%bk}xoiYKKlYA6rql30=ayr@L9QD0dk4bh132}Gz62O%J0u|K09v2Fc3~&q!5CVy2nsdT|_lG?GPjYK<4I|Guz9Y{9af!OMot60H$iP}~+CK!_ePqu9&(6wg
7ebcF3WCk>pTUV7wceCkJ%@k(78c_q_2|U-hC5sr+`QzPI%)M#@y*ahXjPYgvkwj3{p=LmO#YlDWAcJVz35jg2ak+2s=-B|QbPOV-dZm94{8Z<<3LPS=k~z^
Tu6ZDOl=Uj6)?Fx8+Y2wc&1hKHId4PF-Nxnxn5(WVp8R*;nSt(gjHQYsS>w6@F(Z-hU$t(<bTp(>=}ZzDnz;6k}+07p?5F6f|AcRo=DDlL`Ma)iHTepa@;0L
Y5mOSb{xV63<6iT{QMcI36LwZpo<gCt+4@_Vs~L_r*_q^w%5aqdzAn@la{AOmm;69anK}^Sy|D`EHHU-M2|m*KD1j%%7a_2DBz2u2<^Sy3fG2%qszwe##bZj
$msp1_<Hn=szJv>&r{o(9(7W5vN4Wsf`V9^J+D$Dlg)`NOvk1FtRuug4M&~Lc|6k2GKYh_k{WQJEeN$=zBZ-~(f0h=rhNUu&m^m@g9h>=CnOqDIjTYLX<qtz
&*eDI32zDf1z`Oxn`8*eOF6)HaT?}%Ef;j&Szt|fkSOk`^ZNq@6>ZrmUD<F97LdLGek-k!rfU}Aa7#&q>@Qcm3>cSmaIVeJ(rJ@_cn&Xn5zzvWH!J})36>oZ
w-WaZhrtNOFYUtorc4;hu%FVTIs{i?fmMnkJ~9yJPu{z^EsHrskg&sA>qxNNVmZpB<f=HPdUcw><bd6qB2IlRsDjVs48}k6R0CEEENrLz#d^L^-yg|x(b=S@
tI*Kxq(0$?ywwDsLsbB}E)WW5r=hjDeu-Gwd()_b%((Wb!qj+zm6Y*j6I%L&piLqWOQsOMu};Rph~^IK6*Hc|w-PEY5%-kWkk?372&U6jx`PwM>NPY?RKb2r
!E3qczheNWD4`wK7E<KBX$q7I|GGIbso_o#6)OSYD+=Ajmtu}O57x;^?Q2Gx@MZ+6&#y$CQKsBA-Ygip;#OEaY|@kzD?0w6wcS@2V%e$n{ra&SM{1DHSu((I
d<LR>+q137EA$KnU4IAf9kv6cx;?hf#hs<{*o?41hSeC8;4k+Z)F_w(Tm}4(dPc+=?}#TK{HKD6JRF1~W|Ok+s71p^_=#GBml8hkthMh)yoMq~%-<#;65BBn
XKOqZ&#wWuQe#zb2i($9@|e>r*l2g58k62Gv-f0R76><%d$vu@BiRmLJN3#IFZ{CdrzyLq;L6)@=+JQfJs=~)@%&~4n`E^aeTq5MZZ!ld=iLcY<qAsFA1x;g
{85TZ&z59{FE1k%ZPd`>u2#%*XGC>yGmzr`#i;0ongLLh9%f!M#%V5}8qN~Mu0I-_M%Hv*YI5#XG+ESWCZ%AR`G|L*=sbsWaTgXG96w7$PkN~Lu}i7_1Kbhi
Y%hP;Rl<mT8?<?yu7;Doj;00C8FTrIIO@UhNu=-lOg4h$oxCqdqQA+{Z7ryqbwI=E&GHSlf19JQ7*<BWa~uwh)-2^}O&6>*WP$sKVja>3(5rc=cn+}{9ET7k
Z}%)iP-2gt$=*#3I=IAf@$?)WdG@ZN^n&(Da5h5$9JFDxW#NQkQ=9-rPIaN;gh~+|+!#`yIOrbr6Xs?bE?)lT4(KV-Gdsr&>E@z!>N!l8!YoN2QL{Je8@Cf#
b(txO-yVE(lT@D*<T-YQ{xlM0im4@e*$*otOX?vCbL)U`FwfN?znDB`XNzA*@ZyvOWMqfvIl|*AE@9uBaJR!+`R0*-x;X_Hrv#ZHNP5nD(My%KH5P-IbLiz&
uOiK8+bP;)uoaA)(VSQcljy`<Z+LrL2jMsSKq8O!h_VvB^_R<&L<sLETgqXwe)k%u5iZ=wTxlCLaLp<~zmsPpOQ`m(LdxvCqUlU|??EbVP(nXUTZAQSbMc_}
!#=kx7Z<@gO~VXUe`!adx$R@v&PY*o`YD<$t9(|5k3SxzX|GF;C!|X>Flwz#3gmk~V{i#?@S~90k*A#)nx^(tCeZi3697La+2duqejHTX(FTpqISaTd6bkuU
6vs^92inGATiA`Y=rzf+K?$4AIsPjnArZ}RTc!>}%z-Gfac-f(pJ6|7eBkvc*%66^@lgVn%WeMf&GG>QzqbZW-Ndy%X`u$&eu7U5lK2_LvuAuP+$tSy+F3PC
kN^&Z(p4{T^=(&37E;$ubtd>?tSnm~6hEw0=7Lzj|Ar!B<bdssq?geYCjh70_5v3A5+!jPku3#6VU1FXlFy}RF8D&Qj@pUj{D{Wci-QWnm9XEqy!ktY8{=XC
H@SQqepxv*_}3Bfl<RJ#8D4|0j8#&QyMIFr_s7TV(D@Z)>YHojadU{7g(9(?B?g0%&Cc-vW@Ws)$GL|aEXPP;oO7hY?PwEA@ps6o>Js}WQY${0l%Dpm$sy=w
3V)8ZDB@V9<K-USI2DOx>v#5_cXDeB!1ATE(G!VDw{1-y@}taSh_c78u=`ejzLlu7e;7K-?Qi|0TXb|Zjk-w=Z%G@A6Ed-La0YkbeZLZ(7pnQj%i`u-mS}=o
%+bI~F5E(IFiYO9f5BoMvKj|vV%#}E*NWj%a?cWS<cyUegINK_B4GW%GYRh4bxB!LTn8Pe#5XGqNjGfX=X%K*y-&@V!_p|U8FVWFhl}-@DaCl&t#AbZhFiKq
o&hQht@q%d&h;iirzBvqe?4&$E-!j@mZlub?g?`@VAx;E3tUVd$FwH+(`L$vO8V=8JT3=f<5fy^Ybk!v=W7}<Ow_SxqPh0Q<5*$@HYI0<7_@+AdT0!AoOqA)
5|r^3i$MBZ7+$=2B#y48%sxcwRrbuo`ES;cldiC6@(L0LzFuX2*8t1j&1b+|+fz-gNGVYg3bg(AF{C59Jb{Xyi6F{7pNnYfqA*yzNlQr9US3sd=OaWB;hA?M
I~1OgtvjD6NAuqW!p=Pj)Ez(1!@5riI||X9wBA|73>+?htMq|zmu|a=&L$T1iKlC>#1N^LY%vL<!JVbf)|xKraE-CE?I}RzVdP9kwz=c^U>3hjnh2)A|1E*P
*^)dyZ4hmvhok15cJgrDMdgLNqF+|Z-lz8OsXNLFEl(z_yXCKwV6d{}1c!z`!zdP6xMdga#X>V_Y0+#Ji;q$}@zxDfnRDQw<CKi>Keu3Vjn{HL>93&ql?GFU
Ik*Su!m272@tT@2XDhFKN?BtGGE;PS3(jVb-6R@Z-7L}-E01FWx6ZqRo{rC!oY|jqNx|nm)+HblPBqmsIs`E?NZ?qj3l|sk^x&)&@d$R87&jq4p?}%44Li8j
Sk=1iQ1Z&02M*UofdGGoZecXW^-Zf(TANK|9S8v2wPW{Ao9kB9dr!RC$gyH%Vn}kcB|kr7<U`t`MF2h1B`Z31#W|Cp>>I5HfwhCx4o6>QA~ja5uhHYX%PBf)
<wjZv$7-28p2e}R%WED3LHw~OYemLRKeP*U`pYXe#vWuX7Wl1Jb|tWIUIMG*?yk7@kwzPh=CYHK@oZWQMx#zD?HC9HcgoJoMJ1QtK-I+R`hn~?-q>B%7nOwb
2@exa7_pANaQLhpb#lEU7l_3|`X&2jes>WMRrab21?PSd=^F2wpbba33Av9>raVK<>y`#RU=1Ra9>uT)JvrvzA;+)#8{XY9V3T5czd6W)mJ|fEL0qx-SFRbk
Na49M3W|TI+{3le>D$9NQX@QAhJ!{F@-=T01*&wTj32b5F!vdkr*Vc*pt&0bK@RSlM{$VNtUR0RUA-_&^|H&p(Q5_%+a#=2FO;+wt(nC_rEVP=7o$;oUg{<`
GuiI&*Nc<vaD<DC%j2;qHnyCzAFoPN43kL)CYl%f)ax{OF<DoOtI?JS;&#@q&{1u5qN?|6A!oI^YK*pW@1LbOVzNLw^#%o0!EIc7Ki1P$^`%dmE`OAvWynIQ
kPH62Qs<3lc|n6k`Aq?UV?^DA`$i|I77P>Aw~UDs!)?>?X-a2yK`E5wXgHs{ZtBK*hjaIde{!8$;hD#b?SZ-#prOj$nA7+mbb>v=jd)WS&)Bi-M@Dx@<kx)n
5ws%QNZP4sFm|8FkRlQtJ#+yo@MXTZz#I9xtOZanR`Dk_KxINFGQ6!Ovr!BMUrVXcdmDLZHgN@gFg47MNth0*-sTu+7~|U?*%7KSD6zMmMXDxCyg^55VZs*W
HdT-%T6rGBVEgj~zcNOjVx^~|`dLBbfn6*S3-N%)#uOn&T+uE1=q(x``Va&5Ye>sWXo@-?g7Y|bedc6r@a)N`I6Am2lDML4m?h_5B&glxKkC)<_P@^mfP4H_
y(vv!^p}1mIab*_-Rqfsw`1472e*4>>G^s$9i0(zLeW3%fDlTEu1Sdqhi>Ub%s%ssO1P)GDvG8#H-g-?i2eo*As8=T!k4k>08@m(&l)58+u>46oysYac8ewr
GqzrfpxVmUj_LKeIr`je{zQ?}UCrL@)%s>2ev^Tc)SIy?>}Lcunar~TTOvl`zxH=H)<<>H$c?sT$%yHstyWnKris?+@2t6ryk%%r_073uJHAax?DL5^+YDFR
`uJs@_7lee^yGbO*M&Lil?}X8=L2@;5?LMLN^16Ii&?=p72LPNt99=EhEWGw#d3A;T30(;`OyTb9T4NV=l!*_l*gOJe`c%dYADL+xy6TY4;U=w!1p`k*Q}T^
bm1;C8W%%lrew3Q4fCd^!|8{82sXpUiJ&9^O%^tENlJb{nxc7cZ+#RX)1iWYm7PaMZ~(anEv;JmzFe-{6vSE}1*m+5XDU3F4!}o*a?}sirgm!JR35U0oFgJ)
t`ev?Fd(-*+-;L91g+-q1rpz5ohA3L14mJBWvGCP1z|o5W>rME_adqhdj=me)g?XQ#du*5grZ`QcN#@VrMq^6J+@Zc_ar|7C^|MNTN%r9Vu^9qlVt5)ZxEUn
{P-N!7)t(9mv`p#KU^}S<+2E>BSKAr5M<s%g_{hQf%S<T45}G0qJR7hL|UG_)B(7WHj9Ew58cP~s`__n>-HtNP@>BCalM<WJG`gH<y&@Kq-QTA)?r&<CE_j;
g+R`A1_X-R=NJ`u1Q#raCh)x)=dhgmAQpL#;Ib#dWy!l=A1II8hr6%2_)vB&DMg#gGZrc^$+^en4_eKaNu(;otI`iO<c24=ncbWHW(Z!C#9avZCQmR0r5({w
IzuxWB$0)JwLx2$S97n*L-ZP>aB8Mq>QqA+;hJ&B-%qv^5`s`LKD)`<_cVi!dovo>a1dEfUfU|4eqM%9QB+PS+wQZmK>ercXh0s)NF|NMaIL5A`hf*&qQeXc
HJj_3%gPiCuRxyUBPy#V)GQFn1ne<}`$NFG2x3SRU%X8H)M-wz^^+{ow-3C}gI`QqrXO-Y?EwaL$U`HZ%|D5nb9dk%aG^4ZHh2$NL|nBEHI3q-k>}2MtQQ(n
WP(%WBWr&<g3t4l!#g5u6{X_8tKqf7lV>MAfi*$aj=XH{W}rY`1-(O_QDpDM24Zi`T4gxkJl6TsK`4Fw6Puu%hB>WmTW=z9B1HG(Efm-ip2h=&34l{VgwBx2
d5CHysTr3B4Ixf#qR&fK1xz~n9VHU&08J6^&6Xp=&W9pK4a<R^F-zv%&urDzvq?Z-Thg-y@UQ8xg^iq>oe-i<k13pGK;`_Erg*!Nzey%;eBcm2bU=y$7d34#
5f7J0l}meE&=vF9-+i+dg(TI;*3KcJ(b^H*7E(f%Vc_a{`v)$Vr|K0i6|k#Ur6UYuRF2!UkDU2OA9TZu2!V#m@13P-NE$Z`^FQ?sM8zVoiwAQj&eN+ba@o>q
V*?a1E-EJLMYX-z%$0C)$R)N$1yC*L|C)1mNf4rQ@Z@;tZ;ty*r}@IiXix>q-1tKePPG?TRq<Z0N?O;rs?QmPbZjo67hH|t3J3H53blO7INkX!V8c{+H_?uv
`(RZx;&x^h#c5XN3#<2^$wlWKzhHdVO_|6^LO0N??-xh8-xTp#ZGq?#92RnK`I39O=|rHUVsX#{HB2{sRr@!9n5nAk+PEgIv`m^S+hn*yOZm2yqpMsBija+!
TD)g>oAjc%f5vzItgUZ{%^wQGu5r$H64#f!3TQ&ViklZ$vz}u`l49b&gsXgc27)5ZrYcdQ9CQ9Wi$xeu?j@y(hPtXnM;z<MeVCZ~`AcyYn(dOT8PhG5O)l8R
iP*2(=L5Aph7Eq;Zdkbl1_8zP46!LkM-_#8M&l|}<1Et;Veya<dzLMjtFb<6n~`!{;k{3uGvMMLxrSTj;c~hPuHSp&>Z$MnEw(t_X}`5f!r6Gumal$|NtqIG
rYWkaymIle(8KZ3m)?~6I2+qf15IYazfWddD)TFc9D)z3QPLi(HPyOeZ=Y0(Lup3(PrAARl?^9aPKGxC*T5N^$_jPZ>8tnz>s<2NlhS=fZ?9%MP`wCB@X_5J
>DP=Sr7;;*NEQL2O_dME?ubkF%MxZw%luHZaS`3Phw#$sF?+VlJounA6;@_2C9i^uzdAk$ZE1cZT#$S?9Zp4;qdR(Jvn>xU(4Md8T-=l&6}FZ0|B~PhxcPGQ
iGhec{E4hPOoMLG32G<q7w9luqn+!c`Ul;mN-Lz29Ng5LIKdDRbV>vEY3}Zy0AtV1ghtH3v9DEqH`A)3>u$cdI=~&nk+}_jwbjqiYt_YySs5YCFsb*~fQ8d2
ngMvcxJy9E@}fy_khy~ko{{LpTA)pAHR%`CkZl6^S_>mDH5OdZ2A6;9XXC_9wrn~_yh#;@A{$7(8{#k*Kr)p6;MLt(2N`kfZso_ZhL`y!Q>T6{uQSfsEgcz_
cyc}mMBs*c$zW-+jgLg;b}Td1NKJc&OwpGRur+?}$hr_%F&i$Q3mN`sQm&q5^RlTL2ZaVz2Q4MwdY~{=$PGi1$^iV<SH0UuS~+~~z1H%#D!UXGQUrg~$()0>
%Zx-cp{@%$+CM9!fMl-q<sU8|x@+Fz`cf47dw6YoaB15EM@3F_g{SE8pxwmh(LAC|#2_!aom#py9$0gk{s(C%E7helX5+0sNU-StgEsX^-EY~Ykr1qofLkPU
-b-!4iV<)4A4_O^neq4s)wRNSAT1g?z<H2lb$7TC>jvZy9Mvt!Kbw5xG`7Hn#zqj6J#YbRZ|%Z0tV^Y0rIJUXUw_<I|8o`Z(|&C6<r<-S1!fw4g26A9wGcGt
_!O7PIHr7`HjsAG<nOylhioP-A?b2Q^SBd1O!)bd=X-S^SsrVo8MHMe_jgKx)8O?3Xn6VWTs2!V)G-S_FR+nxC`U#UHkSthi+hSdZkN$p`OQxAv$Tzu_j06x
;#lqRh`Tl&N}P3LBD_q2%vrxRiYsfHikjV+RAT2(JX!Kjm8Rkg($EUmp(}X)0Xh6NQmWw!KFH%WjDf|ezh51{G$#Y99cM!FDtr5&++0E9+KgV#jy4k@*?AJN
MqXR?fj$xbix`%o_UC(wcr=}@C`EIPU?M&*KwB!R<Ze6Mw?Gw~HaD?gxl<)q3hfYaMpn|v9+Zl&;2YBS8XR}G2ZA`X4EhlDnhJ+57k?@LYz_N7#t|A}$eF#4
x$vfXlG`HXeopp}xjvgYHHdN!0#{+r6|KZ4(*q#mc2vkbml&IheQuw>3t0ul)>%@oB?2L1h^G)In`{j^59RVgSg94HAeRzCWv`eAnJNpER||0i4)qKzlw8h5
;&dM-)|#7Xj>7u+wK6)O&Y;Y;&rpa}B5r$Zh4}mkXsNT^7?zy##w9vH#lUY+(~wB5&~7pPqy36;?U{+&3l<!F_su;~41gL9d*>z7X@rPK%wTdyFTt3gz2Fc#
GVp0SN=L+wAjowS%&1L9y;MSyQaHx`FyfCvWU<Z_L#6Y@3DeFgQ0Sk}hEEE+`liM4JVUXKF2w;O?S&?tZ{1k(mbD9OGOfzVlM!b>>DRJAkVARY5$l^aD>xTF
5WJz8P@EIazq!&{5)EJ=0b#Wk*JUq*`0!>%n5rES+e@mIz(oFLJEWVsg7WeL#;m>ivwsWD=b)5Eri567kLe}uBrg;zzO6Nf_Tq!GhSpC2g2M&@0Oc~-nxK$Q
t}Ro$0#Lpzb7XbxfQ4>XluFkcNP1u4H&wKb&drQh57y##>i;tl=nwnd0?XR9%4N}i(*3m%g*%sL25ofX&Gdjgz5L4nmw<;$n#nAA*Gf#eX($PF6xpj&QvbRh
xW0`y3*6FKocRxD1hlF3b~KCobrziY*%RHqD8ar<o*BnQSxEm0_7$E^a>*=Af?u#4I)iqnB!NbLbnor~ZSPeG^KfVD*lA>+qb;vMh=KLhfNKpA3RWXm#MpRU
sX+}?h)VChl30-$hQ8K$479H$8M_^3aWg#m@!>$oKRN>2tCc6o|CA5QCN5Cnab@@XR=#d=A=-e9lvm<cx`uDL96h84?^ny+MoHc|9s44j0^%{6?1;>}I!adf
kO^Ys5uu9rgm`i-PuJ4R(_uB;iTuL~JG}eqaLw?$Bo3$-Qk3f5|H`c{I{Off3MNG!6#EOJd0+>Vgg$(HVm7aGslcS1DPj@LiThXkVS8c_MTRAOyD&>y0YZO`
%1vYz8g?KikK;cjY|GYj^fkjh{b+7^;6QF2@*;%Ys8wdo<JM##hF@Exh9={8-9>IQD9WYj8Xg%wdb?KnU7?RBF9%ox7yRZpW{INd+R{>qs{+|Z_FtkT%K8TM
JjefO>~Rhr{CHpr=$JL-o<y(SlXc%mxilN;y0qO&cST;E_GUyHKt2KPc64A%D4A&W+gpX`a9s_h6Clq$sx)mNYci{oNln3uLTT5$14W#x|8v$LOqn+C<)z?s
BuJXRbu?}|D_Vc2kUQ1Fg=}}k($YJCX+vyN<SMAX69#Ofcb~@UN=raxVZcyvbK4a~ta9DXbWoGdE{!vybLDDr6N8tGpAM@{t!e6wJ6i;eIeq%(3ub_zs7miW
?mMDGv^K@pwLr{h$Q=o635eSXIe{owp1UK5>ARuzvq5mURpEM+D8x(W_A@i`&HHn&kllq`>^tIIsqhJwT)Y(|Tj7q;Ej`*hVJI~nvnPDg+<Wbuk@w~e5}n)b
zuWSi7{0h!8(rmga{Zzr1)&|H?IBFVFQo9?*^%xB42LqYUT`&kMZ#sqcr9q<-cil;wpjK<7e!80Lfvi&(eI7`c6T2f_^41Ou5YNI9KNS?Pa0OmpBvE1=7yWD
CsHa%u<nRk>WV(;MrthH$+^v>)1?DQ4s)J7XSIjmg|2kN-4JxY-$gD{lsRn7DofQd2T8`XP7c`k%N*@i)Z(op7P(+f4@nzU-5-M)qM+dHA&<|t?d3FMc;=Xo
E7h>K?uQFq)?;RY6qq@EJ^YRMQYzqOI^m#>%oG8$lrZ68q1^Kki0ee*d!MmkH%K*90nLHfvPBzNU2@o3YrJ_kpLRJ}QPkc;SD7rv_XCBZDlR<Ue}vS}MJ^$~
A_*qZ4g)y-MZltKfv`%(S}*0+G%b19-o|`e%ylWpTN;Hd!}!1lX`kGm@EK@j&0fHPN?$=Rpg<bR3ex3USd{JzoTQA0I+|ppe%>TwoY}4{&ZwGYrVCgTK;WZP
h@@bSL1`EX%0#F^WSBAZGi^m5ewM-Yz5>z*jA+TX1n4@%4*uv5t7I4?<`Ljxf*wRa+S;paoYEy#0Z&$btnR1rD62gxIiqzf-nJwhS8dLV#L3-@<?Xn_Dt3C@
x`DA4lUR=ddniv7X$GylMJrstSP!p6ek-m0alsKrm%%>n;9}2Q7CU*-;s2Q4z(%ta>;)4GiB%H{W=rs}^`>0%k6bwEyxG@8q3I`zzo!AC5;F0it2HV(Ul7hm
49Q^p&!p<vCbaZL*$&<;h2%G1A2)+4Av4dw(#GO7zG=4o7Dg=N2>Fne{a!3e=a7zfi(9fAW!rwD33ci<cYC)vY{Mw_$<xs(;2(yu;?V*zV()ADI$Ya1ZNAYC
rv}6n-rJIShJtcp$!S9u7yMi&skJsqC#)rP{_=zRGqncDCYM`UWT`QB53PI9r?0zUxiIN(b&eD~+~Gg#CQc^IDD!G7S>+7uimoCQhc$Yg=q14gQcvKf0NY>w
*P?08bDF;A90FZ-gO%bZ)Bzk%OxLU@xDrfh`q^5IPUWF7l8Q!PUct+A3Ayv?*tc=y6`N4X9N08Q*2IBX++Z(W&BGIlSl{<x0H8t=nr_BtRQI2Ihf5y0!j?Qh
rtO2<TR9eYXCu%6%BY3fC<CbJ#s2PZ$9M+PI&(I}N}hUD8_vi5#8;;<E+kwex2!m@jG^wL<&_0l3Z6mJ!LG5*JE1hbySM#??Nj*WOPL2!33Bv6x^b1ns)ATj
8-f0zrNqm1lFj9|I|sCruXi$vSreYT`8}*#z_U;(+$<$1EdyaXA%k&9MzH$^b@TiK(cB`A2QjMpW%5E^)%tkbQ20FWxqy2Mh%XdbFS&!f0U9epg|g|69Sh_W
@kv`WKbF0y$p5`gg^=DS#lHOXk6b>%?9i3l*!#kS3qjFILThxMunZF>&L-#*4aHrXWd2I+D4r1iDu3LC_PmjA;`Z@91(yJnaoDyLUw^qL>Y^ibF9(V%@oeeg
3ezUzprEpK`-Wvsk()Ez4{yc@nkry<1DWL+=^TcF2EUim#Zfzwl%3UQDG+*{AQ{Geyjhh1!Xfm?Eo(Dwsr1O=LUE^F?3D0cO*YePH4+jU*P>CzZNk`q;jF6p
*V%mOJRbfR=F%t7wPjt*FOuP(Qco*f>|+%7Ey#veahVyW9K~_K6Vyn6wV!|#Sf^>96R6&O4hyL1A<S#+uxYS5S&N5zfEwt999c>7*_C#&ZW~aYC_vhp=@fjN
-o!y+U!TR#eY!UL5knVinfWK!^_s?f14>K5*PO1J|3kO0*x;JEm%7WseK%d}-bOX@aY~x0d!eyB!&yUfsq;Pe!)iXOnRhdQB9YNp$j*&^HBo+VM=qMdEU3)4
OwB57LeW^w6b|h`V1eM)nz8RMw509!nJZgL+1O;RD?zWvs{h0|;YTb*jl8(4vvV@He~LC|3O0xR0a#KCDG_pgFcXUrSfGS)Y&~T5?imtccmm*FJ0WZcSR03#
8>$4ZR<xbi8gb)qOo2j{M|Be6JCkj;66HMmx8IHBT|Ityw;??NNm~B2GkKcmQ_$X1kj<>!t=|7*X;LsTIexoaQQQm*nz?m{YpI(8EF(cCn5O@X`vXz;vhN7(
7*1-5ni5uB)kLYxwE3+{y8%eIj<Bzey!-cqBvDRZwdpbpRZrj}8bfdvGwB`nY~vxxXXbi51c)6xk7K<kY+s|Bs^zZ5Mhq6KEfJ~-7M9qXhPn;I+EtqkAf9^_
utS~Cdbo7yb);CKf6J~pUIdu9oxxP$<gRrgI;jX8F->2Wwi6EDcX7~C3&~A0xiWR=9iY7Y*;JYtV5-_==z~eK!uj#QCNOs1vJ0d{`KQ#;v~b22B(IRlv(zWL
)}<R-r8{Z3vh)xbT+LlI0OdBg%k8(9j69QR0OR4>-a#q?dN;g}9X;S|Zq;O`qKZFrvZe1UYiH2<xynQUUIqX)jXUrTVLMQ>TVdd}3kjKo>Xb6MxkWC7v6KxE
&*tiG)I5(0NFmkv@EvNcO5Mz~F^I@sb3hRHZSw#C^li@p9~mniM?whaidl=Vd_$ITm~N8{ag2oT?1%SZg6qqmWXfY|aX)lMIc?MWED&WY^TY_FZ5YpdzC>i(
THNZkGmmQeb#xz62%c%92Rar2tG+Rbks~_`I2wfB>V~7XZn+?dmHuO_Vs-$w1?-d=7gz>;JS>WU5^{;f4REAL^*gePX>an|2~gomy_0EU$Xs!i^jK?4Tpett
eia;lg;K2Hf|p62?Ae4jL4r#nl(9C;W+~8;#Ccgr(3$;6I73E_1G#%qO@ZrLmF0@EW=BI9G>m7l+4H8*+YVYD4ol?yyZ=1cD64h_bOuvSOg3veL}b&@mTV>Q
d80d%JCW_RF;0qhrfPswz=LOn*BNi9HyN1%KgCwY4&<?=pbRUbl`iw_YDpCtuD*tcpm1zYerD;=w{^n6w+9Q1sf7a6g9&Q)`!emyi831PuuD4i^(qm?lmzPU
z6($%B}mel@&B`O&`6c`%joq&agUr1?pfy^j1=6%%zWjc(NVuo@WV>5D-Gk!*>%vulYtWao`s$VyZ#9+465_h1ih=D?#mv6hW;rM32l+NIuj0^I7@46DFeVb
;n0ED*x6Ss&y(=)9U*S6hiu(QI{endNS;t-Q|)aqUUh4KB_6Ug)4oN5P1yPKy&asBlky=1f&9bT(<N4})GNGjnncT5o8U3i3Vy><Tgk*Hy-!}~Z3Qxl1}E*h
JL8M=AhS_VHvis^^;1sG)RexM9^!RXQ8I77-nW!I_NBPgKevJO{V7;{348h&bFQ0XKI108CA*+n{J!UzQ^wRsX?GjIC~tFx13X0iBcASFL$r~4c{;QVGTzbS
77Y3u&{}Uua5#R#uafhtVaK%j66M<r@y*ND)@r)xJ-Bn=_b9e&AidC3g(F|$i%S)TKu^9GK^zEePr{oUBXZAG9EG6KfRtT&oyRs$S}O{ySLP>kf~nM<ZZ(WY
X^0O)+1!j%io=%?18d5|#sUSw`j)}4tf}bTyIfTMh2#TJ##bRkbfM0Pomjk+Fe@bg>`LPZ`Vq|Y?oZ5id=wymoG$}mI<g6A$s=OkS?A*KUaXzV)wyWx>vGwW
l%(9-yhi)Ya<y;NAC{fBS{5<s#%bid*OXH8w&@lg+g;P+uM!t0Dn8F;9BxCU$UPUB=OGXbBp~>-ys{_}MH>1W|2v$f(CV>xWUhKiy2xdoOZm5091fExWHan9
E``oI*Y{|$3DnbuU~1^mLQ;Dh0Fd!d6>Rn)+X@sGF!><CP606`>!3U*RzAZTGymGd!a4akebNa-kquyBSTJY8TA-(_PZ36s!DT1Ir&H!kJoD=H{(f~jyZ@nS
(cX95qpq-Qi69q}BrnIr94X!A>N9@df)-WcMi&<`^+q0{KH$%pI5B<Igk*4lhx0Q;n>zFSoNbi!<7V^iZzrJ=kghN%&6%m$2J#TwAlx4k?j=~YpqPDK*5#;T
YdA<~Pd4eF@LZqa`3)a54|WFzyA_j>10XE|Y#nr$ZdT(Xy{Z8BqmSkT<|f~#6z-?OXniYQ%$Q6Q3i#bq+>RN$DlFyP;=gj|tr=7W?r7QYWrNCCW*a%C2G?Dj
a$8d-1k6ZV1S1hlzty?-DQe7nJFdA<2rb<IgSdo-&ew#nQb0U6zJ15H2(-LXv8*C1BWVBkX=ha3muyE81&#t%;pI+j5+R_8g}FFiQQb~Dx4<pL8RfLDnyLey
TqBa1C^%DuYkCC>xs2gJ(ue$fCxXX;EK(?hfG}5052`4BV_l{P+uvuN9haV;ufF#J5Q5C{<tRCOF&-lLQ1vC(u>eMQvl5GV!44M5A&@;GC6V5Jy+D$%aC)XT
knGkW-y`r1R34>FtVg?qWSrZFT6kavQ)#RLJ&f=Dx00$8?wX-d9FX#MTb+<KdG-EhG$ug)2egDrN~3{H+9xe16au-MQod(-T?h8M64?{+6bbc`#A|&M7n8h{
zu|;GDi*QXZ6;{n7Pb%7-c%hW!B;d!f%U4b`zcR?w}Xe^bfYYkgro&mKnI=7%3<Eucw_jCMo*x!zfl{-<8rNi#UdqzEpBE8=zIDHV%@v8g-?U|-I_Q+uKNQ2
52ZxBHXvIH`Vl8uxuxCoKy$@^s(Z^L({U909QwoG7$+2t0NC@1Kq-sy{OL-xg_GBacQEy1yY*R~@b}*E@4F~s(p?&F5@MntWDL!GXtrjcd6=~fRYQrsQb7GN
>I2fuYOe850kZ9D(07{sCV|EHDc*a$%pzZhy;83n>ad>2VpEV3c9r4bMK;XnPI+cJB9JNI2^=@vg}f6w|HrBAA?~x|<PC@7xB~i3!o-xw0K<lcuHmoSi1od9
_O)|=BMLY?*Ao9pNlNTL;e#Kb)AbD>7fF0@Fe~Bsj<J~Md52aEvu_h=$k{dEV#;wrP2W6tI7>uNxr|sIglVA!u3ICzY{FToj@Fy3@=9SpluUEp@>)4g$P<kB
f1d)LFPDCuUX?K*>z#JuBb0js>X_q!@F6ntr61B3Q2CTh*JNE>!$F<Lxk$<Xv&3{YBi1}kOabJkiZ?4$t0Dgp*T4CeycFb_8aIfc){0l+Wz{V+`y0eCtnU;H
qp?HQJ+ssR^nSj6#CWKK`*G*g%^-*6MA;5;+!GgXkik)irkwK*+-{CX15$p2OQC@^h)Qg_wPKC>Hexz$oo@b2Cec6cZhx9V1H#L?&-moi>l9AH_da*elejf!
Z;h>Y{^uvKHE#0u&S)%H^>D|Kt9jeupt+a((9cRt;xOxw*s#^N0tG{g%+Y~}67P3S2ZSz@Xxb1&O8e^tDe3dmoYR?97VU9$C2JZ0<}YC5Zd2ubt8|wA;-e)f
^O2`uT~K@@AXq=9!Vnwp$)4GDzP^*0=trt*bp)4j397=SLK33&QrhYNG&t^R8Vb~YNKyCn`wpLtoZNiNQvDrx?JA((*l=j2#kgYSU3!FrsZ;`+J$C|V^vnb(
na=pb1+h{>&}8I9P68gznR|$EWkBY>6t%GU=kZ56(e6@0N~*P`L)`J!udoYzxJIxLKa_hE8D|`unpgTde&)#~uTJY?h^C0tI|Y_tXaeClJZds1Yw+3P`wu6w
r~=i1*f!Q0m1d4+-5Fh6+m_=UR1GZ14Y|g3b^xQ|IdX~vM~FpSmwl~0q$-2re*||z(qa&Lq8TzG@mzPsLg2dUF+A~uv$8U#U;*1R4m8EODA=MzN6|w=yrPFK
@Ug;_tnX|tjtnF;0xlJ>5+y_=qG!dIbzMA;J4(>R!cx)(K%hlyc1^jfXTD1I>1O<+pa|$MqyshgY;>i*TK)Tvn-7;5Ky6Q1RMPQPTJ)YJOLZ`oX0hSL_Hzj5
#u}|OY{@b!qcC8t*L#?u`y#wz_<zK2mV+fWI&=*o#HOK$r2*ETJ3F)3!TqnO1iebBa!C>d2f)s4ST<*-qiQ9<o+b*dX|xDRs<S?NBZz{FM~N~g{5IG`&cTzF
CTQ3Y0N5Ef0Ru9|-Gmi#C5oZp-XNh_B~|UpnJ)Nz6~1A3kKLNKMIK6*NFlzylgViNu~kk)mtpta;@Tb85dQ4HW+KbZYVbS2ryWJ-7B{q09;-u$dNoO&0e&{W
{m{zMRH`HRF(2a@5;BQ`ltH!7Vj-{6h4#)q67dC*AuCGvqeFQHy1_jqKL$>DUmc16^6C3qX)X#1BKoq$PRrr;d?|X|9&+OYs_<5SM#<`I8y}?hRn`#}ANzSS
cvJ+yf|z{7Wk8}t3+`!bTGtD<-Zd?`N9cgU`_kLbY9vG*Ifnfxd+oQpoZ$-PALorsYCR@FuZ(D62luAKoaA>83J5M;Tr0e?(}M9!O$d*Kw^I*n517dGn9-;x
@+o&HY6W@s{1)Mp3ea#^wEC{P%3EZqwsrsiBG<2a(qb?D)mpVVL#8q6C!2e7w<2_vl!0LtGahRZ5`j{}MEkz;|6VprQ>S!_1Ogr&Y|?BZx$Uc+m!g;6)I**I
;$X*i?d{?|OazZZCFOZaWgvk)<GB)UDCRk;qF(fT%dB9#DUPFc<B_h~HR1M64JAcCj-(Y!f^X1ItG04sB$^t8!r-PAJc-|?u;wty6<5g*M*mY?uPmqOF;Zxi
>?m(o=DBl{uO5!Ol|{qj<nRS6Aqv4xHQ~@|sHi=witF<h6*{$ku-7;1#Ek=!IVW{nNOvkSI+g25`#XOCjb%o@oiSTyo8uM=%K1$yqv(C29&Q8kF}8A(uLSfv
j=uiI%{4aA;^k%bxw89bDUg8B_KYEK#Guw0TwaV}@$7~dOPKyl1VM+hdQZP4$7(Gn#7VZYRHI;JUrrN>{aM1{qrt39uKy^{92M-`4@sBI+_p5Q%ro1W-kL@X
()7(lEWC4N9<{R$A(VMcEA<EdR7za@2=UBmOF5)o*7pV<#O-%-arA99WB^%49-Bf(Ckf_-vkOz<<v6C^Hpg46<>O5-mnqZ0mTenZw5DIrtxVT)Ref%u#WOK{
uN?~A&4Z)T7Pt)26T3UdLIZM7fGWVXcK|N)YX$59*lN8#>AeJBXQ)Ie&L9dZo3WsphyC8uKzU70+?t_+aRBKP<RJb1L}o&Lhhqx%q~uE4`IA04-%J6U<A5ql
d1Je%gr#$tnX2NaO}!NTsZ~+bP-i>0ma%AezK%;l%U*xBusd1@5e&QHGdSa#>;hIon=}c*?2SaYxCS_#=gy3u1k3lv#oyfRJ3(6<Vn0hTKp_x`L}xI$nhOzO
)e(ssLrNK&WuUOr%~CxTxcHtO&~R#E*nOGXZCQK1kWvi)IGI}Zm5WL(IqC$)8S*2pPDR8KZbOT8W1w{2y9UQ7*8Rp7Fu$vib0$S0nR#6?(xcPu+!kEzX?K!v
4+P|ktMKy|We4<3HHemV=>JhsJc7uOS+Ek$SMqp#^3LnD-sPRyWVV+39$E10y##kq@t~ef2E5l(G&*fFZzzB@KlE&zA(gFlpgV_)LcpI~u+H%ctz;n(rK3c1
(6wkvU?8MCD8wGL>|1LEu2ARJuX}$N3Jq3S9bmOYwinDguwd{LWbkgng%$yM|1<BbA>)6!lu6CWKSR`Xm%aW=slNs2tCKeG*NmT?g{nRL{OJudd}(o|$B}k|
xmzLOP;;z%F7(!+A>b%TA^0v}Y{xc>#N6k0<>>+U3zL<C_%H&AD4au5J%0Oqx*QnvJu=W1Rev&}qI8;)U2spsd$shJ8}#J;Rd~&re<>hw7agjZ=NfG*_3ic{
xx7sN05tO}3*gco$U90!jv*$tY><BDhur)3Vl7_l(I`TCGxoosT1*rbcEJ%S86dtyBerc!aOcvQviyX^JRtX&cqIJbr>~s*z3xotb|OOdpn6>k271|q1RuXB
xKt+LC{(f?HkL#5LrUi#Xz3oz`XQ3E1?p8Rlh!G-CP<**>pFJDre!p>AkI0aJ_K)Stl*XK3x$3mV5cPuH~+AnXpi*%m2B-l$;j|y4)a5Lk|seCZAk(y0j!pU
>bfNw;XBkS{1|5y)@p@$fg!y;Y61kyE;ngLEqo>w<*V#htE}MiqvHiJ2wu8}%(25nM+kisXo-sGJ$@N&oy>}vfJG@ksm4+J8l!~gAb&^`vwGmvn&5mFDOCI#
)h_5}P;ZT`T&N16o!mjcg=!vrJXM6$caegj$NBCa1PalbE87=hPdRNas5+w)a60tzo!f(e@I3<<<OD|TO+)`gRE@S#2&Ke<r_IWqeJ(!B?}a!Qk~tZk1<||u
<6st#Y$I3LZ{yXkp#?TJuP<Y5^lPB&8hPs;CN_7~e3fFLF<5^7!ud%JNnyyNrakMdCwlc{Bd31&;7jOHM^0&&Pp%!mbv_EXeb|D*?Z_xuL6C7$O(f$Zw3-iW
2DKqH`+|$H_eD++eocYnHFQjSWx=WhCKhq?oFS)ggA>|WZEHsZPJ?>)4T=AH!`f(;U5!^!uN3sx>O<xXu1gie7Zy6kKer(wehfAno*EBYU@(zk$l84NPoeL_
-{v$VRI8~lgGr`Z-Mv0|ul;4;(&(8R&8lh^o;Y8HA_&>mIlKF(E~#7KCNm~z{jO<6J0&^Sw%LnV5<P8bIPpD5Y1&b1=;DS$5j*^}zb9z|RfLYqCqxYirV9%{
g0o|m{@tj%eH)YK?p29u$Dh1T6p11?-pG}<^ljY#lRseVcn8SMWuh|Mjq|`)l6M5gyo)bpS!X)L2B5sXmc{$rw3zjTLVCz@a&@MJ#C0n{bf{g4dKu908;mql
{NQRKd};oih@fx|X@n^cBl+6>Hti8iP8w>jf|2hOc}>ut6x-kK$sY3}O4i^qL$qguB82%dWZ`;mYrvX>P6@&y1q#TO+|e5Ykz6U_7{9W@c@^qbRH4;sPemf<
PCM2mG*ZZ{T1MzJ6THAZKwAi*xVe9xi2!yR@g()`zG0B<0py-Utz_PVaf=et1^e32!!-;5pFBg0BS6t(wc%k8sZLK|)pJMJed*<;7%Q?ws>?3(eo`oe-?LWa
`Txb-UwDA+@Br$oaEYvBGsM?OpJ?~NYb7YMAZdm3pn@Z$HaRskMQ;pX0NLK?m^k{Xx_%>>JU45*OaIzDsydxDm-t5Cybtcu_=C|jegT_|6K2spdgE_hTvH5-
5(f+^v?jKI5I!y0S_=jdKswleKd0swJa>Jb#7@AFH-6a<Ugi#x7n9GYT*Y)JsHbr1CW7Cf!HAr+<f7L&q4=Pp`+lw{xff{y=+p|S+|yY)$B$?$coS&$*|$!3
<Ts)i)<V#8t2_uL(Ux(9l03>L$Du;+5J`$mUpj8!)jhRp+p$XBQ$ZVZkAHtyW%=9l0i+#YCEX;F+&>#!r16Jl)=C9#CHb&`0+pi`Ny~3VCmYeDwFcUi3jSb@
r|W6Qb@d`K=PkLv#i$P|b$iz>Oao(p*1*w6nImlKj}7Ao|1O=T&G#AOCM0aKz4i;WCf}C#M*4;kQao?5aU_K0ZNtGD?rEUS5*MS|CEai88_ZYrd!mgOdR87K
2A~!me?YC^(IdS=grDpi93U|UFYvln{?ua~3s#6)FdBzBdEUHusWjUi@K;s%9g74(ZhIj0O<J3!Eb%}bE=bfk{qcBo98Mx+vse?kpuW1sh0TavOx=i(&#(XA
?f(yVR{y?QiRpP2jsytxo!CSE5WOOPF0Yx(;j2i;f_SYwhEt8Uki^fd!RQ&T?Xn@s01I`psx`vFvxcXzj=ML?k$o16j2z^+@44EF6=`39yDl!eHN|8&X=<gf
-zg!8V%FX5f;Y)!J-5ru752iPu5by|4(&Eo)hk#d04uM84k_D@jaj(xZxMsbBiBxnt(yPqY|mYOBnCfWJ>L0AMl4f_!J84Mu>gE-cCBhQT2Twl4V+yu;#aid
lo%9H^I?XCgTjF%;f%}QPU@}-$PP#4-S}0w>r!vmju}z6*$aGb2XrLvC{3g~H$IN_3jrc()-o4W{BK^};k&5AO1znMHvp?y%pw(P77R)#)vs-s#n#G&qu1GX
bCc7?+Gz;%HArCjxL`|W__5(yIP9^lKV9?!@u#N~y}QN+K+oz+z4iw8RwSY3Ox$U809W7OI3lW{u2gQXm8-sVoDX)j<n?oyZkdWH@_;sSaapo1N~|uZ1EL9L
8BCE%u_jMnZ4|j-7{c>uvwqq?rw{WYE^KJKv__;mx>G5EKaH|Nfpk^Ym|$eeXSL1s)65Zw>%%m}ALS?(7l#hNw^txu#70|$n%dNDs5~y>rh!ysE@<O-9Y68p
)={EhGNVyZVaSO>3<<eZO*P$T7|s9Zokb`h4yirA;M5xFlX*oIM<GnnGx-;hsgDE*7M2(L{UW+C)RB5(s@X8+kPch*h!aV%!V`bQR5AO6eH1g6cTHJG;-Da1
BOc5|1T5!}ag$!65xc95Ry5Y6=`)-!dHKN6%ET`uI{pnbmS}oAOaK@XO3=Z4!@&SBQDOrSeL6fSrEu2x`zz`WJ_&E=d1YcX2bX5p9CjBZG29kFENj+Eoar5;
YF$8A7zTxe2s^$$YxWtnxoZPvz{N?=oP13=l(hd(vX;Xp<!0~VtU_}?5*#$99#Xw(_<|qob7L|rMRRg**|eX8VHj(-mkxIKE5gYj#-<22C(y|xgz0#k4M*=1
yh;#CM1=3iY;?SIY*9_gj7!ql8b-(%L;N5qo(5hBAJ-4|D`c+){iU0xa=J6F3=*WS7VEC55K`B=f-NiGRy&fLRC|)H`jj|nisj;#Am8J2@=`Hs6;rVzSOF^$
qT20;RpuYs%NZi#)M-<$xs45OfFdv<fCChQDOnBQl^4LS7PBGTQXD8>f<JOW8}5JS|KjAxHU}#pTU3^negjj%Swc+X{4aWHc_kKnJRuP%E0nzYm?1ZSK8;kN
FM_%LRd86InH6voq6{q>{&pk=^lmf!@*?oUa-Lw;=}P}sqa0s7q5BX4WnS%ht#94bK1r9nd&*6WusE4F^E>I7cta-i1(^03i9lD2E&!es>Esy#%n|=i^b|%)
9)Ws$dwoGmvx01*)R{Lppq+gztLHK+M}T+G9~zoh#k=)tf8a<=t15~;?*u?eCNKniRn#U&IMWu_;@w-$)VG-v0?W+$r_9(qt#?!hMwI|n2xIrCeC%t#@X5x<
22@;i)0`f_&=&aqt;Fx{_io|5MPfjRdnc<9m0Douhp0>_*JQ{I#$iMxwq5%Z;KJ6!b%08ziV<&JTTo?7^t^KG5IxYEGa|L?IP!t4BCNexSK_v?i>OvhJc+l7
#dP>(PglhmUl`Sm&7FXfM|upGQ=*M7Zyt<cxdy_1q@$<<j#9e=;Yaxx<{c2Lkc;wRHzS?a0LrrM2D^KY;`}+zXEPj7z&SFr@0S6C8bDU|o<hpXwuC0UR}t1@
Xs~x1i7D*tv<}2fFyHFk;v331nCSX(Lue8uZ;MK35*EZh&0A^2a-M3D{e%=q<({xWcK)c02lWKt?QflDnFO5ZK`g3JLLJ)Sn_o#>uL(y7&5Uv?Bt@!NQXo6S
);3M5Vy6+gRX@I>gpDo-a3Z#7Uj6x8gz0=9&wkI1h7W4f>W626>|`+hTS8>9#KJL}EzG88d&VREpxcY>GA2^LWn7*XSZ)dau%?Sdvt(9iXg}n?i98SWVBCG=
zT3nakkEL5j)@!zQ>9xKu1(rE&qQNg4P9$Z$&x2<U$6jU*UG??nPtS3Tb*`D?GXbl8Rh<opU%uCzUdWrfj{R)b8}{|1D+gx?$V&SQgjf42a`)ibr--D(Bjm2
EKV9JR68C92_7R85agUiVrSJS9vSl*v&J-L_S@?KF9>*fy5)Yv`==d$faH1dyQvf6XaJu!-ehr$W?or@<I32b-a6?o`7h8d1D#X?ut&4XFI;#I<^{jy+;wTT
WUp+^FMFm8oWc96bNl!#*xXf5G5e>Nb`Nd+VCeItt#TvP_HQUUy-6kAk_Q+&@Bnul5rBp*r+PHRItE@AJFnMzm`-uSQld5ML;^T#(z{$S{<CpC%nFkm#>`kY
pz{shaXjf0q20TCuiiq3&dl+I+9}-(9^UWOcCD^<8IdA6|ABDH?t-d?m$74B@7K<Qgcoi7RrHAhRhoQ*GZ8_5*IKOJ*V{;(S;(5En%~k`@B)Kd(Cc#to~$Hr
!VO3?Q%no;tq2I=`(D^fJQ4uPr_i0uFBA5--6%ZLqXp(?r@FmPi6#(Kg^mRa3jnHa^RS)FhK~)g(GIFt3Id+?<oZL|K9iZVfTQYQ7*w$6!oY>QSz9k}mPLz3
(54{6aTVcWH++^6;SL1DAABIdrI0L@lz}jNP0nfvv!tnig6dE|uaiDt9y*>B_d>r^jah?(hcWqxSrLWh{D6qL>u!c%rCRvByT)QO6!Zo0vXGN7Fz}6T7$Du^
D09<nL;gcWsT1WfG{rR1zSBA8n^^(eV9R?`&AS_$=J#qf=CIInHa^w1z_xIU@wrhjsG(M!2(ppWJrn%~y#ZBY)5L3};+f5nOsp~|a%>W^(&?8q9Ke>kFM7XS
!%{SRe7-!XVA!(0Qx<W{w{C+;3LLb1VF+wK<F9L<21InapZX`WLVA&^(QX*HaQ|14R?o>pGb2>Q9QexDa3WB6T(Q|FyWS7Hr3Ce_BO8YghsKY3P$6q+3Pi}1
b0gk;B~y<4!K95KmI<~S-p+%qbn9d062MXN611cQkfOe8>{<4}YWtmIuA57=`fVJg58~;m`M>18pZHb}PJxsonl*ICSVHcL%d&Mu1TcmhKCZxuT_M<ToQON`
@6~zk^7JlzH2bes{`OJJE9csg685ZnQVV0{Cgh0YVW}EFNS(10A*HhOEzXJpcImEA9mOp1C)`n@19Xs&gNeluC>YWesQCzegNS!~@XLwoPHH!3Hb~SU&fE$$
NtG#9o#Zg*GOOdD6aIVEak<H}pUZ%(5*+zi&|O;16NCR!8z&{hgfo+>uC0*m&Z^WnCB{Z6<^n4RX~Ct{=dLi`F!Gd*%`4KMZ**c1m!X%Imd9?a_q|k%2lW2<
7|RxUd@~e!`&}=CZKQ{cg?srWfQj%n5K_3|UB-#-2-8}3Im&bD3?P3pXZk+3bclEii*6U=U_f}37Ye23mc~RDgzKTkzjHf!)Ls+L>qEj*dsM+(u2xFYm8)QI
?D!(_z-#o$+Tvrq%1;rRSIvwJ(hqa3yS7-+@3WcDW$KnY^Ty2ogjiDc5owq^#)lXR2SjN+?wk=k=dGvidQSf48}n}7p0W~Ue67@D;;UO!2*5)(df4>PRdCS;
fTJ8mUp4pOmm^wvEnlr*P#5lq2E+Mp8q(RDMwDD3wyv*a;(Kvjn2Uzx#_0caw9^qfVtGe6MW3@odpp*`zb|35E3%qf&{-dBczqIKlWj!dm^zV+#q)lyhvnH_
8A#KioW&jBC0(|#L0;-Ik%0r#$B16n3aHxg8S=0$SWp~3=qStfRI*ht8h)6b`+&+LBzt*}-|1n+&JwGlD~vbER+rs!8Q8c<Rq?H_vNfj<DZn4sb?Fb@k$XeN
B@`qGZRwIjCv({L0Xk8JbC{B7i7RbMChH}GK@f@h{&Vuq*!h=@Jd(%LX#Vq0?&y7fQul3?loZK%qp}36Z&ay|s@~3l!t=@1(jkkDv;EcN7|~|RB86N{wIpCB
9R<-b%wedPz3Ed=HR(+kfC0)i>Oqh_UUS}fZ3INHxEbUtw$YDWxAl0KzkDq54@6L~)Fw}!mJ>b*Z~|lK@5Q3r^XsF)HGIo_bl6f~knqafZ_W2yoL~bSP*tr<
&x?*#m0273t=%zR)rXBxXdmhIuQ<iM4)>j0EhQr`@MT=(OM1~8R5;m{r!F*B%c0zPwrb_keHG%;4Uq6-ReiPw2c)s$Q#NF`Fg~W4FLg4V@AWe8&95HH((8dW
lW8L<2Psl*eqg3brO(YmTlN{DXMmea8}^^PwwJ#&6{ZlFy&fxU-;rMaebQA}5X$jR`0B|wszEyCwdhIJrV189)!GRDUo3QxEx9t>K^O=Q1c|G>(-I+5C}-wT
%5p7kfK_?B$Zq|U-{^~d3|4RgBW-pUu=T%Z_S+&lIgGmKrJQg2_r|Lc-%8g+kkT&YjKjKlt4mXegGYGBKDJ`DVgI5hTDh4MLy5yntiBDRBY36bYu%&Bn++;+
=CMDw(g3C&VbQ`yiZi`+J21@(d9dOfCnRU9duWJiCDRrsf3D+8rvcd(wO}#NY&amx^Wd+uACi6^%#UhCQht^s+(ulcxc|!Fa(~B{h=uP#DYplq=SFrL$jL!M
>HASbq{O@DaG1yezI>PgHu>?A|8Q}3@4Z2s)K~r_0@Vi<FCfxtRXPh11hHX01zwTd5Xq6`_x{`tggZh)+#FC9K(F@@C~eh)0ZCr^qs*of!p#oU27ac0Dk*q}
nB8&lT&7`ZY$~35Qi}V0AoTPhz(?vPY+_&x;8V!`FJ0vme>E4Pz$MGO1WbUMhKZ^BSN{%`wZVujEuL%ZTpeMDQk6JNV!`~923=eA#Qg4#PCuQ9qaNur?K7b}
@uO@`L7aC8l9To|jeSE}!6;gb0V;D3kWnFDy#`5VrsH?}eY^`@v7GyP6XLAuZ-f%xR`?<GQ4W*kcf~G+<F_2YE3&z^-3c!h;HyrsTdpu~<9;wflC@0rAs(FE
EJTA$-guJ%^6aXrMH{2ETOnAg@5z|(Iaq4Xg*ys!^VC66R5i3cESYZVMko>CZ7l?`K9!r!f>XN$40w7I=BXF(fXl`>jmVm-ZFh_9gmuJaNk6smn`T-4hl}8^
!XwwVXl#G9KaLbSB#wk$rP)|3Uyyj^F;x^Wx2PPWig6$)^n)a`0`<qfo%*Q9tZ6ygy&?-|>@)R69?h+NV<nv4ZOTm(F0Q~fJ*0l@F$_nK8qPoRj|z<mv$dvU
Q3IpFvcNRqVgz1y=iWm<({%pUORQS|;uiUNsOkY>{?BLDtaW%)w-QXfwUk<yc@Fd6&E%v#d~?3lEy|j9ch_HCJ=Gi7pHv-STIEcZJ7?PxNZ+a$3oT{RvL91%
CGwUDia5LjIr-u*Y>*~8Wa@?`bpGmn7Y{DNfOKuyp$`C35;POgnYVDMLtETUo7z5lbMEVpU=D}Fz4e=m(<8kkJ5<H*?6*c>sKg_IW_P~&^bO!h*K*WLEOP_9
C}%I1&B|US?2n_@pmK7agK|H~cBLRCUnh-ts;iX|Q0#SdYU|;cR@H+%o91!!kz8Bst@Hs#8*j~ut=tvt;aK@)ae8rcCS`a=SmAJWU4pIfcqS7{-i>NGm=!BL
_`N37w2o%?Ei0cwB0gJ@ART)BtX6bOf?WGWE%27HdaNq9Do(FFps7X?^ROBZuhqPEFnzN1lMBWi7`W`G(gOx>#IugRbuJgpjQhJDIr~~UxX^L3KGfo+i2L#;
Kvg-p9^;%<Pt48kYqvxO>5qo#$pu#4<JiW$^e@K$KVSD|sbsS0wOL^gys@{i%7N+L@)?E8kOQ&nC^Rxd<kPS)S)v|Yw`HN%l*>-buHdwP+kk}9#-+IYifk<Z
cj}Zy3_IO8(R*Gpd(Pgc-MHzdxJ`&ChHU-ov4Tq{B@nVur7Oa!x{MnBiv{bESqpdM>R_<3go(7ACYV*vys)P+QyWvMLdg^$sJbR(+&G@6UybPF_In6r_Dq<t
VKGl-1#UvzRKa5UpjcZ0_HlIvqgNwH!i5(V*Rr;e&%1U!oGe=ADdD4ZVGH7|9vL9c7{ZHue>T&(<PPclKHRXs@a`_c4&*6)J^K$1s%0>66ka=tz8nF;Q2KS`
)ZylaJ(~SaSBl+n9lu|n73N=wYF3<Q=ZvNP$+cT??KYdCXWzex{|Wo^Y@7>5?okCG8)Um_QF89oMQ7HYbA=rl-DveWkdPsJ#<VEZwLv{6nX+tW#fh+Ein@Ub
vx1_$KMzs~L7(g1qe;4%RdHKpy8Y(PiCx=x5U3f&A>6${jWJ{g$X%FLZ9{)0)u$MUOu9nLvyDGR=}^EZ94CPOq83+6zN)!(PB(XGb4q_k0dLSwRwjpZ+Vv3f
NmDIF&}euD1rJ~Yk<Pd@q1HQxS(9mAYOcwK(TN^6SmTCx?P={OJ*6S{v$~`RKNvpDZSsh?g_>p{c=hrI<3wG_M8yJE+l>aOjn`9TY>^y49R`<qGrJAxbN!w3
zh^}^#;(W40apK<zH*qla9n<AJkrf#RqfZwn0>bWsZ22>;vLj!_Tur26d7TB8yJRIf&;mf=wleMYy3#72SL>rj>?FziuV$rk4L$Jg++jtKkNx+wAkDWXRng5
?77{zVmDWa#057mCrKnTO>9M6J+6hsQL~CxQY%g$(F^DLBx}`0p`xN13W<&A_Kjai_m<xOu*ScECNB&TLyjfQGRc|ts$Qu#J5BPe6C`97rLx0~hFxk=nt)Kq
!Be&~0cS5u5ynvIR$27e@;{|N6ZN0`2r!DU9Zq+~yZ9uk|Ao4QP5~=%h<F#1K<xoy9s9U+kCksm4)}Q8^dW6oNq|=u<P-W{t`TRM1lvIL0^r3Z#rB*b$#_A5
dzeKUXd$t=zNv3|5TS<fT|N&KP@h-WbwI?R>^vK@fFXV9fpXPv!zaC#hLWMM$niLi-9`)8WcQC}jZr&{b|3Z)EDqxKy=FgCE}W5DOA0Z)(>x$SOaVtWgjtj6
omJtGv1V$97!NiyBWgFfn8#xYY(k4Oh4Bt)6=Bp1?#b7J4K>ieHpr;>ZZ?Dhp{|eTkz`MM$6d?PHhY;Djvu!$Y0bJPeh%O82Qe4(NoD-uO^v2f5%UOU?9E2H
OInfJ5gB=dV{JT%P>W_eY9(=FPPcsvm$xL(SxzL&4Gsx*lUSWtDoGeRZJ-U&vRs!r{q43vlBY<?Dm2UCItq4LTIQm2+IP<nENhuLdP7t0!w4A_tL$Qp5eR`*
CM|KV>n>rT7Dfel)4lu={lzXZ3pWkQIkToKqoB$`nV0K<L`x-r>EX#z<y{78Wx{xrd#R&AL_DcI>-U?F?Aw4zFKkQp^-y%XG?O5k-522~k0Hw7iYZ-Cfwi`K
N9n|KyIQGhST`<jXeq#(RbDGTXpQ_N7WET^*5tt!e1&b(A1NeDUr;H_Ftk3ProO8OrN+uf3XGE5Nq1h#?H*Y=$hfJFC}{;hIve%WA`h)a@FML53|pvJd;a*4
N!3lW;U{b1ASY<a3X3R0UgYViS1)MpirR{ExE&YSXHI8Tu#qo{BP7!m=1nB}$RxorP8*-8E=SBT1F0f>?um&b&H4aT<1CWxEC~tX-x_e(Ydawaoc+&u=%w}G
eSBh@w~rMCXhw_+7Js5`2S(PIM)s(rWs<T~Pqhu8(A<PvS<QL%nDgS_xd~D%Qg!Gk)19*G<r=uhm5>LH3$q?H-0^6I9K9gsl*y9hbVsNzWz5=-M}$sm6`p>J
T?9thtg9EK(<d-3hlCD%j6i+>2=*&MLBxG%Q3(V3=Ph-ztdI4UzwzGgMf<*szUK*Eq%uSYlm~DWPM^A(R)s)w3a3s?GK88B5Jd%~r{7Mz$21Pg((G_iH2yW7
G2gHm#f>GC=JXA;np}1A6}V=jXR{N6DoZe4#2)ZW7oBVMOg|U|tHuoe`H|uqZPpy3Rg2ZcMXstPko*P*Kw~3F{m``PWN<Tcxx<+^dIMJv48y`sCzUVZkJ@P1
gB6Tsaj`}oj>UtZTdaNeYb<HH@dN^~afV6ZNpkhmuAADTync2lCiE7{3h9;`BUFs&{7eu%AWU<4fnA*Ci+#!xQE86LJ<Vir2+HC#yvK_(PS+zcUtCs=+)E#}
;L41)?L9&RUD!A~WuoVCYiv-aC7SNKc3hu#`{a~lfW^J(W05DVyY7cxKF^nVTwaaEWJ;K$$)A`uRED)r-x)okxqvQ_;gi1ut=*;dhin`>^Ql%px<n4lSU)%%
GUb#!+|n)bk=5f2FW#SgKmBl$p*S`EI@t2&LN9|N>r{l7Bj!VD2O!N%)n>&^;IE~?*4Lt?jJp-nt5F{}{Bofh`HD889w}aT>fDT&9GhShbQF-|>P)^MV~?B@
1pZqn`be+jD#&mQIA|C2uBF%6>Y?N;ddx91|6;30eof#jRTOKb04OoM($8lLeL-oQ>&G$1sXMQ&0pDwSLxTD>%CYDO?c0*>$jKfIl&*5ZYS?QDLSz}*+H*)D
z$vSpACOEEy^an7bKj=dc+#%L|Bmt-1EiP(e2`VyJ#@B*Lro9w*hse^S3_tCwwW572M^9M;>cCOnP`^_R_8RSiJDHMkIy>iTxL>b+jY&VEL7dEp5f!~-uW-1
6;^|Z$vc4%_t_k)HcUt-7vU>n*c2t}ax~8*TRY{lSYc52FkCdpied=ceOTKFf08tT(Zgp0@h_I9pi2pO-Af-zB9?=!zOm8cRyTZr9DnP5YRPDj%@8dcws)oI
Ml)9uGZ2UM5aoTnH_H%t0k42eq;o0MZ+<611ov1`XL@}3pTZ+V5=}KgMCB5^;n<IubX%&hLPXO0NMml(Ob7vQZPNE-nqjB?)<+CwD0{C-A^0gDw*B46G%v#8
KJ@}N<l#vgZpM05?HwxR2eSo?O!RTHr8JdVX!D{H@bTFL%ExuL!&eg5p!RvxJVq!cot#}StBr-9Vvp6rllpAVnFh?6&fJX(iC++DeP4OD>_&EZ5HD};o+}!g
p>i(bbxPgw5{GQ4y&!H-|G`cO7xkT_<})WaZ+P_Jw+CX?B&qp88LxBZNSPWB4TL6zxM9EowLa=<DJnmPKHTMR#%zs-`ZWWv(Ce2leFZRle%X#kzi|%Bc~abb
zq9EY$=M(YhPFF|M%h%(eoq#^J10W0wOH!Rr#iR^7DAQRF4N0;na?E)*gLuiM-hEg9VMY!J;sD+i$od8^&P{XRB-VBJN(J+L#+-XOvM!Z1+y5P)Zj)qtRb3<
q6<TQf~c=vC4omozd=4w0xBQcgZNz{Jtt(sdl?|w#xEt9S@QXfOZi<3+t3)!;iuIaK<hR;_NT*c-wp?>VUy^boyJA(heY?V$)OP{GK;>TS*}f^d)!yF1e8qG
#lu45NNU-7Pq;X(U|;H>38@1^?}h}P&VIY*IbM+qZhJ{8i(o=sXfm=9<cfmhoQtUU_+FLu8}r0CM7P>sBz-o!)4}QPy|~+0T&z~LB7@o;U%KB12G-czeYm_o
%I2Pwe1=okixiVgggi|TZTY|zj;?eL6>9Qvv(d~G^KnyHs5!QQ2Unz6x?E!%GIH_w|4ye`ZjvP^r1xjYD2O^M$?1!UNA4UZj37IX{iS*Es`$l$=G<V-LSjGl
^9^?OY!?>DWi;bDN*7M8vMJqxv;70+8`&x?>fPhow2>~wdR~?t$1pMD<qz)EEvzuuGuh*TdsVTt`eTlO&7ZCxNF#9Hwwvx*dDoWw@!qE<x-V+_ol<Sx+DLfs
l}2a)mtUIF&qWNFq-x$L7+C`Ub*2&6X1#~-GI%g+jM{v4%GlpQ4S5xc!Usj@bfC>eGzK8Y@BA)#>^@Sm)6E~SGKM;&NUL(gP59!FQv3CSz8YnL_9vS;okjJ`
0PoQj3c8`1`80p{a#11*W$TXhf-=bR4W)}ZGPhzu*pWQrITjCEfEZI)hJ;SlsM6e3TiD8Nd-Uh4CBL?qK<OU?+Zh-QxdeIn&i#&HGon)53{yXzj0)3{#y`BD
(}Iw4vbBHrLeqVJtcQ|cUMmOIp4z23E0QFGUQ(<2-o<Un{_SO*$x3~7V!B3=>3s)!>eSeNa8l!h&#{Dk_RLsV_N7d6nAJ7gH*Wh+NC)Som?9|jp*w=Oqsv&1
>whckvgPM%nf-+#9Flgk_!q@+_wO)oR>3MkSsQ7TGDtcFR4u|uXu$E_AAterpvn2IZ~TczMrq}=+rol*i`-wt#Dnk~1*Da(I&~s|b(WOqIewBKUUILJF_i~U
oruMwyPd&|=E$;aj+$X+?0L&DX76#Dv`55I%!NIcvd|18uxD>|iooFC_FBavyPFza?7Q@<)<EJ11{vo(%dzX*;vtgbyG8e<XPUM^_D5Rbf$Ina#p{11S5EQ;
aD+VC)<S8%bw`7Gm4lV-wJaWO`Gb}y*#Y$ac1q9x@qC83AP3bN!b*{TDdmNcg<c>RUMp{bc(Z(M^1iE+;H?v4-VcH{`pfY;;Chn&gVl%zQq;G;rB4@HJ5k<j
SxTY>MLMcR|5F2TyEq)$HV*g|Qg3qIs6^8fT@vV!_OKYwM(70ZD<o)izt2{(Gx<<Lwi~7V=#Nm-?cX(>cs!1Idp$-K?uejc0?WPi&*|CEH5_u7d}8&Z0R~_r
l3v}7RQGhW{}N+eEzB-f()JUn3t%$0FC)Fv_W+=MptUup){S!<od>sCEQAZgk2E0a_ZA{0%S_B}$o<D@^;}j|PbMVooHzN-?toJmU__voH(cD0Jo~Z}2l}}k
3Q0ZX=$Y?iq7L(Lq5qffdJIkaNQ4QP_h43%NF$7f4QUA-h`PuOrW~KGZXq4gK^ay=8t^cWCS_p4#6BNg`JG>tyxxKyq*%dSUyuh=L@%B`vN^9ve1Riu6<nRb
sXw^=PvE2s>ClB(I1jT=?Ynv2hBRv#8D80HGvq<K0sE0krgRweE*FoJ8=*N*3sT4DPxod3I6<7Tn^68|MesAH;!On&Z{K^SaLeweOXYAiyQ?_dYKy?>bL4&g
`;c)2_;m+;9J<50H;Xypq!<uF1D5NDQUCDDN_vSmACajRkCVUuMmDf-l{HqJ@l??9-7R-u1bFnc=EBj|cWYs|f8!rFFTzN-s!z=YSF~J0O9o|kEQAr&X>`zm
1KEPB&Z^;g9Q)73KgXEAE%Tw_5!wK1T^wSLN_i_zKt7+tKQ-+Nay9MyRWWH$oj!kXepDawG;hl+q5Q3iUcg+LXL|6GoeXR{H^ol-DhJ~1Ya@lT#gF*NR=mJb
f57@aobJ<|sC+wR{yHay@*eDz_m#($W{QGPJ0jy?vagt1Rt%1!+Z>mjr{K=`0VFXaI8XFIkq1e-1K=c+fz?loTNT%pjIe*wPn(^<g4y{rKn-QD)vGU837v~U
u^Pn*<x!B?N8`y`o`j9S9!>&l=$>GQ*8DsEVm#HG=YO5>YvZ_8wG-9&BrN2-jf0=6g9<ddyRA~0krx->O}$H%^-OGN=n~^CbDNRUq)!SsIA+Pu!3!hZbp+!v
F{)z2<1q@F-iVZx6e_5srvK#s`@PQ05W`1L%Y#jsB42<ql>p%A%JUadNfVM_58q_|$#Na2Ro!~Oi^U=d9>r31*Vn6eD*-ulzn~9W)SU9=m5)*h{H@s27(?-Q
4dM{r5xW7JMv)-j|3!38>A$oM#d1A$(T4lCI0>))LaxgGy~j@bMrhF;&3*drGdRh8J!yC6*t2k=tT=sokBTR;!5gkrY4LijiT{)_G0o8)W!A$9lfI0TgRu$Y
Z)r)S4%8gz39K1unX&@{86xHTVmXd$^4@i=4co{M{^7dtOMmVtS%O<#-4s1VK7EIyc`2FsDv9X9h4N~N%~UXWL33YK({*Zdg>8Q+WjVUg$~BL<MXCHQX_(3t
P_nXR&QeN{&T87#t;w$9wf*E6h3bWRH(pDn=UeZMj|?@d(^pOt%RM6Q(be-1KCRAZtvxY5-wFPVB(BDWQylKykcLK`%Ng6Q$`;Xz_iXCNATggs53mqYk%(iT
V5vUX7*T0iSM0xh_6FwcAP=or$QAXVk1Y`ul4cJM6p<b;NnxyuGp}aj`QV}=KIuqGQc)8?K_jF(G!fw5KAclGjO!ak0vsKeBpmJ&#~_)QvJZ-XBW1D|QLb{!
;R>o@*D<TJQOHDK`$1aq0BF=Q&76M!=aX-uMn|lGe}_gsRnRqDIm%dF>wOa!s5%_t@zvdMsT(;CnmU2*jYqZGeUN2F?ro#xFOe?#0%OH{>@0fQ*hMZ;go(4k
26Agg(J&9%wIvo)rf43Nm9Y8ZMmspVjgClP>6v3Zcb?efDyr$Jt&LMb%QQ=Awl4Lr%f{8`B|yS=RwpRrh4@@6>z|(xAtWkZ$)~!pY|y*lsHE7)m|Le~ojuu4
UXG5q$I_VTdi6^}`qCNX6~51_ob)gEGl>e<b$K*)b)v<VC-V!;c0p!3Aof|II_1Wy907tT6@5Q4?mv0EU0C9edDl&>f$4L}YYduafqSye)~ama)N)#<Qq?G*
g*Y%IvYRVwiw{Q^U~TYtAtqs9r9aaX`39|2qfN2-o9RY4Rhy$tm*U>lAtZ^Hg8~9K(Y?sihFVl+Xa*W28N<d&j&#rU+6w(<(%*4#x|GScC<mirs&OU?B?8gw
1(jq!LoGx!WSu)X2c*&hO8CkJD96$$BEZHpovXHJZ6i`}Zz~(d3l)GxU+oQ=CX8qVv=Dp9o6qPv)&zQ=d&(EAd%x}h-s~g0eV!(cP!O?H_x{N2MTxw1RpGZ$
AjO}5xmyBMudrkvl+WT1;Y{j4<>jbIDPeIScPa{9f7^^-aD22n#|!1-C65gcFxaj*q1u3zqA}HwztC-7{A#Rt)fXI1;OVK%KkcYJr-DmPF%Bzm%5h^hN%NU5
Dl+KikJaOZM9Yqhj!q;SD55c8E1uSFJns#%1JcGD0`vT_BgdS;_Lu^nP+e5+`?7s_*-QhmQ*B71OuI?}wu9~Zkikn37-wcN7EdLNgk2-Tv4gNq_=}%><N3uW
W9}DpYl)22k8lhPyIca)Lf6&t-76(Kmoi^*K6q8g2bFBlP&ORKzV7}mz_7PRUXFzbm9r)(>^b|v_Apg6DQ>_6D?7-(>o36X!A@PmBiK2b&vYkyS?Z@ekOoo!
wwPy&AdzX{^?$p`&%qY{;O|fan?s!UX*TrnCAiT_W9rC~P^;sshfxxdsVKtmYQ!FL8WPsV&o+6|qydQZ8dSZ%RJpvu`(@Py2PlNd)2Cw%1*0%Rl=Spf?Lqdy
w)`Apch95l#|h)M2>tL_<Dv5mgg?{qo-2WJGL-Gxmcu1zF!8^QAD?n5QQ+CVHXcFdBQVsok+-QF%oTm<Oz3W%XXYy%cg-RmoB7o)dM7weW{I>QJi4`k`(ZGM
1Bc$YSk~SR#^nol>UEeF#$WS-)Lz!}^WWjLt5sF$4SXZQ$8*g<{S$ToPLqH!ZHm}#<fp+7i_sv68dW<^e<M=@yFxpa%e7mphu!u@E{WEh-`W5wE3u-0ibtrt
!W83}j(8ag5@ZgACf}EkmvvG)jOZn84M7Q%Z-l<W7do)Ey35yuG_npz$pSM68(iJZumKvdq3k#jcCnnyBjhgHRSD=vtw)%j#juuqc^yt1xH5~(u^G`!7ksT!
!BOl=P77BY^}V8K1vSg8pBbdj^KX`^O$F5nzqEoWo;i_yiva~6Xj)Yw<Jdjr$NkgNt+0`26MGRh#f6o`mQ%|qLgJNt<}$ma=HB42`;@=j4gw&#AH8|z&QK_c
M|b03rhz18!ubS(Dcfoy(dNWOmxM2Qs5<S5=b|cll#mHW7-cqa)=q^BSLZERp+Sh4-G19Hs=}G^d)Ih+*3HW@(IW1`+ZjFr+Br*NHSKE`aIO?%E_=Bi>{fC)
O_iygr85UJlax9c{{ijOG1B7^O{CF-qbsS-ZS%~|s^!N3)7K=fOLA06pcjg%mc8K(MmZny<;W?x#-rheD2rIEm2IjuXfDD{1`1RRYUEDHo(VgVABo+{e(YHk
pYc4YZX_#HMXF6r={J<Kjq8U88R8me_|R78`4KYWWWZs)N!to0<(<A!VD<Z^EFBQ8M6Q8w32-AF!shtdO<*IGo15S%Tz|5-3or~}?8zse3X(sl@vR^lBeK;i
z8K-V^%Vz4r*Vx?fL$&7F~KmY%k(`g<1Yun?iw)mu#JnOXebXM_|3XgaV7{q`*byn!WqTRPDzgH!$X1wb@y!6y9=h^;S@xJAQXHW(P5t>Pfo%y#bJdDDudxX
_5!8ruM9CLi735=y|vh27s!nb>gW?9Zag_Gtl@Tis+kn%xjQ=ae;-S+ZWQP4+~&=O{*V|1g}V`;z&Day2wH;#D%x{Bh!N<}SZrr`7F4jwbg-FU*S%Q_dxJiQ
NQn+<&N*lMGz!=8!KkKkIU&Ap6fP$*7bBp(7(kTSK9i{8Q;B$tKPJ^bAuXUIf&{sf&Eda679TU#zEyZ7k027<@tBS)w9t+6FDop0<;pZaeve>)TrDq<`Js_?
c6N_?HFNP!COurD6zR<7ht))N?^6d~LCMzNg<l>-ykGak@1@if#QCQ|@zuvCqBl=!r99<gu1CCR{RErqe-XFn!S<3i{w)Sb9rFUfK}0m(X`Y;PL#FU9eBfYZ
kvg5?z;2Sr<Tc)th;INdqPhjT?NVXHcxVEe8hPi|08C{v>Q{I^y?kD25FEy)Eo@4TfHgVb6~HM;qcWe=9G@i;i!R5JL}VAu!?K~O8i$F+haux@sQfnn>N-$I
(93}-5$?N`&?E_c=1G96)5KApFIix67(8;`A5uFF^-dXtCXlkTTQ1{$b9DZ0ilo1ca|yrwEimuNYCIgUG}ukg=MlV#{$$tGv`@4^xO<-viC$t5a3~N#EZPwY
tSV^&_GOgqiise&*9nD?<v^@~zV`+$uwbj4fh#eI@UQ^s(3M@oJY*MyTr_2T4gssELJ!;IExkAH`fxXg<2ta<9?Ap-WF4O&Zp<p<j4bBJ<1|TgjS{@-i;(TC
+j|{<37IW)AZ!E=t!h?ky~SS*U+?gZmIKCouU+HT6!y*SZUy8*uz1FPSceuZ-rRX1gWFF|q)+RAkP>rMoHuUQGw?evv9*bI&Z7bET_|W24@m|KEkb$Ob`0@J
exj0XV`U^=Zx73Ak=2+ltQ*9vU7GEEPL!eCb32WUa%+iapE*&sZfvbyCw)yF7oJ>YD8C?`{30Loub#Tw$(v&f_UG_5FM;A0O_k-w5^hnJ=NX)I!_n=y5ldLq
#*O{MkJ;uIpn$FjqTuD3Y@agOGmOcF_O?xl*LlC`Y8gBuRUQ~-OeyT^`o;+-Dn7FhV*u_e(!g{+JQ2s%*RV!$ykv`hvIJ$+rPRz_rkS32<!RRI!gQrU_g&rn
7SvsfPu{fNk3G$jd6VW*#4FGh*t{yH4J@cpc5cH$ywwJ)61L$r<=>EsHVBwI;12d?E>h3SO@JweovZYPl(U$0mtv<7#DXpT(jUF;s+P|=(kPLx0?9bDaCUM0
F;@L2a7|gX?9Sn(O;z>iGIuEHhxpMMVLu2>ko;j{*8&az#pk;2L%H^cMttCbDv)O$AMsM$xFvB2j9w;CbdiU0)ZBBLKS`znqj`)mmQs19`*>D5a+nK$SO2&7
=`m^oS(E33QS5;v<+m|7Z4%!)jy%Q!Y==q7SnN`<=dY|K+lY9}3nHWtKG*$^ESzTX`8+IG#!{H1Ar>LGmBhO#VuyirG5g)1zf>@$(T0?f0^{DN$@W(RBJFn4
%!tw4pFqTB*I(!N(}IK~#d;Qs=p1JD(DpKE6`>4i!2QHki<?`A2DY=?>t62(HjaKSt6Fk)4psdm8X?)BBY?y3d>%uQFOcjlHF0?K4riQcrQ@~uTMN12<`;6C
ADCe&{lK>9`7#(@A=u|I5sKq>q*d3De{tDF=>T}XPj}dgw92e{P^13gY{2n3QB~1;c_*@gxhaK+D6bqhRj5bmkcn0lsEyFMVN-~j+6&*{ayY4`N97COD{N}!
_3#S3Iy&G%RVZ8D(5DzTKrD2{_)up9Z3fe)(-hSv8q?`&Gqe-=(S#l-G91|TmY?yXRpY*WYkCqtTl4_W?n~$}XY2L=qQilq>N?SXH6O(t9rffuNuS16lCrej
V_l8>^G?9$%77h9T=#2cVq1S>K3843G3$DSw$B3j%~1GAtA=Cj)30?d?Ye$c{S==t7{qjRGlj~vuwSJ8xL`%HAdi=}lh|D?z=u7Nc_Itpq(V!Rj!zuQ$N*&a
%Fr;|gtfg19m8y>>+N=co#39gApXz&?li&Qrm$?)U0*v6h&-42ptaU+C%OjEsLh3*RC!{j>1c1E$1P4QXML?I$gO)pajH!{_%3#~mrHwog}jQsP*hqOCAw)Y
L~_B?(jbjyoBu)Sum{uuw;2lljG?)k+!tf&eu)vT0%}3Xp_J7b%<XC(p=!(>a}yZ37+?>LZ)=E6-@LVkv^!Tpl(?vMB|>wp>-XlBZw9H?5vsW6k5T12kUku#
b>u>j7EKT<-pCJ36UTB`T{;k~ufJA9%tU(i23oHu+qD~s_O%%U1uzW^sG&8`r?)mLrZ4a+qY%Ug?R%V?M6tR-oE2LOPP#x2?<#S8H)n4nPMMB-w$w#+w@#Bi
ai#3$m0<2Bb>r5fP=A>zqzv?ZLq*l8ObIEgGQUD4t$H|Lg0&@xYQ~oI;()%RTLE-OvM3Fedp=+k7a*0r<-g38M5=fXJZL0{)i6sdXxy6sXEr%?`uIL`GEC@}
=yimr3vd4GNK3lp43ktW*tn84F+&>}3E<mcN^B=RvAfpXc2_%^3?pLsbx;t#(QXQ|*IZWk#4jmPQNoA;$lQL*PTV%J@|3WP%&E2OGIc`Cep<51y~=W%h!j@P
4OzP|RP|Zo*K7UuLdEs~?B8Xa<p>vgy>Gb!-f<jUr<j=6?C&beMQn(TxQ%3W3cY3HNG#P3TP-UrHy%YR#fIXxXQ(sharblP4u5SPnw4xdjITX%DlliQeGC}Y
oi)=NVUx@+u<nUfFyjDo%XSfTF06q1$4gYqBR`~-sQdm^BjuTpfwPgw*0a3O@Yt0D^yhw(b;3Q32ZP6S?sCW7d#wo-fEiB6&p|B!ENFAnC}B^=C;krPc)*eG
bN-}=6VzEyj`!)^+hZvzA4`jO>-`}U1Jd(;YO?6Qis##@&RBVnqApW2Y%Kj{mhK?k6}4iC&y0r1nCYqa$Fb)#QSB4U0OlS;xwnCaB&(tgvLdp@*RqJ7Taa=-
{nBKN)753t9XdRmnsif6YbHM{3fZTsFR*i&mj6no&U>;ChG@wDe8Jg|D42wx%ZFOEq0J8|QV>)=G=lJM86dC&F8GtP&e{Y==cmRlJS~g(f-LS6tYA$>lKAb0
yhFOGc(QKmMPo{(6Op^f_$1GuZ}ntE?oMYjHLS>ItGx+p*4HV9XgTkzQ4Z}4N5~FDptqDBN09X@G(9fe^z#5@qDA}KhjjSsX%!GeFYyoCIjKmxcju<O!PGOg
MDJp!Fjl<r`Zmoc@<>_r26|M8Wb1zF?ZjbWDw5av?Il0)!3c<3^GR^zVR}^}xz3|h_Rpi&w^@2qfK#aKc=XyroZ)a)LhJoKHc;+Z#!{VIMu*QhnVSgoIr;*<
4CAr})B0yt7=T*%duk*5$LG>NVE7TSk^f{~AfWb<gA6i|x_<Iw+g=8fRkzyis(^|@Wcd3zDe^S!P8bMxah}Bckm8l8>k2j5CIRM~9a@fk98xi4WndkC)<&Dq
KyoqbY#HJPr+VTIVaVTKA1v8Gy8-tPp^gMNB#@+D)$T@L^n;+D6!i5{o@G78^-E7fOO&wA-PZg5*c1tOOwf9GHB0I>mE!-5`w4WZ^l5_fB1sr3MU7PY+gqy<
l@-=FBkErsE+sodvR=Pye~(Ah3bk{9pV5J!G?5Wow`0=^d|J-D^==6#YgkD8uPMXZD<+d0sw6AKmLtgS4En1KH1#DVn060qbwC><sl6wj%U0>|TiZdg-AYqO
*zL#u<<rK<?-`F5aSm-iAYf4~`|Uz6SHqS<kKsSWW@d+!R&y~g9M%q5;$I$OTE)k15$A{7bkxJzBFXBNI#V(;c7wiVY=;B}qc7_Mp!Q*vbYtti+!JBi#u+{;
BO^?iK_e4mOAhkh&O&_PLoSwx0`XQxB1Y}yHZFU#H;d!G4Q+7XX|aK<v4nuscf5Oy`-E5Lxp{rR5fi>snL{zRF$9GV&`!Ty4lW>P8Q9Mm?VR|-k>KP!L5CIR
dcs~#F;p>RIp*ZnP^#f<6FMs%Ehe&Xn6U2F8rjINP>SsIFZLqMV&%7=j+p^Er4XGuV_4MtmMjOITTA!$*xW}pIOZU7CkUu2UfIO?=i#~U^NNhJ|A3J@_JR1E
S8(M3`}JzsADVQb=`CMOcywQ-7+p$eEfwMxz^Jxu2fQvx=|I}C*(yomWsxerVNh9U)NLBk><yMt{*i?Z0V_wNZjWzSf1`>=A)x6-xs<Yp(dTWdS_zSNq73JJ
3T?ohT1mD+L-R~r8t5!nogw4lg-iJ6l@ex>1!KD+9q16rqDl2DU+HA$+NXtF*)m@V-GRnaocxjVdI}ejW(0JD_w83+X+EQ6Ri4SPMS>Q1g<WFCW&@v*efS7y
g1<L*-8i358je6zzm^ku(g5`~k!2sYa!`k=3!71>21`NB1QlR#!_*AXOe~UvoHbUwNXDS=@PWr}U_E#_M#Ny<1dcH)RE_kY)Y5jY9}Zq;QrCeV5|faIJb0Jk
!a{&-yBMz{1Y=X`8l)Zt#yDU6P7o7WOhp}eU@}J#o-VS;ovi*4gyu$uQ6zC)qn*L!(e(9McRNs-gILi33!|Jtc6MmsDHy-Ek5i9lVqQKB9;&X*nRY3^{%0I}
3J2NYppJow=^rsTFG}@|Au49q_kBp=2uhD3s=0yoC{GPxG`BiDqn8vz=yciMJhAT=M)dG&Nk+q9;1BCzA0(G^ydsECf#2jpDj_R^qPTBcFnAZjA|K{|y=o!%
GusL)hDV~wsHOPeL=pR8qe9oC>Eu;xO9KDMroQ3oQ|J=kz}G3&Oq88`jq0wuz`nyqt$V6=lEhsE9v*srO|mvdH=|UELi|NdC4>$6o{WmlHs^@7Bq#HJkB}}&
rD9l7n;n53;NVlG&>H_$I6V6Paw@Pc=^^z%x?D5r7lU5-FIOZQ>tN4fvR{VP*8oY16_C%)F1;QjZ(sF4roDPPw$0>QK5FcQ9-+4*PNF{i$?qsjaVyF+p5*(t
U;wim@YnPbHIqQiHN*dMX)h!8C@PEKfl(h|L<m0^G-ZCQY+KdS5S9$|ErhgflDXP87&Rd&oh(7F($zWB<`A(Y?5=IDIt16UKAQodzY+uj3=wN-LX43FW4<9R
rmxN#eDzP+LH-}vJ`bqB2t)?;efxmm_)$H&_(D{-dribZ7OJFcH@W$ru{XUcdsXS;PPS*?HUR0eww(u@QMqu(RrPMI0+orkcJ;=rs<r}RyYKS>WvF{VbQ+6k
5PD*dv^|<&XQL1v6F(w>Fu*Fz1*b%CVofTeFN_=0Y8<H*!`PCw+L$e`vUcS{pM_A#%LDWL`c!#*X1Wkh+l2N1I@MR2CqQx21LU0HN%7mw?p)QFo_MgkoB?FM
iwjSLi|*^1vT?N!|5(P7@CipSVr$ug$H&MUPj>a|m7sl23Mjt!=bY?b>y2ExglxuV&A$WArcIXTmFvhKlCon_q+VG+!^wfOs3u^#iz69oQ~7p`#MGWGa118(
F%%IM)(ftmD!>Gjr=C1-a&fYT0})u97C^>x4c&;bB9r@!?NBa|7Dol61u~^2)I|131VctJDc7Ft*&evYeDd=q!<@Z1T(eG&b0*t>k|uGs%i%W2f_D6y+*lC2
8XOvBvdx_7ZwsWa`t)U;5%BuC$H|CL7NwENfUi6D_(!x4s~OCEYl|6wLhWWH@DhkjvS_1a46EHBAP<N1Piy{Lkuzvt;ZvTNY&W7EiH$#g{)wtUz6!_~G%Cxx
>YB!e<BpIcA@VC6G543IX6mpP?`P%K1z*8QX0VYkTgn<=eMWEbFQeB!R5Lr<b1!EeOR?N#wNl_Y0D8)ju&60P6$%pG@zn*$Wd~>Y#)CizYr#bj_Ff>aGiq+F
p8JYpq=Glym(&r%C?f0Os~=}~iZmBLbe;txZ90R_zjZ9Z>K@+W4(*e8q<0xHT!Qlg)hsc_qlbG+I(#jgZ4-}{N%3h(e$oOu&rNl!DQ4AmG4r@$u?$j1+c@)8
0JTFA)*S=Ph3+=ZB)Z9F@yI~CuGR&&Kb+Ea!M1Cz1eF>fr0ZggS_^0$R;O>zsEW02V*wyU1i?}h<lQOEPCWweR{jTW!PYV;8WJ$J&6uslb<xiW%D3t_IY@^9
(Eb-9RBz{zk|*M`7O$Dd+MknpCF5eZRF_JnUq@<)#{;o`{{iriZQg{++t!tk3D7*vnvoyXq_MeJZO7NuYd=dfac#GNU$?vP!u;hesheL9<g<<%Ti9WXtnrpi
j3(gr(u+FWW;fy~d^f6_nYHB*nbZUP%oLaW@So?K&P)E?7&irBzPS?g45#3c+yPd3yc>e@ZS)5}@`%rhwB6&!GYpyNJJqjzP?vfX)K1CT1Na<@zSQK_8FOvo
(?SG(+jr@*uU?2IJ18+#KV?Uq)I8aCk{?6oNgH~}JLZ6E0LoYOYZIkYc%SbvhpVXr2OfiDky4rFDML)lbx)KCP%*cdl7?LyJD$7rSJBD~GdVEjX~*k;Vr*?t
D$6WzyE}%h$+Wn*_RiqJu$|PSAD;9l-o+kcu9(J^Wp@YQBCc`b>*bBd96WxB`A&X+UEl^^Gq(Stk~L>3Hc)$BC5X-ps4k|t;J_)Hjt!e!FcBgmFX%^q;23*+
E63Tms%N7QUpFuYAhwSEHy#uygC(U&cVfFTmB|V{m(>y$W1%gF_eB6lBTl9RJ%*A21J8}HoT@WF0NtQ+CsX-Ul5awGqj)GFySY2GO7Xqb;o8Yxi_Zwkq7$Di
{s-+Saq?hTVfw6;s+UMTD37(BHuCQO;V40Rob}{9<Q^N-gLXwY5}6Xv;#ZA+|0cbVXl)tJ{EVo;5K&{S4rf9le!Dk6MzsLAh#N$XoyYlAY>xp)nY>Qv1pix*
D7XrmN~I|XP#a}nLTDQzQs3AgRu3_B0%{`xvC7$DeKyBzq1OE*W>H^L<@Q+^LUFXyMX2plUVFP>D)UhR(p)`GdKwzY(ViJzptO%I;ja;qp1tb=iM`c^3V#1o
wAs;`Tfe_6VTzgT^u&!m=_wFn`!c%%gS|Sjb@DZYG|RP)(AEMG6SYv~VaGkmU!0WH#+^(ol8{Xt%C()yf@UA>#Z@t*sbZHrB;_=xC<)xgH{1RmeoWKw>`w7d
`PEY;*~+cz%@mJ2R4#+4Tbvn9_7&mIN1>}dHWc{hb8XA>c~6-DgEFA8K~~sR-uky0Botl#ckA^pJCOJJo74%C3evlK*JuUmpe~+ZN>t)qluE3Sf3k36=SlXG
B(8#)ID2DADeHqO+Q^6bHuvkKr}79ZtCx%>tf6Ppn1=lpp~BoI(-C*t8=3PgFX`<L1c}4)b6~Dq!b1|DUZpI$gZQs9UJXh$DkM_*1$JPa`vt9MbR^nO>`u?X
ZTYz=eo;X0;fw$-gB?g@6*fL7==3<bQzB_u=+otvWSUeHQJ-Duor{cA5Sa$^`)5Ty&<U&*x1jlDr}F7q1pv1NPeZ&`F$&W7+2lo$@Iq0k8!3Gv1{?Q)u-MrR
%KS!uLtS@TD8~~SUT|}V%}hbZcDTIfsXxcr!4n;#FO`S7#9amCeYb(6^tu2Zhm{H$Z5k{juSs#Xgm_w5Uu3VqC4Dl=)t;b38QJX#^Z<?-?|aBkd~R@BEI?S~
e>i$!o3?C9sblRTPCDHZdxs>|s(L?mZJ4L*T6@YFRJb?zIR-sAV}m0K$DkSd4e5B^91xu&UqFS-5nGy^=$9UyEg-OF*KPUElsv}o5|zrB2^xhvdbh6Qz@N&M
X~bb9CYrX%zAa*@15;1Fg*I!sj}JTV!JYKmuHxUgcT{4Xp9|gUcSqkgr#bxIdT2_W7xhUbJXR+0a{ZE3Y(?keaS;()REc=y=#|DB3pjj<o{XCrJ2xT^=(jjG
2Y?@;g0B`tj+FO3peAb-1x@qG?DdmIVMsG7G}8#TW)`sEX_%q$)VfKqdXYFi3!mEDd(3N^L)ToPf8fZ5XXZ|IXEbPfVF#-vfU>tmpHzk<t~5(&(z~`q|3&Ap
p?;&VpI2ECGW)EfvvOD6K6lBh52-ePDh4ugjs@4l@Tp*-Yh@XC1W;rT#S`Q%^HEHYsyUW-YhZ?^L=vqSid&fWI|b-Opm-PDE&{TV;WX3ot}F$X?^$ywop{kA
EdR*ZUWig0$3NGh+@|J;B1qcOZ3m<9pwnSt%*~Y;smRfmt%0IRToB|ru)B`_<|LSSd*I)V3-xz}n8{L<>c2uKV9bA|rViS6!{qCFB>a8+(1`zbkeo-No%>gD
c~W2M=wj40c2NcLV+Yez^e_e8AfDerI9$NST7BKh4`-=&#y$@JdfgVcVTJZD4rBu|(-3~ZF5K2BEQ%T}#4(ub;Z|cnm4oQy>A$fXo1nWel(<wS;XkTgaa(JV
9RDp!w@(g228VmiVdG_=IPCNdr-f}W84VdgwBH%)w<g1m`iRQ_5kf8F#tUONqK|S!Gs!9-EUS)5wzs`M*oP~Q`U&YXJ?cp+_hTLP|3_INmV>qQzhszMrnu5Q
eH>r$$QF6_UpAYQkS?-UxU^#mHeuB%@s9>~GhOem%{&OCzHOJhzkLdW-`r!I738|<Gs>;tR=D@R_UW@jjwy42ciOXSh%I$bODDqf0FZJ}S8{KvP^$VjVRmTM
^SJ4B^-JvUGaQTz&i^GCgjU{J>6;h*i`|IWoxL`vcDQ2S03k<U2d9Zi9^dq6`erARGuiADyf{~s9}0Jrl1XnVZGYdUkv`K|o8fjAdepAluu{(<_Vjc&qSAM9
NO~^6j%aYw>F3CU{^zlPnn|fncy0F^`OKkI*6@Z2uAnP|1RDrTGw5`+!?eq85WxpyQHVFUZZv(Z+*W*s4RlZ8PI(f7TDIL4O{|Nbv_8EaVuz7yLH4FW4FbZ_
-=QFo7I_^TqvwlAIFm;#;hb*(R?{p(501L#Xai2wp7hstgq&7MJvc`pWrDL<@~>kDl%`0fwYZnLY`W<}ziaTrVtiAaj&qtl6hW>k%yRB&RV)>Wb2MjPA&xis
1|VqDF%nnlW-wT>4K`lT4n%y^C97TwYqu{Hxy;Rm1a8$GFmTpFWK{R}D&x9A$iL-kwyB|}G<E(^E3W6QYj|)rI0;0U#kXjQrR|L)an|({E&7}Q5x*HKwm9Do
GYv<{3z#Yd8Jx%QwD+I8s5hdNuV;L?Fxe|i@3L^W2fwCokH~pkeF@=a*^ITQd0+(gL5#+lE6QWalXr>YLyTU^V-$63_1ta;Y<0h`=079J*CmCd67oZ9)>Ky5
S5y%$#w3(})}`Dfew;->P_1+Foe}Q3_I09FR-9}a^?rA9_L5<c2X)zOUWRs~xaBBWo*S}_>6#PZ#7g%7G;}X91A3fncH*ipiSbc7BA7TmL{6H&Ff*Xzz?9D%
3l^7v53m;4=D!^G_?eod1Kdz7y2i+##uGA(c*<g4Gq=7TY*1X&sRK&hp#lr_KbKvj$_9PE3=<$4iIUHqk>yyltUDGE8!glSPDvDhg)olqd_K-nEq#=OhR7t+
6id}a_e#C%&_a9f*z+l!gCdySc%@PlZy(!*DSZ?Yo2gZ5`uK<_+!|kPML_ehWJsiT?+4c`N3IS-pdd`z9-kBaOQ8gcb=R53RS7hnN?TRjKxk5b2oPdC&+`6G
(^oV!1%A@h<(5x<qHOFKSA$Ft{(FN>ftV4G|6Q-f7W(bh#E6a^c(%Rds9~;~bVecCgd3le@wc-2(@qPmzC1j)_hc6Wh3^iCxt<2k$`^nq(zev>zme8WYhRCv
4@?jq7y61NWv;t>l1DVHv=~%2Fp=cKDOFxF%$_6~3#1@an}M%m0Kd2kvoql1&SID_<aQG?PWUjUGuz>m@K;b~$VqF#KUbha4^%;7&c~&qGDel$vf{|p-b)7B
5O=(hOVWN+pXR8|+BcC_CtS)EL$(6jEG*aXLGdU!G+uMb;>k^y!}W{yTGFa`#=c{#e8L=vh3RszD8q@~Jj;Ug#iTgm_%4F0-h^`*iw1i#6Q+KvbjT9lAl<30
@1d|0KO5j&^CCpMN=iL<7VT09AGg5-d{L}N<CO_TSSw@(gckXb#w=v1u5h~71$ju34)f$upthQ(G8EL~M+eyZyqIk~f;ZXcHi`h}T4r(LW<{JkYP#iJiQfLj
VgdA#zLcCPkMBv}5f3}?h&CZwS;Q;erT6~tel8$qLO8<URe$V-sTA%nKKBG)fm>*I@?5sI@S0m{!0kZ`PGnEF$-=4Q;4PdI{_)xMNtN(8;WCWy7-a!(R2SBJ
d5q4*4)d#!6O5yG@;9ZnUt`onu3OXD4^I9&j{*fweI>NsCZ9Hv|N7bJS!Xd{S{pTf5jRT@M>46&Uc*J@tA{U4LtjnV14JF`;(?}gis!Y5I+@ol)M;aFckC0h
Vim}F*%T@j)?8ShCOLzdi*QG+Zk&6aZYVWoM^aKXB)*j-Gc6fK#qGR0nYoO5jh#5Xc_1%S8PQb9ih-lJ%$oLRzYoORI#Su7W0H+uZK&b>7o;j+k@`I?zJsjp
W4^TP|J$Hp+bM^{jT8DGZL8)lNJQvtd~`T>Zj&Po?300!EL|ql&?0v8!zKO7%@sFfWrvUX0Xch3Pdf4}@Su-Y+t;)yB#lC!=SfXe9kk3*4^q7x-7Y1Z{bG~A
7>zD=y@Z;~S97u~H4$R5GaF#F^dD;JXq>Visr_G`hxLw4jzGX9hoU!Vos*_fWCt|+8_i;CyqCd{4HHnB-?kN5K3K&}#*3L8baVg9gRto8jf26vGX&T2f@@VI
p-ua4it2961hCT5+2Tcl1mt`T<r5hKY7+a5@)dy)vK>W~o!Ljn^}~-){zV<enVn|-7x6AShOW+YaIAoAAb{PD=7qozY-mfrfn-`ZXtwS<G(LWE?d1nBr(j^W
J+FD}zH-w2P^mjv3uy6{5CQmQ)1Btq%X3nlL9WB6<2aO-pzAYOU*>;hF7Bc}HtGnDt~QfWMAk(<zk@0<ztYOIAUzQ@x^ranDdcX$>NC^0RU%u9u6KUgHT8Wf
WqFB%K9;OTDeYntXs}D01@KOqNPBXWd%CCe%!MN7&m<}BCdRqc$wK6U&*#75jD1$TAEw4zW58~q6`pFRXBr0&&ifNd?qi`}g<I1XTDz~}noE=#*l9*1!!PQm
JkK$`-it#npV3N^Jib)9&HvcC$NoJU-5w1J@hD<nRB45Mf$Ua7#^-NjsXN8_pN`eKKqJf0FCs>z9SHPWVp5Len@%OylvFt+OdX@}wVZBZlZZBi1?Nd^YOiTr
f*1U|1@!o6J%MZ9WGep(w`9~5Fs7bN)qKB_ls&hFtnJiN9mbGYfF6w}pP**~l2Y~rKz|Fa#}KNw4Zh8ax4=k3kq%f{4rNnHF#GZT$bTAn+nb+?T0CM#a{b_x
ZbD!sps=5~P-P(Xz!d5=CcD?F4yfu>nC<bod_*W`@|>Ut%vnbbj*pW`zSY+4q5J<y9#)&?qW&4FeD-nt7zPyQ%GgLeR@?Ase3&C(iLh8YR|T7$F`p6C6AR32
>7DV6-UcZnj-v9p99efGD!wZAeAObvKO6%mt*PJE!co6WW=4yW$)w&wH=1Ka_5SpL9{@iqOT1~zahMBt(jNLhEwlGrVn59Da;gBKba@8kst!gBC#tbXUZwE0
n%+q+TRosz{kunXJzm*C%^;dTt>`_&OEGhl22=Msdc^-txp%*o=-sAZVxaAu;8eqhdU)lT{M$P7DUop8a@`)VOjE+F_4vB=-e-G~V(ero#7lXV%03vd&dECR
J1@Ly)Nth}mxHlmX7Gas2*F~DVC4nCpvhSB2yoKqiAze$LLr{0u<`rY+!5bf9*yn`dvn{ZpWT#w3QE_nkzW#i{vHXO;i`;6_AVhR@iOtU#8{lnQ<++U&uFGG
eMEJ4I9Az?O!cY;#2Ncp3b=IZovuM^@kZu%)<hWTjr_W?*(#z-FDob!3AqF8;w(-iK#pcF7Fm^nnV2^Xo`m|lhGvI3x_o4uG=;4BYAtVN)#Yn^C2*Y>u#^Qz
%o{ER27f7~u*os`FD(%)admi~MemR!@`gS49m{D&S^1bOFF(|_3O6`^zDPGAwOfLv43cS9Carm;z<OzlPXciKZqk(Z{J?x_HoyWSvI<sB2U0|XF+~5OHies}
pL=oPdLmta1}rnA4t%Epj(>q$4t+XV$k_;)r3wh`GG&!S7LNrchr*yqsGpWhW;tm-NLMmPi32P5R;!XMG$8ln$_+)OX@YTl+P06lAlXK3<p-jOV&zVuuRK6p
l*zC)*?1^9pi<ZWhXVb{=jMn0%nCg6)TGmZ_%W=ilL28o?e+9v%^9IF%@<XaGU<=^jl;zVWc<C7m-aZGY=AP|XirOJXe*SEra9Y8bEdSYgt;6i7Mp7LSiLBX
ZA$Qc3Bod&z3)`c?)fRfs1}l8=&(<6n~pC!=*rN|DJ=7bRi6>1aRwEKH;&VA+Ytb(td7`+-B}#MOGENCrV8R(<7ji2T4wlt7zn$sxm5Qd?g3G+%f)KMj{~~n
dVto9ejpzMqQ^hFx`m^U!(Mjxh(Cj_9}qVR<Se42J|dra5LcJ6M)$zYd<{W`Gj~s!B)#+Y3du6DZr~B(`Rag$c6oiUVUXas;g9u$I2fnyt2=^o)6%iAav;W3
SCZK9soe$#O)9E3Zsj^xGLE+<kDETi;u*pCAbLF(c1{6TB_<vIt1(VrS$S6dgS*2o=%+2t@%htxCwaGon=rPicJTh!fUkS?umuXnS*Ye>D+RBt%NcEfv742G
_moUd9<GmG8ik1iY5vJ9p1_c1qxu!VS=XYPT>W(`YD`?4nW7s=86=n`v(N?mJj+pkaULCK2SMjQyEJe;aBN~o9JmABw7KcKIn$O-+~*d=ttN0iaX`Wqd$Q+v
ym`J@8Hq?8nf&RAU$=__^L!x_QvfOCwD+V7{VBbW{7s}cb#v8XALr6bbjqkz9lAUYeG9PCd0MV8wwNvyz!#KW#MGNo%Iv}<?QXI`K_u-uxI0g_1eBUIH^fk2
<FBM{&)uHdEGuTp8xpa_%@F(nl6q|_gv@Tel;beo*6%5Xzd8Z;b@sS&;D>r1w$w3tn(?91e(0c_9vAKF|E|TzyoU36gC3|;u&~BG!jK<=yXXR>?!-sbcvwGI
Sf1mII(k61AgQ%Bf(ATP9?`)H(f7FOvwbl+m70xk13Ow<od3K>#`DI{VIO2AiV^iI2r4GIXD~02irrTSfvzM`8VCEpV+H`j&%=;M0aVEpRAfEA*poM?=)p7?
0#3=5bquFQ%#DZXk(n$8G}-CwuW>$x4kU*E*sh|%aIK;}Ji3WG=6ep~q+c>e-0$h#Bgm$zvr@-0B`ph>{yd%{0uARRfv{Hr+0o(ALt|;pY0H^n5g(coy^5iD
?Yy+|VdRcXqIyfl+rfZF)X8U?N|Xo>gqDy1F}*Nq40thPOg6_BW)=G)N+@_PQ2JdsOT*<6%xmz>F-m*#V6<*d{aCs|vsnx1i+)^mF+!}s2^_h~(F~Pzbs|Fm
$0VA2hzUdz#^3pb8f*l*hg8BLrWWEdhk}j$>L||Jy2z`lT?*XY?DQ%H5kEqQ=)2Puomtn|xTAM9u-w7WUnagm*dpL|EMDTJJ%Adk!<u~@WUMcq_=nA5D#`MI
7fqA95Ekx|VAkOpRO)GTy@`0o=gO^>S@&r7X@NsFZK@-kY)FZE2+u1ZSpSVMw4-UbkjU;dp-HauP72H9$IyTzE7(=eZD)E+4SI&Rjm)>1hRKWmCRuE08JyRO
z^J=Irh)%-%Ru#sKgNlf&(jdGWeLP_%xe$QRNz)WfywkdhsScmI@R6hhpS>XJt9TO^6MW{X>br!mRy!v#KxM~fFc>tGoN+harY$0456*M={o5BWw*bwuB^?5
1cMGr$`{U*+^FL>GLBU~!<!QI1oH4Qc&+)b*G~WZ*w3%uUnO`IdqV(VxZA|o6;zy6GX7V!F1W}=i*xfmUEC<IAY@%V=&{&ayFU#dh@ZLYy*`Jt!j9Y?owrmu
@+2J?xD@51jqOu#zZSB^naif|c>k2ulGeX0J_4?^dF(0}d@QEhPi^stl(~O266)co7QQhwb-K&2lJ*8qH`vG{M4h)8*)r5<6$8LmU#d0_iw$DSk_nGPRGgmT
Q~+5(cb|DG7L>QV)m4Jf51=PABSoJG(M8jdtkK3ZY8EYM`&s<3`KV}x<`TOC6?-?wP0|1YI~xM%-NFr}!Lmz$Gd-Aes$-)J_VWICBUqk2dzjJ1?5R?NIFLDw
#4&$Q0%>qFe?-2SlBeVEiZw6H>;tH?M8!tZus;Kvy*aLSIio^VOQO`oaxW0O!@pOHq1Tv~Oo{ORgqCWCUGDplMezT_j!g1fIfS0R=ojiwb%VmY^XGt*qA!A^
j=u?TR24|0W4nX!i@=AM!6eSjHuiUD7`iMWsaq8n@}hq?ZZk>Iq&GR*Ne8iaMF?6<cEQ2GS<~jOqW0vW5}3kz7U1R}FW-|G$%gM6+|zIoXoTw=_dHI5|4;}z
2%k}qOmwUj!WK25ePveFy)j)FR!G6+R_+9<7tC{6xIeiv>Js;-hi%V?Xl$ZczD89~5RxUaP%I3;3feaGN!HvH0J!F9`BXsuql$Hpx%=|<OCOAW0|C*nsoK1@
db7%;l0lwd(s?)DWmZ5LID?%6*<-3{Oda+D=ji@sfSJlFScLjNL^feaLSwK~Zj}y1?R*gk;Vd3@3<dz@^yf()oCuHEfXIBz4mgTU4d!GXJQMHK=0o13WW#)o
EDvIb82xM;IS0P<;S`Ag5E8_2Rpoyt4>nsMx-DN?^Y9fC#7XHpXVtWdqbz2@QiUw~!GE#5RF`#L>6W%*vv&G()L)3DW(Xt5Sk0FM9N4>6g;L3}>Z&3l1<>+>
5oF<wJNc4jL5ojniM#P%JIrbAK2}m!uxuWOGL51Mn+x)RlBm3J#VH(l4*lEuBiCH%QVChV5=dOwi8ym#5XMms@#`BP+e}i!oTygU<W}1FvzmfeFPmJ0z6={6
hKe*ebWNPu)xSE%u^WJNkfBeWdngtM2hWSh`GR~vZ6HpoFVp!1V5bOoA8aV_#Ztp|&ta?*DTqAoCTJuMQmZ0?lKvrGSs`~@deGZ+SIUBy>1t%LLskh`X=7+2
!;QJ;!CUV<9Z@1(5%eD><U*#3rDn8?q^hgPqr<Hi;>$Pr(5&Q|?#@YB8|XrF>IG}B58hwWLQEAV)DY~zLBZS=Ey_MGtTy#pVmfNNvfvBd@D=TlSMM+-xtf{&
@s!Bp9M2BEq`!o8EX=lDw}n^hfRv&ZIwbM16qqD1MckP81j!+v4=i<MYqp-=lHIWx3OCX>6XQ-Fp>&jTo;?dnIpqJ`R21@Y^z9iC1uCavn5$F*0E&Ya(U_#-
yg;9_*WIO~)=pmZQCV-|bbU1s;0EGp#pm{^xq)I^;#`T8arq@)<by6T@5ftzYM3Gi3b}GJAVzK<<M)z)M<97{J{%+*rpR&0Q3lRJfD!Az$+95r;Ka8GWgXZQ
qLhz{SG(YqGBC#JJJW86I&`HD8y+g4QB6UYnr$d1v&usJUt8UAG$grkxoqSd9C%LZP}Kfb*a-DwVSYfaRy)Bq)>izHFh%EPaNT6yvGgnv;V1Kr>(eXOwowGr
2fzolBGn{%HkIP~+G&XnC(G@}*(0YXqMSFHPuiVz5NE$VN*}3Z5+L=?8dA%^x`nh$xl;FHb)QDb(G!0rO}ycnn^6^qM7zO^NX5|C5fyvLu<P@Hn}HpYMN!jo
p?)E%Y@Ed!dDB}Q5rBNY!;-^ELI*nAip$YNw!A~}RQmMt#pSSrzlDrWAK2bUlP2O$00DBeT#P-^C-$l-xM)>3VE!U>J%h+IZURR#z|FpKK4he%!(4Q5fr%Iw
8=pR(4X{*N1Q~1&Q^k7gy%P1;UCPosAp8aI&A-W7d-vaGV77~QwUUBCTl_b+ZM@&G2JdXVKdXr$-uI_Wt?-nf+g{j0wLbNG-Fm;(Yf{4IUC3!o>+Z!YBwP<`
rk#?i?0oZ_3Yx3tM;r|_mK@}J?w`(G0N{zoS{u+&p88p5p<m504ggUTk#v)>|MukQR7nR7Tq&4<_bzlBi8N)qEYSs%u6&8ZLT?5qJm;5av01j9{m=yFp2L&<
_D7`7^4%l+GsW^SaihZ~F|``wxDzV^=N{7^w)ezF{ET^C9Q`3DvXh01iXeiG()_A6!fE}qWket?&WSpduge`CnDZGg&3hojH|-ICgc`jvgI!7@#Tb*L@yYMc
FwC+m4%xPpI+`byS5wiCCC`Z7uxugWWT=NVw6bgFydIa`&oA7ZczuEKaR;|tWd<0Q-Dlr+QkmkqlUVAfsfhCtqZ1%tCp<+7nK@u+<_DB5PVVBufXNZtDMTHJ
bKgf-$hJ{gl~cz7buB0+6ak4#bx3JGl9(6NE2`XZ)Br?uXP9&{A(wXP<VU&LN$hO{C9Cu8<i(Q$MuaFYgJ@FZw4UR{X@}b#RfgnQ_1?Q9bop#qCBcF2ZQX9U
7I#)yVrCxczODt4-dO$TCJg!%YIURsXZ&!$ee#hJ@zRHWrFX_GJ|H#U`nathXLPP@kl|xKAg`bSV{JGMvxjT7NaeJKhO5LAaq*`9)7D@*l8<&W{cPqRt6bd4
Xw|)B;De&Yva1<KKex>E>IRe=;hL};jF^CIiFyVC2Op$&=a)k`fNd?(2V1y~y35&?o8U}@jZLb+%Km!w56cS-12s|x<N0P3n3K4&r$syeUO}ZdK8oil0r=O9
_C|EzE&%bwt_d=e2Stw=niG}=a>E`W7^6|v)JFpGHIG;lOj)utawa$$V<j7Uc-$YA^x#9;`|2)xxn-=aZLf>VO=J64wMbvpWnrsr63#d9U9)))c*W&Yv13?9
E&XMm{icmyVTCSy?aqh=YW!?~>;9ix7_C28f+g2I9A&HuN|>9|y=VcnyId-s<aA;=2p}WG6YOmSGLJ7N^X<Pfqs`rnOWYtx%sf?xb!1cGWUN=X9>Qq>Ncu)#
(=f#$EERF1p%t${?9i!%YtA>+sN{^8*S%Tm)p|E*;HyBy_r$l|X(o})bak00dF-3EG7foN?9i_2x-8k*c@FHanz`rl$AfK(vaTCKD&-h9i=5ifkXpk~FCONc
{|Hrl@Q{_arOpHaGO<;K`+&Leo_is*0FQEVO3EBLNehh6MRG*B-ap>sgH&MwNlRfce~Vn+?}06pC0Zt#2YNq^r}U~symRj_w|1PJrexuTS(cMqaY+=Foc@x@
KJaG+0f})iGjhFj5zFkARO?rD#+OiRBDm{TS4C>=C8ZYP9OdMR0v~`fY8U=K1{WGzz`2~PD}ud*T`&*>ROmwaLI)^L#$H4H6Of~{P&QKDqL7wqEX|Fh1cbB0
b!NEr8VLWBF+XwbhzS0Fhj&$BjB3df7+WF+3%DGOW7kSP$<IAhbE3ntb!~;4k?J!I2vEDES&upMR>5xfr-Bc9yJ%@MptJcE1|0BLGHRXR*ncVJ^e*di?YkHl
Vq=)(xO(>yj{h{b)${D{P?GqIqmaV;M}iIZa%_LSI4qR#<xrdt5*DBULCI?KwekF!GmNMmS6bp^RHbs`0!I}%9X-Kw2>*yZobCm&cuHI+%jELbyGy}}W|f!1
W1%668PX=4c~(eNG+BGz;}AZ0OS;IDB2fQ-D@LMlZ$jXn+xGSVdgQgDH}47dSm40nXpOVCqNg^)42#uU3b^j8Iv$qaY^dtEW4~}^CZ=9T_gOK%>fj0l!WEa(
_hf}JLcz$Tt5rgzvc^<donc}~3>30M;6Ex&3Y@T7{@wdx&#9-z(n^&hJD^LrZakI`ntVgoRu$T5k!N%$UpXFU1zI=GpvgL>*L_lPY2kh+`?doqojxPto&;Ug
$a6gp_9yX2s2rcdubI{Tzr_^d{Z)f~mc`udKWV;lovMmK2R;$cq4Roh910l}doaAP7Mc$lu2v8J&#Wa#mSkis5GVUt^s+}M@OY97RYOZ6uk9*??+#waa;=sh
4y+ja49Lf2p%C%ad58lq)|Hc?J|3ETGC656j|s32RafXeR@ABSm#*DUQcTfxUn;Ex7I`;lPX5OTRrWqQm&9r?PoKDeV4tn-1mlum$usl<O(1=PRdn_Z-Cj49
<+TK%4lx8N60VyEcL68EVc?jE{d=21>6Q!bSO!eFQ%F@eQ9dJr+*_6E;BsRZcww=+`R&)~sNW(elW{DL`;2}B9Pmd|I)9MCeHo7M1iP7f`$C_Tk)>g3Xv9_!
LBIk=E$l1o6OBTlycR)0p34YorEvVnz*<8~1tQGZuUS?}V-An8C<6c)&eM2cDKPC);bKbPnxp{TJus<iIKhLQ4LZi?Vv*X%(k?``E<*vRK=MNSbdI9+zDe+r
{?u-k9q8w5bs!}W+u!a0^mG;l3y-U#xS%hEprL}2;ta<$`~PseOJjjfl+s`wZ)=V(L2(5Qrqg=%P25+V1q)+&jwCG(K=}oCXu!(@n^HN!MSqH50tF>?KP-Nc
nRnhK1U@ikOB79thGjcKoBzW6@xnzJ<#TMUm90qN{N6A!=Kc9GOLZorxpEt&$k*|B&w2^Vrj<EBL1jGmcC~RAI*xu{I7k+|yy}+DZ~{2pAp+CU)Un#PzzO3(
Dt+<J5TS*@i!Arp>ixDywKK43>n@&0dAZOV<f@0A=0kQ=OTsjOuONF{k(n95*Frm+cwPY{@&9oDcuI$2u0a$C29v)2SC(~gcurY@{UnR2wv@FV7VJWk2bWMm
#53f$fatzed8$)@wmgJB0Qb}}J7&fH08ZvHCGBy|kIw-_lmdpMGi`5mlVBaapNP?k$qxaZyoumSdnLL?<{NzC$2*FMOa-CY(IuUs1kp1SQl{xz+|$zO<T|*P
u}&q|mFmY#))rHUrlAn!>%rn)8G8-&rQ3K*hW`t1sM4gT=oqn}eESZC4M_vV2cY^%!MIr-07<Bvb_yl($A}dv8qv~5UNAqd*SxA74O^tZ{{N$kx<jS|V2c4V
r_&fo(cx>At_wUC96Lq-+1vGW+CWN=<1}NR@0G#k9RcFImh|_-xOx=DMG4zq>%b(`<z;jM8gT#1vDZti>j_#UH!ME265L%SYnl7pXsR4zsk`Lv3IniqI`?0{
!e~AiRV8xy@q?<hiq{sO+JRQ&mcc=uredI=oydOl2m+~M*6I|5Ux#!r0rAc(Cg&R!hYY);Ngddf;U;T9oYQb=E%0sf*i2vYsKXM_q|>9`XvA2fhLc!Bb9rxB
0fcm3+wP@*KkTW`Xy6bq*Z!Sq>m!{GWch&%71;0rpuozfRZ!@St5Rk;i3jdikUB90`Fze5Hr7s;GzNg_)OpLu$`(G!>>X@Jgg%jMU+y!L+8w%ii++M~NN*%W
>1+l`T}}n*|8B;_Zd&I1C*aV@K>o0A2wl~<bLBVXDea@EnCp;74H!GGnuP2S(@2U3rJmhK&Da!Ck~+a3<u?1OgxiXz1<^`BG#Nr@2@zjLG!WMa1b;>j6?4#5
9lH3M1iu4@Mj_@yZD)&EyLc@-qpdZjCB=N$Oo}y1pco{tEjI$7PmDL3P>oRY?>DWKy5oFRYZP7T*w!}roL)(WAJwoXgIoa?WP)<Yr}_MK`f90o1Z6l_Bs6Pa
n1o8;*`UZ+${P;X1jj~}Ol`mW>)D`@j4ke(Je*d*d#1j^1AxH9g{x0hlnJG@uJ3mSr7&3>8pzjw1LwlGt6ExP5&Ir{I0sCAehO!PCNl7f)b6k0;E_xJBk%!Y
wFuLRZvK;%u}P(W32W-rJF@>5livSXkej~duGHQ$%^r(q@v(5qWOEXCVxub64S0iGyvrbfJxmZOk!8xaCby)U%-h8;jk4}yP1e0Wqf&T#R?ucfRKht^5FIpe
Md1gwW%8B?&(9=0{Y;c-&?fANrK3>_$A+)F<CEZ<_2EQjfd4xZ($6Ce3mpDaVleADdIr9NmO6VHsc#%{+Jn=%qz&bP=oJ=fZy+{&Bv@1pFPn=t{sUh&SlnQg
s-zuM(_yDakt|VeVzHT*a?T0}n0g<YbI93`nh;dvJo>GMNBm$hn3BQz`9cSU84Wzq0vTyeLxV4zNr*IvypVWASk*X$;q$&uE(?u|i<I0s9{XufokJQpdv~-1
?2*Nn&bZ_>U0%f@`1>Oa1=b`s>u}BVnAf#0ScewHf)uTY+6=J520d@lt}NIeM$Q^@$>o5XD97f!h^swXwed>3+(ma65Kng`+@??RuIy%njeF0?4h|=}6NDd*
0$E6Xp#P7sBgS9%WJx%$Li56*ybkeMV;K#C<STqAFglwkiJR&>o!rte@WE_Hxl@6UwO<WXWZuwlHoBWAhplD#OL(%`K*ZuvaLA03={6W-k}?J3m0TO%?db4h
oD<T_So0Of^a*OPPJ%C!ye7ewFA`{I+L%#9;?1HqIkJ^fu9(S#0L2*k23)TIU~DNk<z>AOe%^E6?6#+%Y$EUX6x9!M41%;;7<X(A`P~OP*m69oL&?gDqu#h0
j&u@aD#E&?GM*Z-J-q!ASj6NvWR;<s5@O%oE00e~236@K0GG<jHna=Ezph!$3Ar7Y&GdnnLYdSuf;ILH4Vd`w0bK9yBloM-BxUmIWXYX>i~c9K3Tk6Nb5?x4
s0@?a#AP^^uwdCb27(Kp)c&blts+8)Q;ihCAIn))NrC2YTb?f-)l6ZmjJiB^b6PVoOnVnyVY1ad$>=wFv3%pxA<H`!EzBZ)1HTd>tUu}T&;f{s%!WoBi1;bz
Kd%!lJ^Y)Tq^`~`Z`cw#{i|~-Nv$z;y&gzDvL6WI6%Dzk+ZZlH-X6yWe5^4t+s^p4+ia1G_uO_Jnr!KMtz7CgjQwiYWCnB~JSEIHdo|y<q~vXQ6Z}^avWW)9
KDcI0Le?z$SrIC>w*tT6(DR!GZvbIt&H5T>!V7zyFznib4C#0F#4#8}Wg|!sv~zUA;`Y3&ySg{i^~?aN%<)<|%+8d}OM*#*Exv+3JF*e3Zht5?|Fr2m+f4Qr
^xK~iG<ve+b0s19)Rr?Z2kAMOqgX+>4K%6(T*r+}VZ>l6r_zQ;S9EOyb@y?s6KhV#R)&MWVTMjVdg-5=%Xa5G&pSg4e>d|*;2ZP-+>urP!!bN`mT?TKLmNCQ
1LUv3XX|(?{flY?W2TCx={_9DPvnmZj1qJkyXvr+)Wtolcap?wR^O>m%zFLYEy#H}c6BHanFi>zY81kV)$oR2=&DpKwA*?DONIVx{UL%rz${=N-yob`kcRiL
>tF^W1_D%v)VMYsgx!zCFnF;1jq|B&lv2XE6M;xiw?wQugi5cU1j5Yh%`qe2*<p`UGZ%L{xQ7%xmU+Zf`Ef$R818d8K8%>WbpL%-zjglFaZdwQx94ZQ&>f-U
CXX?49N%*QnaGf$h0=ir7xT_H0uOq_LO60oKYNy=?}kiI7p+b#!@zR}N<XjV5#+$)<U&g|W3R7x>%qRXZXaJ4dY7RoCg~L~!jdu(=ueCk1@eW`mnNN#Tdf&Q
>FeX)^an~ht}K-J1n(5>DTId6S?-2TUIj+4G(Vcl^*Z3~4sG={2^@s=nLLa-Y&?6=AYcnFR9*itck-!bgQ=DpkM4<ZP$9DKn^G2uiRpxLiiyL5h81+lshu~t
?Gut?tYYh!eJB<_MfD%4YWFr(HT_UpIi**5v^?aWr#$vJbFe(+^St7`35|wem>@w<d&H!V8A2ksNKP929q2u!gN<gnW^ECr|G7#F9n+vaf2PBs3m4yIFJNY=
<0tD;e8)0I!y|uO7z*~33K81!3N=IjsxeYkvADR-BbT027{4~aeX!zjfE0kGb}@!0HE#N7f^S~(`%o$DI{!&%+QjU}Mr*2m1<?`_u?|x8U5j82LuiV9hy-8~
>lNJpZm>^X8e_SZ^na#nXy>L~Q#CuxOvkk7Ya0K#7%{JHP8rQybTlgp-m$I5k(^@^D`SzaP_bakm9bskLIjhpO27RL9%(-7NP$E#N$t>=K@1Fwm301y?%L=`
6f)8d6`rPaS!Ak3zXV6=Nr~@Fl^#9(UNYEn-@^eGYEuQL#OUj46Hj5YK!;5CIH(LFH4M4St5T<X_|Vk*PxDg#h8Ah$4+SrT$Gxk!Ty^gLVrU;BuY~K9oe^HO
`XY_C#fixnP`@?~0RbQNwG~v^CV~2js)OxFmcXN}+n9wX3Z}jF{wjXC<yX*Tax;jx2v{I9zC@|XEf+=skHkqHkBA_G$0<0Dj<*3H+8%tAG&XbL8+`?VI17ta
KlxOtlJP&xhi>W1-J!ROQZX_%lfu;EAOj2R=>1RGz21p6C>~xMWS{5{@mBI@4xd~B-TYW+{vnyX`BsXRa+pEAZ80puUN4R;fcJYcOKK#9!eY(Um!v~VA3NO$
!qfmPr3wd*10LC<FSL7KA#RpyGbLWS&WM#SQ+*o4DeU{6TW0`>NMjwH#VxUf9;rKHV*|}wm$eN1ilS2URIAM|1v)rlz&iQj1Y^oeZksbn$hkN#%sJSUQ&_N`
P&6ZH2j&&XZ220E!Vl$gXd5Xv{RE{dO}^SXQ4bhH*88ESi<Y3n{G@#&I*b`Xc<H(lr;~;IB5avu`dS)-ciEEoW?fn%TX1IJ$k)##$8I~o?d?$}*ct7XSgbXk
Xy7i59oLUbykkFZkpkf(9*MZZFJ{!hVy&u<v`IQ()E{CADkk|9r{E%?chZw0Y}f3nS6Y^vd6|5mi>@6}6(+JANV&#|4bKhXO&Pnit}DTFOVk&W%eooBi!UR2
1i63pAQEIm-h;9SgC9kL*4EHS<SutyGqx4CGTibEEggfu<Jyi24jya(r3LmfGKDF-75Rmdb-l8qmV53QFt?<NaEMPt;Z_~+$}kMGqo4f1vuyNm!<Ag1Z%n<V
lFWFn{ZwA^k54aQAg6s&DVjUxIf|FEbNRNh%UWx>SUh5ACJv|8JiXjTlae}*%F`C1pQtC-<Bs8v4Tx_xk8`aqsN4kZ_B+%N11xFN`#b%$7%I{_Ugr&hqkgsP
-oN;QO0$tPRJgG_E|HRPGH#o;BOeW=gvDdkG-crg%S(ve-)6oyd?%~P5UhXsENU&HoqSq}?Jwk)-}K_<n@|D7YDLkfj&ZsS4McGQdraT!E`3JN<_zs^I=?Mj
;!~{}lO1=^m7GDLT;SdzsHhlrPXtm{5VYE}{-nL-!hV3Px*f>0_6~l*!Ky2dj-$Eqw)HP1{T+*cQd9&{1Dms4i-D%S4T8M=%kEXy<hAgigPZxFC>U~u8(>@5
)hJ_88hcX<mcMRMF+Mg3Pyo(+Jry|95Z(Rg@>41XeivmZU+W+|B=T^tt1sX>6hR)c5kO|xwe((&$-&C&Z1_KAbh#K95pvl4#_<DBwbTZ}KLQ_ody?=wodSrp
K&cV$C>a^^_LFL8+(?|{%x{q)Knq<gxV$3wjb*ul&=E7p;Cy<8`D{^;Oq&I2qv!V!QxQrZ%IeqJrM!~8sCXg;1q*7|x{v{RnkB;KXc{G}of><q=r_9H1?lC;
3|I$$naYlU1jG-;iR#Z7$TrqfIo-u$_qQsF4aP6UTv&}?eBFCXgrN|!P3_(Ssthz4p~W~XzISo3vB@P|&t6S`a^shnzA5RmoxumWL<fCtn>?vA5y<2!ZgYUR
4AphO^K~FmJ8&YYlrV-Q-V}}q;l++<(>ta4{1G3pUqj}6e4UiAIROcLPyPo8p7F@=Kqc?kz?m0xBz|#E^1kjsJuZkq%)%BVRpODNtd9ZSsx})b(G1L)6!2Q!
`@OLsk?Qhdi|d)MKE1Qv+i)32cdz-1u5BF#0?ZHSF}o!1U!H6bC1_S0;L6y#!9=<|le^S{H_04NY%3N(?6)lqqDp2lgBg-|?Wrmd-R_DJbxco_%W1zC+WH;6
Y`r=u-W8TC2*dj7=Rb3Q7L0Zl4G03?jKO@eKi!!I7)!9rv4s5MwXq*75gDnyV9lWJeMaze=s76E`d|tvCVv!uXQSmj*a4KhnhV8h3L$BLJkn`eVJJzW6A>N4
=(muJ@5@OtB46G1A2#)XHZeElh$57X#J2d{yN3~Ypxp+`FcYEQyVlpOXw7Y4>zQU)q>r6nRyu--{AZtvEU{tHTW7uFMc@clag~H^l0PqNKhBH2E7d&}KQNEY
t^>^=OE<X*{o$7{x3K+mb8jR%2Q5>_A<e1KNT6kq<zgMP3@S0(Q`yiLev>4}(0@pG8xzO9DQ)^`dd$Y@;#;5A{PCBfpCkos3oD;HkI#sp%6kX$oT{7ZiD0U-
42Q`1sQw-%OaB)J#tYM@Kv^p~qbYC!Zq&(J5nN$x-*6#Qs&8(iu)lh|M`QG7XA~qP^M@f6%!*A)ba$InJ4Qar5-OQ39ryb$F$yR#v7|&Y5Ef!AjiA-C6@_$D
m169mGXsxuac^do<@K9_Ag%OFFzi?f*NHFg4%VWTB{V2D9$a$g4xwd;C30IRvK6%`1R|>Ew86?t^IE$46Ewq$R-hQ4Fs2vV2L8R|Wv|^nE(m#`SjgeY09Q$+
gtY@XkbQ7?L6x;ps`l0A2V`H%6CA?*zF9YMipGDl*oWFR*(-W*OFX0?7m{PRH$$@y1ffv4l-twjit5jlH>b!5#!b}Wzj!XDb{yQ#!;LlJ2{%J(r9L*E8(D;K
Aphgs4ERuCkkK#3FiK?vnhOhfK@#<^-y{$9MUt#l0m+SOaxciuZDBgKQfG(O+D$*tuH2Ml(sPS(tb>dH=GiM=28Kl`R6~t;S}zwzpi<>$wO@u8JAJId33Z!Z
D#w$4J;NdQD|!z-iS9i&TAn*CLeT(+Sg#&lmPdqTJJHtpun^%I$D~m(M2~;3oD)1ot5E5g;LBpniEwW{e@{-tKxyK2twAAr($Bwwnb0n=9>nA|J@$?&ef^+I
l8y}yl)7ASd=DUV8=Y|QnOFsAHgYJ4HB0guF(eEhSb;kHE;!f~hn4|2^;GZvv^VOPSanir|M)d0;zrD<MS5k!^}Ma#2u}?>NLw=pz=k7%F5Vd$`R0$TWF2VM
&`k|qFNsR4E}{IdPi8(Y?sYgBN(jPbZd;t7CAzo6#F(i~p-@#qWm+c6*zq=50%?Ja&l96O7J=K#^h^{jPbHG<t`t<ts9)JD?m8KeU?%wD^W0hFo|eC@y`$$q
#B)jIp4UAQO+&NLF1xpxD4nr>QS;KKCW?_eE<Ui~9Rz)yKL49f?R4m-i)zKqt?#D3!s_J2&I67?>Ugs==-U5%H1ivvy2r%bW$dm~y)0~x4LZv`6i`f}SFrGb
Y}$@<`CAhCd__g})BwEHP^xUmOnUfVwY$C~w@FMBSTYm$NCXvJ_v{0@1>fw15|{vDw((N93=YWQqU6EK$F7;Nm|fc~1t$ZwAAy)L?=G#596M1U6JBJq-8i0^
?t*s&>25kSL;@P3B{R-dMI5w(P-CSsFOV2hNZTc2-W|u&?A8`vWaX-5*szpZq}NP0YA4F~5+K9Q_9ajgnS$&R526xK$1CYj$hH`aw>qC{@2^*CTybUKv9QUm
H>Uk1>0xAD2jC;oZF8Y7KR;FhRpXts`vcS)6B05d&zaoZ`%MAG9^*uIOoXy3R0upEqVH#zy<EC{Yq}O9j#Yj>41Y-o(ish=eZIH@Qqb-!VI0OAkG?a+<i6ml
f1FG#OSNUJyijVm?go@TSSe^i1AGUXV5pSKaM)A__@_G~U04)CcWouV-edSM@`kl-tbzE$3lCLENuVpOr-R9GbWKQzCrnd}Z%MXU!FnO1##@7%VvM<$d>_~N
dI;E`@Jx$%B|l`0C{yc%Kml@}7`KJ9Ip>9c^1b*#GkWzS*H5i;MgA3&DYekIF+2HmnoI}ASm3`)2z@mMM({J}gT9C@57wA(t5rX1r+XwkFYl$Z^!(vsKh;*y
eNJ6)(+yNCHA^Jo^K+s+ue;aJO|WIDnN%4Byh<E~cscIc5bXM^#UzxbK7e1qJ9c0acuKe-he*2jgHB3l7fDk~NKgRcAz$FIzE_c=?{vE947;xVUFV%=ty_^E
(3ywWgR<{?AFfMbp<RF(`LeFjE#sYaGwG)VP(T;alsX%t*a2sSiLk@*d-obPICxDKTYMkv=B6{OAG2~cFV2>{=E&ARZHN)(&;zK^qk|Nv?XP`hZ1qQAe*>)b
erlgaikjl91(IRuMSiJf3k&!gpZe!faIH#$?!;UGCqWzG9pv_oD2{|!?f2KDoI}a=%@m3It}6S4zM4B|&cnqxaW3h*-{N1ZX?_oiTJDc4^p}Sq{kwS;-jIAG
1Y+cl{vywNwa-s<n^;sbe_J|0B`rzB39%T%s}6LI1MlLb_Xj(9=Jp$}WK3CM3UvJ-K`6n2&=ETjzVl%Jp@;JP@|-?sWI|;aU1dgEce5q0k-o4_z+-1EM&#l~
8et5}oELZ~-3e{I&8+huuNXiUz>^f)qfr7PtNs15;t1~ppimN<SQH#4Z6?MsgWw$yNBI$f`EfSrP#L>!->+SbwjBqJ0<*^|6@OwHrc_OMX<q&d_{XSyYO<(%
llhx_7scR@@7g%aw5?6tebGC{UR3b)?tjhj2)84-SV(%Pe6fQ8IrBKw;U~fPSuP}mjB=Umo~@79xHo9sZ$-RO;|I_DzNO`plJvy(wg&0Pv31AbRr$}<f*>x5
)nqt?**9L+X$|Xe_E-*!%+A<ovW!qLrDD+(-`5e@#PyZ`G=YYcUunzztSZo}@02SCyprO%sB72uTA(##2q(}(mi93XfyeD>z!uurq%ageY3UqPNT89t3Sm>j
4x`;Juax+x#Cba}bcadxQlM+tiWulfz`E>6!Lkuq6&1?RCtVHNVKii&Vc$(e0Nha)>a#a|Cm~7|_NkT)DC1LAEP}M1uM8OeiA}<QTuEssIx-mKFPtcTb>Q*v
Bnl7)|BWc`nBa{uAtUY?ujaHY?+_|biu#8B>Hu+88UBY4QLkboi>p%G$2-H7ZE$5eff5X9YE^fBg?1Im?UQJ!HNpcaD)n0Ore0Z$7P4WDRQuT+(z1I>do-A%
*v7f1gv%%piHzNw;6pAom6PReP&+2~;!aI;=A&kH+*qPiygKz(8rlOTgh8IVTYNrU&A=7Vc^z;y9mizUPv~K(Jce|kj->P2x>@*+@yi5SnH&9l7>UsWRCmPx
agaaJICd(Mhv_sp`ctxG$l437FV=<wOWKk{<t$W9M(cWDTC_I5`dnu>N_4t;g5`--Mnk1wXR+#sOX86Hi#=RI1Gx;t#T3OQ(Io03ylYdPEXg)*I6Dz9-RJ_V
e|LhyfmY{Nyr1=^k9Dg|9@2zBMsUTd;1O?%kD=!An;gJ0)gv%h|H>|r<f1@j?lpefBDr>IdXkmELS9u~In<MEJx?F@yTlj2t!~24Q8rp^&XOKuasO3)H4xkK
#toR-o-Cd#&Fy=ZFEo#@l<tc|7;uZDH<HLkj#WIJBQ_pVSIqJibeR~r-3B)fXN?iN58`^2{q&q5Tg)c0tmZ=2W3&+OQdDvM+VSaGM@=2cNmZ&Yss!oueNZ^v
B|^51w;cuL6NLI%2myb`iyB2W;X_%Pb<rMGgniVEhb1xvCe$Y6a70-8vyIB5C;1xY3EVR00pE!uTqNtSH9<h<z;I4+=NTZ0CP(&tIno$4XzS25GFgdv#ZS!2
?{zJ0DTUvJ$MMk{B@0lVo%`iqAM8CI(T*OZj{4Bl^u5xwY%&7}{P#zp6(*w)T(ji=zVHlOT>ZCwb@=PZh1LrQ%1cje`4P@Yd)V)tPeKz?%|uLP_zV_$4CE^v
!kX9|SmT}@ihy}kMoTH~&#L8ec_5zOwu&jAH;t5l)#HtXpLOprtHSG`9p(MNRwV*x)$&OQVrKUnbGo#UT5_De>mOG*!<T%2(RYjQGmy!k^MCKnKi+k<<%W4@
Yu50@7NulF1EI;iU7-LVmaCN9k%(3v{IHpFlH?TMw~urHCv0k?*#Y_BIKCOWI;W0CvL}gtGOKzvRyl9HRs+a6+dLaqD<)8yE{&|qPQCsoI8e%_dDu{e*VU!S
_=YbwN}8+?UI>zCvJxQT_az{A(m9G{I<Raj4UC&MS}m1oea~;zdVZ>)BSECW{@f#Ot9}-PrAv#rn&jRV;zP5wVu9?QLUPo=YZ7lXjs_r&su)m2bNZINC#IU3
FsjqpM>8T(nII01Y8JIS7i^);kHILx^x6#s9wbdU=LsIFV=^cDqQM_KfgmY>>lAqfQnMOMThSYHSRUUT#;#emtBNVkyrEe!ex;lWjE08)PNW=6_<4{v07#@v
E)frCiU7z5PC(+R$|7b2Ad{*YjsuMRIn;@}Zh$tm+9v!-n6nr6a0tyb-2!z9_D34qkYe8I*WcC2Wi?!BlVgzYlauyah|3m*2Rf~bCgwMf#P-p*Es;vsVykH+
!FN#e$9acCQe-1j#iE?#-SEVfW+UwdCiKr+#%(I1?~_{>?kV>I8xuhgp{>ms$~7-97pW$FZp6BQBy<BIhdDop%Jgek?JpRUL>%qU5}g}lAmzakrK@f;gj-eL
<)uIbPbvrO_AgG{v8gu4E~F6Kg=M8Zoqu$zUV%^h-Cl|rSqz|bF8kB^xoGqHfUvB4;PuO=FPXG@hk^(+__$bs%qeEz?kdrLSjXo(EILtoZ06uoC7}DhYDszv
-IF>R4oXWfG^g#zC;S_P4(n!nmRO)JTr5BctfqHCoq*<3C~!-HM5bBdTd2&;r0ovm3e5dlS{_=l!49G}L}FK}&jY@YxSU6PT$DshADPwj1_Ni^C{7}Q5q6G{
XY3_}uxP)CUd^$I>%>kU+=qd@%RRK&FSFK4mxoQ99NlN|6k^V~fhEq>$r%!8g=BZUevUbOm>$y9pLv~dc>92d=d~2_7&RNQFRi)T@#eV<HbRWl=0CZefhUpL
70&|aYruFUmT>!|z|6^<Y@FC6BaSbs>N;1G{#ZVnh67ew!3a;i6IE~$`*4J$a%Z_ZNoJLt#)9%8^a?J<_4-)lfehe#_@zZ~3uN$UBq{6g^c<3Tbi#RmIL53Q
2+I5WKdD29lVDp=V*nA77ifEnX&kJUKIJ+f_Tn6{g0R9nQtW!CbE;v>;B~NFw!?YYbR53BmIG@Op^_@OjAd3~Du<X<mlJHC-wD!dV(=u&ps9SU>dy9@o#m|y
pp%m)T(Ia6++wTXVs3aCngyYR!+I`E57i^Dss>WX_dE+V5RSm1Z;pu75sXHS-RGHXkI81Wy{Gt`wTcJ0w{>+ubziUCp@&se`+h#QF*lXQq*%aczm9<i^>i6%
svz^df#Homj(PA;ak@;PBjxJJAxK%>eP1ke5^8AD@-_us2>47cH|hAN!uodi_^UGM_O{k|#e8)IZP<UwzWz@soSsgwnCuU|GM^2G$PgnH<VmTW98ri%V7Z*<
>dQ{^qC$kVm3pTRx19bW)?qI&ah%V?P%S<o&&`}ummpr!9Xb;zpCzR@7O&j@>1ykJciY5#C^~a@5hA2?lO*B`FX&bif+$1-f#<ygePwPOr2Ac?V%Z`FPb_2Z
(b)C(lq9QX9I2n6`A#&sn9>hh`^*3S&GMFVSk#Pn!>Gx5v%XE_xJMqiwhx~uVAhdOqb@|kr95>yAqA8ZWDs!Cb0Un98J5D;yMqZKr}tgJA*eYl55Cz_z?ge0
eq`O=K`V?y$WhqWmar+QLY!SzIeO3K$pHJFTlz<yDWBLZIpw{~6KU1_aD7=R*i+DEKqhFD;*D%`<z4aa|9$c9Zy2(^ArBut2wNwIuo#O?Sl3p7iT_!<J7Wek
8Vr&cW52&KcY*=eR0W>M{slP$fB?1u2~!^Vw~oa18w{XzXT1{Tf899rxHvd87F9ZNm9Bep*?xF>+V5e)@GU+J#=w2{0jFe5v<*(}U1*wJYlGCIBr|R7lH$Z?
nlVYb0DY)A-X(l=4l@Berj*+&k9yab3tnqpFb`5-?vxg1-M)`rf7kKO*TPHi$FkR(m0Pf7mf?c_KvFRTNprwMQ!j`-<^>tVp{DD_j2oYX+EJ8or1fRTgs1qr
V>SEoMTM>c)pT%5M@Xpl!57=<r(bb05+pB+R)4(58%MW2pv@rdG|+3M#tdLg@E|8?Qf;1uR*ZV&8A#}XlwTKGc8*>1v$Xm&u!GPgu8*qpY$_eEQO%C#JlwY1
j<T3&kSt<EernAs#^WCjkM{;TLL!W(r^+*r?K$Kv_6Pfz_tO~@Qu|JGE!BBoMnf?9H5f$0Q9RjKo+{a`tPg+ySpI~J7Y1!uuef$P_}}KNv9xGyjNM6>IqK>d
tItw5%OsNep8j}^g3}8MKOxN><C{k!^-nX&?r0E*Ks4ZI_6eML08mOfAo>n{;2Nnt47}{X-F6awd1M<y(4u}I^$Q{aUG@i@dx%J~`B=G48bKi?&x}Wa@0w?j
hHJ(vWgQFL-jkIan;MgK`dDCrRV}xff1uPN16#v$9ZXv#QQ!qn@^qZikfa%lM~%HxC&!uqJ6ZuP$UUN6>;M{v6f9w*j?&GRs`>W&8@fnV=9FTP;VX^|Cfzy5
N2WBvp%E)qnAbI5mQqG7+QNz7^9KqB{x*m|aeKaD{xsOK8(UEbjSaE~TE6reEgq1@eEW<?OK12}H{yQ`UiDvGh`go8P5j;wiVD7%nk0*UN!5yS>>$r!4BP}P
KfgUrpa@X!#`4=><c7LR7BUT09;Bpw8YUQ|R;THEb0RhVFUh(r^`tO{N4YejgHkK>RRGO{Ix7|(n&Kev+)9`1&xpPHz%t??r(0uZ;E^)V#N!q8ox*LG+HVWQ
{!MyHMc(TJTs1m^xVk!HBlVmLCx51i$ff2Ky^Kb1$x(7aYRRdDe|PhbOjI-!{JcLzx(bU4MeljIrSta9r#|)S`=t&o<{$n!dw6P9c19gU6%)QHm_I0<w-m(r
B@mxSFk6CZ`Jg*>Mk5qO@(df#Zg^-(DzrXPAp({`Bds?B6AAYV4Fw5?`9$U2N*nt{iP{;7H$^b{ycU7dubSKgr6-2CIBWSpsP4GVh8=ZlfQi%%Nf;RJuAI*4
ygQJ6uhOsUmNk(M{-dYTy!A4q85Nv5mN-bv{eFUa!1rEmLGQy{XcMw!;uGk@<%2+Df8-}G*V8BGg$Uzy4>?$=a_rXJp+-mx$V8R%P^GECYHk5Tb_S$wB=B^I
FI&DV&zkzvV4~N?z?|$UD+2C4QU!^*T*{IZ#=5@2hGdTUI!*O~(VjIw;bYVeVSyvIE@g!VEFl2xZkQnl@Rn5&(;Vv&zn(UBMIbpfa#>$N<=f>KxpEBXmAM(d
>O$N9u+M<Fw~*R#V1p&yq9g7hKz*2}C$ch1B&!JTqT?K|hU_Tn@FComr)vH#G}=wX7_X;%s>~HDGr?ntTwpwV0R+-doxDm54k#!EC!0EJ*Kt0a)Y~odnE#%x
YDc`>=MwQmk3ux~G=j2xzf84>+?ZAs#R8S!HeODDm&1p4zO*%#6rc)bzT&la)sBBOEJnA?E6p4OzjO=-ADG(4=+>j6ORt{G;9>pKSQBZfcSFjONdayBy`7P|
#JTCc7QP3?VbVXZ32#L610oR~yfR`y(YZ;04g$cK-*MkCiz!)-Y@VmsY*nysD+96*V;YLX<Y8k}Zc2do@f&{UlF4*K->1QVmO$Rv5Knu&Yt}Dpf8cRuWQiB&
OHRf#_9&CQu$u27!d-kti+lhzFNGO2t`TU%Bik7_M;M~D+Fl2mEa|#bZHZt5wY~5yf9}W>xngjAIhPfiELhN&nCpM9r`!6oHST3#gQe(rbBNaE-P^T{{)Q_0
>8l$5SO`>8<j}$3u+MMBaP1?4hS$QNSzY+zs-|rzx~s1DfK#h)+_UFI86hGW&DZ*cg)8OAS2IyA8lFGJzr>J$mr6*ueGePzY9c&u^R>_!&#@&L-1V6SOwx-l
A(2NYcf57=P82p~WO&5#4z3%v<{~w2Y^7}$C_K~17)gW%hTV{VP6Zf<n+^pC*b$!7v93wa(lva10ACNm!h{>=T6zN)5D&F=CjdeF8in+iCp6MiJ)O<Z)8zOB
A59XrD&_OCO>+atpwOyymVnY<1o6@?$GH18jV@UfSOPa6uorG5p51u9`@4j_B59TE;MrT9_dPwwaPZ9dm{ADAYO-ub_r>&RU6jb>eJx*GN4001-II2{Df3sk
@wW^U*{P;3Pj^gNFx+iod7!-CUt_^Bbpq2w!Wb2}{E5j~3QY8BNu#9*a5do!yl92s<{$6p*`B8*B@)<QyK(+wz6wYGd@~MX0{Liv2j74yCAiw!ry+lnM~{ih
&J}@2a=PAdPfVx>T2zxM0j$6T@>~tBgf&C2)(h0WP%^4o!H7R#=*-k9N*x`rin_6enirvRC(PATLsp&IR&IR)iyFObGkPkEbI@=?jbU)ngn<MUd+An0mHL-i
VrWiDaVY0(BG&b?H3<*%i_kkHn1(MJ&WiW&lKIt1MEnq9Bk59{FhnL?%*3;J?P*`5KQ1c(GnU%6qjK=?_z8Q=RDpRlzYZ+SC;6HiUw8PdFR2-Z2@g|P?@Faa
UZZoY&Cap3Pdsoe#miZM6s%fUBl^=hulO@xuZR8qC3@5rqJn=$%aNP2;XI|5E%iuUUz)S78&m69cN~mb2AnZO!#gn~M3{K+lEY<TPWCldqlac;m(lzouL@fc
lwB%9<W9`;@*`lN(3cJNqaibN-iR-vSDj?R{nQEUngwls8LZxmTn%P*3`;WU!UlI+#O8M&i}a)r64O{6$&VaoiHy}a{0b_Gs}X=s-uwh;3lv(B*p28A^8(rA
hYmiS%BJRP_>f?d-7+}0vfX*xV-rY4BBXM;fg2{R>LgGA)^|@{@JYH|NYcP&oZLv3c6ln-NK)~Qyjc(-P-v!@NX%Zban}@zM58|)(b2W8O1h6TGOgEBAMi>^
MA@zG8xJrFB*q%@z@N^|RbOEj&VU56(3$MlkKbE)Sn${s@j{UAWH@$pTtzU#tsq>Y4rUel`mIppDa<~9mrpyCe|^s~hCu3wREgEW`jBKhQ4B1?K_xp3<+l6>
JIY)5kJ>fQHYU<#%9nGKN$lZXu5WYGlBHr=m(lewLJE`HqOMtRqB7S61JVxiR5=I->b{b8BHZ)@2zEsY6@`VuLoo}-O3Rp1QKH#6YfI;g&{4-&yGdRXWaR3R
H&cAHO~Glo`n=ASc99^BgD7b@Qh6}onPH;FqwkDFT1NM>3+q%U(-^4jCl_$Zuq~<a04e@c$SWP4Z~{Okx;{?S3+-k25Z+zLc@)~Jtu;u(-qS7tTMYJBt$NZN
pTl6&qOy^W4I8zxLcnP*pn&%joL`?8T7NH}9%H5o0TW=yzd9B+eQAGS9&=ITsft3NGIFEChUCbAaW5(VyCX+9@E&E;{fEoW-yFdTHcQ<(CB%a6r#cqn<wgWp
a}u97K*~g-poVOImn#B5EUTI6W~lfYj&AKD<7Dk!`s#Xlt2f`bU5=}UCLZJi;=^!rwmJ;hoxrrLaX`r8PFKI{?h#OdKUvhC9QJnE_LGY<Jgr1|6Z3wSfAvxp
vh%4=w)kYAFI=jSjc#47Q@WoaY4Ph6zf9;*E@WTP_$3EpM)iuImy-Y@JXZs&5k~!Yz)D)i>r{2%5T03jsw*?&Be-O4hN`&%Y#sP@SJir-(qbUr3gT^1YD*t8
fNnJIO<WT|U3&T3(&~G+%ckokYXm^?6nxFdc63@u-m908e!Zx_b%QL^r4eEks4n5sgwEs~@EMG1d!yp5nJs%TnhuUnTQJ7x@$eV9y=@PICyg|oZC4}oLYZAL
l^h3%0(dD&!B@7{Yil##a|>$uyuYewVIjDzUJ`6fh`fi$Dif-}B{d;?_$`+nQw|vv*!<gou8T6hIQo6t;C*-mE+mO0u8JOUm%tcpLf<)$9*a@KLNR9gpp~HG
u<gqP1Q$PPT^5xe2>m1&J#Gs6fy7wS5`6e{1DBd0kcJY$X}dF+@}DM-P6d~r;c1VyK6D0mDbpp;#gxx*nMAE}Td0P^N!whhBl+HCw%RTV(BH2gWx_KS!3>#}
tc7%0){N~417pHNCg9u3-LV8WEdJE$KKm*y`Zm{6=)nDH`Dy1LL#e?0r$v{15t5(;{O-A3-m)IYhP8fLkiyh%L-cc{s=k)V{Mm^PHk!;HHZ@Yln#W6^P2W7O
+TFoLjFlvMI>E0}eZGWn%2Qn1ZGq~wrOQ#zwc3MgDy8~U!K$s~k1=I46uAO`U9pPh2!RV_Dy0@jPsF04s--8+5juv-xN`oEaS${rb%wDmxSU@3(`v~ZpZwab
8x~91J3cbM&DBl>$EPb<k1&v~qG3+urTGtlVSA1?+*d19SVs%`4Kve0BO{p<IgPTH_j-G)KVyGqaz$A&cG;16fN*hV{fW}kjpsZP(+#vmx_epu@mybkI&=<D
w6urv-`Y|+LE+5wKEhC4;3|)?@>zk5f)<mvcYE@8obvm$#rR_sPHI}BTiFiD*W3oxQRge)ijJhYk5E52z(kx+7eK!VJvv>p6)@iEBFXOt$60jkhXS}utjIqC
aAf6OVLAez^=|VPMmY1jz7CxYlMqON9v|5EW&$<nV`5%+R;2R;Mfu-Nd5{M?&=<p9n$o3CMG~Z>>anhuqvpsIU0vyr2ZRg~_7Z==Jo%=J;Ee#eF|M~}eg?V<
jgqX)JEBTEirqL4psZ`rd7!F@_Rdd}_!3MvTN<XSB4CDG!H#0K<g&^vF-panqHButFdV2A$R?zcmem&L#b;xFO<|Zp+Sq9p`f&4_h{J$J8jx}!e4L68RN!g~
Ra3@$R)YyA7(`^JQ@!{j;%I_6|2TA*Xv^2hOkv{A@(c>2LX3yr=;NQ)Zea+~Y5D>``QRkls;i!aGH|f@zx=}3I3>u!I6BsZk6zJHdnYEsE|C@1*u($3AQV!{
)QZ_`){xBCU?xW^KC$xVZW#K2qo)?mVGxA-_?Q~0Xkmheh;@~BJ4H$|6^Q3+J1wBTvHun>x$^D9JJ0RO8@mv7;^e%Fx)`jMp@aas6UPc0<^8+&u?E)#*YXp8
7cPb>ewZDg%+t!!33o`nv4}%V{_`(}YW%mj+P#BY`TMRZHoaxvN|~1M7Ft*4wFQC&(uHv?az}h2rR4PaP&cjIi?l{05`O%Zqs31j<3OC1XNg4i02V)(`IB|B
w=6k_Cn56`b*mV^#W?8sPn2lS#*%3Qiw*m$FNW&nZYSp4+#qtye+t2%e%u18trI+cO!5-7GkITfvdB(csQXslnJ&;uacbR;G6&`{+L9^EMZ8l-kw%aM`RW2_
F^hPGdkcB%kM+DAhz;Qp<_r1xDTon-JKrINw9?8-M6K8;rl7WMAf?Ax(5gGJoTgoYtka_Y+y-aOUe^tS3ek^IyN>&QfQ;2w9Be*<-4Y$iq4=8ggjBWrKR2@!
NZPztPh6t+|9LJWdC|QwXcSf)18R%0ds=VN7X)T8vij6Pc}jjfXQ4|z2XsOTE60uL5J}}QKk-~<xN}e5MvPuS5(MRE#&rPQ;HnB_yOnT@=SfEJV*V2~=tTT`
AL75ZT>SV>2y2zQbrPTB&&-!^#v@3+V0`MM@j4(+xtJE!Rfg4BN<{;Pk1(0$xG}d8w5rt7G)kV;t3c4x*Y7rVHm?OF&77sHN;|59k=N$vy=#f!D%v$trl(^9
%AD->jYpG|KOeqMO#V^^fAaiK^^^KFmed~7F>K5XJsX&ZM0lUeW9)o_CweaBlqxA7XlaQ&Z~n;2=>yd?ux7|!OD$Yq9jt$*RWUdP-F~lY$U=q%iEKj`Luyo}
xIihRzt31gJo#oFtCwDW4nD8dhc9ryH){$On2AdNSQZOFCmjiSw*)1hUKvdBa&;4fFyzg<cz%$*elxC#P^O`0%>$$S5iJvj>W36^(eZn>?=&c0ey>;5xL}ss
I{1<}fXNK^#O19pEJ@LmgDJeTc^&(FDzAj3OKbl*ZzErpu1P=N*qPvy^yn%6v<`4z`m&s_GRM)`ru2(XWFA|AF@T<c>8Cg704FSJ7GrWc*Jm6wWahF0O>wYh
ga*Fu@Wn#C;ke(U>sRxOSYLzkR(`j;%fd8UggcOTpmX-)Z5p|URVE<?PITX3s6ahG>iNji1(yy@eH83HD`Zie<;q+@z2z#nJ3D{^??EcwePx_4LC3m_n2QDt
pLo;_4+i1$jv`L%j-()el~yj=(;9u9MdT=khvv~4#LxAgpHFDtam8&x7BDAkK#dY9`l_Z|d9+)AOSM7=`S{cw4q68%I8jc<t?XyPq}D9s5qo}xQlx?xr%SU~
+yTIXCn=|24*v%b>1u;xgJTHLZ=^;hjRC+0!3?fMFvw}vgDV%NbFMI`liaRwBi@64vI~m1LdFd9b-a8>qx&xkK578gOWeg?Obn>E+vXaqBq0n8+l7=TOLWJV
av?q&D$>XR1~n$NmirBmHOfchH2ax3WT5^EInRk)gyI(0yt-n2^+nOt=5j3;i9<%W4`X5=_rs0kZB^t|gZ`>>lTYeMoIP1)EAaw>c5aBt?@LrBal!!AJQE4J
d>-BeKA9Y9YK3yS6%$jM<(Fd&_@=Tn9c5^&5`j&`NK^oCawdL5YERm5xrK6v<D#0cm(SX#toYG~=48#fS=C*o;#&@wJ8yGDus6;cWF9&2bfNoV0C841IIuE%
s+0_~$xPt!bTiPBNSp<}={3yq02<<WT`r#n+1!D?=E4PQLbb^5I5>Sq`v|%oU#q{A<}UBq@+O(M_b-<1`F_9Sm<;4qAiV{$%3N^`u0Q^5q%+~1F^=Z%wg)EG
TvtuysdO#|Bd3YTo@6Y*e{BF)AXqvqIFT5zSGjs5g!WcloRd%XeQc&c*1~T370_`qu;*z60sf4Dz+-?;4FHZ;hDy=G2HEUjED!b!TB{_mK&13|V}-@uj&wPJ
+B7Yo4v}PEfUU;z{py!lVi=LFryDV#Z;J~#=0%lXKEz)s<RWK7?n-*j9Hk9cVn)W8z&cN+;v?yJny-Ub|6IW&@K2bL)s7$2IRcHg%}^i|?vvg{NVfNP{OWM5
_gH+{{kk0T=##*u_|Wx=qM=^1C=`C=oo(~0a_m>Kc7vao(-_J!u6~Tkg5+I;avCLAlR6ErtUC0I{2p&u8#034)`<WtwGt8w`By;;iQ-Se5UwzIL9*lSq6TE7
X)BH;>bi4psjde3{fOx;2gx^o#CD>rrm2mk*H0nFDfENr+Cm5~Gj(Xbi$$e-@cKgr>>0XxY;3AHFv-5HvUXbfO=b|SLiY=<&*>kE!*`_qeaw}n6lJDKzF`F=
N+dD_$Y4r)NGF##pQH@$;^o_H?#I&}#XxbZIlY}&ECty){kOkV11jaso7JoT!hKYbDit=>A`3WN7q9Jk%bfi9zQP7Xa`R{JPAm?D#yA+yi~t5bMluIB^5QUW
Pz}TxE)@{<7Zg^@k+6|es^8iMkQUS^q5nRN23s=QT+^)JQ=Dg*(TsYekAfS`(oSS@(q^&OG*dejn$EZi<Yl}Rih#+0UYozKQKMBvL3Iku$1Q$J>@_f1Pcam&
QF0|daA9cF2S@LY1T@t=g&JV&5SPO@7$$ZX6#k|>4%=8%7N<}ad?k;aew4^4DRGXW-O0-Y9q6ld;k=Ql9Nz`moR0+#PKlO1hk5L=W#|)MsDUIu(K%_g+BU*m
03=l$8cH{*5UQCf!Qh^vWN(}HVRs_q_O%{Sz~#^>5q&UsU@%lJNC+qbPz$?MILT~|Bd%J|bMPV;WB)FBKpp(7UoYT7p0TJLcbp^mio!jQqNG7cQ0H?ui|c2=
Q#w<-%iDEZ4v7VS&jE^OJ=ND~Z3D#@%{z9faJ7^!7u<Ndo;-CORIECH%w$;4y#20~v)osYsbq5=c?On+^NlCgQVaTZh6H4jbvGcI{Vlac$hi8*t_;55Zb8#)
*7|e$HDIq-5ZbEwgpDO#I0QXZZp(Tw%f*c2K;xiEdR}_4ZT8`=T|VX8zp2EX+T^FEer+QPyWfgw(S=EO=oiLoLgb4%j6L|=cW3(tEMNX|>hBJgwB{=6xXU2P
B;*Ntnh5K9OGMEFYR?^ET4I3;Qd!*AO5GfM`D9Cc`w4L57#gtJBgllCV|!t6@K)_BH_7md_mXV?+4BG39_L1uzY7XytLV1a*lMpUYibJkuoL9)7_TDwsYbbP
TUL(!Rn8qpj_JCi8d{CcejVr;GQ8TLdEx{AM}!Q)ijDQ&z~!!;1EE-@$TN{(0W~>EiIRS|nBPI3G>|V^$08aMR+PllHcrO4F<AhT@kXD)sp0I@ijrALLVmrL
!^v^scGOuq?->RN5^Rn_9nct$Rikcl46O=ph7Bx}X?nMBa||FR!!u_K6C^;2#<pdS4wDD$+rhmqUMz5>%_&K7XX?Ks*0RBf#|>)jMuP(AX{aFy_aTzQC9s8`
4jer=4ZlfiI_`)JJlznEwRThlAo%IwRqHO7;lHv+8Z)Zh{K{&t$fxVyD`1pBpC=1dFbi?raL#K)L%Y?~vA#^-9Yz|k%X*4iSyPuVRnU=|g*JxmJ$>ry?9rI+
w38Pv`{{`t+iA^u18YI3vDX$;krsfd{b!#0^;qx-(4J+!e~}7w?>AY?SyuHI`sYb(L)(Tdy2ay+w#(p%RJidS1O$1@(b4(rEE06*kzL1~e+S2^!~D+6T&EOT
4+CQFUXIF$Bm3W2LL_MwFmC=kuGz{9FhYfLU*<MG<+c#0RPjGfqlY#~b!iZI^3d&QQK?|AjHO#P%OxKYo=NmUY~kuq@#q8u4>OpEHWPboTYy3N<91$#o(yS5
D<I7G0=Nv>zM{$RNUDczXMVbftN*Z34_P1DOb_l3<$ffmO>UY!?`4s4yaslj_`i3ekP_TDA2Xr2!EZ7Za7O<a`(|-S#GKII-d_}I*g=zH2gaATF(vnA^9ElH
9}eb1VA*H9gEOYVmaI9eeT!@l={6Joy=DZ<D5=bN$MN}yyT$^<cts74#7T}U%$}RtSwrbe0k-o=p&kgaZk~A^Q+NpV1ZHg_{MtnOch)}`8Z2T2s%YOVoZ&wR
0k-QZ2k9?R9C2E!{>y?6-e%if4|z2w!nhO7?IY|5^nYx|ym-zb%k-ai1Af8BED=~nCxR()G~m&yliEobkoc8ZDTq)cJG2v*r4f9Y$Y(6-gNMro+ve*Un+q)D
@yWckHq=VbKy1QPc{TO^M%!W(w>2&8UODF`i9Z~&4V2dXxd4SzZ}T)>+UBt!0*E)KArK>aN!cs~9v^P6`~G<ubIg7um65aT4l>Yl*Zt^I3v%G>i|>SSDn}|A
kTi7luc3o$L!BVt{*L)?`K7dHyC4gm?&N6)JJJH?-;~QOUbV0n55^yMUqcdbTHHX(is`M^F>9U)L4dXXVRy4F+EHPQFu-w0+^3z#^~zv5Zh{v=|4<dGnk=gE
jwRhE#>+P5)@VbI6k$ZeIyoiLb>{(lqi#F6hDVi+<VjqsHo1FUTs^}ow+4qWBkj7o5LhVtn3PK2&UQy&gT_^f6FpPp8p4-}ciWT_?&L7@$-$Z+inN`I*T$87
NY6rfSTQsSG+*l9N2!P_m{ptFWYcfcdB5QE07sH2to|H5w(nUuw-<)nw+^74l^Salp0cZ^xxM<-fc1_`nY+-;0-CDXJg;cXB~jQh+~HQ2wbbEICF0@k1CqY6
FbvKZ3P?+@7uyt&khO~5$nl43FHweY!>1(lzFLPaRDENboy?`cdkuqi!{$*n;K>QF9NhD)U&0}F#j7+hUz8EZpZWN8@dGH>rEl1}u>(<W<H<_DPCR0+i#SaA
a}pcyw!i);jK{G<eFg(vf%--M4y*@$T_d5kM?-7zv;`Ga8c;lA3~k36o81et(k?TrTpHAg=xXU*udGX2b^rao7)$F97E4ce&Z!lmS{q*TfQuqE-`KSeC8*%J
Zxw7b{i;@Kgt<+Ih^M>C+;$@#)G&iW6K{*?YwU$Z5g|x@tF=KA8MCae&|J-_7Ilv|9|cgoIwQGKv)^1`z745wN+At|iQr>j+ROJO(Q>I_jdFUaSPKa91udj8
a@63iI1;2FU5?!UV<fgTXLAbm#8gg=j{4cAcuF4V((EqVli&wn|Jir*_)mMLbMBxFMMn!bAm_ilTW6r3c&GS9=&&xKAYU9~bMzQ9kE_*u$2IXg4c)>Ej9#pz
m{-q#_f-6KM%{+P&{8`msMq)^AKcuNj1TdH9RV|)63@+Npn)wos7S51aO`haYPSb~j&JSynekkUkPB)Xz4EQR^n!j$hwNK~)h(ItH1S{DxAn2%>9*9+ONN)C
_%~4h^&&rP1G~oVxHJ1wTtxP^{m*t8F(;*2Gigkoovc3re^QL@fy2ZlHjH|6pT0YYz4MSqu>hWlW{2XC@}hW0qk195yxM~^#`|x(=K@VxDU&m1+a;s<qLO0&
&Ags6frCIh49*`@Xz%f?{I!POCgcRWndVn<KC(+3K!U>V$J}@dlA7}|PBT)@s?}mzKp&z#{*N0+P=mS$7-j;`)xrq$4I%Bztaw$(7AOial-NgWZIf?G&P4rG
Mk~^>hH}T;=J&Ox>L;j@h~90+(_c(;SuC0TH<FbAJ_40e*@X+bKA_kyW_Bd<B$PBIYiPs#tPgG4gEZOupPjkG`m7glNHS<0*%HmmmPVWYvF6<Im8;^b{4hB0
(H;`U*cgWIxJc*HwLy31t45s8`1N*FTyvk??G1d;IF0w=Frew`VtTp*R~Da0Lzj?u#KT_gf0cHpve2-T3Ny1_Z1K-mT-}TXC%tJo*m>1K-q^-EM{1rXpHXSb
>6k)8_pG1r1rIqfs&9`tHN!>93}aEpw-`NmvZ-7;c<yDlJi;PcGozhLwZZAKsi^SQ%aEN)(6uAYDBgwZi=#H2IX`7i!fKf;a=NOS5*f7E|NC$2Df+1u^Tfse
JRyBK6|O$3&)N=p8C;vNwK}Rr3LSG@=6-H_0lKlSE(-buA1+OE_M?l-;*SxbA?{wM`@oK2%{|PiPdYjDy<DsO;c-r&nk`51EPJ0!i9Nduv#wR+qk=KA&*FlK
6W<y<uTGi=J5iGVRhtLH^$QK(<+VOS`5f^W3tTv|PapeXtQX*cp^auBZY_PcZKqc`E`X98b2zzQO{FTQ2RFr<*F&PGlyZ|420WQ_S5=~om4f<5MnaaHK5Y4z
T@J+S5sY@u#fj&EN@Cku2e1C`+e`A(acL`sF{GEPOB4yvh;Q;H`?LIEv^(v4zZKAg;>Pl*ayutkkf_N4X=rame>vypj9&#6de(2qbYBMC*ZmTiP``_Ku2Ea;
bKIj<QQ-YjNgrOC@Gv_nYUe@q-djf;E)Ou5m=KlTp$nlT^|i$PsFBt<GoZwq*4uSvO)t0bpa?>{q|*6zLKH6^x1T}Wub^*1fp1`I)ZDOpgP|tK9M6jU2JNWL
yT+dPgzNju{_c&M1V88*gm^Z_wRAVI?>Xldw4yhfCY*6=-3Sr|flcux5Ws(PD@z>6%x^;nhAuy>$HrY|g4|N*4BE_h#i5F-`t`Qd6i!;?VtbHt{o`Y07I~7R
kpKpL_#g<^f2iA=h(zFQIR6w}y88hs=J23P=LIP!Gi*>JGh&qFFe0z7izrSb`mbg@mmNM#rE@6FQGeQd8jnugZW^UN`~fqS;Yc6aEsQhp*Bqy4W~<>Ee}Ksu
1rV-wvY@TSQ6<$tG?qdz+5q;gCD+mkzDxH}?eS1wbNSwvkf6@>4KCDXuF(S~c4Lt<BN3PS(*QP;nU}KM1aOjgcD#fkz2vV_0MRbCam_tes`dh_o|1mUKvC{F
B}X?(ph>FXo&)*S)+u@L5r|-<g87yI$;<CFx%rH%9;2$JE?xB|OtPJhqW=TBy7!y_L6YpnV$J8&=45{sAiEO|_f0r9(u>ebmwvH|3;xr(x>Jt#g=LX!I2YD8
-@A%8=kAi6-w0anxP~TV_0~z97kG3}O$hpXbzG&Tf%GT>jjjVw04J~A+n4r<MWgO|BGfeu6b;s)HhBBr#g$^94Zl=2otAfOhr?;yjDdR)-uPs93LeHeEpJ>G
{5BTB0fAKiyZj_s%Zg(DwRrCVy#qzXibO$45DJ#=nbegJSpwn+J$*U{pxxt~&x3!O6poqt!ByFQ8S`6Oxn#c`VkloK4{hi3<o1hvGe;TCh}bI!3p{ksA*s7&
8d5A3xWg~G$6+<V?hKePDx9xup;n}bn~jxO#)c1Qcl!dL9a9U;Im+GUS}+8bo=tVA`*sQz+uqun`sm&yE439(5}%orYr&SQA?)1t?nd6$$C0E`^ppf=GeO^|
aIm7!KMk_Pz@t<7?9H!CsDlQU4|9J}j5GOZj3&N25!y!H(bxe_A_f~n8d0J%-V_guliO+lSLLCarp|Mwl6XR{>jKW}w|v@<OR5x<vp6}sF!T}MAD$XClWc$;
A^L#9DNLfs-T9D~CWPBnn+q%^zaVOvs<=x%%(Ezob(f}JC3B<!r8LLGJ<Odcr^WI;-C!>9{_Rf*@~=Z8>x1P5vC61=S^B@V3kiVeT9KZFAtr#p=8TBpPy<Yt
+G+)Gm2j;KOBOYI8`?@qC6`QcVFUh^iAV*$K#Wvt9R1aAqmPLM%Ukh_ppWigg<bF$s(*)_M{`Q!lD1^p;ZVsRaCogOZ~<m*^$~-q!c=<6;-KF}e}h7j$YLad
J@aBK_dM#>t-oFm@VS>wfl~K6z6>{zf#o!^Hf9mNLL|zL_F<X;+U8~tkIQ|ROOABQCmGdU8yOq#?%}97a-ExXTXV&7tyY3n)~nd9FR!?=*~=1e7@pgrE^l{_
g@zP{I+**@Os45wR~*gUyuRAakD|9y0%_Ha$LyD0t{mM`(U1R{f0z7Zw@8Xh_OvF=?z)y?!~#|o(Kt5@mFIn5ZZHp&%gJ`m!CxX?8&CK?KWB1ZiHul%8r}|r
I>Bb20q!a&d<~~L@taKe66iqEA1}R8ksz?_V9e&OF^>cut7<xPpL=6%pz#d+m*~Ij{!$3$pw$qd5eOp?I~3*UwC3VQuYACnRdWpt%cce4tAo8z^61@u5&_Fn
&zv9CS>W|%IF0;friG>^eb5pQ>2Gt5Ry)qpZz3d)HQB-Dq{Q1<!f)FaJ`nAW-Fk=9?Ihl1W_r07^+4fbpr!F(+gW8)(B;$UV?v`0DGtdEJ}5R)1`u{G*TF29
+JzU-x~H1HQ*X|kVMl~nfp@y-P$Ie~RfO=!H9q&sha+?y99IZq15yHK?C__0!5=_a11JUsKJ`=dEUmI@iKrw)e`en&_w><P`kgrC{}S&L&shC?ZwJ0N`Evl_
NdPq`_xa!ks>Y{~$n}m;`4s5ws+d}|*}eZ(>{3tax_A1u=_HlNaL2k22kAatVsEaMq1->1LCs5{Aju$OUjT2nC!M3aN65)(LQzDg0sq}a{afzFhS>Er`>@$%
pkdc%)#xQOb}sfDX>zl*piMW&4yh*+Bi)!k5GBdGx&YhibL&C>qX|1BSeuwp8SIlnKM)8ekvzL^fm|%*3{?F*NI7d%dSLUyy22TNe-69Cp4YqitA|V*=Eh8v
GRZqZFRgL>uYT=w{ybgO0C_@`JTEJMxCTU}P~&`S&T~^GQ-yZY@UW7sl8yj%eaWNYE@>}ky=0_@P_syTN?#3z8nGm@=1i{&t{*SE`^90AtJsRNK_;(7Wr}oP
$(_bw|0s=={9W)wqr5<YbYwQJ6~V*%qGX=eMqC@&X(k^SsP0ne?RAmC*L2NIv(#N!NsF^M&HYESelAESkp{MZs6yHyjQwX!Q;$i9EM1|IKQ-g;52TI8T#-~?
rf6)W!+qf}<22Q%E{YFq%G<nDWY!ex2bCE^b=4ZK@5>3YI>BL=m;*9ltVvQU>tvrzMAJI%yQm5#o7@4e`S@C2r1hoNGlvZ=_}qvlpu{AB5($A!Ywff!1iDhh
jg5dZ(AUE2&T?i6*2{eE{N%Kq?%F4QyB1DECanlEN~;>5NciD<mVw0&wI%u_NF!gfCcKY(y~|4-a^K<arXUnWKn%YNfiDtQAhkJAqEABg>@Lp#Iy>$!zKg?x
K0!gEhDI|vA7z(3GE)zC*{sHEL5f5U1PRYpo7HFhln=SA_h|rs{Apv(g}iXfX}n0GXHD_`&Zs~)2)--LZcre~Mo6VlN@HpKEo&#i#55lN<ZFGGtMByI7yZ`y
a$#t^xj?K=E&6>LY4ENppVsO(mw#c#d%Q$!S(a&*?pV940A-QDuzF`Uk)%$L%f*`sLZD2#WAypHM{SjK{k0$OM7qsE+4z~Inh{6aG^qQhOG$P9+K$5Kf*NhL
6i|H)Y`3}euF8*lF%po#A0$&pWE0)U(VhjNiI4(vysH;M50{^9Q+DkWPajB-6q`U5*4{H*@cKZ4{bGG>%pCqJ7hSu@{(&^x{`x$XMqm_|fXPRd6&n7wJS(*Q
f<p>vMTxeLHXI7SKQX*R>44iOy~kJJPSu6T*LHdi=}BhyG4UE$9Ku%*L?X4%V3-9NWUBB}2^{I!8?(b0P{SUJ7DB?!a=_rJarg?vKT<eWRzvYhaGu(jreeI?
&IFA%g;!pUr8~g&o3r#awL2Oa*ZX@H^D}DsG@+p57+BtI?sV~uCz(eu{R5@Uwy&6pl+JsQ-cOc_uqSp8Q0y>gQ9ivb$3r*Yd<5DSOnaRiL^K^%_06Jssy4&C
*hC)6R{<ePP&js1=+Q+C%LR=JS<H0^(_wQPH1*y@61r`#0U#_3hC;k|In=YJS@7*dF0v-R2LtN*9lzW(^ic4khY!ug&8F^-<URI40XUr#<|BC%$!p9Bj*Zdl
Pp&S$TfA~<6xb7ni)0NFQ}!caGfMwY_EGq(Lef9zW+jOThw+oCl|h}suPEopUkOOk@BkpiUpy}cs0ev8>-4R)bvGv!c-JiMoNDait@8<2(oHGWQ{DFvt`8^a
12`_EJ6@&!f023ddv0o(QElVWjSGeYK=8Ng#{yBC%N^j|LnQkN86BC?jf6!Ef@y`a{iIOP48{7`Cs$zVOA2Pp9C@FmQ#-SuTWM2lng;vMAF9*l@>YF~yIYW;
KIk2iGe4rNe%8XKter;HW2?JGLeJ^dhh=Xg6bj~3)60$r3X9vD4>CLJc)`z-T{z|~tZ-v-QBX}$ZFD-!a|csHW}}09nyyfbMXDhxrB{7F*V8)uxFIx^{P!Jd
1|FJyy<>XkbFys%De*2d&@I9DbI<zBoj?V4_<#)AbP*hwmM4(cKPQWnhp6UW2P+o`G~iy^>I(_FyKNVO0r^8(+lScf8<Sw;O=mx(D?QSe*^Y8DM~vl7g|!I4
rR@y#jnCGI6**V@o<&WacVD^Hw98&5d)TKxVo^blAUY#vN`HiujUTITBj$VI0Y}e|iY~oYD#pO!)_L9g6Y(7)DlDy4mkFYBT_j&~jGeJSMJ_S8*6j178ur0F
3qfQFpCQFQsXCO*LRDXCxAGYKNlhC4*s=J^xU6Id*(fjk6G@_OOq#J>Ga2QX%-2-oO4rLIQc#x*iTDCTS5|;d@p#o_COM7p!5hlnQlmSF+i#Z5bPOj|fgmou
w_qRsMi<C`zaCyPrGI>DoERy~aw8?TV*!nBQ-nh@owAk>bI?$p)Wjc)Z9w?V&8X`It)Caf-sxwI=t0;?KPSB=awI5)xBZP+c+XHA&IR+LwxEN7#o>DrloxaW
ZxwaCo211)ro}TB$uIMxO&0W!X{w!dVb0-%Rqm`c!_lbCUg<y#;Aph4v4ndFGOa5ru?)C_{8w*{gRS~R_>jn-b(4yJ<S1s|p1$WupEf?&Q|2cJjPy<D_r6=i
p-$39gf^m@SzH-_fh8o$azRhX7%JLCOc2uDyHSBCv6soZ6JrS7;_2JebEwn9TkXf09h5u{#3QAta3sdYu0M5P36Z_Ey?3gw{l?6(1a;%maqAZIntmslHq*N~
0Pcb5y;6@`y<$^o<O#md^k{~~Rq;G+0-;74bRrX6QRH=f(78wuWynum9GZ&G3)`A<zM7&ta(rVc)Fz&y@FLUxD!P)F1~;fA;3}F258)qNtP%pyh|)(SlSsA)
G(Dmd2Rz&cIr=8u{-Pkc{%y>00)+|P<4N5qHzadWYF7xl{37p~b9>Uv7K^R{wL}>Xc5i0qmos-8Q=%{L^4u9(k%X!zUrozB2OLC=f?gt+yM5Z^+{qiBUMZ{g
AQyfVK2l>z(Ui!zoTvqgT+<M);}1re!*P#^F5u47vDT6T4a~oY=>WG2!*hQ$R8+&|$^e?P!f9I8cjZgO;-b%rj`1Q2RCsw@fG9R6?+JbJcr=-){ff>O8M(W1
o-?dPM2p+Na%)?+*Uq?{96akgfx^$fRqd>VVd-nJjqe+>sMR{;o>!^UxC#D;9WD>y9kXq*o&+Wbn{&(-C7@1v7?$t}rx?!Ur9dLOSr8T)MDnAIehOF}nR(O;
A+O*xRC_U$*M+yph;bdQ&$3fn|Fo!4x8F7TXny%~QXlQy?jFmg0Ytz{_7*B82{j2N<cM!r)@dy=pUW^92E@qQU)guUjW{JGwhQv^L?W)}KYybLy`f~ni-hKA
n2Bdw=goL;&Nfa|M;iI8Y43WgO&;{KhUgb80QvU5Z3S@#Uk$0)0S#lhNq8U3sJ@+-TR+kXa@G^ubZ@hbf|010!lRq|k6L4A*lQF?TWGmgO?{A{JlDvbk-wUk
jqUk*bkZWz0DBMzjbwJp6eSjbRAg=3C-6%$6S^$Pi1ICIZa(-85wHskFqHkM*Ec*&18%-=N9y2@4Y1GEB<3N$l86Vx{}5jMG2fENYNo+>A^Dg*b<Vg;US^w<
V+7L&4C0qpK7WSJ1AJUv{al13rg?V*RJVY}yeO~>;62KQ@R5qS$;#uUrkYI>QUe+#X#fL~mrIB(_k2Hc&I<U;Rcx||!)o~{RaP<>)5fJw2|hPrk79(_R|)^-
N5kz?+F~nHuMS^evLu;xh*C}zLm)NLGwv|N<5<n8A4U6Tn}}75#RgaDy$f9h^O?dAHnmbclw$G8feH7=^Vdw06-y%+QTVB^D$LIX#Cqxz*8|?|&oi<kB_XfM
3xqpqAeK&tox<fL^BM;4>RQp)b~uu>6(5h`AR1vLn+$_|GF53DRN)Ny3zXYqPNuBR7U*19Ie6s<y&@pWuGIE_TvlUB;Z`91R^ZBIY9-k<Dr9dsDj$SQBY;F7
ggUCH8hpXn{E>O3>!-Y?mo|{v#+F*93<1;d!xQ2KZi~--k>-t{a>CQqhyIHkWE$$blirYcx-!bR7=dANBjsPa(yIPcHq|xVp&kcod>ln!mRQwqs*<@c8{nEE
PRK>92{Th~bSmIWwb#BNWLTWT6ZZBT!_w`sz&T>(5tmDD=vZLL7RL<RY~d3XngJvid$}JO{9IYr8=2sv>yD5@^5^9G_a?0~{@_{`C)UK?k$W!?$rXdiVj<Mg
tOykpe6>#`x-bpAV2?V^zl7)3FRD~{)|&!4oF|h~!zqbIUko*soX?X}YEX<?$^!>d`JiwTN&P-?VSk?GBys4A0;gx%#IUZo!n_I1A!8{iOl{8$dH2P}LL(j0
pv$AQe@YoW=t{zmKGB(btb;kB51)qoR(HHH)i#${B^D^G?&00BEn}xmC_`983(X>^CxF2hovv+(iOgpEfti+uj=Jxf*$Cz}SjE}LvZ&PE7N=gy!mNc8d<b(?
>eGy<L%R>s`%;M1N{R6+`A7KpO`H|fO?hx8^`HYr$!T5(yzBK;Hm#qrfD{2L$;W3s^Fweri<(=2pK;)ODC3ZI*8#2_<*oCxm=5_STXYfrk&4%?^Dto?BXlXr
+!~tZY{_IKhx3TwvPVWuPcophGWYQIS3uZ;%W;z(v%n(x2pSAXR9d=c1W%HsnPH2H(Bee<LwKb9+MX%WHyw)dea}Xsb}axy@p|MEa&m>j>wr;h3K(4GZ^yVQ
Pp7lQm(5T_o%@5f^3+!E4K4%z{i|z@+!d|BrpbOzbXE~O+odVfmWBKu?#60N&nd`d=QE`Qd-eb&keoOW<zsEqeC!6uS1TmjV0eTg$Z@V2ombpvizp~+%_g!9
Vyn5!uyzPEKEvt7ZsZ~64mKlRUm&_-qhc0(ZssLs;vmUW4v}wrw8jgkNTwf?VY85->p?XCAxl=4&R@ivAgFwRxt3|qm5R;eJrl%XSD;^!(WIvBP<?sOdBl2L
^UOHi-VO1_5;#jIXr^^>hlFdGr!-ssOhZ{#w#qFruVsFqpWwp{qVf`9K?lQ)_`s79vbYQ#y>62H_h1lfHdVJ>EWj=_Qn~0+PAD6zdR|f-b*%Oc3f=w~0>g0b
Fer5^4el#SU}cM95!3@K_Ix%?3ruc$#;l>gh7V5>Y0hZgTZ~T$vv?qmCZq0x3Sb(b<Op!t)-2Pkyq5~gHoc3)q89v|p>(=XkzT0O&n;5kAL$~LpNami*c6Us
tl?5Fuo3I|1#YPF@=(;hkozuiCq0AtuS<kF>`(c&lmQts$ElqR<q5QvaTk`U{y`)KXgv_mantaJmTi#^YDMx({Ra(zv{-l?)D4joJzQHd;CRwoG0!mZHkPQG
un#PsH_cG@j{`YjU3(!^_K>S<6W0K84+jJ=_>?SiV}^4C9X#?9-+c>mjSW~{8uLL~f;o)8ljLScWhD%a%#wpcp&AYrU-Li4R>Icw$gh0vZo*n=u+4Ee>GTI|
PMZ)NL<Vm0mNdKAf=L}8#TZNOztZ(<`^QA-=|VF0o<oW|<WzdfZB}8ESw9Llk-$Yuo*33|rR7buqn^k5nlPMZs9;-Zk&c*NcIl!u7%Khe>F}bL#S5;&@!@&x
0&(9QyQM3XAr3hxXqIFq7BEo)WDN#^_Kj2#@Y^RvJ&%=Hf?lNKQ7D8C)@M)suXfgk4SoW|aDs~U{hR0>OW5j!EM>PhIJSO6dfLr^bM!tA7mLATK5Mrq@Xz_z
BdKasx80B;JwW6?u>vZUpBQ`F0`}sB5F*YjfH>NIj0=EU{(hyGr5&*<_~8Y_#xL{rF45(}=H`<qgFJkvnfCx;JFCaPb@cX=hKVkP+&H3B9@hgd^MoPkCqxP&
AMk^!eH$FJ72jKBU8_+R4N~wa(s9`dxxpyzf9!I<?~q`J(q$7;|7v)W<$RaqY5%O#9BIgOB$Jwb2YKOV+1!Aeg3PcDyjqsFs@F__J`f8d9BlBp2cn`{MP%qD
6JM%8fr;p?I(uT?^uq7Z?+~Pp9E6<1z>wnldU9>jk;ui^^8Z#G9XeSu!t5_x!tBf~EMB<@?THmo^?L`aTT;$MAb5+3DVAX%>^%Kj>+m#X(6cc;2li|z;9_hz
2h7Z>^f8YrkWX=ePA`xQY_3N<BCIBbLlKiE6g6d9;Iv737q&G4;aMK2QQ#aYk}~=EZy|q;q7FqGsX7dvVXW@VTX_!zq^{6<u}0+`8cQ5xOSkckhh{J>LBEM{
H5!<~j6!^GxIZRda~Ycdx<9eRPtiK@!ZLGqLf6OAW?Zrbm3kTPeW6zmg$(a2^H#|L*YuUwTSeR8Y)`stWPB>!ogL;;J?3akB!lM`@>Qi&MadDcnzRRdqax7s
mv&&Z{i-OOnoL(9RV(nC--F{HIGo29H-Jv_Q1yBC7ik^z0-YMHE{W&2nnYm#-b>oqgT)0!RoEbyjDH5;JhAP)W#ppFk{tjRGlT><2A_WgP@Mj9-RWxzW@(G1
JgP?giA|qz_xV{zbBakboDc#Nu~Uri0V|}_K`Y3kn#npKG?Q{83DE;q`TrBI7k+g|c)E?vCL9a!Hc%LD0OC10O>Nt)*7a3GlIW9UbYWJ46XqA^>t%zvx-f(B
XVwA_K8GQij=$U(O4xf~4G;UDZg7xZ8wIH#R#s7pN^qKU@=ckj-awhQhlPvLZt8kNkz(k{7QzazQ}p(^8=7x+1cJtQ6+w%TyxndQkHPnT%1^Tu#~%os_fONl
&MQvpG!Fkn+9c|^ZD&F>Rc*jwgVmwSL8w~;G8e=2#oBxY9ZT;^`tMe=s1l!&5HGq2N4Sup^o!rTT|PGE)^PQUYL6Po%=?+l(bz@vjM&T3u_1&8##P*djBUO~
uwDps&!;EriffOL6c_mytCe^ozMrt9Dz}M)HgqQQumGc#Is|AkN+h1-N%J>NKa<tOV?DK~1T&I&c>Q}X-D`<sNv!=A3P%b#G%W=c-i3Te9MjT;m^$rOp{s{?
@frlPWOlp4*n3go+QW)2G~v0Q0pvmVq6ZmDKE(Xy)p_U*8b3e5!6|WOKvKv*-NY=TvY^5Z#Ck2*kXcxxB{`2Sm9~h#p{ubvz;Bh?)C`1|hY8->hfZ$cJM=nk
&CQrj@!bNcl1tQH<%UCxHWhq3-eWlKFyA97h2to%+8UGL9^JGec0>51-lM^($C>`|@l@R`mrVUJUAfzS`7#JzRlIqwdJfxgXKxb-NqqgwU24f%pVSnv<49gG
?p3i8pM02~weiP+w2YAsFZ!$mT@tF`Q6#uYG$udWF^Gce8-cD5#nlzThuxIl96+uZT0M*4IU=J7<zWJab9VWi&qkDCGxkPuWZjtJMkRmM7U^i!j$ly|Cgvw9
_JS8Tw#bXJxa3cVe^zn+yRGf58{=C;)S@R{CB>7W`@$9PJ1AvUu((VWBi)xct&V~%6G41rkGwjS&LGn&IK<qW-XO8Zy1Na{(%{PDV-BO`fcZvWNn3nj>FWg^
&3e6Vr;jagMIsZNI&yX$=zc#~$nf}k|8jM@0}j|onP^f1wlw)+?#z|4IKW|rW%#vx3z+69JdO<~${=WMJa)vNOHAn4jZjjnMb3Qa_=j-;F6#h3Fnl`5>qR9c
xQ<qvnUq}}!>lo{W7-cszB_+qjVCUYU``oyZg@?Ojq04U?+CoVfk<%jyOzIb$-b%|d>xk>++!eH(S+T;mn}%w70ke}_sFK(Y!tY{{S66KET5C&D{!8yE~`?H
!mniw`PzEVvb)VPb3Fl-cM)j0rUMoVP~d7iTI_CTJd`FI?*1OQ7GgAM`(<asJ}+a2f?x_h<a1u9T_SS2frQ!aZ_<*4U>k3vjBcr?$-O2U89@pk486z4og&PW
$)oGZ31c|O@+)aqSwQ6o%6*CIRjg?Iw}_r{tM09b8ASK%RnE;vqH8X`5}ekSLJs-%=D)o;wVL{})%$-*`~|J+YQrKGNqLth?nC?}KDie{rgu355M#e7bW8c>
)4wTg77=>2Vwa`wcq-T4w<Ber5cnbzp?(?&#zuX*5o_Z2c{0vKdKhyXv?)L=E7v&IR+JdDlp<@D@vO{&X_Aa#a|s=mAdQ%v0~?Y4cJTr_y-BQi06#!YgQ#mX
+N9%sHF$ASx6cC6{pdbP)QF-nKTywGH7wSFR}e_n=Hlc@Plu8o2dd{9FPX#d+rPd7$Wh3zxMYxU@5S0)x8>}OG+3_dwSTLA0ATA3p}DT{EB-VqLSRnd3iAhw
h(>+({XI%rYweIde_PNiCP7%!-VXt+6LZKcMy4V^;2lY>=xOM6PHlTPI7~F=kK69ayO-T`URMh?-O<j4<%z|Mwn35_`bjR8aFo@Pl^y4;#({0|`~Q^j?D=^I
z<8|1>}WL;RzDBo3EtRwIUxQFH=)_&8lKe1<iV_*-DwhRCqK*?PvS=5uzeXLzo}UpJ?-*BayB9Af=2*g!2+w<@FUk1>Qb}795v%R<?jr@&gT<p_mk77yp6Ek
X`Y?6l4kp8J_8CZ7zHANCCSNUaNANq<c4ciGbhFoE;?&;Q*GOQnS_f=DpYWpB={>wpEXN&&aBEpCM^9ZXev6J6d+)Ay0)C_7JLa7z{bskQXT5d9t$?#t%~Zk
qEu0FBxXX;(xp7H5aJo!;I>zNu=dEHNmuV$ok0&rHy`m_^CB(5ipj6Zfu}bS?z`aiLWKh34Y)I^<N849?0%Ar^rC^U7}WpMHiU@>XVM=A>=8{H#I`(`b#HAp
>hXwZn2l6$hYv>qa-GV@4$HjGiv1ceY5sE-DkZc=1W=g{7_mfS)fp1*^PO9g541ubbyz!+?k>ra4JH`+6S&8?kPy((bx590L0a^gMWXN4W#!O4wEzS$f6$DP
^9CtCk+y>z>L$?h-I)mUchl)bAPRXe6PzD)Zp8ytWHh+eQhnE-aA75aX~b&}9mES|E{zx25J=n^9K<tZ%>_3qHiylh_S4ARq<VcboqsAyxO-nT(i@mxF$^jm
(9aOGk|GoA-4{4@hfAsbRVVXbR0b2s;ZVqSW*j&WmKj;?R$xL@#}^YYlLHI5{1J8Wpq0b*dER=s)240hn)z7iEqVPRqwIX*un55h083NI`#m_$0}L(y2<87C
H-sXjV{IQ~lJWja0vO!y=$8Ih8(lXG69ck*<IcR$I55bd!5D2oo`Ew@G7M1Uac3k-`%T*-U~w=MznNLsXn~GlHEf7YWMOzlH&scm%gBI<H8O;cJx-2Qjx@Qe
^0sy#j%$FiQlvoNY~!TAc@w&xtKAlIB!irah@wif!!KY_3~W_otG5kx8xX|)U$F33lb&Qe;&KY!H38>+>~zSFPb0*VLWQ8ia3j+2J^tYR!pOBJ=Tpz+mr~WP
%hReJBHf5eaeY-l64CW)s#2_5iOaCf3a8vBZ&G8g&9E2?n*QZYUr6f$`RAYH>&rp5;9y6zf(oNM1xrJ8>FTAW^M@e~JhOvtNrY(Z?)wbw{b3GF!UaZO#iXyN
&^L;X0moR~2+VL@umFoD@~QdM=?yvhJKgL4!1Hq<qwOKn+kQgiRj|L6_(nO(zd8FHZc3zKek#}|RpU)UP!G()E$2+Q7o&65m$vCSJDK}hj3C9jCt&_uQ}gT)
x@VCwe4cf8$V{sJwYQ12SrpGA$Lu$Wk$&q;>+qA;HmYv+B|c1QriW9RjPBNY^rLPXEGCv1)&bHGYc;)RMio{20Fm+OO#bdB9Q@C%TyF#6;-x(r<WM;;OJ_%Q
vshgklNCHYFLK;kW9PaV|3^3nanx}VOO|WqG7X$NYWYs=j`b2Urtc{O8Tm_v%ol~qxw9O9x1Gk&&V+_H;lbL`j_Pn%M!V0%nuilP42oP|wy#K8)IL5S2{f7&
z&AGj_64FlpExci_;ea@*7q!shc5RFiT$9{VeuRUoJ7cg=Sy4g4V&Z+#YrNGr)5p#W=7|qpWjMNb_TB1Pmov>ExuUws$hI`z>)v>YZUi`PvIg1{W#ql`ya8?
<&25%|8~nCYurv`O1{1`e7nHfTxDiEr|)rk4G}SGnIO$8<`$YL@1lva*WK@MCmy$Jrs&ez<Bpz3k4WF_3ryGp2*QGLpzhBGlt3#gkY7DFiAPnJNaPcVLIMU(
HH%T`ev0ub&%C6LvvD}TaSPLh<NQZ`U<^MLg+>y0+<q9%{ng0}nU;^xWm#ILd}oB#9;NZ%5TD>V!kv!r*1uFTl5|1RLRo_VHc}g-yJ)_N!M@>NKyCu-`U3;x
!WHOG0#mMA)1@rb6G@-M!(#K53%kckJ5dNh8D6)%*2j_dgE6QX_!WiqGjS1GRI<JCqu>VuUVIO`ITGmwn!W}nh^m!3pF|{GfW)S^Z?<@inQQG2fBFB#+L#_e
(3J$vQ;<h!R(F!JD;!>NLnM$IV74SgY&=lx`Zi@9D!)`gdKF%mlzn=OHsf_f6Mw<nEuQXJHo+fM0vymTW_c`;(c`mcBN650;Lo^B1V7XCNi@}(le{QlF@N#=
YvOilYt9#1h6+#>s7bq_NVLVhqnR@e<PT_5X0D&mra>~rP+tYlr4X@O;=7Qi^l+><LM`W-yK+c-d}`4iESUgLDg}FveE2EbW+S`PGTKPbS9c!Ps=HZ4T})fl
vr<=tDRPAdFJoC<Gk8{a`>#aODm-8g=HATbkHzyT#+T7M%W#SPuO>rwv0t8l*;!~gt5?3`L|`;8Z}|{IMszbiaF6qeS5?6Dbd#d-GU2V93wN4ER*={E5L5H@
W!?s#7v|5G5Gm0-L$+3}N+GY21#$vDf}RO|r_lCBknm4MOh?esLYXwtw%Z^=E6d^B)|u;Lf|C?zUH==y!>9TJD2Q5>z|836d5Y99F!r3}wo>zq?~SjV|5<3x
AIkD~k9b<+n)vL721N-2#9@6=Yq40a1A(<H2Z3Z&{e5FwQdhW1_T7*zH?t3#V#OuEbBvd2hl(5_si|?kL)cX;?^<Ux5Yqm^5;9CCcx8MOl7HpMR$*LM1<?8A
9u&q$Wg$X_A1C5V!qXS|BV_nZvi={2k?JT#DUnT~Ib)B>jEhWa;(Lx+K!qX~0Em+h_%?Nq)w&4hCmt2~h;97z9m^{JWXO63_i><d0$I(oK|N*pj&&f@{2!~w
nauNaaMo|1D7W8olsclZXMd<**ADQ)(qGYo&YG-Tv`7D?)EGFG1!#4gb7TJ;PMe#$#kMeC2=sZ!b0)K_=lg>;@T<<hcyF0}Kjb2XRi`(Z=MXtlp27h$#T@S(
WRI~k+jB_1Ot||TK(P9{^Tv0*2x49V?nQ@rqI^NO(o3)|+>?@6(&T+3@Ie{)w2)4x{kSmv#A`_zJyf3kUltVT#II=KX(*3}>b)`StR_KUwZRNctUA9Mfgghh
1P$RV1}B!{OdC`)WMF*QHF2Cw$02ywvE1|&=SHPQfl%+*=C6^(SwzRbBA%&0oDgu+ag;*r%TmW5zC2YL+E{Vfcv(2E<x^?A_^Vi8=?NF%LZATb`o9G65vz3O
?FvMD%t~d;;fzEQXMmyh;UsqXlAv&z$Wi!-^Mq&D`dqWe7dWr#ENaYQfVU6YmGv2%PUkne!FEsdCabIwgHGX}OTs<1<aSsU4J=~3Mkuj-S|yC6=P6Es{SpZ;
74|>aX08%4ssE}eah_~PMq|_yc4i9JE-Y;MW%)_rqntN3aza9mA7%3VX~T;hGox9F6KGjoJw?G|{<vf1K@492r0%$;LSZw~jbo@rWON=DfLkx^R$^|l=m|RC
dQr-Ey0|H&G`NK%HP5n7N!P&MTVewq@sU857Z@sXOhq@|fYJ;E*?~c+tvhyZFUq9q!(D4m+mBo1o!oYccOd({f>d~x%0uG)gn$`h%YgC-{HfJ{Toxo!*PuWf
Y2O-Zckg@<Qbb2OQW*iZyT7iL?D2Sc_9(<FELmkffGJB=4NS|lHsCP9GPi4={JF&VP`z7l00q&Vw9<ulO6ftJ)_kR~#3jj!BsR($>>?<zLu)!V5qBwy-^U&N
l=2D~sT?ew9AdIF^ZkVdny}YL03G(5>2)raKA1rueZ~}c@XiF*bxdfnNVDMe-J~>ly|+rcHRiH=lGW}0P=Ko}8^9Qd4o^B;JwPjZ6rj_p7XYJka-SahpK&jV
d8`>6HWPizp2niu%Ti7mj?IAa!;5GfNHM(<h}I-G#)NRfe$`Bf95u4xFz_)t&PJ$@&b6$W6@*9KRyX|I^}<DiXa&6d)Cwtco6UZc3x+S(41637FQw?qR*SE%
yBMsv#jX!n**3qtIExeM_2=hcI~Z-rCvFd{^le@v|FW6F3*rhYhr8Llhn>+VtTrI#5Tu%R3`zET-hBh5!S1H6l#ec(#*XrGcrVp{xM}yd2Z60Q`vP!^y|^_z
y^X+47czLFtS5J!y9gMuGQpePrUl${3PjG94+cy%V6+%_Q{7by2CERn!~HT*t*r1F>P>AZEjHFE=3bgD_|xf65?xiN029Ak1jprDhKXZOyOiy$jiDp*2|0pQ
YpBfc2^(gIdy;ECcYs1uG<I@f&Fz<@x+NEvFj$Gv{z0ZBA#9whj$78P%?&5ggUL*^s&69Vh9S!(%+HbVkjTPyrfKMlE$=oRB=Eqa0FN{>D*M=6i(U;9_184S
!n*k(L#k)|(GbN`nDx;QK8rrg3mIx6T!1;90+Aaty7mwWO0MBENMA+i#-=@+$DQ<$Li!*uNEu}bmjWTuF!YO~!e8k$8+vlCa6{z(M8{NMij;&}r+h9B@6~A^
Yci1k)GYj7YVzFt#~xAFySZgRJ%UK|8%5VJ_+#?3flg~s{8iYYE(<m2z|2c1q+saZ)kqD)iHYYL^sM>~;b1OV;<eRW=H23bw!7t%Gn{+>(XQV#5qMuM^o2p`
1TK;?rI$|KpRu-wA9^o7&OrlcosBs<tu&U>S>qJ6=e1UYp3{#-k5m?`EmHk*Jb|ZIHN7|kH7@xIL4f^p+7Z))T9{k)Ef#+7_}l5DoM&H0Eo(ln@kZRmSWtN}
X{kF2zDd38F4~NH`Tdd0y*hI_vRRE??p{<J{x}0yek4$@YAzx)EF^xKr0FtC3~I$Zea6X-x*BdUSV)B7aK&K?WmR!4^<s&eG3Yg53fO(~-!Y9X$`BarYriP2
=aY)a^(mmCzJ|2BvrHf{Agoio-dD7lTko^gagS)gCT_GLpbZ2M<(oN>odTqK=6aL=0pPPY_TZriNyi5w%lzKpDt`1K4tC3|oA(_jId|#eU<F3OMtU48JzOmm
oZH^AA*vcx{j+FKBF6;v82m$~vsz|FiU`)?FY!d>Nnkt;ZvN6pdUu*gYL48}>2@X3vC~2DW%cM`7$^NcmbG0fsABS9D~e7x53JmwK!o>~v+!O%19&~Jn~6tQ
?-Vl3^|3WnwLmn^tnXJB1%XYgZ-jenFb-eeR3#tOU(g|*NO6(CDSTc2AE!%ALWNqLAC_IufFFQm;*)lz-B#N!j34|Z{1}_9wBvoeUrOk|c;8e*;V@iGXd13$
#MPOLC`{eFIWM#dBC}6aAs;&Wwu3=*eCV*VN8N!4=%Z2%r;_x}E`U?2;$D$znk6!r0yNm<>6N0rOv@eBcKMg#8&Z%vRZ~oc33s7rzQnJnJ=>H**Cpy$Mrw0+
`d1s+cL;LNZK4HBhhkB&nR~dQOl@&_tB7#02_;F~cH#vewcarF?!4jT=3vSO#6#2tFE8fG0ZUxe@N)H|x%}_eVrztRS`E3hDjEQr*-@UiV+roTA$BgwrO3P}
?vXv}n6fWeJDi8kg6s+~=x=>ufzM^E6Xq?|F?nZ}F=}OZF(kfJ_rD+=zl^5}z0tkXq^@J>t%}NID!r+FDC{N)Okb1>vARG2vjfJIsCFNs8!)&RlhtH3hZ)mq
B6~Z@N6(>xJZ6Z5Af1q(+W){v4QB<oj=~Xb7~P9h+<2%XimGPUYS5{Hxzi%Ui2c;|_NEj#Eh<4XR&)B~QLiK)4FC9%-g=-ZM{>RtjMo=tUK8ubF)Xz`M>p_p
3|Qa&e>g$OZSg~=^+tk7J~j@Mq6PyB{%2*NmR6p<WYz0r*pe$EFuvc4Xe)2^9>U4OODR&rnxK~18-$<Rcpw4^>!4KquWSLKOO&SR;ou&o$na>5Y2@<f6N2<m
@4so=6VFLH!B|VOE{0}XJefB??hZ-Nc806!!0scMD8F&IE=+T3GyD<&6bkh%wIY7Y2W~hN;I|Bm@Q9lfv#lSpvY#=AY+|k_S&yjFeV%gt%Mj0QzO=;svoa8K
;N1xHv98hsl(EQAPYWg0nz$oLknyibKN^d}#>1@wiE`rpuCxk&LsQ}#z#2^K^!|ACfPZA>1n1F|%$^#0@SHz+0=5pt|IKj|80`Q24XW2Ao;;7eba|w6e3pQW
tTPxc<JZF`yw$pqukewyqSclJWqAPA=<dGq97fo|Gr0JfKwl^A;>(HcSK&*@4xxs9gp@$NZjFX}Nqf6U$vT)Oy4Z2h$RMM|(EbrPbQf3#jBYG&=Av=)hsk@`
cnrf6FHxPvL@g&Vu}n{Ti6}M@h<pj|IgM8H2y56D;AdpV1vLT>3P|m!R(pX@N2#Po$YkV$D-bleUk;@vHf6N?hO7=x{vKzUXI4rbb>LRBAx#lO*f9d#3IN-8
*XcefW?*oVB8j8$rc9*C(98BOY&2D3(It$Rf<UPs9{+zq9^xpt&O6@Y5=#zokeyPnXq{Om4+)ZGOxA*gWVe`}HdjrQotD8o&lS+AGF*&%)|j-u`S@=^Qfj9-
Dsa|<3m)y@c&n#aSGRK|$kZH`c*1@4Pu*7}(XL}gg5`x#lD&}eArS=S)}H?Ai@jjv&Ec~5veI2AVZr+uY;FvvgbeNY;Tp)v4>+@|mgXK2naqwy-7t<EdKO~v
yCb=h3#`yq5p`F)dQ!Ni^O`bffwscO7X=z5r)sDCYODJJ+%cyhAz?=)&cW4DUQ=)vteMbrEzhCm=FvV(!eIWLik<G(Uwq!A<!wUZ9xkn8lHH%)Y_RtF!a^`x
K@#u^@tI_ry6UslNi-$ll$A06txD^={~#P$IPE~8+p&a#?oRE5VejLLPsP=+ID=K(UE%nwMeHF*R-C55D1YNgV#G-^PAi^Q{)JB2Gp)^7FU4;#+w=28O7>+M
P;QlGZd$I5R7hph(5mgQAwl%nr4!}I@1q<(b;wJSp>ml<nU{Sm2WEwYnP#v+(2rM4Y~T+NMLj?QpHx2pvIN3e^G2q#;0#TD-Af74cdPudO2h<2ne;$MQf=p$
1RLnRqnn6>+88-;HL%z#_%FRi`5E$tQ=P7C>RtT`-q-Jy!2~2vj6R|$3B~ix%LDUWE1Ell5x&HMNz91K=1J_&eB*XT<}!p1v)=xXO>c;Ts-YaK-|lp-V37nT
lkM#mlLv%Q^l_bm0_5i-LWLC0JWFi3;6Vk<=IeG)YCXi%`t9)aN^?WOJ*cGku2y$ztO2=YQ2k;((zPXH5h|7uEBEPk2{2Hwa-N{ku4Jm#$8*#S-2PC0vE?}J
NdbFddyQ19Biw1lWC4X)MxaO)-5#w~Q4L&da72(#WVn0foO!hmNVbGK%Igpy-Xg0nGEfa+^@28xu3M%!8u@m}NRs1T!312mb8Bm$UyYF`4hdIQQaOcY(#sqs
2m8aq7bne8lMOA1$)!5u1}i93hi=M#OJD0w^v%O4@}U8Z6}FK<cNtjt>>tx2Ebj9%ajU*T(&Q`NQ?qV3zVNM2C+ql=pvP14y)vTecpxqJME|-Q@<9?=W?`6{
BFIUM!c?3w$V_BBNgMf}^Tj5RN>&gHN?Aa>4K?2I6rw(-J6^%O^?*B#ux@6hnjTXpi@%TMjt)vV+vIExK*LEg@WZGpmS<ibSr530he(@xd#3vIAZhOo(=8gC
&C0y#xRXwL5*(lr6_2<5bryb29kgX*&N-csAHRS-j(I&d^7&}A#uK%fIc2T5a{XijpI}}oR&X*)FEzT|0>Ywc#ftqJP}}I09R6eZY)#}|o`R9U0a6jc!sra)
jOZ!*FBRL9EnB9SotSfbGNVnd1+>^alG_vlUP7b%QxB(5FJb|3q>1{cN0y!S1Q(YeWO=L`dGSa01}3Ye<qr%~1a2mgd6nIG=ztuqKk|2I{#i1I_B8<qHLwYV
qXT)aPfxEWCFHuq_=w~vLMIxOhny1`M!!-Yn^CMETwUXyhDfJuhGcMW*>)j$Vt;JS5&#@yrFpW9%{L;i6@v>MU8~<Luf=2?*Ax+sbs+JOmXU6>iVh-olBy13
Ts;cr_vv9rAsLjg4ui05a#Do-<QP&2tDEF9zp@;y;H5Dvo!ER55Wy{#8F7VWKTl6LBzdb^=Y|SLQ-u$WF6#otC!=R?u?93$ANSMM1y~}?S*8tR6fpJk?kaNC
?1;aONmgn}XgloVr8;Z)u+A$8P;)Vb!XTdbgV;2FfLF0wb}FERJ}xfW_AX!UbFSj>lpwCzJ*U^H>!*9;|Juv^nT{RON-Qx+4+^%Y{y+ZlDWR#RWm1+!obd|E
gF3mkEQxBw-F1^=-PHpj=WakY=N9eBhR<w%jVL{dKX)8x2_gmN%8?r@raPsd{InHE*Fk`I($+eh7zq%qurTeEZN8022aVzW)YQ?ge;#Y4Nx<F66CYx!F!F=6
S@KJ`0l2@QWk4sV)e3^}WpHZ6wx=p<we|fAMor80=F~!@@j!^uZMfimFZ2$vxAPf=^^sbArN_`M4Z43VMc;$437cQ;{pCsit+%cR@Q)%@IrgS{PvfA;FvgX_
8|(s~_24QY<a1kZnY}rMY>-?R{0jwEu}&hMDdvw6Rpi?f57i8cS^BWm;`MkslqRJJcD2gVGSF3~_A>swOvf<jIoZG#?8*42PrZ$gqSJs-`oS5omtK98!wR`$
roy~9OmcCSN9%yeTZs@N>5cK|aqpQ!h!)X+pijTzKmt!#4JvaR{um81;VGQ^aRM>iR?qufsx3F@SIACJk{5^LPL4zP?L>hiVWW&(5p+lxN@ZZ_-6?ErUK5f_
0rhRbA!8D8f8-Ys-bissu|O$#$jjS~;beR0wBr5)J9j|u_vV<SWKb~$WaBGF%Nn`&|Gr(P-?xBKOYGf}C6NBtn^@@%dw#eEBRp5{ZEFzJ*!?PVsB+)TC*M`L
c&#d6b*EWm&MgFdjk<P?a)+6m0p-ZLoHdpNo}VgbA!5m+=-SU|63?8U0R{N@7iK9~98d!XLEK^0p5-Sds+J<bBmNdlS%lXnu0%vJC_*`9bBSOXLcA)Tg?kfv
N^#$B<|q|vmD7Fd{PV3u*KjcA4aQ7Xls^ws0WMd)oj|z!GTiPN9EFdFNem1c#=kNv{q5dW=ux1-M!!JYG}6!590p1SFO5<_;51ZhG~lA{@;J)XhrB952!)N;
xHIYCTsp7)ajQWkv^veF`^TuB&b|aB*CoW?aNjD5E?U_LO4soewB<NU8V%+)7u*nMnIECQIIR=yh@_;@4fJ=AEL1}|wC+;T?}n4b{4CMFs?Je?DdH~p0IH)&
g5yb-enMq9bSlG`1s85q4xJisq~ibp4YyCYnx3B(00H#y1jw`$;V{pSvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/treino_gpu.py')

In [ ]:
# listas/caixas_profundidade_n5k.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/caixas_profundidade_n5k.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;TjPsK3xDbnE{3$#X++%RjhR73VIPD%bn4W{@$E7&2DlZDkoJN*}-)|=DtF5h52;7TQNp5sG4~_tg)b$*hb_>k9iVt
WI`5799U6>sd7j*Li5R0m_u(-D>1(X?(_T=v$}1`=`NPsXSge$J?YJeSo6A@*KvV7;>|WLLqf{<IZ31(WiLXCEee<)5WJcZvPeK-{MU%(e?q8{4WP@3t2i|U
c7{uW21b6#2F@LY?UuzkuR^wXphe%gr*$eDv9`%k`7!IpAYNP(Q7t4G0_G8=NUKYH5B^c0NC91EITciJTmgXet*V^83aMY{vOJ<EP(=m}L?w5Qs8B>q)(whX
i2l7Nm!t#cDk<5`K%iAWvjM|6B2!)-f}(yoYVpPoDZD$hwjCV>NbUDn8JAv;S(G+u>tUbGU)C%big$_O0DaH_!gQ#jx~a#HseQ;<TX3n7utQ9Zw00cFtV-S{
J3j2$Z&Wo)*7t<s^Y7(U9|$XF$2&1=4b2iw>yZ9JoBq;=^OQjQT`lg6BtBy5AmA%ufdWU2gx2*lu;=`V2n|&)zi-`k=W01uOBO~;g8m3Z!h2z|%f|R48<nm<
16BwaR_g)^E3VTWrOPfwk&&Xs_n3%vDc1uEqw5TxB(%Wh^xwMy9N}|)e$gN-c|kI?RXWP-=0+ZlVeW3q{tc^AW&XLm*T~?J+h%Y`a%w<+<|I_(4lV|2f<%%;
Eln}p54g})u(0EL!&`ep%FUvq^BW=cvyqAAj5H09+qP9<9+3nh4PLp+*c4A&F|-ZBbynDE10W9*0*aFknPt2q{A;MJ_*Dx~+(-HimuToGn^D}X4iYXqgf2YC
nig>M4Qk*3$3#v+emH>2Xo8Gw<6fqF1^P#BK6GIxV=AeC8OccKWknGzZ(1&8iGPvT7^O~@21YuaT8qho9o@Rw?of*P_?<WKi7E+}Yyx)Y`97s9Gmb51PEVFG
D=`iZ1vEDsL(skiN767$)VzQ&*3G}Lvk*C<UKD$MX`C9iyH*U^&b@RdiQ<Gk6%^YtbS8DQIIeMLvx?X6UxedUcACVe%xR0A)X;~?xCuno*IC33y5`2rd7Z!U
z#CCO^{iQ%NX|B~niC6_#Y#fu$;vB9D4QkBj7-W<Z&tJ;gB>gS5KUt2zBT(Ufdm@C`;V&YHVRFUyBZh=i0_EwEaU!}L**V}e^mXPx`10LN!s+DZ9sJ^;(j79
`NiCl3;$-rrbp+$zx{|E$g=$c{BT6xf!x#-YHhL7K``CIg{)bUG$dUcF8^1+YeZ=kpCC?!V_YGy8?q3_w+jvIFJjb`2Q8GF^hrFnc3Gn5k@%+kxuTa1e;h3O
1(dby1MD(EKUB+YhVX04udD<y>S~h*oV0Xwr72K))J6?^Ovm>WlZ1LubzC9X$4)n1MiUEy5G|@&$3aq{wCcBN{D4>i`4Vuwa;tkyN7xHR`gr2qKk8Oi36_<y
Mv(s;)ts#eN{Yd3+yaYSD@WXMCDk=^qZnOr_dXC}p1$~8fYDE8ZpJ>r`=W_7sYcaIcZC1`6>_qdA{ppCfk|Uub{s&s)sVF-K=7471blgAb|7*zvZV8X$2h0g
PI*aaxZn%8BttUMp{o{pUVP*ZJS&?MX(ZcYnqnLYJ7Hk93}I2>+JMo)JJk~YPQ$|@Eek%cNy?KvtT&X$cINqmZrzEZnm@f}wB}s_yd>AEusSSXLjj|s1OsHc
4%wrQKS_f_Co1OPwWkdbVy*=nI6&4EzGU?*r4Z#3vdQWLyT|GY(&`%mt_s78xXb&rUGLC2VWn(_(`lDEFL}m+2_XA<Wxh8IX>P4c^E|VgBdu)XRF|Nv+!dj4
UhfB=^IJ97wNft)p9kxOZut?j#gfWT+i}8Swhs1kQ(rIr{)vv}(Bk?=9{yI2Qq~@hLjnEO(Q1{FU*baRj~vd`get7Tsx{PJTZl9jrU3i#fVp2OAEBF?cJ<nZ
AWco;UAB6dOnrBf%a~kjeGMJEY9dJ?A@=g*d3Q`aFo}Q&MBqjX5sym^pp9RWT{ma!wq@XuEX-Mt$~2q)j4PIM&eqcwcM<HuUDhSS?CO(X##0+Rn<BhyV7@t(
IFh?cFQ3aKnbkKo9D&}|QQ>Yth*;d-I?ioSiW5`yVcOSA(kPXjhhS22KRyN$j0bBD&W^_fXI9#Uur`f|lBwmV<Je=CenWM6<w-)D>@)~oHLejChWQzQ&@3e2
(zh`${Nm}bo`SkmEy7nQi|dNV`BYB*a>nimKJKRPC2QO_DDzAR+l>}izU<YNrdp}!6c3+hHjt}&QUbH*EQ!f$<Z?AX22><~w!Z4&vKh}S&~QY>dXJ7%`@wR3
JuUIGT$oJ8!^?6h#8`SrcFnl&e#yIJU08G4M%9hnR8+~lufm&8@b^ba>rf9a2P4K1EfmI-y~#rKEqe51!U(UFt%bl7?VK7o;!V)V!hOeXMN`4Y-*@~e*Avev
f8{O$-Ld0pNcURzsYfd~#&+-<Oep59c>dN|t)YDlVDO|=AG2%PL#2Ob%d(BaFw2aU8v@X!k5Y+^T^{BDJQSU_`7va?SI3pQHDEw|=MxEviHXO}-jjzWCIVZS
L(0B!jcHEQ%(|ma4KzT0bn2AcvkNt<x{|UP9y5bhmH@Hxt8uy!{po>_rAN^89Bayz1u!?AY?E|kG8eKkz`#ihIzfo;$#J)u+@I~CDSra=0~=l=^~EMup4Ax1
R%tZ1&1WBk^Lb81Nm_a};k725OcKM6u}=E8bELp3QQECkfM&sj$PgtZmMs&$kB0-hvkbH{KC{2Znb}9Mo+R3h^m=)zW;TF5nIChRW79HXsjvzj<xym&YdWK%
QpjkTq_G5SxBn6;k#B;jQC(kL%w#Z3f^Q&baa>ZemhKATlarojr+R~a0jb)Kt`-{`$0jBKBlg|6O=vu+z|(5gs)!|#kY_bsqIw&jRgy<KOUccBPxh=jU@~$8
&?Wr|Srt_i0BUfPr&Y{=Nx2}nL}OV%rqFB<Su_k9Ig#TxaRkN|QcD~?5ej;klEi^F3h;U~VxVpzkHH&TfI`;oNhM14*S>O;*lhY}(jov*Y_#Z53Kol%ZikC$
WQvuVS(QrGy{$X&r)?l)1yndS#gm*-8U(+M+=XrUFE+0()fcAnFk_jcKeK5M*Uf`Kv4=RmTa@CporlCrIF09&qwvE&`Ak|zgLg5)#wl7Ha$NrY@Be!s<z}fE
`s{%wuJP%eHvmuj)_1om14xrgmUqqt>b}#+B6O%$OcYJpxS0hSabGyhoWjx{jA34?04I{9^Uvfs6hr@gbl0~>H>M0bBIzaU-|W_*qzZN2HnjMy&&*P}L9}Dt
zg3>53$4VvJBKyiGoDW7c!;ZaJ5(%TKML)$-!h5a4@m3Bc|styRIiO;f)?UnnaQ$aWv%Kpjbz7yFEb}tcvjdrv=6V{`lZY;(*-{BBfA5A*_o4Q5ko414I0=G
_vb6CI{aQrdst|tI)9?1K7!AN!TCR7Pcyqe@UM2qLjg}G&0G9lax9@VZ82+@VTR%n_SU`@rl*TRo;|S+dmWY>>*wQnu`u9Iem7gtH?DT*3gI$aw|xW6V=BU$
Ve6f%PXFd3>7|els#Pr;v9^6_3D{7F)Y@A!$HE`m{^|{eYcrhuX{*=j_+7S^ZKh&U5;w=XhhSpQq8v)lj=9h_2an3-l|JfkZ%lq7TPFN@!CAROD#UsBtf)1w
#GydChS8>*(Js3$;3T;ffz4~zAk!Bk^XSCdy0S~=>y3N*FQ^TkbX%p{Ky!e%9xU;oUAVB`EN8k9a#VLCTNe8!N2S=)8ydaa@Cdhr;<fnoiqsSxlN6RT1ZJ;|
xPS-9=<fnAQgCm-N;9y@*&oDgoQn(bu6`(a;Vq1HW!l059_>|{-j2X$qhU;z4;+ZIF5rpdVwT+eRbQwp<*2I$%u1QJ#gV~-;u`CGnl^yd?d*qf?B#Q@)QI8i
mLB*u+&#uf7d2@nh_@6?rM#w^aK6Lk=--Wgc|4oeTZK;IobQ%qlhjW7=8>il7?|+9sE6!5QI8qfQ*=m)_u>zEVnvA!3@XhG9&qd`ME%)R-5c;phnRRuxSRW;
#iW49{l7oppUYhH=ojkj)hig72*aC;Idsg{jj6r_+es%$m4p?Pnfl?`;O4PvOnxic`Y}Uo3sa#Sc%g89k{{$ED_4S2J*Db*E0(rG7}8`V)`1f1d)=K7Xa|mV
>-U(HTTqd>2>FH-WOP$lWFIw@`7CHejQlwFN$iAd{C+*$jy&)5DX|T5_?>3_#b=rDS^_ED946Uju@Xt(I7H%P`%JS~mlxQ2HMa>Av%pamt5I^M=z>0|r@Nug
*uzM-7o<_^{XMi|U-oh_#K_CB+5=hX@p<seJ=GI&R6}IY5kD~|mUYGIy9sR-)&v=sWr)!Q{$hV@7RE7e2u-nqY{~NuR87hVQ-#tEc<bpMBZ>saJwKPUK^i6R
?{%DlG>92A&`S*@Ag_Iv!3*y6ZwoSc{)i7Os8~k3DDRH%5BQ4ERW2w(7zvdEXw5;{WV_6g^Re#`ZBZ}^H<U@jP+uNde_;qN-<}aA6@WGNrTfODcVNz7WayeB
9B8&pyaYRF71rfmptb|jHft^o*5<P%Xl7?;$50@;a0-e?T)>jlm@0IDiy{gA4(CoXGXZvgK~^iNFBub}K|#E(wvsm`>cv!V;-gT`7Z|FH0r9tj*D2HH(v8nM
7=$1Z6O=t0cq@jqCA0$cjhK>__`_sy1pbX_em#?sJQ9G1?DKd~<^@7AYEvA>=$XV`76shvP*zZ0G8bN0x+lZGbP2zrPJR(C8*Q{t5)+OA@$^#6UPzx&I4OAF
eC}8B^%#(*14<@51G%y-(390+DBwx@rWTuE22W6*^_<?iwQ31anRCq{ca<@<xua}(r-VN@7{V`(tk-p!3JNGlac2BY7n0>~y=UL~MzS;zzm3wueoj;&lSAou
$1%sJQQ#(Az`F`K=+)6`qWX!7gQ>TQ$v{zFw-O{_JJscYTX$2{4yBSIJS*Ok-3YWw45Q_L32D@})C4rYXBpX&XQG^JAXL0!kb;pcMikfFxqBj`5_u;r12@>v
AKx`mXtnHP2%W0MM-*2lML~WiEq<D*7ycGx>~!#%a?7?MXM(!`xLMjM!log@Kk{a{RC=l!gukR1lH9C`@TB#@d0X7NxnCfUE{O$>AiK457`-qo;i~J4BK?|F
cR+vNh;+(o2jhy}ESFVDAW~q;x?-bqGodz)bm%S`TZ~#}*Db#KFdl4Cw)u^UC|GBcg-r8%b!$mOgKMtDWziTYOKNeIB5O@~!@A6L#B&i@fksbA4^ztSG9I+D
g$Y9m))-_^7+s53nf9V)mSnS3?#)8FQwj<xmU}Y^`SzL*NZr6U1iz}iHA8j4aVK`*RfD_TB_61ncC~KQW5~b&N;Vk-%YBLtJt#>DpNAj17tbqN42^NlHE2c1
^yOIq%SPX8>xOu`3r?^9*E_>&z5R{W#E$l>ES7Se|A=seE%*`w`ldp|349J4Mmj@95C3xl83I4X`+xTKWRGKNi`EVQ$;d!kAz0gr_H1`uiZ)Z|AjwUM#VC@b
RmA{6Ry|6S^Vj$DF6Z^<5-i+Mbtk{6B!(|=QnEUlA=WG62X#%1x+y{7;fk|&cb3)tPD~4Hv^yZhg;&;vMfO;98j8Mq|LS6KY>mxy@ZY?*Umy7!$NUoeflZ;i
cjeB=J=88ZgqBBDk$t$LdIV#ay8HrRF{c<M>wVJ{8U>;`eq<0nTU>m5!~ouZNne5Es&ss}TcK%yL!N8@t7adc$%>5}{N{VUdbQ+n8LR(0txWvIySJs8Th2Fk
U56q{4sxq0B`AFUWT<tpJ!~!XtkV(RzC`pf6Bk&X+5^1i#EDAE1$?CXKtxq%NaIhw*<v;joa%vxNRV+-<nGLDMdcw@WyyjFMroB*A!vW~tkJ2`$HTYMDGKGF
w|{}VjlQ_kdSurgQtUhS)dOblEN>e?l^INpx#)G9mc-aIV0=qQTEmpVy*$G5gFB^T30|hmXP?|fQJl#;q^*}HE2+UA*e{B|*~}l`Fn~Z)?`T|(cDvJ04?4o0
V}*x2Ed*(Q6)Ou8Q&YG(3HJo+3Z6lVUhoR*5;0I_BUR>})yT>QZ<s13VW7I&95%{0{sajqa}z0uuQBM1A(Gu(B$@o|S%csVly5qF+);vII$Q1+l}tM7LB>HY
cLV|r|1Hl*ss%*8%kBtw$+5V}xmS~c%<GxK^pJ&6A4qYQrlcGMYDZc}F&)?30X`)nbtzCv;y4AZM578Prc_gj*VD8<VF6>A{I=e9jP+iA>o*IbWI5Hk$v+WK
;}=mRfK33<^e;=4bp4{#dSD5(?DCh#k~W#723lC@jo=vD27d0}<=(n~KjLly?2*u;IOR?uekNyFa<1sgNd=`g45WBZt%lZYKDAuN?vTdFd<mIqL7+m(U@clZ
OTB}cUp_=6H2V=vmWW~K2~sK7^lv$tPYW?^I)QZ8Vt7Oa@t;saA1gJ$&9@75drzBEc_8R${3ie15b1>7pEA2%@#a+GHfLi$;gN^`vr>MMK~=_?UF9sojz*m8
#4@h~*M3%k6B_ZMc8g{***9|T9O>iTU{hC+f*B%q^c@wp3s~QN*xxy)(BsD^;6}OnZKQS7sdBnE?7{0iC?y1KEfFOJDu??}qoG6gQ8AY+#(!BE77}XDEV_1t
#nl%Nsf!jn3=*$n%H^sD&xr_uZ|Qid=&xppsxT%GR@|nQOqp8pyvd3N@P}oYWYZmhl&Ly<<z^))(jtA303Ih?EE&KHz(cy~c#6nxtZ}u&VqYM5bUdWpxPt5J
<#yraGL&vxL{o#eBtuzpk|>|%^V*>pq>UX%r=~XTCedFbt#?g{Xr1suX^Q>&G(o@h-b-z<>Rq|{r}DmEof|syc==NAd!NN~iVicwzC1cH2@K_hWVOdw!udbg
V(uFJonVS#E+ZlL-#x!gstEtHzl6U8@*WVnq({u@-GGJKY5Df$NLvtvu}Lyk8Gz*eYP-+TQ_>Pf2T81*iTh5N-Z6fQmdIddieM77E?J{7%$;hXU$`|`ok|uV
5^m`mzP+D}Ecoq(oG0`UD-03=&0+O1dLXKYkUOORHrn>Wz0a>K)n(+)H|K5sf2|76C(7y@g98%y==Yl<D&v+Fd8Z7770{c@;BJdj-X+;-%>LqB1{5p!sbkm}
slOX-PD_qv5JrWk=8o9uCOMU=L&1_p)n63e^fY@T_ahhM)J5HMoEDShxkknPd_(5bsFoTzAKy^#R7aPB8P;t6q|EF*45}ah9_^J&690n^<68vHHtkTpONG>4
8HZNW?SW+>YAh4-77O~>*7^cwxg{`@IYQ^U69y%NwYSn`MZxNv{ZK=J5rPXiFhFxtF4N5|fXzWV$`^+_LUhs^MM0?Ll=m^97Jfm&97kb;+Z<#HY~h$L8M|Ak
oXPa4qm9jC1B&{oVYN?jQ#K-I(Y@g@+WXt2%B5(VZ|yR=|FaYiVq1e~H)lYQKy9;~QnC|up8F=P$cbGyw-M)kq(z>80d&7YgY!UcZh<?xv5=g?1wl&H0gchO
Z_Pkj3+)*veR}EPu;8ChHuV$AxmFv%o8HAN>}`8PDa{PS5vSq|2#Ypc6mNnYeOqcOe?uYSxkBLUh<p>#g~tjYn4s4Pi#ec!f!DVZ<fMt>3<dy8i@$r)7{9oQ
9a&nRP=p<`;D2Prq&cpcw5qT>d^=H#xn1J8be(#c$yVw4<Te>MKO;pxI$(iML1?ERc-(q@Jc?wG`1J-9j(4+FIK9NQl5Mb&K_J*5veq26_>bB7qPEH46jrXl
S5C3Uw|A-_b|uW4jvG}Y(m>WBqSm|SAv>uKnEsq!?e|Y7h1%JH7XW-F(LAtpzWi$cvHGD_FmlpUzTg33x`B}Dd;oHMN$h3yl4bG4!+#r{sTb+uz4(It63KrF
1YU&kXGEH=+ZiH=-;X28%A;52TF-RkgcC0`<ZdQIUi$fabjL~7585b;zk}JnVc(CBEPi_5gKuWcBtHaON^Y6-)Gke9P;=ZRQ@$$NFv6h9V=j>DD_m%kJGx8R
jg4N|g2|+<%?(+C;KP)&tcIVTCuI1`fMu7J0KR3{0?V$=ogYknV#+v2FPyT!X?<e|>KI0`1_UTT&@T2nf^Vs{=2Yik{(0G_e%lH~%vN@XnZP@e+#KPKyH`?)
F*^;MAEtolcn)3pqf<bbVanQOK6))As(r6Su<;^#Sa<o?2@vpfN=fiq$pT;|bm$k0%l}{<<0ni@?i1iIN0AkVaG3A@e#&25o@byHbZv%M=ld*EZ0(w!q|y~&
mf2JSCzE309_Mr?OzdHI^_+Z>>`fXqv{iF0s#rCnM~T^mFpze?+oa>?CzY~Re~J7ox|9jB5IGPSZ9d%dYmkbgt)l9+YDRIcf}P6w)ZH|$<5#%JiCnblfV2bS
8`7>%1K|Djn~mC8o-g(uf$tuV3Mk}##B6b2f@4!1=rZp1PoKE?)V<+b6nyg!Gr^|LP`!fo@b_LmQIEkCAnRsdZg33=^W&@aa?L|z>L}L3Yh;l!>9c=_?brSn
2e()O7gy{Q!{4!gGSNv!?!y5>vdv)%d1g2OL`^=Jma@IHAzR!8`-v#5NtKNveO=jsh|?{>m!gI+oq+L3>k8aJ{$v~kTUiZh=cs@tMuNlB+{$jQcz<bR7|kiw
29Xm{&K0RE_i;$vZtKvPLVQ0Bi>{__GK9PY{T0VGH#ySj-`c`cXA1KU^_ranL06d62vu8n(3`FhRnTwvk^(q|=v^l$YaOMvk@krFx%;bl(#OC5I5Iv5t*$-*
2W<7Z^3s?J0o?KZqp{Ww{p504?Jx?HB;d!$4?^|x35#$fPSYU#|5E1JCHZ1K^xnsMkztCVAM~?;1w$Ckt(Fk_UrPcJ*_@2K!Mf%DlhBN&osWHGG3BbFw4>_6
=VJJ$@3QQjxQjC#<IRI0I9Q07cF+($5DjXRe&^MgW7Q@%CyFJBKm4Pg|4L7Ys>xtK<9k8Zm~XN%zwV&)j}rxbZ6IQ-_J{iYaqNW6w;*7oy1J&EXtPdp!BJWH
M!@HBb$J}Or?HNula%7^UnN1<k8&}6+B*_Bd&eadkrIdA>@UHxravn-JAEYI_8b+NBjir3clC<%hU@eBv~UzD3g|<F(L<u`tU~Vq^TfNC+|2*)3KC1jY0!Gm
D5QMPUOG_*TdDK=nd&w)C@F;YZP@x>Gg6X+S=UCWYw*r-gz(T73G{7$0yWT>ihS@Bx|XyGm?ptJsuEK`paO12MOeVEt{4dC$wyfC5ZJ%Gc&~N>vznr7XDWaq
!?VFtsyCK_$RTF$UX50+QGyf{Dj|)uZ8!T@3hH=xL~sJD5nU3jT*J}0MJv0+OLpKlHeHy*ySFIkcKj`A6f8>6G2zMIq<aOZk{f-r07}#;amT`6z(2UA3v{5$
jP}KHP50P+pKNkT)NJxRRfGu-5O(kTU=SxXbw2-psfj+fQ6C>?@($$bTDYmKv4_x!LF2SlByQEZ4G#_5c`>IBmJkCjf`jI-V%{8nqF1n$_u}fgD%YgESItjS
m{Sl`If9nu#<*!WQ3kjf;B8A!{&X~n2=iCe5HHv1b4!iNSXZ8lSOi~Kg=$VOtE%;uQF6=AuI^T4K7W7V+2BcG<kwMc`4-1j_AjbmgS5XSt%5WB8LEM~N0vrt
PlN@mZUk9@mv#i$m}=pNd2AXX=}JYmzj-I+s>}QH_8U0PlqzwaaNFrIk{aGUw|nx-t_C`hUdxH)NkaC>DZQCi1+5nT<O8~gDpV(&+p&s0huwveU>CsQ0NtE)
o9srMV*J$fXJw5^&-HzBy_UR4ZhPW(-T$#OBfT{~y=202uUN(Vm-?;m56&&^F*E&Cin2eFj}-8Up$d+m8jL-g($oh0<&Q1o?~{$1O_lKsS_3M*X0PQC=VXay
xcA*C`0tV}4~eu$X{uC{l!@5up4B{Ie@(*T9u0A72Oh5Dd_dmWLnMSO1{IzsuM-;Rinrsn30*lQ7hcOLYjv(x>KaMa&0AcRUF-HcaVWIn6=}<feJFGPv=*d!
&NAfU^r45LQNS4yzj{ObPm7b45QY*m#CP&+382T2)mPZZ&q6B9zmX!|_YZ(j@BR+6o<(yDWr}Gw$WY?+Iim{WbT}`vmQTZPs)~?x=7v7#p4RGPb$D&e%ARmb
^^x!tb%#<!vgq3Rl#tyn<rnOy$3{&~TjDjk5Cw`g_)Ii(be`O3`Gr1Ar#2>_UDkh~{)_VLBiah|tJ-TQsnL=lU5$sloNtWA8D}t~kFpF!1k}IPr!`L!5%`M~
e8GCNOvYe@uBq|i4o|GqLYyrI)fV~ejWOI|9?6zeNFh9NZ7?@H0T*WchRpaqX=*kQ#dFrPG5^v{2}Eh})EK6CR2f}|HASkgFk=z|x3>_KG8*ozC&^5bdfR1V
2TA<2sf&2ha;p#6s;3Ww+&>|-s_gp;n71&7kqMINL6?yq$r+SD?PM09mQ0lhO_4>3`ha&zX61?JJKx<$b2!MQJhNo=_+r!7FU6gzmAGKsSy_bj2#{ZUz0rK-
(^bSakI&Z_9x--LRnH88z&7t|K(y$?ls5h;lwaC$ErsUDMPrx_<8<An2sVUX)>j^=Gk;D3rKYl_AwkP|kc{qGrZA6lI!N;At0x`Q;(!KO8*)5`z=frN-(16X
VO1?%7Bn1j=Xu^ZniCVQ-N^uJf~D_Xa@3Eh(8nLptnRb7c^=(e>~J0y8Wv;pjz-Kz7vl@#k-z;pUz-iMD5cL6wgGFXKILFLGn1oSy>{&wR+J)`b8k*NaLy3n
Iob`(*B(K(C4B@drtKsSCzB}!BKpo~vKnbpa0V2{cjNFc>{J@;-tMJH!L-vH;)S^gc=To(O`XP^R8x&AFn<v=k1aaST@`cMectz-ER`E38M{UH=QMxIjS6qJ
XKE@w$0HAi;ZojQBhA*zS{3c8zX`eq|IyimSk@;I;)}C%Lh{QeR>g7?xj7quzIN%Ti6iKO5>TKSB?#WBb;GW`r}raNvBD2oCfXi8-j+)znWG}rAtVJrgLQ`@
mOAvNVsU5SD~1`~@;~_PIUCJTvzm`%v+AW@DqnAS@td=SZBMN55h39Y7-fghVt2EvXH_80fPP^p(M9}6vfI2=xhTk7q-4aAhdwDou<9i^w&&4D*~r@<cFWdj
%JoToYz6@Zx0(q^RUPt*s2>L`G7>Ny+aoGttZC?L38bR#f*IX>P1@5UxC?H~L7?s_6XvU#R8X8o)vprEW~Cv7?3^)JRmND$OafqtAZF}$1*5sIeKC4(JGhf>
I7TewnlWcwCr+CM=!rbyG>K_O9Eco#2q@QYec|>1?HKx?*en}P*`e_oiL|N0(NE}P;DSVOMVCtAFag!Qy#y+)le4LGN^N;UuB-aQC6i37%U4PtI9*0|##K;0
KA!-m#HO#%N*#)*0$SnF>A?d@LvO(9q2FTUwr4c{8@6vf{YWf@BNuvEh&P5CiGD~SD(d-r+qoGCd$)zOYVunw)%o<B`gv%4g%2H6K=?ksW_ZJbVL}Aj&?T5-
%6KnaNy-&EB}glVMY3gsU?{8WW-E%rKacu&YWteHanA5kSLS3Edu1~?S+*;p{CAVvdA?a;BY%$Pp7VdOVyJaQJMB!eF;bd7p(}<23!UF^`8|mSRPJ|3?1`fQ
gKlTMvat&_Bo$x4UguXHek#|vFVcy+4+-wS%P#rxD(1=R1O;_&0l>{y-aF&vwx4n92r;c-YLJ*2c-)bD>c|g5i*&hVgR-w_C>sKfy0|~eFQx-@d3*VK{X_9y
hB8%e@Y@68*bwmcbYd|l)P5Swu2vs<X|N~x88sdD3IikgK-E%uGb-k^*KS|MQLM)9I|-O_-oOOiC%aY~Qu3$?w&x^1n0h(PRjKMNo8o6_t(H2LQ8Cc+0P#2$
T`?T8Z3RNGP+Pc;{KduERiVRzZ;RlsNsfQIq_vtP44HLV4e+ZsRv>x>bntO*A|6#Jb!q}O76xK1HMCBrA}JUibg_+&@R2KZJyjKJN4kUV2dl{tN_~}J<*vjv
Ix@R3v9!waNh;VVTIInnBEbz1^qP&|dvYIXDne!$ddyuy2Y*yQA~csl-1J~$iIH@l7EH^@r{9s?AxuR0Mx;Anm9}|Tz}={srauF>;cmBpS)Cq|7!!2BbaxX8
0hH1YkJ~!an&q_=en1XY42(B!IMOpp$Zjl9M9C4i|Mb46>GiNlAw$s^g?EJK_CHZNwSo3!6U-f}AX#AvMFysftyt*dSh}k*8($XR2s+e)6Qr7C3psrc+3+9a
mf5`tZd(bI8F$|@((t1TDLJge3Ff>@W$4e{7oYm*+?<^WhR_5dEq*;su>CpxE*2rk*(vq1On-CJ%}Sgy)@Rj=yXYY?BVtqWm0U!S&`)4X<_b!3fg%&eP%tkM
w3tSvoso?_jk_A4NIj6pz2)C@KbNX__}MowIoD;L?%20l4W$y560cjKc!G0G^&2m2bMn2K&~sNR-+9$EMvUPeyXE2zOKl=@^lfddaDN{Yhe!AHSFump-Q~u>
6-|Brr&U~C&<cq#qBOTr0&dMVIXrtKs(>g^^#%sC+y&pmF<m0jq{6^~C<G`Iq`M`RsoaXb_CeYR6hP>I{nJYFL%Keao=pwRyz@v?SiJ7EJ91<V3oRxE3k`F7
987FvTPrde9ENVIOfp$Xw7Ag_DyK)es8qZ9(yXb9fo*m|8Xrx1&y6KfW_HNL8vCo;I>GGg|Br+XR0Re^6{D83rtKLB<F&`9GyQbQ*fMU)C$DN#7e$X(D1%1M
fw4+lu5zE&C5{b<QUIE=Q&q|33toymSJEoUIF?w7RjYW6HiIDfU1h>gFxBUDS_TQg6Gu@p@^_Hp2EAx+C(F!T<qdHrYN(+q;bS5{G0LgY1U9KtHbS5cKh0Zo
7Qg``S8(Hjw9IX<=hpfy)Yw7`tj|~PN8pw6&^2-?zBI8Es?gBaU~bc$4IrL{t8`n9;M0fBYKvu2DQc~}rUWZ&EoF%xSn0z32#D<in|dv<y20Y!$w2oQ;pD?Q
Fv%T6P;(VMDrm;%AMK8Syj7Je<YWZez&>C82}BB@%0_hJEkuTK@_KNnRQuagy_r_xz=y#Rwrq1Q4Gu@-TlPs5VLM?bc<q)>W&Y$(QC~GcK3<MW^~c1?78`bm
|8-q}7=L&570gEwv34v`HekWk8S#A!)-|FI5t!gKR%qQEDDP!&_bc%+)kjoaG?p-ZEvK6GIpesxnb)pJhx+5DP^jIZ6M1utMj^9Qjf0;+rXIpe%b}|BXGXa^
q9Ctii0iNtK!L4(FgWwKh!IH%asL6yn+f>#G;W(paF?iC#3$ln(s+3<ZNv2vOrJ(&a0mrwDtHxY4_?FbGHc(<r?iowHgGMb)p5#2b=Cyea%JHL1TY7R!{ddu
U*9oXvC}y56s*4SbcLun)@Y3Pr2Z&$1GI5;Ks3L&qNhxyHwt!YVy}4XNRN4zSR6*5TU!@q?F17lbtnLP4X1|(z*~AqpBh#Zs5N$-o>(CwM(vz)S52>Ab&wi~
8J#e<PT`ytj-AN6F(R`L<B@O=ngrOV5K){<;lk`Br~u2<|7hPnBG#xmS6S|^nmHn9vQxNbCU1#CwIh5E(R&u|&M9%m)=b<(@(%QuI)Qm!{zL{mh?jPJj-hoZ
^T&U}3-V0oM(d}1cc;>;y3e#5oih6-jc>ao00000u2d4D<E8&!00G)klC%T>*v(OcvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/caixas_profundidade_n5k.csv')

In [ ]:
# listas/divisao_r4_flavoria.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_flavoria.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;!SNenOy)f8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7v-DGDY8X~*_v0-{=i^@umI~GHRNPa@`5dWMqjsoLnzp;lUd7Ji_mA)I0;G(WU`yfBq{}ZaufwqIWAMf7Qh>!WAa=OY
+>LTd77ID0ipvQ2TQZ4Xg|G~xnjstD*T&Y(KBc<;S@<Qs9U7X@p5cDxn>~aJ<5*U+=Fmd<`SQL*sRepR9+$*n*W8psfRPrkCKdNIrhQ?HzMoA8xQ+5wl0@;2
?a9e{Pk{c=5n=P=Rnd-yNNouIVpA{xrho;Y_Cvi+FrdpgrblY;QH_I!Fj@khlcwf)S;bumsVm;TNmjwTB2w@g<G=RZL4xkI?vq$JsSO`oN2rhW7`QW9--?&f
qoYFghxI~O^%eM*;4v8fUxwjwl_s={Hxyh3<SsPjT|kNg+`kZisyYxF($x41=E$gzbcg!uRESu&#r|PNsqWhV6&X5KC^+0#bPK>poZc$y$ImBdRvV2HQs@@j
vH~>Js_-Iq-gVLoGDj8MSLkdlSQU7x@Te#bn;qjqaX2qN97**TirM%M-m`isj4JE%f1!N`HrS$lJj4>hb^{oRCuxSU?o(!Lo3nnj&abd)Q*MJzV6CyYbiA2*
6aC7g**#P)O*LXZV56cjBLVQk_-|<M;FZl(f<<5sxnI#7_&KN4T@-Gv<xJ=IA1@_F-5Ewfx!@<tw7768EX8bic0^14+D@o#qe2g=WM3liL@cwO=Kot=#(_X?
K;jWG?muZaj<0)WrP@}TWd|`a_&9okoi?6fJILF|VV|!PRKx-jke(@04542%HgIYm6NT<_oJ@E6VOF|8RnSOj{c3cAIfNQGjL_j^W-!Cm<vKpsv^(kn-N?zZ
0g&%#gZo0xK){6?yEfw|1OU+M3VoCD@}<g>=Vim}&Uh~E+jfE_4!JhYvP#es%>)n0iOg3To+3u@N>qhTS=d6>N_tD905GXAki7oWM6*2rbc!_xyTu)k*?72e
uJC(FF{K_1&c9C>!+7qEb0x?D)9d4S@Rd{yeN(5`*}sy4jf~z7q`pIb5~jFq2k)XPthqBHd5E5EJMu@{H=OvI2)2d!@gX&^VQTV{W%J8?uyw?$j-lS>kNqeZ
5z~~TJ_^#-%GEbSE>|6e7=#`i&2QOQB-$-kVC-bR*3rY{%_#9`0PwI7?LDWfhB;ucm$!zZi8L{o6Vus|RL_$4)SkuEiuGYVo{rVA*^9Gt+{J@8x}@ir4@u<J
YVO?<d+SMFx6;(n{M4jnIoZY_fbT1dsn;g`zVhf2xBJgW-{F9*^ZXCA4Rfr+Ax{9}FBUT_cRuNM=EwKs)C~+zM}vUz?J6FHh}A+k6o`i?6VYb6y$MyB=x#nq
^(#FA(BJkQcjHvIR&Rb8-*7{HgdtRiGj4Cvl2tc@4VdWj+9Q{X5zyI=6|LWgg%vepws!pH>Bi53gII+P1cE+A&lLL&-EPx^7xLNWF5hKw*pbs>EW4<2=G4{U
0;cefFv}mfRV)|{sx2S3s6ZK|EkwO0iyY9boi(HUbpXS7XgF(Nf8mZDugM_`S$V!`i}SUvUmC?gfNY-fpv4P0YEa)I*xaEg_XfRYTWQc0LTKII3o{<yLW=)W
p+hA^Bf{k!qv=^!H=+AAk3fW3xTvPZoL7M&e;K>o3!?O(P2!t+lZ}%+8T&1PvZ5+M#V^{wWse@eV2sKa_4zU`-AHhrK<#KJ+Ro_<qGMVx+dfVX=#}XeIQy_(
#;vK!xPqOL&5@UYcYPFSjT9{81_2{N=9bK#qU_81uB6KS8!xwBn}JkD&UgZ&@p3-tN6f6VN!IC&uKDSD{vtrWb%6$c#|58-kA1fSmuOQkF;)0gby~<alA2)b
mHlJF4*Da=KN&I*QhB;Iz>i!VzfMVR<QCrb$@WqJ7S8m94u*N4*!9Iajew)kEQ%XxWeDyw>j93!%XVti^-3Bm=@-y~dW08i`jJcDU{qo=@g=;|&HD~T1PjWX
w-8Nb9vuWCjHGiMU$!e5W!n>J+5A5?3zl2^6}mD>2VMTyCU86=I1b)c>e+#?@O>A_d{Sm}b-Ip^o0#JHEk}#cF49zS_#?vMkOUQv=Toc@NAs?I!)!Cc<tMV+
pC079HkrQ<+N*dlLr(|#2=|T;%4GIq_vVB-lPwj5Jgt4P!5dD{4Wogfkj7|R@-sv=as_AQjAWs$tMePRNt4c7U|M!p8ij#^5UP3EY?X<fGfsV^@_S3Wdohzg
$WL3Qe~-ySbU0R0$o0wFtzRkQnXH}N|1A6}Rz{t;^p%CJR7nEc6CuFzO%BEJcOT}}xTNKO0MP=8m#pqq{hK<rctxfctj*iBeRoCcb)M7=9#XHep9$<4Puh>`
l*eTK%q}*sHgpg{cc}KI)C$;Ji8>w}3pTM79szg4QIh)61A+cmZ4W_|%ElV3+0DZwV_hmP1?7(ce!)!;#(V|(_`{k8Pn{hmpIj_i1JX}qWehQZVD!_1z#brB
FKBqS^{HCmcui7#o5h&U=FdmyOlinPV|&|O#Rb#|1G1~x|98Yxj}5cg={4f;GS}}$P|1qfJlh0aXl>q&Qls<XR0P7kS9jgE0ag_lP*(O|M|5#PD$&3zPx76#
07wyADPTE7A!QpSep4dT=XkN+xO8&HeCQ4gmj~WeC6e@jNoaeWM~&<frj$Agr@*fsI*$J$#c2tlG@ddTDh)CcK<;KTwpQ10Q;@xP3CEo$(nBXghkG76fCis^
tvoYl;Fd0f5z+D6KK^znR_o%6_ktK`?zzX?-TAt`4I_Y)hVmRYA|!nZ8&)-F7%tx;8yKxl6aL{%*0LGPxWoTw+8Rg)RqKi6n5{;TFBOA0!O!afI0<!?pTEbG
v`+adqsi@pxE~c;|M*r*nvi_i(DqJZ$_4J1t%WO!EG>&`DTMUI)>E|W`xEMQREE5F`gg#uTpkH}=otb|mG)KAc>XvnW((i$kI5uv35e&p0f3sQ=M01Rh)y>X
glwOkj>vfn;*S@4apc<QM+E`v*b9d1X|?g*^LZT8-i^0lkcCvn{XBL5QJnpfjv+rX+iPAeyY)p>-&K{ZU@*u-TCk{7(X-In&`4a@avEC97Uwk(qH%E$RsQqx
(Zrr_8KxiJ?NDZg;66Fxkznka^6IQfg)B`RkKIfPsP-6Sj9<&{4|ggm*qtND%TJ|SV0%=<P{66XRKVuU0>)42L#OPcW5dYrmGKOqTVrP){52j@pbL=?f~Bl;
jnEd?kU$y}TJ!GZV11{GA_^UQ&0tUH?A=f(yh!>&;Jf;%?lX@cEQ}h>o9_EO<Y4Kv{qdf!Fu+^BS*G_9{eHSaRbP>jID{paUIeJGH~;6Lts&dnvYc5ZJ1i>+
kHsq`Mf7&-q$FU06#@RnK7H>NXwxSq`ww5j>r^pqrptn{z3cR=T@e~(YJSfA>Y2Wd+5D!j`?~bELYJegM?HX6cd@W+@ljmFTa+D=yfrb?@p;P-hd>~-vz!Lx
=sCz(+H>gsm*{yF2Tgg)%KKCN(jA;bzs}I7wwEdO`hPkrTmE)gV=s=G$xzKi?P4nvSXG%stz8r<IA+8Jd~CvU+F1^12_HE%Pw}&aK#zuzCU=B~D1&@-EZk_&
915@n+3$+~iuOQ0e-R*kAiyzX)DPYI(QNva8LTC88t$nuX{T!5XRXhygQlrD7=9}N!grkcvxJE`<GZ<dI3t!AP{v=VRMZwb(d_;1a)RfW1{8TzZ24R{UOOn7
{))a$w?=|xLr0Dcvw5&eFhu;Ge{Y80Op4mUlgXFT(Ez=`PyG)(DysaNCOm+Dj3~H&z^z2JYx~UzNyEb9J@pat$Wx+9&f&W_YE{gVvfL4fXgPGDij~oB{-^RO
dz}cv)h!Ear3gSZjV<q&or+<ys8zDNNUXHEx*v|zy=`4%vC=`epkAY#oAd38#@-UJ5OuTNEzxtPUc!J>TJ8&p+#$&!c8L2Zp3MBb&#fidt|&%}Z819(*3`ex
E1OdioPQGZWV!6(_o|lrWvv!e7IKqN0za06_I(kkPq!(v<{<%p-%Z`-c$`u2)ob;iwzqHtm=dc`xwaX~t%4K`XO|A77z=<J#zLniX4JR;=_WtLPLJK0))|ma
?u(1ynFo)=R=QX1uSs5h4MFppperx;_rXOKMUm*cTn#|WV%naHBt_obcU%bQ<C>^3Ic3LQLkbZvJEziMJi_#5madBKLOTAP`?}=-S7`W#cPhox>`9(k6czYf
-i75R8K?0QWf$ETywGKyII;~a*MdoucF#y4rP1R>g6o)@l!oOK%_OM<Msa6&Z_bS)hO9jwmfVEWrr-HVkIUbolu&Yf>We=?%M%3>!9H-MfEJ%BALJ%h-D?0q
yoy>V{j$pE+JxQ&#)-e}%V`sPgoKN19BE?k4ZVEm8xLxjm@fAqa77HJTPHy%8_IpJi95K}W<@$gQjQzyL8}?G5Q49H$LtI_O^$?-s&q|h1}Nfc95t9!sQ|GR
Jq>V=lhbA&$Tyc+@^qdxrXOpxsGW0+r({3N%%f<ZuZpk2|5lk&_*%0zfj}}2wnDtld2FGORJ!1Ir8-XK^|KLi<J;dVE|(<%5d>t5`-AK(Tgw1a0|~A%?Mvzo
Y1Q+HX<73M!s6jJ)lSYjz`ui64X_(bvgstcy2f<C1>Wn$^?8k&J8g?%^mwCy<vfNH*Zw~(1WGIFdZR<$T2L?TTtE19@kluRLG_}Z6bgEZ2C&JuXUN1%sUc>n
_wGmpcf(7O7pO&k=DnS-#WHx0?nwU*4QC_)BiL#s7ENlj&;Vr2h~&#4RK1Zs{PtAx%#7bOPS#{VQ-a7$E6DU?Z!myVv$<b6saJt*L-%4{!KKI8=Y}p_=n?pO
twq+F56>Y8TwQLi(&(~sJ3@cucJ;nwqhz8bSUI@iABi|-xaTh@uI40+N$W-FpL^}P?%~=91-~(W0BJKH;p9L(xGgNOTE0RxLCE)PTi|8;Y%?W<Lo>C>!%<6g
3M1HJk<Qy>`WZYBUmmMbH-igXn9W|`Y*!T8XnloZ4e>-?im!(+Tx2_HnX9X|(O*y_Ts+@J#q2cX#-93V)S2A97M!d*<oMXRD92|r3RAOn92b+Jb-sKWFSdtI
?UT&2AFfV;@BBqyL=-)jj+rO*Orj07Sl|66&M~{+-bH%eA8TFZ=xNHkJhx|Mw)ZA!aFVXRf{s<bvavbiqVPCP)Cvgrpv)@s_UG^3a|0+#uf6l|h%SqLkBg+f
*>tUEIz0Jad>iDc7NTJN=DhJ870!jpUqP`$A+saLoY8_rw?=NaH%ptjTY3RBH=dl4T3oP%tC7+^Ci$IguvrX<TC^M<X|znt=UzzU`kgp_8mkQ)0jR*!av<SL
w|8vuyEn%N@gKwKt}qO5dBg*B!kw*&ZYw7gQ2y^Da0%VZlG!?!TZH$AI_@q{bqy3p{YpMQz!@+%4kD%pD1+%ScWCt{YCz^=z*82xkGRXLK;o#UOkDxY+FD+O
;3R7X*wZIEzu-oPe?laq|FTzFH#n%CHvJ<F*6{&P(f>ljHoL3-@h{L@ZByUr*d0TC7Pb~O?9h*hH$KC4Ba%0usK@a1i9*(QdB4mo%UtFx<kQI(wGwXHgUbl!
6t4T@y#E3?XJ@;6u^x)vsWXO9aAAzy)%t(DthGTtRzPbrD&;G<e)@$r1vOx-fHa|__8vJ(q{^tqM2as}s)a1B`<?xzDK|zH2It2H>gdOPGF{|$P6Ai*A;dh@
glDcHg<<yy+zOLdYCfScOb4i|#+<`jR}`ii7I@<wA?T6Awuo*^=Mg6>`vHxx52t?2_z?kr+Vys&icDZ5=05L;s1V63$U39PwTp<S=tA1t4F3ysq|Bv+oeF9X
wuy9yk{|;Zhn_SBo^#j(7vM(tnzi>e`7j&XZ6#Vrh-YhfPA{_rU0|c(OK&jswL$kyEC9Nx=mLj&`C2&b#BL6`|NQE3J~M0BK1CPh$gL4#3a*E%)AyPX)&pmz
qsyRLynE)=1v&Fp5LDh%ZC`wz7{=1>ix;be-Lmu}T()O~>w-3ehcvj$!ad<;z+Vq5A=YwKV4x1Z&M{evIg`6Y<^185p3?Aq^rW<3T$Ydkr}t{fsX4^9#GKi>
<gEElgQM?NsOK7jYcOtkGI-&Wn%7hllNXnKNCcJ%>3$sZ9M#n2QOQ%2TXooZ4L_uMjenUNd>)Qof@tcmwz4ok{-;&-El4n^P0}8^L)wZaf%F*uIqS7>r6zL<
l@G{fR`t7UxVH$AoI=+q8y$oa^1$owCWu1xbdadZqhrMOJfksUnxNKV0HX7w7f&T-@!wi6Z&a2C<u`rI5#Xp}sLK_J8d~xlFtGWRY|x$M@gGy&_KK>evWo^B
HVA};vx|=^diqH=B@v0-W9-?uEZBh5i61+AFJFCr{i2~f?}prp+n2B77qbfoEtezbFVx!Dz~d2SAX)8LqQa||25)ji1vQc>@zz$n6vsi&l%#E?TgQy;N3lWL
@ipBf=)}#z_$51X>Ev_#2(kcTNnwjJ0N&M=l+Ifb{?=`V0J5JwER3}V?Jlo?-{_}SQI<8i&oW)OC{(cqUi_c1hF@L$<p6jNa)ThyM`;&I@-*M5IHJ%rMw<zJ
ZvOM?nB(Ul`%Yag?NkN*_#6=1)Mju%uOH2Z`#vUx>A?Gk5jQ~ZUt=F2PB!o7+<=&G3OO}--knxN;u36EV^HdC-U!B#j;kQIig!@i@WukUgYZUwxcl?wm`u84
UU(;k@_br0^F2UFt|hxPZv;C$YPCS4jod$cddv9+EdZ&a<f*6twl#alxXg3dE*Tqe5KG5APaBF6E*4nUu6qniH!t#;92pRdH5}Ua+O0P8sTV%a>-^Wy)v03e
$f^j?m~W8xtdD&7Erwwnk_erla43nK@gIqa8(GQrqN76j6wYW^#^CGP2jW&YRB>%{iwz9No?U9GP<<8-(!J>gpataJ@W7u#nvY~gFirFZ_}r(()+-q(QxG!$
O`W=t&(ToA6@h)3Id0DGY!77C)^UuezDwUl*hPngHzs0$BjtRJmBvISQyQjb4Hgq}=<)Qgqh~Sg8JSbQ=aTIqv>Gh32&HfED$HVSNDh$AB4mN1A;Ov%N4?Ln
I+JRd7=cVeX{=!>Gd4lVUF`Sm|FZ@%-`v+;?lK?sAt5eiLmOG0pvKQ?9X68GFXe>fq9vH(Kc1Uxb+>!xTZUcyz;^MsimXA}0^DNX49>VG=ng1$SkM`y;e^uD
xBU@X;cR#T<L-T+-EEJj6V}_sO_TG-k1zFSF>$dP*nR57Erq`Q@sgbY7~$(@IACE(8UEW7?HyUf02Uz@P*uaQG}|AsZGUg0N(I2UsI;B>;I6GpniVGIzWGJk
5=LZVLTS#Eb4Kz7c88ac`L=F%<DD-UT+IFAt@Z1vvEWRzjg;^Pfh{R#+a1N`bp0qhwbfoc?K0E*gBHJtPLaHUcgzH-OL{df!MKqSyfh{PYCM5&jxuYS*(sN1
u3fauj|pNrnyG4bde@3wk3~y$bCcda9P;emk5O~ih<h~XJrP9aCqZ;tq}XsIfhu=W-ZY7sD7eCo<Z!q6iBVS?${+2EYT=yYx4LJV_(>x$ik7Oy{iQ!q-2JoV
GhR%=$Ay|D`=4U&(OyHP1Cl#zmQgIC1y>69Uu#D`Z1yq{C)*BF3v#$J%CONpAb>!{ddHLOnY@*e0n^1>!Vx(XEna+G`k8n9r?rd&I2BnSq^ctf^DD=R?$qso
5*azFQyC?2-XQ+di{dBIuYs${y;3014{M6NKQ?<7Kkqc#OkbN~wB#q)3~Zn+DXbo{G5j~H7Yu9>FjB$#WUIZ!WP{p^#PEV&&?PE4AB%1+dyJY@`={`&=vB);
;P|e{*?LHMdAb!@Xg`4QMizC92eeyRIeJpQbOrTPU-vB@P>DPt6pn~Qcq+9{zg?I2m(_i_a>W{Au-v>j^X%R~A=FswdD)-T*Ll67m90}o1&J!v0+T9^n7J$J
PI;dy7jF_WVc6y%khVdGd-EtRv}`MA+38g7J*ukd<&LDf>gEmouk&tc4%X+@A*sqOs&Ek^JM`zTu}|z@@W1A041Es2<fX>ciAbq}$vIANh!{DNKh}5Z!D$#l
heYijl4dmtKH&+0=!FRC&Wvr}yZZ1CgY{~wU^+j+D6Z`JcV2s55!i-4QoP?#5EbjcwBwv^pic=v-T+VHqjxSkz6e=rxZ2Z2=681+CwlxuLcS1{@Ri@yyTh0i
0h{TmiotG3heI#bYv`8{B&8;y5l_!O#Y!ZK#OGgwAkDcc)Mq9ictt&-kG>(3zp88$Np?BfU3{Gxrczwn3s`b&+uH#D{g#Q`L7Izkvu30$Gh1q_6FWi1{VBj0
I)9F$0pop;E*w$UnwGs0W~=4Ea2?TH)X0>;#_aM<N}m{qU?9|;s_YJ;YeIXhiW6+0`97K0oF!*%m)%C6j6Wi#*{-?e2v^Y`sBnY(><<lAy7LhrPNM3Dek_ve
<D}U)^oi0{lf*Q!fd&|OD3Rih+Cr$LlBorJ4WajI!9#Cd*!jS$bPEWuh|;CFEheCD?94RQm#fOra61Di28gp)NG3!m4L;7cwvIPjaV`GeM6)%9ArdL;wU{<`
fTtbk^p3y^X;gb&`=CKN&6}1nF%oGhQJe!e@S0AtFV&ow`~#^=BI$74MY;tu)w^kXTJF8wkk*@q&Gxk+#}&xibw6#ek`bv1dkf<fB(cUuus}mawDmsa#RR;>
?mE&xm~vgKS1UVzu4==Wfv|>$O72Qkq-(?&!L>=RQzG+@5e@Q;^R3$fP_gsomLvC&OhxUge$PQ%WEFQbC76cXubhG@KhRr<S1(pA535nl)>~&yP7~ZrK!S>^
Xg+=vN6Rz1gy5dLeY7#kX(XAM4i{hM$s~VlRjRazH?FDTo3@3v+TZbU(o4Kjb&O&%mnE+ewf)>OVh%6IGP`z=;TCV|$UNm#=%2?=Nhi6jkZw}T)H1+lVP2Jd
UVz{$IlLf@JITyb$8s@O5fgsl`viWOrD(mH^L#PCp9<!TAYT;(iLB4<ESy_O$&h*0(9M^Uq*ZH#@CKDV)CC^<>Hio@cRtO<+IlU^sRW_`7RLIu0ca9qixAby
^k5;JbZ_@ys509TXg%bxNC?LM{h5!tNk?$uP+$%$6r$*_o?UKn1-Vt-659I<{-7r2ar-kRJ2(SRKqNOe*HCU9>hT1|FAdr96N_dOzXaZaxF;JSKu~7t?$xwh
(7a;p7q$Gkx;+H2>t&2D-`(izIkgn*T?>EkoWG|qQm-|Nvcy(7B5ek@Fp~OzgNV1ARy-c{8jZLXGF*+}hD%Hot2``%4Ccea_#)dW5sQr{!Ha-Zzs?^7KK6;l
dWehol470LM0j@RrXbjwPk3bVSvolXih)G9En89yRQD%NC__Wu?-lvLY~9?08RO)(C`LOdLGfF*E!d>|7D`%|vC`wq_Y^dni|_$Vr)YNs(=QojOuG#uX69B+
P;ye|wy1^uJ$V+0<=aS;6@d<CRGHxHUj&6#m=n65T4OCPwQjInY-1cTELk1%uEb@+VCSV2)sM?n4?u^@Z?I@mX4$4SmQA(2dm^AO0t5Kv^g8UaZ~{k%`RhaN
^f5tgh>f@SW(C3YbI?8`TXG<ku-TX`qx1}(RvUXkzSF-vPY)7dm$RigCLVbwsCXAFi_uzt{4!LVBv9i;)gi*<9eo61v+@uhm`?0GSV9Z-GKM)~2&#kWuH#XU
7UhXKEj|2bH(GZ5iP8i)jCg)wIwT^-u2v%NYu`hg8&9U(boa>EzfMSxTEFw_5(!Dhcg_0`kU(I)0qxU|uwS5ps`LF|7G4b^nI3<qIFqf3Bs-hY7^&CI=2e03
?4Q*P0&>I{huQd-W(LN;@n&>71Ci%p0cZj?bVhtgk`|R7i>cdmZS5k~UGV^}s#GF2>*njxA^2JD35zX*;0T@7nO@D&pkC0GuQA`cNF65SwOA5>0s|oP(mX1F
LEkb!l>MKmZKHF9{_Xo{WZQPw40t@X;9W5t$3!}vC_IAoBEd~OJnKQc#7F+DF--;*J-HuVJ##7hq>7Y~vd(rH{SyWf&+q@WtrA=kT#5|T$rT$U33GrO+H{^~
06}#H<&Qyh>CFoROlBwj%qX(1X4EdQ5r^Aq8h|Jyaj6}F$YU%dLCUvcT|ve_R%T3_rcQ}Tz=A+m`>xsa6)2+JpQHy{TjpaR8U(~)WFBc0I*o=A!;XQi$uYad
x5iBkLT&(f#)f|{`Cf7Mp-k>$W$od(V<H=EL<!%OlX3bw!n@HupsWxpOikkgp5ku3TiojNP_R%;GM7sO{&3+BOD=H3@1aEOygrL0<XKUSO5xXPypo){06Hol
f_zToXiKgE)g!)9&?cKtwre>?(#wawP}*cs(wwE|E;!?ODOZJ-wc=lPwy&i*Y^sgO28txCz625};!$)7Qr>#lk5I6m6tJXwj-!gz;a+M3k-cal-6I1*xjL9X
C@jnqSC9pGwURAU0m(fFd)2Y6sg;fEnW3>a#$%Lganw}LSB<dSUxogCYOidptrdn+8S1l18;<`o8{O-Ox#e<An{(gmhk<TQCsle)hJ+jacL(UeqZ)E+`(--$
md|(DDskx+=(Vnqdc9aGIjj5VYHUt---&KL*^zCPAqH;W88jbx5wn|N1Cq!?miv%@U8Tbe!i0MYG^e5Mt33gHKuPKclw=S_BjqSAxnTiwzM|{!pY`n|lzZ?G
8MCl7^PXIxdXeXac}z^9@Fzq2R<rLMPEkJwK)6u1mkAE|);K+vs)8GXCbZX0;s8sa?b<`MLTi_-VNCUNd3HS<&9JN-SY29`Wf3z5SM8{90;z-+Z;ec+ad9}0
;bk*{nx}J`GiW3NiSK59I~4CqNxt}>P*J?HDRsXX7ca$Ib2Sfw%{Bnmcd^XLNKRJe7B*f`a*-uky0P`^AwIL+y<BC4b~JK$P9nr=%vcw!zJRTEq<e%ri0r#_
V5X54OE_OL1LH9yqCk0dsP}3JSLh>tWe<L#Msy3vSED?JV`kzs&;(epyS27mwGb{be_Kgef9RNu_<Il6QiC>Tk2x|mesI1?c~v%+9?Zs<snNHLKLU)M`1oLt
zFJocZqeQ%A!AYVRj+)_4@HK)ImLZ}cdmck_rTvDnaP>B$Y4_idsn8WBkCo%m5>_^W{^*=+o6&xx4LHD8#)xa>!IsN5IhoxgvqmlqR~6VQp`U#g;i7*QPPqu
{>Gu08Ds8gQK!EPlY07B;Ss*N^SLABov0_hMOm_1XDDosQJ>>)o}oAiN>k)>^ChAVbVch2M4KRB6D^)`@(Fr@u!E?r$ma4HyxR*^HBbH}%?KEQcS%g2MM<Ck
BMe8J7K7`Uy!qz-^LAg6;KMy^pv7hCisY+$p?g>a%ICnZxr;U7Lrvtzur#&5zt{LjMIcNOs#H{;3@a3;&vm8Tf#^<YgH3{9kPem&!FshJL4Huon-W#tltyAJ
=HJ{b=w!V@Tg}C9bMzb2>5(%yC|IWvoad;T9M>V8FZhRNBy{h&?b3lVm&1<$QOr+3=#{z`LDjA9P$RfxP#iqsc)jSYFzo1a{~3|)xJkq$%|4jF2=p8e{=#oK
8ba=vx-*-^#W0GU@Vi>6Xp#X&`$C2(<y&4k-kL~5U=vGLlbG^>%iz3x*P+cAByjyD+8xHid7Suw2g=Bh7~;KJ8OvI`iq408&=Vc{4qQM<&|rBI3l(&F9^m%;
oH&#da)LoDEQHhampvd>rV%)Ja+aplXj?#|dT&emv#~xs<7WidSt_doe?pbLrr|VUm}GORs{5GPni&1DS?T)=c~ah$7k87JveII6(7&j{KttG7iBYqpZ;Y(m
Go*&B1bReru14u26x+wrOOGLoOF_F|b3T<sFT)kGkSh9~S0wkwES=7A%?1UOu`}UL#S(`S-khh=<Ji>BAyYnvQT44|r|8{h>0IWQA_L->Z+HyGPBMTB)o!K#
{1DeRfQ8Eat28cINQSjs#vJ(*l+zV?2I|o}qB(|fRw4>zFmXMm8%B{#BSQ>B!HDp`A4ggsL&)d7HtUM-q4=gwq;j(95pRF#ku|I3=jehcnBU<V&T040q@yt^
?AxqpO_<?g0i(?aDDP+ID_z36pCNi}v8Q=DExYL2gM)OG_$}Wmg`>P&dhlQSDs_Ozt-<ZdE{ZnfwP30Yo#$Pu*9NK|=w?vJ81B8)@ZO`LK&;G4$N`X*Ru{kR
ZB{=>%}yf?-6>#C)=_WV03-N}vFC~;_<lEJxda@hV9Sh$^9uR3BhP{8txb(XdsoLiz&@Yin>V^5I_(8ih$mWC5R)Lxrl^Y1ZQ2acL35oeuVIKjk@HH3`pv?A
=AH5mmOj1qj}6xzWmO4J|IJ?XDs)}r%Zo(}Y6XVIm`<oIIlC6ota+q}U>_AesLd}_fSw^XQku!0FS8ch!n^^&M|@XEE?$=1)>Fn(34PZCU6pr}(^4R+|4~fb
LYWtA75sVMcX;vh6qsND_0O*7SO*F+xSgOjMy6W|C<O~w_N_uRi3mAPr=kA4=>lg;rZ|>jt~7#@NT!*XNvBB6sDn}ktJ=Uul-ecNL=^9cZIw7(RITEt%*=A0
c)$oolouuA40Sia7p*dMjG06ivR&cj(W*#-skdgs8}^nM?23Ew$nIGKt#=b*-psASV#&{y$!;#z{`Sl?Kgv@lzTZuRD6Dt4yP0nW_M!EcYNy8uj{*BeQlm8@
!Vf=>M$63P&dX@VS(!MMjia4ei$)Mu>l>;ku!(zH!D&dIJ9H^(1~sr28yt2YBo=kfnL7QuL2G_t`?Gqe$GkSi0Ct6QWaucu4UUx%dfv1&v`=APrW?lWBB`5d
D|=X<L!lWeR{O=K6yf@nr<of?raatU<n8df<^$Cc^95W3X-t+}mv0Gab911BkXO4zmn4U%pP{ZqEQAAFaj|H1V|I9-(-0+oEsgNqStj^SB9L}dkVNDu&fI90
m~RS{{UzWD*VnA&Q0pc1eXF|XOhX%}@mq!gEOs3OR<%<WLcB4|GJ+v^v7eVC);*X(5KEqRLhyHSUOz88bXwiU$p~nCAQ3f=UEV?KRm*xN4ilLmn*z58Gi#Kh
=cL#HdO#j~K9{h*IxjD%^-B>HJ4^4aUwOk;Ae&!tFG_eZU<NQ|_w&nqK%}|B%P2;rdNU1<(b_b*{d6qGdlPjwm^F>Hasu4y`(<`><&rbD$Xjn`p+8MX+;D#+
jL%kcSF{`2QnL;mGx<AGx1kcnJZ}A*%IR!D@)XeR3qSm%xfEgB=slLmXWP3Bpk)*F;(e9>kV6n~jx{V-5o<Wj_ApFJI5+E{?X7m~QY&H@Q$%!XJCu<3DYo?W
jSp2HT7w@~;b+SO7j>qd$ir%TT^ues!TDlohui1bHaze*f%a9F(I6PvbZ|ex@$;h~0#T(#FfVV-hMo7z1r2O8yH-Mgya(pZANX^hy&Qq~EC`p*^grYxnI;^)
BZ`axwzI^Mf5ebVQQ_tIrtZ1AseHm#mb#fh<tdD;wM}(PG3n%cDA7-sW?VY>uG633+JhLWChC&CI527LhFpXzVS|@RfI~q#M!yNDkJP%NZ0@xgs%t!h*hu<(
F1;srf{v#>PC8Biuep&-WCNmVxWv!us4XoTDNaEEmc7d&-Nzg2`U!$rxR5bj8X|c@h<azmFuxBiNpJh2<ZP<v+|V_!jP3W@2~G5e3%1QYJIryXpa$d2vigTK
(rFk=j~UySE0G5l?hqjn|4j!-NcZ!ohpV3}C4!5MkvBLc9!byMcG%5sAXibNbPN5G){x*^u>oCk%vdK25*ZCRQJhT`ATHPMKR>|l`l2#^Kp;<)!K7QpyqKBp
#dJZ;-)VxJKUK}<f@0Y6iFoPi#41!1{Fwt`gZ@DGJk7FMG=)~RQ*m0t@^Y>w2}%5uCXi`jF+Z3L@<v;47>Hzn`Z|_(a`+IjMg!vXvTJLtm0EEtyV8FmEs`fa
c^VV*&hnn|>;twP#zn~6ml9JAwObC;(&1J!MPdqN{npvF?_H1H<J|#=?7q4cvxys>oX%cSN8Q6G=1}58oJ%GQb-#8vDAFqjUjkv8**#J!8Rztk3t5UAF$jAh
b1_j&T_*k>fw0CfoxK+BPd`Is^#}i^+ZgK?zV!iDk*4=E$^&%`u0f0=7xxTqWz}MZ!|??tH5*fJ&YF2Hs7ab@qsq={AlDQ{2KG^D<7IOq=q!iWs;QHwJRJx*
Sf~(lxfROtW4~-+Lr^|F=op`~%f(>XJS8W{YB-gQ(3W@fMTLKlt`xu0THRf%{P?y2F&p=Hkp5LGsDpIKWQwI6G7Ixazzd^K{xA1eO*OWymc3q>Ol>TZ-rmJW
Q{~b4%|X=}%jY#4Y$gDq1N0C`Lu^v;RN{CCQgUz}Ogi-W1pf*$a*bS~SE17kAE@FRu8pOfT&lk*o;c9)kC2?fFmvdCLdtHv9O6aVvemap<0BU*bKh>qfJJRf
{L8BPOt|D@_s9J1zBbO15G<LM&`l5K7TtxXlwK5E{z|<k0Q2N{91}py#Hk}E{TLMXy&5F?R^GMg4A@b2<>#`i0S;kh<ttB$SedY&Y?yccGTdMX>x_q=W@#XQ
QyuSk6-g?anm1*`Qvo(~*Q8x87!ffzxYCC=g#syOTtWqXzqTi2^Rz)L><aG52x5g$_R>qGz}Xms=w#P*;gTeMf?s~pA&_I%Vh9>t6N|{fo&JG$jhHp)L=y{=
+FSrlHA`qr+6g*h`{I;9;jtzx=sBiE$Z;OT#W!(|QY&@Q4U_TJskNL#5{Udpe&UA;hlc-Kp$}P4(IyI>uD_e+otJOZ+tuu6_D1Q_-o8ybY+8k{2-tA4hct9V
!{w!B3~nnE&OdprQuXa)$ntNOS<l?>{8u4PR>H%!LREK;sN3bNYpTX!Hzk+qc$6W-N*`$Wn)3=>%q22yO4Hyp3|Plt*cTQsm(fybn5$?j0vj<!k8PuXTc#I$
p_d-1t_d&|sMzqV&#-#p;&p5_G23rX&HuZjc<N)Cf;5l7xi8e{K_D(tx%R7lLOJjTkrHez)WgKtfPh|V+O1LkyBHc!@OVM`6$f~w1^DrBdw)LX`9vT!Iy(NC
@RguhYMeWcP%iB=d#415Mn%WwEm++7t_=*3@2@3Ff7Xvmvy<b{Zb;;dQ&&$^x#y5xCr5kSjXM@nV~0WG<8^bw1uV1bH(sXSU|~0GDw~^aG6%#y8{F4Dxn*tM
FsBU~g!VT=h+Kv<2>wYT)iym^!jYuXboWE{Jp1POSF!Q%CvL3a)d=2}vs$7n?X7Mf!DaezE<rj>GE36p254ia;}xjtJcJaVr??JTz7Uq?|KR=TkALbcSV*C#
`w~1>WCZ(Yve%v;+nZ3FcZJSmx|AavlvFgrxe(0hxLL{w<p5r~)oeBTMha2Awl-)iyLaCgz^t)4De*iI4m5m68Mf1}rdOcMSgmrsYLJ>GI`@cyU2y887T34|
Ah<PR;Ut0w$F4HJ@AEky4qyps>yq(v2d7(^IaMRNJ><7hj8n$KI`OV>f0@Fq$hdEHZ7dm~)cyA;oR{zpzS3-<5~q0RQss#n^3COojz)9fpypdiO?C>(ZYHft
=A@*V)rj8*pBUOrqG)-faVIcfC07f8-jDLE+zl3;sZ$WGc53Hf{WLe7fdaS+b;i1O0gQjtEG<vG!4EC=bE?$oL@HTP>nIEIaky~q!r*&eR$av{qJw|e9;6?0
Y5Xa-cyWPWZ_ZNs+`3Udl<HGI6~WE2Is^_>XN_(u3?W+|Tzadu?CYb*tix497@D%HuBA}}jX7@WiMV;0&e!z4<jBNfA2AGaPP6^nu3`OLK(G8zAQEk5lQG%;
Hhz_V9*G71)RP!bRKLa)Z9|VRi$5^uv1iL0`>l!EfCw{X#IY?0R8<MfP`<Qtn$@auiM^EE_0YTh+7Eu=dDb|pzLRIf-XH-?mYFBh;M|yeFWZ(&nh!>bj%4<s
B(cGMCa@-wsxBf3uA$2&#V7tWTFOk^%?F=gjYf^}QGzNi@w0#2K)foE{2<;~TCJ`r8gga7*#g*c8&IGXz<-^#r8c}wgq)VBF%;2a%-K1g(3222x+BiK#~Ck6
DED4tKS}+`%Ea4yd#Fv^FKlWC7!qKyF~+`#PTdw}ZPL3D>#7Z*dYv@a+VD9NYXod{0q=-@)gi&fD;y9z1#Ue76B~$-%0+6`IqAd)zbn;!isWC^;Y{f3Nc0H<
)IzG6Ua{-A>)X}Gt=j>@l>xsbWZfVg->bkP&&d0jGbUHDQw6F0!kr+RAPR8n-uPqlT|;4|na|y_loL*aBm*Ih#QC3nH_QiXi=ijrxc$xAi1{!;%pq@IQrmV!
3inLZeh&S)M;amVv0;B-+ref^Ff&%G_e4FYn@EO}gT?@Q+-NW_s}$b~M_mfAT^T$@z)q-e3dm*sxEV?kc}RqCcJCrmek7}_e?Y!(s|o4(@O@--^uRsVGsSzx
(1H{TVIxZ`Q5o`yL1&%TwS0Uc^BnjA=nxvg{I0K5VN>ecbRQQ%VHYPegYwB;*%$jl+HBwthgpD^1;eSF&imqp+D!R+BmeMee3!2W;+k0sA{pfL>Qx8yx#W7o
5;K96A-Mu<T@W7ft#zc0<V^JOukICAcP+i@wc3PLI4b6s73}KtRpA9jNuEa6e7S!f`0M=(fZSuDj8ntNEg@XCzN(-Jp`OeJanify@CNTaVzSHu5iDO<Y5$3K
6p*g=hq10mo3USju6F<Ev2^v;eE#>3y=B~&Gt&XmAzs3YNRf8Hxlx*f^|b3pfDu7);yHLA^3euKc(F~6_o@jeLyk=Omctqr7_!M;ml;bd)=3P@z~6>lt!M;C
4O!MB&Aoc*j+wj&FMuohagtQJBE=pjrPG9yz2<1#A6pM^jz8*3W$lbZlQ2y@9N)U~zA7?Np%jiKIiaoBC@DTuRqWb-`trqVfbKLn@JSrJ<}t$n!@6Ihn-~%b
uo#<DB5mxw3d%6>v<FmSm%3gLx>&H28AX=<uL5LEQSN(e{kM<m03a~BKgS<vs$}Cey_$on_;0?;qx!REvu+Q?0$&(C7Prz60eN%-#Uh;}&m*aAN_-O{LY9`Q
B>lXif_2F-_qq>Y0V3e-da3KTR&F0Ai;F!7FByHKNYTSQ!t@uLpFV$&*V}t6p266$oP<`n^mo{l2!{ZsEKJt~rr_zDh5s>6n5~TyTHW~bB^ql*v5)O@P(pRv
E!0ct^iI;53%>tXd7mAg6Q-5T`^jrvjF6GfPnZ`SLjnNSSE8N=uM;f*#f2~aO?Vcd&FgNx0S*h>f$YsVgYw4+c5KBM2du#$)S9CrKLNRW#`E8BS%6sNZ$rS5
&-oy^QAer?KkKI29FYlQg>;B)9x3(G^?k8IsR}p=gASEsvSoH>nx@7p-3RMQMboQvxb1<Td$=qXZraWJqM_(ofztg3{dp^4&&J7&4XFzVLMI~eLp*y&^ECkc
eh8};n|@ErBov&X;ow_PG$1!#K3vY=qI8d)Z5NMUr(BMme#Q)ee?{98Mt;Of)-G@9<{xmCZGB21tn>zI7XI5pZS{3vOW?3Fg}qCT(ej;1n?VsqKFc^xL?5jZ
?dylRhv6r0caUseKn;K^M#6L8s-XTXxc$4(ji5=7nNI-_32g~}()<XN!Npw!?ir6piX<63b7)FEi8<sS?>1MC5d=(0hILCw0?TUN1!;=X<nD+?ooKp~KT`_6
h9cr6_Nm*adaACHQ2}sAv9Lf}dA1S&00000&(l&U%<+_Q00FgY?wSbzBcN7rvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_flavoria.csv')

In [ ]:
# listas/divisao_r4_lefood.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_lefood.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;GF3VkX-;W8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7wR^Z9<JW#Oex0-F69)mXKZ=wHqQ@7FPl+n8r+<lOj6-HXU%@EHSZlR1OGgQ7u=ipFZ-N)3U5&haau$1*Tl>2pRLDAw
{;Bw9$?h^~2gF>EaCZD83@v(sEYd2RfM2)p26HSA>GCIV>TOZu1sCwkpN(l#wCg{gVCV|*Zq&<L78E6u*rpwFO*!P;KYJ&4JoxH%z<!ouRIA1zQj$cgwm2$}
!^axxSZnvF1vLG!#wl2ELCD39fq1<6D+PPA$5q^D1q3(1K#^gMaY?Ij(eJY6R|mViVStAu#Dp%Z2||T#n*H(<|3+9T)?{uJoOEjq!Z*Nl@7HMG@21;+9={nZ
;_Vv1nz%Ps>8$+*qUg0}0(0{i_)qC=J1!2mQ<i&NNHy#<5qnSz4t$7L{@jaQ`xWSZ*1h6r6~yxD><SfWk!?&1DbyUTz~H!U3YBDqH&)^=4PkBV4F(6m8n|l*
@|hsWTu5I2vF6-ROVhNiyFStv#C3P~lPW3~9VxBK%|>{u`xBp<9PNSuEk4bqQGy@9+hA^C$P{7e+@$p9hsS-3UB-YJ%YhTPO>7~$M?T~+8u~rqWRWCJc>#IL
8>)_y-nEVpdm^#7(*w>zNcvG+c(-YN<KI?URvjbT+9tJ{ij~(LSG9nN)66%gLgp+db4*l|zlfJjRJsltwj4&@&;uZjgV6OpX%(%R^({r7Icy|*Q)M*Es&ZLp
;1Z|)xfpJGsS#PuAL<(oq7O5kobD(~Y}+o+40UKw&qS`Q8QkldX>J_sY(w}aWas_g)%+e{A}D--mgjC%NvJi!aP^4y65{}84ndQxS?)ng_xL|wbT;+Bo;{}a
OjQ%@wD`SOTk=&7^Mi`9ytXF|kW<;VS5-8#CuxSne_D*!p0R^`oQwudB|>G=e_*MH#MCj8H+P?@X17f<Q#+CK9=mLVl>wQb9@62D(4|#eBpyc6`CtiQ7S|^m
Zt)^-41l_vD+2bEIv3;b{$XrrN2y-`iE@pP@W{h7=y#yJ4rJrb@u1;r6Y-DAMnvSyQ|s;xOXXicis)2kx1cK93WM)0wXyhVbq9|dLdF3N{F(;1mq1>}tcxyh
Nf>;lZHQNi7yLfMh$Hairf%-O+MNpLU6LceUpSaUCFi@N;2GIM<~}86NkH6DC$LtX{2%}JQJ17t#zHYlyd>#9qbhhby2lPP7SU$_E5LC2L}ADvI@5X8b}p}w
0yh6~u-=9rr$5foaZFW7*yo@ESJ@=j=s`MRhh!1B!J<iqgPzG|<eU~BLZBLCi2nLm;Q$3Va3(nGTP1yH+=}fYK#xC;lan2O*l#%o=UfPs4a#|X!$zbPS427l
EYL?yOq};KiygtXu4oO|m(be5>f9dtCHcpop8N(>eavOi_5$~N(thF&v|GtNn;iT7w;7}`AF`PamiO$mJa-rO-$_Y09U_TRA+V|nn|?@G9O89W*e%Cr;6J|~
K^`=<7R~NB>I~%RC#_N11Ru-4@zHbV7*jjNHw=~s0KMCN)M$p|)9f@B`{<`xs|PTO?H9i&OzJ2qp$nXxh_kKc8fmT09_sb0$^y4(&a3k2#Pk!`MCF{R<P|G0
hecIunLORT@}G_jGTd_@NdRXuNSLy7e`!jWB3E#p$H#_+XznN$s~|>_hZrPjqy5uja-Z$NYBD(n24-cu8yohTdaWaP!e$>``HQ@(!6ab)%kZ~f{3AWy*8&wg
DPn!w0G;C{wX|gZj?GGSK|5%o>}#&W{8G`+1<-yX%0{ZhT$CBL6G<r*rcGGIaP&lyS`c~8{1y=IpZR}iU(xS6M`x~2&Jxa{-CJQQ%V6ZV1bEZnw{adI?1sN%
IrbeB!q%fQ^br*wEwY8=#h5pc<-vOWFfGzJUonQP9OB101xfGZ_nlm2Zg46}l(zhXKp;jYB7jE~ZcU;oe5yEg!1<LtkWEx9-S&`*nW1JS69axEkaz2?q=F|E
{v)grB1%{x<bg@3Do+H1F~L}LatrHHH{bJ3xSW0t<vw%cKO;+>!DK43l7D@LG4vvFSV82+kxt<_J_GycC^nTDt*&h`aby2D0koS<13v$)UVZ{|XbGc|p5$zO
`{v)?^Fl7;y<=Es*RuQx<cEKZD{UFA*vT_dIN;vR8FJVnoB+Ie4m04y6F+OW*TRrqf$M_xs?#Soix9jPywBnjP}|CkeUwzoE6HQ2!x<gFnzEnODWu{WWukW$
3&vu3eL#}jbDWmunuP-#cdMwWu&r4Bv)}?eR8rglT#BLAYMG_{N<V4XFGxWRZUng*C1Q?5Yt*)7>y->7AzrpKF^_p~J%6V^IpnmB1ZPKMk`!exREJ`qkaW7L
uUg`x7mikNo(#?PZ_dl~fIuqQAPjh)h(}+KzQdfFR^`?E+&<kM7EX4K7eqghZ5kTO23-Vc3K%LK?xjo_!vd8s#es80!_omIbs5{Aa8-)be(8zo=b4`rslCZN
%dmfVZ$N1H6OiYhpD9Yt?fn(b#yX{R{C^7-VC!P*8n5RI%u*0(DUHKFTApYtg+LqR7+*(dv3iz~C`Sc!OV5nucaVS=rq4mg#REHde^plnwy{&amMQZ8M581j
a0<=xINesWo_g!uFy0JjX=ph%hgAR`Z_J%3b*z=J!j<LghBp)v!Yu`ih4=nsdLcSa=`cXg0>N<`Jr(`X1QyKX`a~{U83UcOX+1(bp17apHvVYBI9BL6d!KoY
7khr6$*>l!5yN0G9CF=|*28ti_4PGHcqA+XPUi}cM2LF3pP<5!%!%cF7Hc<0QozoG<d@n=4DZiK>6#07K*!jvpX<%}uVb)gUSO8=r>!V_i&=F{AH?{9MUDK|
pV3)o)kI^?WSSq6kg%i}uj;}?FCPnL=w98IgQakYD#tNk%!nSjH-_qVe}^%^R_pW<HmeKfD~2J|O_5`CqO_@%+H{hqQ5o*}%tbUf#R@fHUlF=eoEHGodg`&_
4&uhPk(|Rsp=S%O6x(ml>7cJv+j$vHVf!T{``yR<{)v`*OHzuPyrMKW>rje32E&I5jSJc&CQZ-WZh>+dV76~Q%K5tb9kew+U==$X=#0OQLw}?$UrGdZUc-+>
`$rWt|M8&2L~hB=tN2_&QxM;(Z#`Z|_YG>p7(WpE1b?g&YK3v9l;k*CXXo?uy&VsFp@vvt#Owwtc*0T2A=<k+Gu5Qz_OU<I(_TD(a}3s`DmD$*d8U6k6TYgg
`BeKKI5<C!25C6_Dy~E1U58tRBDhnlgpNOY_11xqK;fQQdpDSIJ>FZ0EW`6oPQ8hljl(Jr^+{aoz`;<w!kgGPV8$V8ML(z$_EWPPgW1=gK*#x~u+&f4*>Zkb
6wSVeyJG)ZT*ipe^FqJ#IW&B~W7gUmWQ%@Oz@wSyarv%o$#pix5*J9O6E~Vk7L?{lw<IFNO=KiKB6PBYPM`<Ma}1l7!40P-hcC+VHph<p=El+5YJ!SOHJmbI
lZESpPs&-f<PksSHr#ftCurZlB%uQ5g~~gJgLaP3X#xESW<eWc5|Rb!dYAa(BK)nw_oN-X0Mvuh4<Aj+73#p_itkDFhAu<$L!)Pqw7w7}U0eg$D@SZs2e>v;
{q$2LZ*|O1ELXTcm~|sLh?vTf2iEd0jaF3sGAS&JbOMx=sZ(2-y1Po`&<-ZElCa5Y+~%JNOeCJRYw1j=(l_+r56@c1Q~nASK_!n}MaZeDJ959A=Iu`&m6r+h
qhQS4jU-tiz1!o>5&eZ0zG5E2CD4U*NYEHjA82o1FU$!R@4H1EZ@Jc`NMgCFCz&)l#}!4Wz|%`D{yq(2k0CIDV*9@GPEC7LJuT-@9ekRWWLqL9O^Zm4xmTw^
bOcB_TmXqg6c>X~O+7fO%&v>!%uW3Cz<3-;D;j89<;R621Et3zWRK7;WnNS_)XA!ty^Zwn&x==<2)8%mr(!#7dts*7^$aqGtCPOOaWm;ptHs!zcrpQqC@1v^
#G>RDrd4T}xT?|>r4bs_cuL#^yIN?9_$%5*;)WD`EKOuX0RWEQj({hRb!nU+zkYHUp`IGWn7OGWHx$#jrd@dGpu;-5@72g`Ex+$!W4Ct*U*NJ*zo=xCS9o$U
Q5)8=mp0OKtiqUp^$)M^wY;DWu2T6#;V8ZP@QNxVB=+E1TyJd9m*9Jvk<wY)cw@1jr|g3}*BQ7Og9Fz#ywu_i!3<YLyS?()Ns=O>Ci0iM4M-ts3yeU(oTUuz
K4CE6oo_f3Cv~0&bNx$WH^5G7&U0MyaKG4&T@#$E5Tq)P9U+9tf`m*OJT8$y^82yaO3^8&+Y2f&pbel={_q|xw{Aa&zdPh*%vp?A3AF_<=KLaqJW&XeN|*Ud
<wbJCF4;ocRa(iJe+4F9txrTA8W(%t%ML$bK}Y)gFDx)e4i;L8`#adtN!jKA4L4t4S2-I#00FEU>bU{{nx(+|vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_lefood.csv')

In [ ]:
# listas/divisao_r4_mensa.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/divisao_r4_mensa.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;4Fp$&|LsB8cNL}xSP$f#0`*|<eKar^QU6leZ?$b3vsNR0$c=dg|TTJze1t>-L(C2!;vfPK3uSTaI!ie@*Hz{`DDo2
4x9(I!n&3mcTetINjmm^dM75R9~)J7wY>kRM9dNEoAvshJ9cQ8otbPUZ&_o%o&?Z>B{wbQgN{wP;Bsxs3myQ1bEj~v8<}<0zJzr9YohB6&doFyX{6r&knfRV
VdP7k?cDR1Bj-}IvNfkoVFwZ&=hLc|o_Cq>@~;jhD|Sl^1C~RA`R%bl0dgRTv{L!K_Y{HoAXJO;y-H}fZw=i!OlRO1F%qX1d{;lZ9$PmL$q{aoz)+<;=y(k0
)kc9L0u-;$U(Iy;2IoLPC%i)0nvaT%6QW)ve|!KOb<}=<pJ{;)<HwUao29_)shEX|5nn3iV*Fsmh|~mfFh3nYvYCAxVZDHg?>1>>Md@3~(`P}6_Ohj`<<Eb?
y<oka>@C%0cJj-NEYV_g4R_H&EP2}WERPpGwfss;U-*cLz{==^UR^*|-*;th7ip4l<_U-B;xSJQ(Ne!2d=r_WD47PLnLKUHtPCnAfJ5#gtNN#2ek*2We3}kE
!=glfmxoB0_B<ibMs3SyFefaLapi6fb$#-AEwe{*XOpIhaf;P+l8%F5#q;(>frMG5Ie6L|XMy48)`lMn;yl^IVAoaaIr}*QSE@yV78-9EP5qsvvdnGK8M%C(
c@P}&WpdCIm09X?lWV4UauFs^Jww}bKF3QgKiOWmosR3k{M@1m-~82K@S+r_CbcmjC05NIC0NDDhnbJ6$ibLh?vqYxCHRN}=h#=r{O~HEQ$qB?+C)|Bhq&L)
+vzfQG>wcFBtdWsbdhRNI6jbpP?6OYmveKjLPK0Pxp{=yO4VY6zJ$58hlBcNpn4c}6ons<H;oSJ$i}idb60>#2FX-VD9J9Tyj?B;sYw*nP5>Dt00Ha;hgkps
=^M~_vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/divisao_r4_mensa.csv')

In [ ]:
# listas/externo_flavoria.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_flavoria.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5=d{0^)m;EG8cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppbIM9w`%CkyKXG=WZ3duqfbQ1z_g`
N2W}#Ui%oQe%&YTzlOL=Ph%`|Zq2Dm3%W2wqX%~!;@OW`NchuDTF`BJCUr*XV$#Qcu;Z?5h8Ah#NLWAV<TMBJ1(15_b2}_{&uDS+uP@I%DZe9%%k*?sd9Mcc
?qGL`Xt@s42wKQO5(W(R2o0gr&g%~-{E|Gg#r;haP)P@^Iyc+UEJI2T*SKP~a6ZO~#(VQSLBe6(Og}Z;2iQW)IbJl~9aDhwjADehrDV!{3*Io`co;w8PY{{p
p6Q9Z6@DYfuP@xEihMHfFTwm~@-qDgyPlg$GjJ^Jz6*dmGM{zu7-gfQHs{J&`jbx2V6L!HCu(&YH8$UqsPoavKkF1B;^irZ<uF;4Mj<lFYB3eAbB~<9JbU#Z
5#MkEB~+p@rn(y})~uOVUJ9`0B0n-Hi&nVvyGT!sI)5l-AcuEEuLJn{OE><CFPd%k=*ApwXx}Qt;wgc1{pM;!*I5I!kr+8CoOBm;Er1?;RbACx4kW54sVU)S
ekbdESzz*sSOk{Q0DsQCQe{LF!&)?e&1hKr|2N-Adm%%I@sy3+z%utA;aU8X4KHs3GjR0Y$jPSHCM|Mcwmw2vg?Bu@LDj))(W6}z5lCQPE$Iap|0qfy!D0jt
1B02O{4b`6Oy6*49UtnWekNUa^TRc+F)R{pWUA$#qk)M*Arj4rUFJMH%RTXNL!0M`B`kW2r-fiMLgk%ewh3<DyDrtj6uYi@Z=@A7?4$wZ_`Dp!@?})RNB1ek
fV|da_BLQ%`l1O>KJlG**Tkn&+?3&{o$ZE*i{y!EO7cEN0U7mF_OTYuXssL9toqTIv~Nc$j2;HU%OG3xDEuCEu|v8n+uHX3^d)TGCCc_M_4IR~=R9t(;jSJe
9ODMlp6B8TlFy};{<VtrB<njKZE$R#fVqUzb1LMLbf>9Qap{UBqbF`*GXkRqKO;I9Q>`2L0%BR99w+!eUoM47+l?l;_@qFM!>~(Q=<*xfMgof0y40D)X=me$
KpmpY5EtpMt6v%s5XfEVv~S;FE)4OBMr4~(_@Z2ASjiUjsrjW^kNexn17Km@4O{<jn;$t&oUsoUtTTgSaK2sN_tsDbH1>&5Rq+9JjRpAO#f75^GJ|L6ta<=f
2}LoPw0!A<>jp6odc?~=wE$q-6a1{Ms9JDnbbOBfct)#qe$azl_@K^-CO5xGt{X40L-c4=s>_i$IZx|L$k1r)A+J*84}8%g8d>z&(a$(-gyO`rhSJ3v5DE_C
ne8Y)EfR5AS(Zn_t%?RsO?(Y0<OSbC-po4ak?_0y8ON$m4>Ax)=2bVp&UeBnu~7~?2h&;L1s`*SxJ=NZ1(4<*wE5e{DGGdLZ)U<L>}M?EWhhfIep{!|YTj2M
g>Y?jj16)+^ZB*CMi{C1R~W}t^a2s5ITwE3g^Z|2FwZP8-Er;kj;Ab=J-bpR!jD4%l6=m24gH0(qiY1=4P$9_Y6q2N)ro3RCD)dk7jzI+*On)kI4&PYQ%vTo
llgMAg)5nhRX})mIe7D$OWqyK_b(9Fq{X8xr)V!y2ptkY3i{j_@aWu>fJ3f!DZRiZUd;{|#M;pzG1FGdq4se)%Cj{>GVFZw+TCjD&pFYLok2syP`tSp#6#9^
__o1MD_ugIY4KNFY-Koxi-_T~Oj=g{OsE6G#!_owerrJqp1rCw@e0$)LH9r8COtHn9fq^ciRIu(Ru*8%zgHtIJNT@xf1+ihz+uZea5`Y;-omSe85A%QF5$xd
l>~BbrxPdMi7bJiZTp5F28&<N_C@>-ii*bw$#HK#XoSwnyWyz1F>b>L6jkmGvZ(7s);wmcpUJ1UL-%~;6&<o*aJ-O4&Xksa%!=9wW_d3VRKPVaqg(pk+7!J)
tz6dH7|MV?@vZYm?>!-8bPsv^S;{H%l)`Etrp8J~Uoqo2ad8VJ-(jan;OW%Pateoeq@qJb%%=onH0Ubd$Ji$mknJ;4PyNC>oSc{A7IA&ZXQnzK7@ut&jaZxP
0!rdl8Hey%a94aad;3Qo?ER|)4HZTwpVyiZvqz4^b|W;%#wy+W5&`Xn0=sr<P39(FgIs`-(ZZ9F@pvLr<=a@3e@SZ}&uX}16c5v}o5*_*QUTo6(Z@HhVv)^S
9e?shQbvrFf+}^{XP7B!B_OJBJUfjcp+=4Z%;HzDx2!ik({UT!L(;|etj!Q6!SM;KzU&qkaEQQaO);u^mrRs*dfj1OPAK1T&ht?eL{N4$*>5zSuUO+n0ekZI
5WH;}7+Wg#?TxjUeplWVcac3sCN=tZKFXy!e;=@xB=`L!M;IWiu`ETwMbg}w0&s?HGdl}kChNF#x7O?<(T3q7Rg_rJkiq9eI(f^%BdP@foM==H+bn#S-GdcO
{|F@_sSP@%ZDe1oSL8$H<Ey}DSqV|kY+rloC8w3(*#4k>&4h#Kht>h$2qe<?@lKWbCoX>;Cm}}5VG@G7veAKg`{A}uSW`xwNww){Nwf(5!1=H%@vpN+wee1>
BGu|3!UV-aTz2f#xg%}&Y5J17$_FRSIo!<kN`ufu;+vB5?FW&o!gIC1-4ts7D+b^I$;^>V75BpG3_vD6hf%LxQQ8Y%hHQ3_bRIgIrU<S<_S9Qk@CHvI(FiAQ
Duv3x^KR5$_VPLya=U>giagbnqk^e<^i;^5!ihVYBhQm*;7@NmxlBlY1(skvaZUuU!JX==)2C1f>A604wj)8oF`4*p^XelB6sIVV*1MhNyF_9af%rx-qZY=&
<X4MptlWynHfn(t<2MZbCmkvDQQbR$<>}KJzqly<@X|(cgy)$_nbgibIVM3^P^t*1(nHgpOoZ)i3sj3f5#md5h$6bHRdnH1bhVae0tul2t|D>#ppB&3gS&co
Ba!jJM^J}HniS}z&uQdP0+1ntu&p@nvbXpu`mKHRCS}4k(rgCx^NIh5K`fc?*acVlIp<W$Ov{FQVM)Frh`}NGpWg&L`Jty-N_I6bKQZg{eq7}K&X_N^BczOF
Znq793Twr<iF-)e*+lK^Gr!})ZffdAwcEOmMhnd{P)Cr~nf&)I@1&j|!XG|iYSj4p5f=HhTCXkI%g<^yD++I~iu3~41zX52@23Rby`Z?C&pWeB{f}B<YJe}T
Y}B0$-Aaz4=%FEYwusO#fw$cG+?7Jf${hZ$c+6n7SfxLs^B%UCcaIkHl(TAcCU?jUfFU?0?qLn?3E-=&^)N9pHnugrB1<q8_qw)2@{f4=HRI7pzB3d!knOnD
9Af_HMN>$|SJ@m3>|w~BR|`jVV!h*+SsUfDQCs3gDY9f(&8-Fv|M$+=uJatHB>w7ze9Aol;h9RV?f_W5qpfRf$o|yF+s&lnJ;yn{R5jIk>eeh0uBUT8(iSV{
Ahh^9MhrYXPm{Z>{%^wwtD=lxn3GC6FiRu6x|uQ`8GyVjOsGs;m2zqqC->!R%p2~%TF~WgHONPP>!+|zQE?@;wSwYq1fT4j3R)T|^O^ib;?jgA3^M_VV3s>f
8bqqep2P8k@=4H;Xu!4?6W?@c@Nv*Tuxx@Sqw>sy7vc(}2LC2Q35&~sbf@l7Hq{Hkyf3FzrHO<N&!?oZs4__n?6UZ;a$-sl1)U8m_k>3+A4Wf!DGJ37^?{&K
l1&~Zm){>spD)SlK|ZH^rPT1(61o&i3QoI&Yq0fk4|6=!lg{|HMIf8_aVuachpfQSqx{&Z38gEH>wEaHmE+ORQ<QDTR!(rqx_ekMDF08X<lUBqfVmiTklx4r
wPL;Ylrqp<{-YX~kTFoi<TO_k>#Or=LK>3fo?Ah=THpCAyfwp`nsK1!>~bWet67&J$PkpmWSq0AF?PV}!b8A;?uEm~b9T-TF`{z3V6J$nbvBJlJ}_~lF#1va
(oj}ZaGo2)6w}!M9Z&kF4%aMN;+}y0)RRzw%@i^t32hzSQFD{~G$t!<aCta!DIE3w-iu8d)_y~5q?S)=Ix}a==RH?j%LrKUX-Xg7ax~m&%Iw8dd2DI<eLXc>
*ahdE5YCIQ)hXM|<~b>P&Ca+ld_M+5rPpO{l2KqLl0a3r<id@Kw=V<<VQdW%2sL7T`Fk9+cn-H=ARDOUzk5&n=WZWNl;(3qMWw3~ejsqM7&hSZ8_HPSunJ*b
$`YWe&Yr2}1otO&obtyoQP!$y>czXW(`(k{ktQ)m0yELQop*{<RZv?%Qd&s%ZG~vq!mz^qA3i=4u<C(9pKTL1G0vK*+6?HeNDE0B_vIb!{o}_VVkrTLA19m}
DN0ye%sw7MIT04n9R9FMvO`t;P{F!MSctO31>d;B5llbk+=UWjs6d^n;CEzmJL|9+!QgMfw_1=$oQcl@E1qvab~4${tXP<;ku9X}VTMI?LrI>x{cUG1Rx|q6
PRc9fg_RxCk7ynOa~xm2w^4P}8+(_4G0VBjp{Mn>ypO!Xd`;4DTkd~Np!G~noZ2~HscO;PM3W*&i+=t)4n9L<>+Y3Wi)pJ%!vU8h&mdO|m&B)k1PM=z8Um3~
TE_qz?!A16L7@Jxv>szZa)M9BKQO2$6I<5%4f1t5^`c=~Zqs&vh`!;x*Kr;YMJmEd!b@gMET1mP&8+nW^KuW{mZL7yqZ<}DVS+Io?lPS7?usFOzWn!O=h6Za
R2N*ag{|WBiVPeBy9F&M8^qW-6&t)jdem=Km&$L7P46S<$9hXs;r5WIpiJJgm}KdwgKUnQjJI9U5Y4zejw-6|c<RFG4*fSm&q9=j9z0xVH)~_{h};>*JOQgD
D~f9v4Ok$rD53oSn5PdoZ)2gp=PPQp@HPx{mK)o=!dBf_8+5&A`8Z89UH~>s2SGFrSbScTt}S38A?+ts9tmEGWRqh|QRDY1e7MU7(EX?MVl-IV*zd1}FM#RS
Q?9x;ur`jPy~5#Fpi&j@gQu!<u+yA<j0AzMS?@Z@gRMcj7x+cjo!4-81i>m!B))EzX@{3n(P`bz;4!cgy4fU8mrZip=HXTQ1VS$B4Ep;~lm??-X>w@mBZfw9
-94p5EHFqJrjkJJ1I0W)mMOt(B`oY9+J815l+W6dt|y*il4RnkEr(O}x2U$aF{#eyxmfX8Cw7G}&mbj8su#EQq_S@^JXdY@Lsftl9B(1<Mfv>k+0{iC#9g`M
O11;zyvNHTqK1Y?A=orB^iD!qm#*|v+G65u>HQ%<cTDM*6#xmtBzoLCrtw>B;&s;f3%;dJLX3Och`M<>6_%pl8JOobOLB`vOfjJ=tws8~>gp|@COYj1arAV%
=tmfVRBc|od#fNov{RP8MWQh8yu%HWmRl{!u<8Ik9&YnWEIdC{o!3qiI_qIwb2;5BLRkS6-ZJBY4F|pya2|e6B8>Sfw12z=r|K1>Rm<q*C3Yrq&mw83<-=Wx
F)^Y~4WoWLXh>|_Jwy0erjLZlbEfKyF(tJgD6*e1b1br}QjIm$C!SLhXcvRu^FtIFDo5i@$U%;SZy+-n$N6BxNL=8@9=GAD=&*0OpkF84xy3{Xq4DCx<*6r3
2V+#l0tGcQBkC0Egtp%6OXa8s&}O&v9t=f5H($z*{=g;KX6+n)yb)9YJ{V;HAti9sHoW=8FGY0~bhrnU`-o?cB(9^!t4Gjp5C2luf<l}g_s@PG1I%DBnmwob
sqE462`ky8Eo4e{5t$#vR*1fn?z&WGOP_76gvsJ0N|OO3l|Z_FNyB&${E}l=CqY8LNfpgofXxu$2wxm&vfS1)TO0qwPoB`+$#~1;l72`inx2v?%Uj$ohRh<{
mFVQN<p%GaQ~c;!B|KMfw)JCl;=te!86kwj^!9$#|5$0Lm|*5x2sYn^f<tTE2Q4D#QyU2j_R!nXpE-tXa-sF>yD&fElZa!{x&+-CyiHQsryiMx++(sus#ZVS
x!5dl({q*J#bRc^-Dm7$om3AFqmcyWpv!?nTF(1Z9vz%ofy3Z9TLmTkHHz$2)urnTc3ISzuP#qn+U65O-pVukVf6=5h8z=p)#*yYe%;H@I@w4qoGjwpU>`BQ
OjX{ldX~ecd%+&Y#7UT#I42D^5s&~dFqQrbNhHS2w-Qwut0`T5UhztQdfElyAHj9<5Yk`mt%jy=)rnJ;pDVht8c>iyQyeSh4H+x*3C%>I3wZxgFR3w0Ke6r(
40NoS`wX4O9dz6I^3T}1g69E*EImBH)R^Vd+6|L#Qxe)cj1HDWzJzIO(|XDc<jN!KV8=9nnCRR&<Dt&(psD-6IV+sy7@&#)uw~93x-dE1q_E~9E!Y&sXe+X_
UQ`u|woMdI;Ymb%j2PjUxfsB%)drnKiS=tETU^hgGd&5$qEA#)+f}OcyD;1Mq4$h$v)QuB!nxc?W;Q%0LCPu>t|!V!9+j^(j?JPP`+|LAd--|1YH<GadV4qF
i+9ZRcdE2xk)<922a#5d%jS3H%R!J}@IiF?VBnP9?jVrj6Um2U5`zBa^eIR-H|mr2+0yhEqqCqNmva1{ajIKpI5kihvdf4R<o>G3YR$_B;-H};TXoRb3eHE_
$QMqeKF~Z*f9sjs;n4%A*;Nl7`ZkQtc9=YS>dgWJ9wk#tDCTPlSOJe5aw*&%DBZD6Ld#c#uTV(J|6pH~+yI}s3|kmj5I5%jdGp`RN9zU8h=Ygf6&WQxI|&i(
-feGN6W}AIgD(UE@wl{sbZOUpLF=cxa{CnIwip1UFT+1kX3#rM1Ubd8&D~hWmM7?0k6OrK*-|WD=w~iuPmrR@<uow@Q#*2MF3_9&@cz<?@@?qcG%UXSaL|c~
*WGcB@VQ=K^-Sc%7s#>*{z4MH3K;i_x?R?fqU@f=QH7CH(;1yeQ1!50R1EenwS0GC0J&t?LS>E3k#Td}eZrH<tKQ6T&}s-?4wXRAj5BdQ6a<}IP&-zAD(iXu
Lz)6o&?B(tYbJtTYDi!&>r1yJK?}#!!0+}cK~Jia)lHge0}K=NwNZ{nW*ZHC+@l3lB`t{QE9a3VN|9Yk;3q%+4r`Ahd7!N~2N0o+qqH*)5PbG|*%}A3H)h}r
i7yIf(h<i7dUlpmx;YSyMfLSHhBPezf7Hc=jKsf(2wF==NIX*Atkpp+UnY24s8Dr<$`$^_J}%&-dqmYw+jJso07`+5Xd1TCX(5%{wfA|8;g+~0bD_mJ4_gCP
7D2W1+(zBkCF0i4!AT%hqh_gs2d7m5(IK=ukaYbvv7zDf$G<&AcVn5oGu4GGo%C%1p5%-q%%HlAW#9ugVviNZTY=YouOg9`x3*UD5UFp~a}@=jxop?@njpnk
4~QrmClC+===mdVV;~9K%gal8cT)l~CwWlM*UI_zIWl=WgCx{5(_(N;r)TZUm!;fT6nb0fjEMs-p3#EDjFT|;8dHxq$KPq1JHN@P{K!%or-@jtLOF2)3=<$!
kbC;?1|W+6SRy}s<4$d5X_=2DGY*OX@&x|qbt$v&MD8uEF=-(A>>~3~VIZtFawZRJM~g%Ur@G@eD3El_BpbXjJIbobz=s_THt5R>W)}WcwdR;0FUBTK2X{rp
|MG^twUH`qwy_qjw1ith92f>1AagKH&;3rL=1Jf<J3;W`#wDHyVL7#4x*pKRKIZ6p2jk(h-t&;iYdAr*FvjF|Jf;58aj}Quk4X-fDo2*BLnCzY9ZjK*b4a3D
-Mb3NuX|joQ(#Xo;yGGCQ_Y5ZxUB=)RxU>5JOHS^3=gOE6(CJ~3DCQ=IH)qKn;`yuh=G-UWIg?Qp+E<pNT&L68ghpm_7FP%i_X(^q8x{6$R~=FXRfs_On1{4
3HzTA@o0`%wVA7pEa|1?l4pm)-uG}8X0H?FqXS5jrQ-WV#fecWE+HNe*jVIGcji;hJvt@OacA9TZgm-dbzgrjE4g$JEAhZ9X{i5m!iZ!8Jz7?*fhuKcdV1L!
HDRh;*tomlY^2@4Vd^+N%Ee%&Zgzk^>7Vq-TBhZGj9C3ocfD^wkwp6%ELmU^&ndaeoq`VO*VLH=@qZSBs~Z6hG*JRP+oc4-m=TuD_@};R%MUdphfSY;(Kdfy
<tTc~xu*oKS>5KIi~5hfObz0rU7J-9)A0{-@;nrwu89D<=AoXppGtRhL#i%-Hxej8XI<YvV>^ZYg)6rrhKBg~@|rwqRSf`twlrg0+QqyPgmZj}^POkmJC8DK
M@7sPz2@-hpthCidIK5+!c14IKvY327;~sQ=g`}Vf!sc8w+G0MS{pT!fC}5jkcZ`3Hs9$il?)h9stVF3#;I)!S^(t`mnl}OavV5=FCxD%WQ>Zv(v}r@(hg3}
5ZIjfFGWLJk?>;X7WX3}oc-+aL9I}w+!l|@ziMtrV3k5)e<&0911Z$k<Z4mbfGYEB^1A}rzStL~hTYU2G_Oyk<&nDO+S6-OEEMj!ms2o!;AEkr6AO-Gu<2#v
OG;7$Fty=W_X_x<yod=NY6xA&HwI~S04zN)_wG^b2C#Xu{E9jx4xEyNM@g>X-n3*M2{Z5A5^DH}w@1tQGHVFX-jq`%)5%-k?+KX1!k03YeHWm^LeAydFH_G9
xXSmKWHVJ}_mDjHck+{WsmedEg5xI?CaD-PX<yTURZLAkYl7PD%-5(I-Kzp0-hmeyJqz%w36mnQbaxZZTOf+1lmyy4ooCg)tC4{)2G)kTnaUhK*dY<1W>llu
rSPQmD8WOS*C70OJ7N5D0NqOs9HX>zo%QoCTR)zMH$){6F-{Uhgb_yWK@-S%)j<xjlui|HTdr?$m|JgM$8PlrBcuNu+R`|A*yogzB57_OF`6*M5XKvQX-WtW
I(8cPW&~$r{qg?qraC$0sA76xmCsd0J#vzK=ESRQ@3we$1uID=@9*@Zz$gj;hI@~inA5Q2fUu!qx7)aL2@ulZ;!?d(MqZlFY!>((R4d5xth+pplalxer|UST
pNY~C{n(L&r!hU*$mM02NEUQo6u?^#q2^dS{1c2ru<u`UsA*6?r+&sY5;l{$r(2zbV+(<dX6QceitnYyimRQ93>>h;b$E7>xjYmB6NcbSS?SZ)fTIt8fdAdR
8od^ewFvX1rRSx)xtmoJ;4#2tALS>;d3x6fRlUmlLvltat@%NeT_vgJ@XS^VD|*7Eq1}BC`G*MeFxA)~-Vd@yxF2vK%?)<<2KX4yM1FFNh0Am=sX5j@mdbWM
;h8eL8Ps6ZN%~W&<(&MKffj4beFRs~!PuGAJmQwVse^;yu7w9LLl-|2t1eqjL+`eRrG%;60xDonVd3r97Omcxe#_jbZr5`ifRZRhfQKiVZ-%|ik%l!M|J11e
wIQYj$@mRh7`GvwHv=sk)O1Xi5Ilq)*mF5+Qe2|Jom$${u8B;BVCl4oeTTI;G3pc{jP2dLP0c-(I$JZf$R_xO5&_B@2Lb=(^yh+SlcR5vR6*%nkPrCo!x$?V
l5?hri?}qg$ou3?mjPDJ%K#Og^+DNSoO%TD5BDn57CYHXK;sdqFSGmFjWGqeeZeVt4RCS+N7i{!%K_$(A7$rrv7I&nBV4pr-eBp7>;YEeCa<J#CShhIo0%*x
ZhBLYom*dboIy73g-49@7_uB@8j>Qq8^}f_T{pItK6o<ffkj>_q#o56s#J(xK?5Jp*)F8jDPzc*?XdRV=X)S(9krX`+^Gy+X){EK3f>-w_6X+TZ~9j$bbTWY
$P8?MtMg|pxh10h^#_z>M+ZJUYyU)Go^$uIbo&zA)M=0$_qgU_A})dux;$<J+q|~s8oyx5p^-(sNoHybRCo3}WD?MmUzEvTmKeemLozzCER@a(1~25b;rL(6
@YF}2kjG8zKxaOlTQ<i|WkX<KP6o}RQzZiBXckjL=vz!U{MV<v?@%zF({w9DKw9Nt{`zM|xDpC=ZC_lz>nB4l@Z9XWYwDrs7cf`6B64|>%Dh}Aq|ayBejQa%
k-W;d4p`+n{xS#S-cCM06}C=L8&)+tvFM%BPIUQ5<e{CWNL?i{3Q<OlCW|!%%v<-rmY)@WH^z2nNHvjXlxEi-k8G6i#X$b=BogsZpBx<Ks8$P)FaocC$0F5?
8(a_&)O|zy+#MpIyd|uj1DT#yoK`NqqXV&Fk{Db3I4FlT3NR2wo<a@$C}lG-YQ>o-ucUJT5kh%2sNQE}C*MyDS3Dj-LHzSczjfjf^U}&8(q~t5OJR&=Gn6Kg
%sPOCEIs^+Xc>u$QFP1jZF{bV{!kSU{I<7Vr60A}G0p@JiBkKI8A%ac(O_)3(qa-!pLfWe$6Fy){G83Csd&wCrW6v`QL%^_Vwnk!FMgV%-e!qk0EiqxdGhe7
2Hi4ska^21ONG_|q5tc64`;BqH(SwR7=0=lE5A7a00rk{|C$}sh`20zx9&gp+&AEHx$YAVPR?_IP;$O9>91tCLCL1mfca@`;&g7vW7&f6oNsQJ;zD<b%rleY
rmF1e0;-6)Rm+1HR)$XzaduKBgBe7hz0e9@8vwUd-z7ZBYeS<u0;ZfEpyU?tD~WygFALmZUWO4^<#Q!ZLbBJxEtlAQjIIA$#laz`sd8qgtF`Ch@qEgjYn&TW
$16l(p?<dSeLpm2{L=W=<>9A^vTqgHvSgZ=FT#W>GLP;pH%cUwljs*Cik`?L0$D1pLfm1j-dCe(9kr84r|!murKf6OL>uyX0~J}c*z^qyzE*mY5&s=U)TNT9
FTr3~MeMoQR?M8BKrZe#^sW8}#kELruV0t!Ou$_cY;OQcUx0%^j9zeyF6Mx3hooysP-;jLufl!-6F*)*kcvK{Gy|%c5BrsHU6@4P@3qufIwj&FKKgt1qno_D
?FUl}-tFLbFhWjbfqB_3P3l$zL9&0q?+;E%a8uRCXybFSA+0b~Npk^m(qj`gUt6UU+9_w$FR&jpya+-U*&Pg4ZCL7<s7M_>AtQQ>FU;sz01bD#K(3fwuZ#^E
#<KocOZRo|NrLKBeQV(~L$1Z4w~%L5F5;GIoHWwVJNwTAFpQE38X$&uH>~SJ1UAtCxi$iTWKYZyl^fB}gKfGx@1rsZ$vbISu|voE6U$yeWn%1#DA2dP7t$Ux
Ewl1@q%0H4G|@0@LpcLhqFS(g24ZvARum3Jf(A@-M#?@p3CY?=FYRyriJ}4gVPb<t_{89)1@$R{)WxS`S<+f@Ar9`etP(hc>MXMnn;>&lj6`0y&H&Ct0W*X%
TDsYS+8F8pAY(JxwVZcAXnLfac%i^$WZ`wAy;<`isg!RiB#Im#NB53=({|RHSPeh)7XltPiVe3FL`FG=f=1P0&7$h^qe~%5VOnG2<OM9T!DgECR-i=0$IoiD
ld1$*l6I@K$0-t}+NvDb*MnuphnZ|J9-d0+oLJ_+nj6mog}uq}Q0=ZH1C6GvF>|YxUoGx_2%A(_7S7E7B$!z4h$>Hv6<r^sh_*qd?z0Il*`A%~p4ISbq7+T*
(~ZZqLbF;OkSYQkkd(j6Afsg237i_Tm+)EdLZ8aJK{z|3c;u(oS|h6V_wF&oL>xsMQS8(oZ9G%9>SkNjWsuzeDAu1`W*t64UxG@u_&V)Vvk64aN}cOI%X?_O
4h&z6)^+3>MSh1CKH<h1lh1@o^KI<tuEdQyA62kF-;F7H_t3}@?77Jrc|tG?PH`6fBx)cRt{pFr4gmJWNW-?gI>|)LSZ06te3H%0>^IReN_Jz0mPDKYV7G2n
jc=sR$=QWU2%Bff!v$vbtvP~+iRlyvfFxV_%=nrC3k<RK0b;VVoQ349C}JyCwoxnD-4$~-bZqTPq_2!UcqS2bAFwDoB&I)uT-?fBo@Z9;03ks|ZBW&-+3Lq&
SY&;D6SZY{^SsvT*})vbl~GO#Dnp>WzJ|j|P-H&UjG(QlHMxtMYMVks?lU3?G)ghNS<A%>C}5x_UCHgddtJZ-#u!(|d7+sx))p^b-MnXk&|}XwlG-aMw-k%g
G!~iF{2Y{MqmZmC{}uX3c|u8WLu<PT)rv$8CO@}He(gQY8SgS{ch(Ufg;3Kmhf^km{|)Dne$fW{iU0@pm>JI}lR+i}p`!iF3F$_28k5|fik9xsuA5MloxfDI
if=X7V~k+RBI9x)=*EQNXwd^oKhypAuE@c4qWqR3S`fZAB%sd@k2Vm5(rwi%!7p}VH|(@|MQD2k-N)bC5Jtawcdgs0k)kfy+19&jOhQgr+|$R06Ue%yN(MeR
-fFD{RO0kBSu|IaCRn<%tjO=X3ePmW2cTMcuRmqzo^v1=o(;d$+T?MCKCy}2^u1qhKa`TR->fbtK+z|YNpDnEv)yNIP?3&r3fGT9cSVk_&_T=Tf`C=%7y)Yn
vqi8v<Y@L=1scPM*(I)aGmT<~kfGSVymu$WsYmX@10#6?;7un{1rwaBr{pj;5GLb`MA?QmpPXMw)QsN@)c=mP8xBY@I_>F%P7HW1+;yVBNyeh12l|0K)p+?*
j=TdhOYYpp|NG5}f@wM+fGZsFIpG!E0s>2GSMSfr)@KGh%+e%ZLJC|s$x+ecNCXv00#+R8A3L=%fXb~@2pmz(^d`0N4x8CjrBT<Wb~yRl%FW%_RlSD6+GX(a
IyEf`g&Y-Q6U6A<^4D0NHM+u1IiixTdjG7TORU$k?M3r--BU}3(G<XDVmT@K3qlupdSI7^u4JNpL55G<zW7N6o+b2Gm7{MiXS5H?o;l@0b-?4YJwu*AmDO)O
VXW8Jk%ildqh1rMCPIEX_Y$GW@;xM6TMrd3zT7<^egg4jA!s60Rnv@J^}Em^8k}XqI?fKp;>x#NuIhMw-YI2NG1;}~#MBwW(<kA-aAILkwQQ1d?xI<4NMElb
PBy*FbxhZ=7*;Hhqquc)RBd|)C=50U`SyWefEiI31f6XQZd>uAk5juQAKOshmk+2bKF?gN9}}nFuRD12C6;6m?2rLKj=Im^R3ND^Ys57CsmT`5Xl0{UJ{vdZ
adTfrO7gf>aAc(i@X->2Ldjn^Fumoe9TBi1kn%y*2t&hr+>aF}7DAoRT(|dsn{xlnlD|X7D8U`H4{d+q>=3|MKXiu8HF%t*XBw;T4VHspJ^eOh&ruF-(~}mb
5{QOowCy-84(^Q?qo!mWSX6|qV&rs)`l%{K2HOnQ@`C{pD~kb7K(VfO+<4L|WGeQoYFL2tyAxhcaUxsI&){gkcL$}qI2!|*tR9H<Q?2a_fjI_=y{G&W7KqhM
QVI!6{GJb!!GJ#P8OJbQze$?p=eg12xx?xGXXeDfK_Sk9F2o=T-z6vq7W<7<+9|{bE6|;DZtILc8%RJw_|{<f$80H%D1fq&epiNG)M;?JxL&MpCMu*NSg*gF
qex+YGu=@;5)78E1HkX0Ks_=vPt^8AD`9ulv;&0(kfY^SM*hZ{!hg~gL~!L8gC=`G)af%5ZNUPWAg}V0)bIfxO83d`RUXVd=Y5tJxS2gv@Ycu&p;!e{9~z(S
ujTsIFy#%Y;xYldmlaIic?JV;IC;rsdQ*D^lv@&2zdmu<s52s$p{(eefl&GNe5=LC99Rg~W56R5_MW)1J-sB6Fyg(chOt%$3xKPMsTkbTyW(=MaI)naG{e0~
o}pG6(&l*6W^(KCsimL~nyeJZ828WR^{2FG&z!}N?cg6X-RKY9phueNKzh{~7^kCA?z|^MEOHBct{CY!J_6qyZ?a{&v**UU1PqXRU!iv-G(Of*up`!F(%{~H
40j@852BX+g|Dx^C=ob>{`osnccFi09|Z_+-yE~Z6aD!d6<WK7p%OlAw9Sce^92K}J^41^pL7XbC;l3dD<6Nh#!vhFz>S&XB>GH>#I@m{z8biOE*%??LI*P^
ZgzqQfPq`<AJ$+dj$<d!PWn3RdFN#0cOOnY`Lq=Ar5p}!OE>X(&gDjk*;!oc5#pXPjIl6y#jSRqD{|fDUBojP9xBPQN*8gdR*aJd-tA8BnqGxM$27i*)0)vd
m^{hB$(@)QWUxAm_J^m~MSF)CcZ0W)qvQ3LstfHoHajJj8sRm|x;wMx3w9&PX(d&|I}+ku^}g8%EVhXL+F)Z<X39~%Ums`}V+G;93r+~VoC^Ua4>7)1{B?W%
^sryl_8D0h)F-EL3M|5-I<*w+`cIIzYB6txJSPt6nm3Ozsuvw7+}^xoJ{pV_-H=ldZW((0f^H`&6e;UKO2?g>bY#<e0zghr>rb;uSBHYey8~gM@3+D-G56X9
C^H@q%rwI#A!$$R7#CoV0GA+rO1$WzUL1t0j5{E~yqUZWMHPi}V{YIApla#qzXfayN+5XHg<y_V+0GwZb^d-|g^IhtjrbyKI0|{Y;NI;<yUN@aE~hf1u0elT
p5i_=1zY11wIDg>72<0JT&l|X%@;}$n?wUo%5sB|u;Xe{hqP<{Da&#xjRGwmnCJdS6o}JuTZc~>3Iff9VZ+YzyP@*dR~%$DCaOG<blTzxME!Lhj>4BtC+yKO
;4zgY&(`$wi=Tvx(HquVqF89kDZ}A-t!UsB?*y8L+HQOP-qxiEHa0!sW*73O9Yi5OMYwcD1qR@lf9eY}+sH`Qr9MpPpRpTl*3@6Pfx*2rSP(9#f-#1{;NbjV
N9D$Ms`|{_gF)_>yR%Clcr(-=+0O&z@@XE8YyrSLhd6v$Qam<wQE_+sXv*&GOlyGwkjF-;Rf3DpCj1Q8f!(l{$iwh$;b5ZhCD=5T(3)Cvqb(A(kK+tlcJ^RN
=R6ButKa3(ycz(Ut=REYdFe;`uFa{AmyjdCe3tGy!A2KIMdcxplEYGFkB1cqPXqz9m|t2%1-RwiBlXX1@>Q0F{3$A+JgALJy&jVaKo%%4k`?PAOsY`nrs2RB
+~oZUgI!oE_;m;qrd*a(L(#}p#=<fyLoTYd*+j{n5z6uy(g^BFVqI$k*iutf;WW__t;B)uhE4JUL|-arXuMi*iGbl4;NHSGs{F!J|Jfy_n@{<Z-VZzb8l?s<
yz1xi6&YR^2&uCJ+MX-=N%GVTNb}g%<}qu_Kz5-ryZ`{<FHAE%`hGAFJg}Qrl}9*o&Sg2Qm#0iS`vS*u5V6rMnmplb|ES<U7IO%`i#d#{_U0tvpuaRh#?#H)
gikWqt2a}tzL8x*x{Tvd7YTCpgvd|9f5!}QUq%?uiQB4@>XBqt)<Pv~kM5aSdG552S*yA2jnm3i)-14a-KHxFR2U&WX9V+wQog#ftwO|K!yx#*w74$(WR(b`
26V$0t7xK?wJP2h*-<i!ScRn&2Tnn$uo@K2abA#$Gh$h}MB0J<SZT2k#^lzTux5f44n7wvel}3bNCk%k$J>u(tInCYzL{MM<905}j3fyInctQiC5mC`<GLFI
COC1Hy7J1WwQ##qK6l<gf$f+WM{Vqsa>aY?Od*u<H)+J#>9Vt9z<Lq_`~S|RZkNJ!VS!?Yu1qzzaS>X$tNC8((&nJIo@+eGL~Y&$hMNM!9{LzcSWXsq73Q@1
9yu{{6({9s9lsO6H7lo5Wh7O#jEQZJ$FU^p|7@^OoM(HU$;ZvluHRTODkZ$rw>%hiu_s6sguXFyCBp0s+bk*GfF2fvg1mx*3-Ca+VMi$YRQ&fI+fT)CnLqi)
lvs@R8H`SB$D)>AUQLH;6EYnkzxD>XNVM@eZ|Ht!za9lms~b9oEjR5YzaMKOb+qo!BAcWp?=K4l>44Aj9CrB1Bgcx*l>I&i+`EMGA-eq_cZVJ*PN(W1^pYkO
Z;H*2mFihNPQlP&(xg+QANVrF%_Mi97p+DFX>nGA_L8meR@6L)Z1mCYq^xS~p?B$=7)*!$_1@|9Tb8%2+B&G~@V`jd1h5I9UvXQ*b(5O4BH4{?Bz<_tBo>!7
e-WZB%G$m)nE;E2(L+suQ(P2D1Fc_azs-1U8h#NofOV@RG#yV8wp_CCFwCW0w3@)L(9RQpTfS+aL@FYvKANn-2-F3E5Xzg3V1u@+p|C+JU$&)M-zz02Xk5~v
dP{gapR(c8(t671>yv_HfIT=pQN<^F_VqPB5aAR$j!mx9|6*wBA#TS`e(K&@a1yLKy}(iD%6!MDuVMzm^M%f6wT9WyAI0`uJm85uZbJ5FB}RzEMvYGwB>LRE
HJxZd(Hn4lmJpHfsJa1sY?{zIwO5@vgwP!Z`)T>oK2xnaBb5sQnX*}tTa)Br(acsS&uxi>tYd`$V{wm4Xwdck|9Dn-a^U07L>x#O12GW0|LvJ?-Cf%!#sms#
=Iwl#pw|vwUzGn2Bp;<UEh_FNtd2XB67_h$uO#D+#qDFZPa$P4Wy>_n!9s+Hcv4W)i0Au49V$-z)WuxVVh@z#Kl#Q>AMtJ$FvECkt#N^==35vk`?mP5??WJR
ib*Q=N6@bU)hZ_aPOL$j?xSXmMNQNP{;51w&{%1ce4i1fnHxR&BiyhHPV*p<B*OVK5>_}C4%-e?E0qo|n)t^4;j$H@4e4|3By3hubTnuZjWGC%s}#y|(w}+l
w;Wna!>Io6`a9@32TxRIO?5|tv!+g+9fWoiJgV^_(*p6^XfeTTM`+!FsuqHz9GdLM^8B5#)z;X6UdhUQ**Y3}Ha*C1pUV{vxT6)FIEDoH`8uzs5fJIKn2@4y
EffCLMxn|p;18z0ngNWSXDlhQs;Sa>!1xYP5ycU~_b*4Bprvz#3)jFlGk1wxzdX-Vn_tWNpW6B>t?|H#Ut-3<#qYtYjrS~z{nats$Smq&gV07*{H}Ubt>r?^
A<?&Wau+7cITzj;65-tB0ZDOR3ppN*`}U9a*W{6NrP4YvxG$j-8mp5C6<$gcbEeRorLX&MDKkXup(Bm#E5DLO%)Ar2dn7?)*aY%>XWIK?L5>iA!E1U#Cvyk_
DTakVEA+zBtDkM3taoYGXfxEm3G)*TMW=yipWsCc5almqEx8N2#R$)0B}h7ONx6Q&F`;8mB%3kSdEro$Ltl44n13%CP2|x(d6wfG{sQ06NB;w?w#EV>B>3U%
Kd=64-I<PL@bHH^+y~DXq#Y0Eg5*Nxap#b0S2yfF$&mW;+f>7*Rw5?AwcEzDjVnqa6p4+gWD%^zKqth?m&kLW4-c?8Enz5NNQa}Lf~3*l!aiv8eU6O-g}9!t
D2BTf>e11l1nr^Mm{>*0`k}=X`dGn+s!tg%jI!fx<C}Q@);EcD5i^%@bVn`enhgQ$MwZF0Af?Dn&iI*?F+(h!gfVepK%F~A58t4uoc<)EIkfk^i?fz~f{?e2
Au?VFrsiJECz=%fvDYeTAuuq6FqE`H4>8N+aFB3l^W!m`Q%Xv?zMfi#S*T!KAQDU7O`~qS;8z$<uDk>lajgr^pt}d}<gZ0sJJb(mUimebZ%H0MQrHS$4HJDi
Q}H2n%oJN89#O1UuY0I<UZp6{-gt)kW?lzIIN2D;z5W0XnY?vbc>Mp~t;~tRfu&7d#kn;_DJv!<U#LgqB4o!V%qk2nL4U&;qY)~;#==zdEK!F5<sE5&dd{#|
<LF8tK1BLb`t|5uOHes45zUC=pz%uv_*O4W;-vx=SSYuL$MtR!^xq2Hy?925fTYy=@#cjcTzAeMwlNx(-|Vw{XSj?KPiq-r>=?GMpoT5<JLg4vX{(7+oFW=e
pN05vY}zKzVUN@&qCE_7c?a-V`uc2F_bvBtX+$o1-u68*(bHiP$dit6jdK6N0=&JwTOwFgH~a`4YZwkGxEV*)jF8$r<+oQR2?yiGnT!IFuRPbe$Nhz>8$0E<
X6rYK`afFSo~G)yY>%%AG7>qZts6d<a!m4hWm|00!o%7hMrQ3p^;uGJ_{J$Ll=bsn`Z9X~O9j=n{#v3J!eN#pRPE@g;N%)5aOT5A4a{J$2?pP)^&c9^wk#t0
RWspn+F#$Q>&XO=0x6c^;--sLK7ZzOc#`#W%bQu>+OR2cb$g-ijQNVNMDHoIw)u*xjP*Q?!4(!kYk69T?r`-2U|3%!As}NS2d8t%xl$w~K5JaAHyk3Nv`xW{
%o{jTO$)w@Xv8nu=8XsE`3pxi$ZVzZf#QzNVDqy1R{ZgRjJ_RYKwO3lfSANgUI9nObq`zzRZP6DOIcB?OanZY!huP8kr3QlG4Sg)wBV8j4;|$Owes-bz|7@m
5IG4GR}c6D@$T8V4wOMkP9L56D${bFXz7BaqHAL}r&86#RpQqP>t<+$iV<}{HF}t-*GxE<&^He+U)xGEeS5fxPs}>VqeIgQG54{nk5e8W=kRDhQ9FBuLNW%J
XpJ-Sk#`V9p?`ojRLBF*L)IkSpaxZYm9Z5+Y<x3?jpXJ?s@;UZNBdh_xFT*dkem)4&_gP3Ya=k2>3JZb=`{x-#lhyAqjQfxuT3)x>6yhoj5jw%)ugyHbubv&
>Q*9U+%3^@8TPl2G*BBI0VhN&j_^#n23$}Tj``E`7v^Y`{TR{h#L4XlnD1_@(BHqTiM2Vd_XwL{q@k5ORyKED@|jMsi0mDMz+#;sc{pRXtb;N~(vHoWT{(#m
Okf`JrsQSoa-HHk=Ge5xW?)N&0>c1k8-1viFvA_9`mP;WIGkLcv(2qz?dWFYI+W11lZ1%@(Eg=!_EK?w030ol8=VPPQHfiM@>Kf3H+9&@7DnP5lAg~=j*)3q
M!i`bms|PveV^xZw!XUTS$Pr(!}*G0aVzi;-CGlY3}2mwo>Ddj6*REeOwR0TtveinfvHf)Yz@p#8xEY6lGn4>o}g9!P(i*Y$8Er1BItS?%G;v%##7pWHY1#_
$fJ3tndw>A2eKHV8xrtY{q(2$fcyO|9cqfU$Ih{0vf>ICpzLHFzb^u~@7d(^2Yh?Gf&{J5Y4M;q`Ft>A50kh<Gr1KG{g6igNz%}cJ!ISxYOz?vQ24PU{pw%0
U1mPH(gMuG&5j=7u7l-q5P0H@ya2r`Xpf3vtW@i&vu~cvbszVtt9`AXaJPnG2CE=*PDvju{?AOx`m!`BBtxPaT&zr!*8ov2Ve%7%OL>|E{rUGBL5gdE(n7b#
n0=o2PQ?mcA>sVH%8fSdfLzv8%FF?^l9(6AxHSU}Vb?_5aFV@0%wGgAejx}=CjeWdcN=Wj*h8-$@>kn@A2b;TvGh;C0s_EptZA1f_y&-N(yZ2pex5;UcAREi
A>kOeE5zE8F+{0=YFQXU*(dNYz@v;=agwl2({%-W)Xji3^{6cv;bP>@+kAbfK=^OgL#vCWyR~TMA)tn(JBQ1slz)B<{h_aY_Hi}VF-_F=MLZ3$=o)BTv}Qgf
YIk&^^*P(y%i~-?VFQ?)ek57M{+{Q(7g7SI1o8{t6tcv96;~EwuP$#q-%Eai0hnux*E|wXnP)^uNUWVDBy`%?J!)#K9g7dkImE2Vc2c)u3$s2WJX)whcYq%C
P{Ae>GI;pIP7xT3_A;`hxJTONFtzc|*xN{4=dv~AdNu1xgmP;O>T2`~JF(NJ|J}IRe|<Wu_C-DsSUDvZ;OCfry)XEhRmw-=g)}1l3fAZ49DL%@Gg-d?sNv;R
S5Caal}Yl27zs#qKl2<HtD>AMstOGnQ)=mve*eXLsE>bW|CuOMbIJXp^l&RwZ5P{zLL>gZF(iD-SBvWhm|iL1+$CkBR&O9@CnZO{Izq%i^q3<ej5P6F;2QJ0
wFa?U8C(HqU?D}QVa7o(`T%k?*Tf6St8F)JOBtP>QVrkTUnB&`&(qt$jBz6vyBW?&{Lf@^<0GUSu{Hv5(eY8Bw}F{A_K={g(L-_Y(^D`h7tZ1|+g4<Bh7|dN
b<kOXfhlB+nDYZ%)3znkL3Nqx2*LL8S>6LQ2fC!#?_r9)`iS)uj}SCVhs*HE3Q@B&q*1YgE>jHYufe3>L9R4?#9fKXcexLknEH4`GIpoN>QGl6-e$(JSk$){
eZ%!f1VtS*8hK$LdqNxi(tyvpeSw5;F9R(g{B708JAyM_5S6{+w7RZOKR=!##g<Xc`yh}CfcP%O=3|Kx*tkCk^!H09(h7#qEqG$Mu$5f3e13RZDRL(>m#4(x
w<1aSG}OO$R>0%RRCF9i^XkGBv!d`PS~j0IgNb<e7|~xYAejAMv3$VD=u_bOR_GUU)C{-3f+{aQw@j9@FtZb{#prAivQDBpdgYL3R`vy8Xy=_RHv3i_u$mAX
ON-#X9@UlNJ~s+MTkm`8JhxQSk@R6T+&!Wzw`}g7t`~SCa|+LVh6ZJ~&H6Tg4%EmZC=2^6|9+PUFeY@UTw>Ci+F3A?r#V^iaWG@yGxxx2L_To@tF7XV1)fa2
I74~0k9fC=U2VCwv-nDk*zbahiYXn>_k%~!`$mhM`!d^%*W~if#v7R~&+p&OX7WUxp;)}uHIBLvI+T*Y{Zky<d4|EUgJsPLMMw<)M;=z1hDq<f)BMMyDSz%Q
2x@8=w4(nlnR0FXYGjKaFN-av_Bm@pIy5Me27MSm1aM9k0<AJ1<mskpH&$i}*IdqbtSO6=R-;aii*~yh>iHJIgiZnPu~|`-hWUN!$7xH0gsrA0EH81nrC;W;
FJ6)gce0Rg-jZwUfX@Jb)e>*Scib>wVLQu@$^aa2$~`A<P^JiV1HplHAoFjCF;6Z1Z|=R`a>!uL;B<<@KOn2?@NqwkYB+b^$=NzQ*PDA3WDU|WCzteGhjnU5
!z5{dh)6J5T8c1YISy;jm{82^X+OcM3Z-dVaUgVBSIKFvdOJQsxInt+8Dwr#u-u3P1s49_zpb`t9CT3B1!w6$lK+Ah;A$e^?-BWTQ?U&(1Ur|0;zWxirLgTx
HRDLPmfeXO6BW(k!_5)*tz6O5ml+VvykE(SfD4n^Ax^Lk;tX{QC7z2`p-hS+X6sVRHTq29T2iCcc<AAPHr`wO(v8bYxV^?IA}oYIYt^>%FJ3Ulrdd8}Z09z-
1A~Oiv&`4cU&tJD+bUf-T8JAKzg}1+*7rWTpD|4z+vZU8N1~>uub}r4pSRL-NfCsPF4o};{$P%0+ZAz(UTZ_=+Uf>o=V`u(|5!9?sKU0b8`gG<6j0YS5C-m*
5@zT1E5;7L)*99O-!n32qZ26rs)}aSdSB>6;1>Lb=s`Emx-7|bt(ha$gXj87#u#MGs|Zfk?Gq77yIocr>h=hqSOZnBz|@au;Xi~ofi5L5rCl!b15c`1WQ}kw
(*4zyD%1OoFhc6xX)aFw636~o>J6Qwo3aWSaGcxXX%20;ITN`%DVbTP+-O*AAS@~(WXwT{G@k|=BlcG`X>;-rXUg{5)Ha=^9w-G=PI&2lgB^i~E(zV|Iwsed
l=S=o6jp1;ePyv3_oSu0X!fE~#`n_GhlJ9+p=6&oWTC`w{z;RL*gEhzF>?|iY1wVNgwj_RxGH8_iL&L?8mYRUw;iFW%;iZ6_hh?LQk^|76iw{rHc-XDULl*U
ep5O=1X7X~cY@y9Osj@HZsD-%&oA9f^oG#_g#09)_qz$zi&$@+IWpJszDCT9TSHyc{h!84NZuV9ynsO+sz>okssnV$hX)L3N-q2V{*~^>`1=k2>Bg}OQ|f5t
H>;esp^2$-N$oDE)(H@TP+k)!2p^#bgHwp%{xILJ41H?r8Xv3ut9j}*)ezco>F#|7$16PTbqr0!!&Q2;s(g>`R4K~aah`1*9>hXj{}p%iYYv-gIjAoTQJ*nO
GOpt940AIl{O*r{4dNOKVB5dFG2c<`R@l8o4!y2!itOxaG$Visv)@37%qDd6>6!Vv!$T5l5AyGB47)<%se}mDlF>ghfOcSn;gezu`S}6Vir@Ih{Q65WJn<v-
-u~lgHwZ_wV0^w7L`1%&epTSZP{0u&x}tSLN!$nhWzSZU#o$k+s@xazQ<&2-ONn4}`corBfCD+ePOs4*wi`f=3MG1kbUJF>Yr%F2ws!1r)5-61DQL%}@eCg5
KOU~;x`kzJLr>-%CX8FC!dkc#^W_VYQyUzW%$YD&IZKR$1~8ZW1$2G*n2Unh+gI`VH5jO-7Q@0^ySHp|>g%)zzvB0%elwD{r(&M|@npv=a#DYw=9i?|G!ucq
GCe#@1+EQL!59QgG~4b@PFnr*WXP2VMzGo^mms9-qI?&P?GkxLCUWQ#776f+(xlCfe-RhTNJoznT^;_|M<p~=M_oSdTPLbbvtJ`JOwbxFe2}=~EDhNN)9ahQ
OzmOW;_vH%0(XTG$@LZNEaO$$@^yXLl%onK?kDXlh(5=Na>#J){O+@>@hTT$Wz0-CkJ`J`@?v^XkLx1ZHo{20b6vP;M6y8*X8u?I|8|XgQTC{7>;E#^KWZ>l
lktwxUQt<)F`l70!~77PKe6)xBE?O!C%BM9bt8cy?fYTnoLM<E{m%kmMIr(4w|*fpG83LE!D3kJRZ~8=(KKNx_cO?XcfUpO7~f#h_D-QmzZKpLwwgTZ9`R*<
=#m>N-%d`we*@kq10Kiq#YL?9O1m4_5#K2PjNER1vQO0Y>N@7vxvyjzlMq&aQo;QvDv24%smY~Z{T{tGrw3vHkFva1`6jo=+X>TxOTl&^Q0+Ebj%$&tiT5Uv
8)&7*gko&$2&`)b@GoxJI0noyQV^Py@=!^2F31@#X~6*mH`MoPzg4WbPi<JBI`NB{*a3)v<=*}AAmXxr__aYcV@k}cPYCx$Er)4MJAAH|vh9UsmjO}d=R4m(
?lUuVgY^HzgTD)mYP>+&S9C0lH5+iQ+gKQtExX{2T2t4YwF-n5D-DDD5}1s$8Cv=2<$R@=41WCx1yM7&2ncp&7?*R)pz0~V2U}SIoC(fczkM#N%K7r2`rqbw
neJWP?8v*+2ZIMS%;Ug@-r{|V`k1@;18c9co$nA+Sq<w>$c%qn`d9j<F-BjEcvWb7B2_Bp1yAcCNvl(NO-AaoT*ug;Pb}cb=I(TJ>~Uqcyd2;>I0sWhRe=Iu
6pKJ_!7HrxZAeTN88pP}=A^P6{z&fnN6%QQk&;M&Wl_-V!>HLmj`-mbJTzQl_Q43QnvAul^3X8Gf`7ak;m+`<a_(rI#TiV1fLph=R<Xo<U0@gWR0!0UuXyp*
^Yp4doi;=8@&tmoT2SWL)K4EnBXJyTA$@NiklI7Ax@#$UG+e9*&jW;Zq<I{#J*a^QI+L7oU%WW;q7}qxpnLYON}NiZB{^`lTnCo*{OG2edX--9*vpw`499lr
o@g%<owhr+oO(_qDX<r{w)NsYL_s8vy#NZV9FJ5uIaT-pbT2K&lI_HGfS;zQOROE!Ewn14>?)cz^gnd0Nb8rG)07L_7t|UQ=X?%8H%@|im6P?hKMM?lj-B{4
m9B=cM=<z~IzEQL)e!sFV>v-SI##UpdXWok3MvR3B*WC@XPy0?Jn#Q|lwxrCzR<K{Bc)aJ0}fU7Y)^=Du*3MuQIUn$&Jq?}B6zURPF8EL@-F`xwk?6pqVP8T
FlDq<ot1v|0Xm2bcP;2t?4`e;V7KqYNVj?WHH(>n73JB203uVFgs96Y)<K%kwm$Q*5Qb=mvohC7xLtdH)bd{$UiB$`qTK-8{ED*Cnmo6<&RE-3KiCE#bH&TX
e){{nw#=B&x(}5;J6+v{VR!kqF&(QT1XY-b1VjqX7s?l?s{T)8;0Zs0w94rt_Vn5f9>*g+lp@gFa9xU+ss-ws2A@voMI7f;xgpy^Z3m-yW<6&?PQiPW1-DMA
(P=e^E(RSp_BQK@aurExN000UESiuE@0@GvGGRi5;Qe)7IDzyU$!KexK8IkQ?Lw-<Hf{}U#N({ZcaukNHPn7xaQv`P%FR&FB7VV_m=VW`q!*WPsa43kFCWg_
W(C|Sr8c{aEW?*0v5wFN?m{IvRn++Vi_1@f@pA_=&n}EV!g@%ewW$U|W2N6)oE8bwsHI~<I~!^Ad)+wy^HYs6Wz}T=7c^>ViU_uGuSZWoTm>X%Y?k(DC?-%d
bRFH!Wo}Z-$iE%B6=Qqa`Qh@%HuMy53w5?7=imhP+Op~}F3{N$>`TNYP<2hOfV-^lIxFr?f%aRR0XTBM(*ibhzfnp|tgWC-Q#dBI;_w3cMV>7Eyt->aR{MZY
#LKj4cDif<l+1Rhtl?HE3ba$^LlmWqC#MMcwT2_r+`v)+;t(W=Tcz3Gx?qpDWu(X5gI7*M7-3!#=I5PlUI!HpKD1Z`&iwQtMfp$`azRId{fyG(`*PT})xDD6
+6v&B1?qUJT!SyMb(Yge;rISFXC?B~NMn|I(LB-viKa~>8yTfUXTB=l4Pq+`R9TJqA2us$UUUrxv7u1{MMH7wm&{e?eQ5Mk%q&+N_6K(-PC)PliG%FoQ8OVf
8LEp;DybyEH4yQQ!|+DOLyEq6s0P?i?O@sKxI|>GOMwE+U?abda%#xz7Q?aUrt7Mem>BWO!tw>7(Ebq^>s-_geS~O5IaI8bnWOvhs{ob1AiER~u)I-LBuTdM
%D7!|Q(G%lkmv6f+b%azTVAjB;d6<?zrw+TqUHTtr|>(l2VJJ`evj$NX_*zxV2_)|f8aC8uOcx?Oy_^YKq@v@{1c-mN!2K}1w}xWvAGP*_1fe7CJQk@q;Fff
n}Q}4kE+I7O<vYztZ_~ei|ZRu+ibU$i-;*!N@hW|kKU&$+1La_&E4Kjh)OEpg>;0<IC;^gMHY#Am8!0muRi(WpQH@XYdc&1wv?v0n+>hsSpBJ4X=2yq$VZsw
A8bJMAe0dpTujbA0=sbM-6lvmlW9#(AF6JMkW!6k2OXhobJLG={drMSz{V^k_5VK;8fI`-Q2gfvvu(7$qK498^m02x>VZNL72ft5=&0HKPu%f{m-ehBa#ZV7
zM!wN{S{eGue_w}UdoPDt;|S+##EE6Oev5O=o&v(B1uZxS7)(!58P_);lDr<vQqEy{E4zYL_K`eYNipli`3oX-%vClwRK1Gw**QK?QdpcJl?VV9ArbE3H>Zr
Lb88x-7zUR+s>|%@|5Lp%J506wbY$(3*qu@xe{+!#8!v81jnmC+i6nDBt&wQ7_VgkwG<r3`Yf=y1+jVr3#kl)2~_lWdR+yCquOhAaQ_|bM#3X`$Z+-u0|l+g
7Pt*lZ<oSuG*5O0+jywEp~^>(QN^K`!?0X<A7uCa^1Z9}X@FJdWN?GN38i3Kg`yFN?eSp^O6QSb0yMEmGH%i2jh;%@u_X*KsYt`>>GUH#l8;*I%DxFu{ah<e
Hynd4zS=^|dIc6jzYzl=OV#*J;(5Nm?M0h4z!;;Q9vRWK8JdL9|07u$G<#T*Hra;%`^uX^AS2&7IY$@dhlW>+l#CHkBLVV}*}Cx`SXJy=O@yR$20&&e8o)0g
4}Vzq?J^@hd$P#e0ZUu2cv@CI0omJdgkoxWi`r2)$FPe5BYyhtvq1Q|oChSL*8f7(h%kJI_cocW-FYre*08SX&8f~7Ipo?Y>%n((%1zGTKYi2~<zOa|fC$V_
(q;J0Cs2rjOiuCPX$owMo4Z@$FEZ&lw;Ri*_`J>1QeNi0Hbp0j(eUUr#O}jiEh<?GtO&~Gwy<))Gh=4u$I}+0vDrS_7j>vLHc~Q$!&o=sn2^sENKq1IEy9|)
&}GA7wC`ztTI&sQv+ZMm9M#HB*@2g!%!y&y6S$$CRd6{C)`|t$b>M@il3P~vv^e?gRBVb@e|*9o)qgd=JvGY_UJlqpypM@_%zig?jao#yh=nxlSdgto#UGDZ
Dz#n2A2a4o1rZo{Lje(RQ*4$9aML_>;M9)D{=Sa;#LB9oo08;bJZrpaLVmh^0aXithzy7HrJ>Jwbj29LG2${#PzTO;aG4&jXHu2gvF~%mZb{Ch06v$eY8cJ{
_xnaV(OJEZEb=7Ne~zwIiU^>IFdJV&CHPJfbFXDuA&0&6afi++a5@<Y9utFT-)tov0g;6MChP4%sCI|3uCpU(t&Y?}G7>Jp%{c9H`&Y7$+jG73@KV}Mp}0%I
(#^5!Rw$O=oHq6?l4dE8&}$i~1^J|m>%P`ncbV|9Knc3**K#Q_m>iQF?>LhhqAKA-9*4nJU0<79&OE_(_F3ZP=LVz=tL#a*GzMWj$kBTyydR)11;#BB+8foH
qFH9Kf)MFM!G=i;eD-o?wjQ3#vuJEcUpyqvjCEe&G9Yy0gNuxh(tDRyh2`m>5mZDViNqbuYWHnLC^2g<nW!M+UWeWO7GF!%#brKV63_>s{=(S2b0~$+mWBWL
CfSzPGZ;w^Mw%E@&KCSW@9Th*uyu7^?>u@<;fc}=?)1Sx))$v8rnR}#^)rYKigspy|JMT?VN-^@SZ1tGDm7+GtDc$|XZW8eX@?TM&z=RPW-(*v&yG@!agO2_
kVPq@#b9?NC1oI=2e`O<P_zD$MGo?=hM1)#!?5h%L6*!;5F~My_hX%rlc!!y8OT!@Qr$uXbexR#E5YmPAUPQaPAoB4GS1KX+Dp4K#~p!+Y$-k1u5dR9-8bfA
dR@jnTZOqJ&1mA!q{n6B`-cYghH8t_aTLFvOE&GPTaUmerYIyK65-j{v=u+srqoe4HV|Jbt(h!!S0BMup?HpRqImMS^v@id2A2LjhWQtC0aa-K+y6MjJX%%i
pVc}cke%xWCSEyKmBq!m4#^_HF2yfc#Xaot>KLxd^yI*7ZL}t)wP#g{AI11G&mb}$(n|c^`KYKOM{B5(DLzp2k=6C?B+zJl<!&9iKDT}r`$d0i3Z$&$$y4ia
A{$I4IFVrbJ<ot#_HB8&LY_2n(`wTkxM-cfngM{uVdvDfDoauhF`U8Y1?KRchQm3xCsXDT@6_a*r~0BGV5P^U@8C2B?+v%tKL9lmlss<9s*n2{C)+l#gj~*M
7mE@Hgv|F-P4{pt6BVU~^;xSJ;!RM)G-(aVbi#oMTv8!Z1>gcsDuqK>9i{}HN!eL0GzWA<$16gIBBQoB#TcDTouGmHZS}b?uck&4nPUqrN53kRp^s3bWUZRH
9eDqVT8XLy5L;%Jgv!q+b1j;+V8Jbc)Pn9XXR8D6|1)80$s}^TJ(wSnk~y;S)26nl1fO~Q`dKW{XV^~WPV=1S0S;7zY$68?vPuz5pR};@i#l|UQP~3-QotaW
wet$~ZZ+1EoMzbIx}%tUTQxD=5tNHZP<b9cnDC(xq^~%*JHscC5a;kFVf!8R87#rn1Bs{rMX^*r95g1p1DUP3^Y^q9ikKy!+s5Z|BGIrGYV?Uu#hx2u#Xow9
f>U{L1Gb;Z;{J=LwaE8ns%&=U$5b>30BRK$oP0KbYv{Mcpu}&yP^|zHs)_XzGRI5Ky3LP8rXG_@a6=b9&5Y-=N=D$vr=+qfNx~hlmR%L<(fek}eT1ODVclE?
M)lf~QjCR(3;ha{h)6AM`QL|3T1;*JW8N-tXV{2xwnJzav<abDcN#&Hd2d5yslYW`BS$O4v|7OH`IuC&t(DPnsZ4)c^+>GLIuF|9yXX3^LPP1U`?7!fIv4ah
*P>}LnJ^gwD=vj-d%uu(a6)Sgw=gS&Gjr|puLw&qTO|ih&F(m7X>y|g8JZa+Ty!famV4O5S0Kx~t!tr}g6GwC+O{x+7(udq5@InFEXdoUWTO|`%wwwxuygKJ
hf7u58VOy@+$mktw>0w35X^q$GjRxAtLEUTLoe0l!_N*7vn}Vs)hKOD#86~p|9hMk{-@dqUbgEN^Cw1AOM}41$k`bY&trM?mu5j!p3~|MEvDhMbT}q|!u=lx
hM&zc!gAA)aT7dAu+C^j<9cg;i1sRBYIx60-j(Tu&1D~z{YLVS^jvV~`g{GufR*PkZcy6CtX<Oa3n$)`Z#^J*cw}-fuyr)+=Xo)ED*MCPyd3mZ<;hK<aGiy8
9H52x0}S933QJwOer6CcMAWqkD#TC?*mAUISz}E<ly`hhSjmdTAxg+VD%V}>5fX1(pr3R`N~Z>uD<2?8rmhIBq%C2G@z%KKQrt?zy2D+6g`{&XP+uNrFpka#
g4Ur5C^a?pTgup-EA%@rYpT&G@=P{%SdC#xuP3lRiJ}U(!3-1xjZQWUPL6O(Bl919lYvy7b4{#P;G?Uw4wEZa4|1q`r-ie2KQ{|s)6;XxOb$UQS!UC_K2@;d
A&#;(%i}58Nh;0GKqK8!zlF-wXeFx+t?BwxiEU%zp0DKNs7e_Fq|o8W@|H0c7(qF?r@qG_MFBW6Ep{o$mz5``Fyuo$xMF4s?TLT3_{}CEhkrAs&j4Jfv{=zW
Tyzq7%2kj{u~En&YRGaHBsxoWvgg$xiSLq2g8Vy|&1uPX6iox-K;hHkysiCs(j{ZY02J_6*neOogVIdQV=wp0!i$Kw2eLW`X&XsnE%$Ml!e=~aRH=6@#yp1=
5XxIDLq?Z!I~}sC_QnunYd<x+CvX7N3lMQnI{UAW(5(S8`L9rtVXRYz+V8n)kXkDhV<if+-kOb@u+#~mbXGWef`fBUPM&)i6@Jd)h{Z^qtvO6@d@r4`zhlr7
HSe6-$?_zcm~pWr>p-DS?EvzVBQ=-bWU353FK#JIKy?l45-D(Vv{g2Wg!otiO!##m>eVjdtry@T#SJ>cP=G{=>T!GlD0De<Pg;!x!Ff#!s_6IjIF?0|P97dZ
;n6M6q$eoWwhFSVkEBv(`$Trpm@&zU^1?);z2P1bmQhm?oH#Gr10p{W<?GR`u0BO`f29%@F=mSY%|y+e?K@Fa6%oe{QzXS~L|8yXK^kX+q_Pe!zd4hL5dYJ*
gNcNnqL}ska~33`?a=NbtwaRt0!mXG{}7#{i*l6X40u749(xj|R`dq(s;T>oO+bntX#L!-+!~2vG|~jH{^Q1C+=g86Q(op=jY4g_L-qA+eUa4?^fu4ReeT9K
4VVW9LXSJrGhz$uh1DvhH4SA>HYtT;Ab{^JJ=Mbni8RQeiy)BaarmDj?4Z|$-cbP}?BO6?MbBAsRiGu$07?`yag?8%&**lSyoBso(7cTOe@aDgmyqBY>ubeH
q}U|_sqY<TBS6sQ5{VT-Y-$iUG|$_sKQZ<}mNrY=U}V<l<=Tz#oJ_N{wVY5BjkWBR;}LbmE)~X5-J@eW(q)&1iCj4g@guC+Xq0ZG?%h&OM9N*o@qAhjq-|%b
LM(2{Z(5Y{5Rv(+d#~6GCxz*_3VLy`wCbK!)HrbuC_es;DLasZKZoX2>Dv%W9T+fhJ{V{OuiuYDIYd^dKQO;UHBmnv&7P{VIc(DIS4zl}p;!;0V)1b(iPZa)
g5R*L>~GcyBx5L#m^M|4QJyOV708G5G{LSfw)$r}>916OU9;RJ(p!U$U>%j+rz9`)ordS)_pC3u6;kcENGA@J)epJDaE&)}>@b~MFuKEjEUjtWo!SmdqBEO9
q#P1z8S4j*hGdJ!r7G46Vm9e(y7VDbF{ux!5lQ|rpjhxG@<CWu0Aj$XrZ%OwI!d)Xmc>(ls<_iUzte08=?3|SG?hVtYexhz`WL52U%D5%@^=kcq*MNX`DxU(
OH)@}?^3qVx-s&nNOONE+-dbf84y<e<b3L(<1E@zg2<=8NC;)XYHR-wlCUamsC}{-*xR7oF!QaAyvpmG-pT|fG1k6{@o&W?r}_oH6U?hZw7wBE3=V)dkk;J~
K=9M_W_V3h|Ji2P)mu>M8q!q1)4~PN5?FZH4OGJb!)b=FNKadOU8lh>MR;}1eTvn#xg;7GL0%QAU$>xf`VR0JG*{}r6#G`Bnz6r8d{~!KVcUx`i2yRKA~HiS
-VWs(D@*mwU66x6yW@DHY0!*x*nbYc>J9-~4VERJ9?KKOuGMsnQhKYIUK(m+;J(G<O4?G5%o!)lU5j00CH$acHyFIWjMpO<uQ&$f7M;UH(_%g!O_)M}f>}2e
877KdRqcijva44;T9iJosZyv%8+>&TU7%qz=MI!hl|VI~r#0NxPO;!{KJ?j(1K({$By$PaVm{x4D2SK37p#}xlj-i8YRRFv!ho(+e^mQ~vLETn3L(6peRUYD
5CTLXBikJ9iwKPdMxM)Q1Juqv1E9k6F6}ik25*4V0LeoYE{uTVe%w|BBh7FzS_9&(@K?wZW%O&r(qhkT!6z7rZZX6T5D1lLpf$n%@o^Q}lI9CIHm-y4_s`K2
^W(PWNRv6~4Vn3SpP^dB!2OE(EkML6>`vJaOS+Wm;Ehl}b{z@BNZmtIF3AZ$apE9xN?jdlK+a9`p%rSU&oNL{_9DdD%;#%(tk=7<;$s|zBT0rg@ruBJ0p}Ey
>gVjhpKIpg<2qdKTDgzT5=8;+tdOYf!{m<<I$E{5Agbu>vQE2x-2NnKy?ugl95&KLphe5LpXEZ9BpcJ3*9xK1(6HflK{h($>B@JfTcWlGOtHZ}0iXv{NKAb0
b`nRcv6t5Ef4t{sx?|vEo7^X+^bbtY`TOnboe`GBasGbu0&|ZNowU!|L4UewkidUlAc1j{<Pi|LDIU^AxwdrWaS4+3>Wt}{PEskAs^)Ur-6gZAjq)^u*^&du
5e9N$th0F61WNp&$4QV9*l1Sa#b~hHt*g*dYx_lfR)<jU#gU+gvrlGmTzbVTY0g094VCHi1!%IX4amH;66cjKeuy=<`caez-lh9kuz}to4^|0NhDE;SL@gv+
m3|ApBBC3Um!OM!=ljI|cVJF?(%koPLW7I0xIC~FV-8wmp-|c!k>#~tWj=6aLtKIkKCC3u7owwm93qnMuA{+fx4Ks3@t1}{5;Ms>u;y|*f{50|M8F-8!x<KG
Fx-W^%u?h0&COt_eiFV7Fmxc}r4=h7-g0Rxf^+U+<k%^JI>w?cTQ}FGum|fV6{Yqe#P43sS2|t>2&eUYO6B54%j8`DE<pZ~n6)x~;h==&oD(!-pVN9y$Js>j
E=2k-O`T{6K8CM)&L(m1HIUy@KhduR9p3TZK+*xxN35h8;MF!$(P3MZ(70A>*OxV3aAiSyRUd;$;z04Fy@t~E&1lq!j;Tx8ZfZ7+Y&RR#Tlq@4?aF>ic|_I#
+iRqQZ6fAW+5t-C&fe|oD0yaR=DACeG08ks_A;;{$_h_Zay%BA79~ZS#KY@@+)<dQGlzYLT7RaXwQ;;cUmZX3dSe3*dG!!%)cRVWsdOv>$x2+)y<~v_l{f{J
HPy^Ol$IlcEASP5hD4iCeX<3mE(t4WrzOCL;%5-}9n0(4DNf=&zXq5FXZNxo#X2Ncw1nrr%&t`Lwz!f+XG$Txv!Sv7#X-2ZNb@M<Zqqr=F1ZTvUI~y*o@O)Z
a}X>ThtxKemlB6-MJRc6AF%&xurER>ZCOLnJ^}H6e8yMFVr7%YtTNOFFr#{Dn1iBAWSIwJ*E9Izm*gt{wW(LR1qYQ_pSp8+CgBtlVG@imwXR9-<SaO{(^09j
G)`BkHNF1r8jD;E+H`s-P{9HtgZc+X>9d#h^3qiPz=hW2h$qV{SvM@}@;^V!h3=aQ1Z0?Tg_mu@6jvuM6Y`V?eJ4<2INyivEsSCyUwh_giLDzK5Xj)1KsB&;
5PZodAU1e~vi9&PA+f#`@<k3i(stt<a}&^WkDCsK!VYTRE*x^g>KVgHxZ++<KM16(mx64>Au?S0+E%<n^5-&zKNe||C<wYUtf>!Azxsa^Nc$WdLaUs<@X?66
DK=jTu&B-z!(XmbP8YkL6SU~95_#Ub6jHr1AVg?krNfB)ZZv|i6B}9^5`(OV)oW$RqbYw!GkAh)g9Vy}f26MSbjgg&WsLOL6yNt$9ZBv|HW|9kUER^4*epCk
P4IeO8fjM0r4FSL-?iuOF`yW4=lgZ4gc#HA^e#mrj~~yLWMsT*hSyl&b(FQ~MP4U&>sUvafpJ)h%|{&bO|g5g_j@?;;Z6K*=rVza)dupYGN$e1v?%iUmu4}e
Doz#)V=qs3J7d5Usq=|EBLf3>VJL;gxP(A_3)&K`vutAFG1M1fzl+V(g3CbbBSc7o_qt2%>TGF4$v$U?<a@XXdsq_SlWGU${q3uDcLA6-6<Ej7J-eOmqVA%9
!W8eD^BoWO=oXAl96Tc*(dN#S<T|3z$&`|qZB^Me!cN+#9GVs6@CThk?f>~c_CmAuGY+N5{eh~*`mC)E(wrkeDy4W;M}Kj#Is6y{vIj?N!7#RRbthTK5SGiB
ZU-^^sW>TLI<+kpp$+!!n${%tvQJ&Opvr0L9Br@`j3*G_A-dp=GPP^_x9sA8X}QQ`;9fo2VJ`{tW@_nnneaht8M>1-VcvY|pQm+x=YHngLx#;4#iWjn=R1<N
ihW$q!X%m)j2UDEMRFIsu>?(=`%KqL3D!VC!Fy<Pcb>~Zi_xaQ$h;8I%fZi==VKvIal5;o33?;4X8(#psqG(DQ!%vN)hPaJrT!<!E0@$R%s#{<dY)TT4@iB4
sePn0jIk)=_v#C3TgX_(wlHI#?d#*j-=s-<dGSdHqw!QS_T#(YH80%4V3KW#r=4;=C1qQd@DZdbPtLHI#>e|eh61@m_2ylN9aavnlFo@nupn@y9(A5~PpX%~
00u})lPe-;{LZ~aWM^#>6rHaQZX#SMi--~mqG}(fu@TzUsp3`kg}Fq8a6oD2UaeX07|(?9^N&P85R;$kgpg006_;owsF$=&tUk5IPi9Vvbroq*g3t*o_!76-
g_}^{LIj!b<xc;0dtuFX!uoS9QrIyX%8uEHnVUy0{R)pd7@=(OAY!B_)K7Ykt0g}_k@<&X+Wrl()2F3X-)T<MlU6r(;oaaW>4P#k`H!h*-hP?(Z>qq0v>WR=
yiHjPs~w*}^Xbfo$-St><v9EjSo1neQF_4v_^V2nGfZ;*)jq8dMqe`zK>;oCcJ>NZ%lQ>0mVbWqhEZYM0Gd!IaU(E9zGg=Y;)K7e(Vmv0RvdQqy2q^Ht5!O-
=lI?`t6hS??B4n`Wu3{~s8anKYjKEYT0{t)&9wNh8F<0dvr@jx*Nu}LGQJ7EvM@~Qh)bN-)-a>S?y$KPZ)K@@lfqND%24HYgF>7n@JPI-_L%EZc}+!{$Ch55
0gmWZ#-!&>r#}-559X90nKW89dsN6q(9$MGyk%|&YF06;$2p=X>gygkX(VnDLk~eDCy-Lj;u9X*(nfH?Fw1p}SutVQ%ar%J0>5Y2Y|{P7rnOtiU2Z#%e@EwD
=?$syar9V}UCSH@5%K~f<m5SbMM188;s4;V{MNk<XmV*|wHbklcY<xD#x^GRGVrSr06pE1leZ6;@+Wm*R%}+5DZ~ijwO~YGxelrnV8Q~8ERY?y5P)^(rI618
WJv4UH|s1t(VNMiM8t?kpeoWr+Ew6Pj{$Vf{FC`W?lr(@Gk<p|?%+P*Vh@S_qMImPZTn{0(Xc}JM7esJ5W+vujhv+aSkhuvkiw-}j3o!Tkmr9S;q)wqbL5Q>
o3rsFxrtqEys?7Y0s4iilycr#H81iFdH;lk3>+eV=ujYib^xlBGRx>ybiALKe!q?C013fw$T;?>*7_bI%Kth>v4&qP11MjB-raxB*ui)oV3hc*t~^R?zflcr
V0MrL<U?y3mVWsZ&Z1(IS$<{%%e#q~nTY=U?#w3nHB-noXEx9aHH3OkCGejJ!)*U-`0I6q#4L6xdi`VND$CmvKv!8Eel<13eiKPj9((AIas%SM!Oh7n4Np4d
_pdyR`kvu>Q{=C6AmaGXZ``{S7W)O73D8)*i1#$85c@jn$9bOZB33oWhuYQ(m&`)AC!I;69=;B;?@E#swzpa`=7+!_t~Uo<9@?92B=4+IAWBR!K>Cl60%Qd{
BY<q1`!-{!^6Ckf`oa7zKm_Chij&(6TdxUKD|Q5ZVnJ}QB`F+k|Hev?;QGnZo2fV8S#=pTZ{CsRzymmZJz~dw&<_KE77#YeD;7(dsrr!EpT|)BbK}hYXxq~m
RwFIN3@R%6i}P~<&a1(yV;H48u`=EB1GXuHrsE>grs$14uwRXL*O=Q7#$Bt!8*AnT@rRj^+($?a9m}sr29GCO#!frCLl35?mEp+7$8mC(m}*|g34X0s7F#fc
jb3LY%>TD?X8+<5zN>xGJhw}2#1ZPe_fd>RTX~WrhZTY9J%{NuT!XdafJSukwa`BmZ|rOEW1#=G8)bR6-}@-s`3Yztu%JtBMu6^;v{3tjvly5F4GXh?d7xiR
L~?``eoMsaqeHrhC>uTlnPNLmIC|fQuP~@T7HjkcKtd0?N@Wv-wTKT4<3FzeeZBxMxgKgB!M2La+LKsV-em2O$@V+ltA)%<V{XD$5iJF9V0QWyeh|b(8k<XV
KCCHmMTx0v7zkd>ES{VT?e`@2XDgt&66B_I4Dqn%QG#%V)jOT9R;e?0U~?hv{$srjH)q0oT(AGA;3m{`goO7Qr{AG^vO0c}6uUR&zt&!76+fSM-99;Yvi_<G
8OubQmH6~xS{dsA+kuWM44V9#$!J7xfe$TLd}psL&#tsTxd;KpEIQ@U^i~khGI^xi$)cn|0wx%edFJH1d{qS;BcM*O_kp#y1hHDbyW;l{)MtL@rBk;?9!j+N
AbIpQ8+J8OMz{K$w^ku_rCd8mjyFM?fYkC25_6O^Yf&9;-rLc<x-T3QNnsuYirx(k_&v|YF!g2KVHksl_2qX|6o=&-_Q;>rZ5P)*2gk?;Hg<gJ8BpM~*WlTV
8jW_Ne|N~&ZIm)&`8Im^6Tuw<d*(b<YUx4R9>S1pmeGYNB<G@qf6JXSjNAMJkLGgE-r(m$fNpv`O329YnPJh9xs)AEM*}@f-@Fpl{npO26$DpCaD(V;+}P>@
P>FA}oj1~tQJf>|GSUFk-TYs^MWM<!up7=`3k{x5Md}<O3kt#h=N+(=ky8&V%+tL(BR3;iU?_NO?R^3)X-<sB)O-6o$bR|NaY34aFjfgle-tU8hqYXW;^??x
wn>&+$%_#ok7aztMSz-`7cM_R(3`sE2TMc?*pMQmY=w|E=O%g4)KOn{*P02#msDnj15p;$5`2#uH@rj(f8_lBnQR0S3hGEqrJCgp^~?l1dYl)4E&gxq9vlM%
g+QABi_ktj{ZT<bp1WN_1pj&rNdV89Q?r3i;cApPLmP<Zx7}f!;cR9;X%o>581Y03wxRv2dy#et!XzL&>bSms9=WK0)W9#q1jGq~Vc!!k(O!(96{7@GU0~O<
he3lK&0zdMq@mdbno-AJz}J4Kq*{WT?xOluniJyj74vyOtU}UBlp^t-ab-pa-9!Y)wrK|P;`}BCAmkY5sA>>tSgUCr@Kyh*zpaG8<jdMwj=J;)84TX7y&(S2
BRHVoxV~tV`ZmxKwyPFOZ54B0Mjijt<JDD+s8Qv}&)5<uSI5cdhq9M)GjfZj59cMXSWZ`sB?w?K1&#)LQgV;mwm*}94!}~s;N|=q{25UFg3fmW%;?F3O+^Z5
{|jld4(qtEX<855Hf4rgUu}Z?*MdV_Omer~Q~L5`%mdo%u{T&Aet>^2YbtcYKY{fO(Y|x-U{fl^+>vc|X3*R{+sSbK)#Zau2^t2Rv+Prck5q^1;XM!&nao2d
-L{7#D)!<TZ==@7Mws=WlOm1&)VoM3*78bp(fVOZ07IHine_{!Y(i|RseXf4Y$+GsZ@WbD!&xgj7T8AU0!2NEsZ+8GTZ>Om_@i!UcD-BS@X;F6EA%eYZE2pE
GpQqv+GKIyR;)H#@1?;7X~ufo*b4x|D4sHC4VX94<=d`AmYQxJGpVbY(wDT&lez~%^bX=E><@irLZULwxB83C&02h<(#`gJq)4)`)0eqc$L|fcH(}c3<z+UW
2r8_>*UOPP6I>d#qEiICt|CyH9i2!7si|Bw31C+lq_2w!Hz}RPFrFx%N~b<?Rs}e|2D%%!)?DR$x;gr+IjCxezsn9?7le-Fo@BwTF!d#0>W<{OJSviIR7qYY
Dg9`NA3pT#*ppF!*OIOdoBdFq&k&%_J9jOeQh$YzEqk{Ah9AOFRu&ODbh%3t3k8if&#<%1nrdL*H5NYwNfEfg2Etf)=-&FuC=O3fm&zbTx1S7^{Iou>-g$m0
`8*>+)LrXN-1fD(ykxd+bepe1lf1H~_`8-fRCeuFyKEcn3bhwW=Q?26QCK57=H@NdKr~(W#9lT$!Le!#KmsUS6%qAOGD_?kD>hl_;+wW^(o#imc#>EEAJ6KY
*fsE4D5^XtRk^fep+km--bwaefEbywDsPYWliQd6aGxDwacrHI4ukIM4%xqn#K8f(DJtWzh)!zC^@4sdK~x%Sh2-XFZKy_3$Vs&mD><g4w;Uj|tL2OI1QVpD
VI$RZQ-}n#-+Cb~b@_6kc^3P7g?ri7rCAt`pdG`|PvWnp=C60%CH?uVYu&2L+nZ~ECX2rz(U~<Pk@!>_)$4jXh0uCtrrHdwnCL(d`Z0e(rWm?Tu{We=AM5^u
f}(EGJgA}P$oIgkG1}7fC-Ew3)-NFaVc9U~bbY^KD@f<xVoCnO38}V>>z{gyzyEVA>N&+@X@yrAm!P1Gke&74jV5vQz7pk29aZ?+T>!zV{4Q6yq%+OX*fj(j
!+ny5D9r8{oG^7CxdgG_bK#V6y<M)Z6kNDV$HWhRzcp@Cg-IVUfPy^=wMQWBMXD|qRH0+#=2qAsF4NOS5#9=lpw5P(9e>zrOTcK5Urd=y&|OZ~tz{4vajZgs
Vk~)!x3j?QQy>5T21dTo)^85vpJ_q*uT}BCAhQhz0+fV)k`gDnHEq1au=RFt@sN#Nlz8@wL*Ka-C}$5(Z;tkwU}=MVz1PquL*Dk?IY*1$zH_jxyHX_+<^b8R
=2|?ai`~(Ua?xIKfM<})+Zf&c@q(=DNT995yq0!)9H4n!st)>l8LqOgXMeDFaRls{ogdS#g7an601qSGvWc3*Tzmj6iS8^)e(|c2>>(k9{aoZ7sIUN67+nMv
v3X`|(h63WCRI}guS#>%mlSYS&q{=tI1tQb8n&V#cs0!`K`etDi#^i6POB}*HznW01mZ3)9-XzmyP56Mm2c7QJemFMgZRg?{Ogb*FP1h?f^yD-pwNrcjnL|M
)tKgG)zT_P4I4+sdm@dds-B9`U{ME{N;OMX8uU9D%39zN5?sO!3g@zxsyjL(cx<96brFL4k5Oxd0MtLg;`o1Xt2)XF;~L-(fFiiKfOy1sV^1NT8>^iN33P$M
OV&9Uj*X><5HKZ3M(Lu<59u+ZmI5#O7=O)F)2yn-_LG+4w6zeNe;Cw;Bh3$5#^O(r)R{346{)W1BJ}%}QN~fMh|L+=6`w`A`y@*x!+G#!SaDNq1^q8K+ViVa
rqp)&`UL+-h?Eo=%6Gg=3|{-7LeR)yO))*Jz;h%ASbYU$jTi#-c6}@~*{Uu8-=sq^7E}Haq~%h=sX{8iJZgJQ%u4c*Kk0|Rx}vDG5OLIyZlI=3U92Bs#{prI
k_Dh-#S)5u<cvVofcrdp{6`P!t+0n@XmqRMo5+VUI7=FO7pKe>p}(R@CG5Dov#vE~GPQ@X)m5CPJ8(HWfFU%5IWRyV(xmhKtdv`~nop)e)**-T70f*#K%Fur
`QI+d0)LzSV$~59o^$~%d4y@2%7;b|EqFl)n7}T<gk5Ku$W2WaTW8{|U12@LOX(53pUirH?|U8e<;y>2st~^)ZX@QV+Zw0S{u88!C)k#BR@X9VXbgelX&Imn
s&DC!Ti#tZ+CmOJOVsn<SsZmnRid5DrqCxc3H=8x2g~pC%U3NrBFZ;bBq-f&ldm^gHX8rX<4jX=B{mkN(kWN01KR?PMs|-=Bpm&(HBS0nuhW-#Yugs7bAh{X
;T>-e(D|Y%@EkgPOmY9F3j4(01rzWm1t}s#LF!f=(@d2pl?RI@0#JTk6!Cm%U%}e2tLr$VJ$enV75dID%=WUZ9mrPtZ<dDFX`vRDyhcx91j1|+fZ><FdOYal
gh>&I^(X}v<w9ktA#++*)~MTg3$ls0T*>13D}00hh%sApV2jG&9D1~NJy#Km+2g~36ljGe`5#R#8UP+DzOYK#QuJX*Zzw-3nTx$7nhf9wfJ9r8V5QwwBqj%z
jhtyjCB%KQxSb{Z*0=CU?@Vn<_R-$Edmj<M$SYbfoM#}=D<LcS{3E9-U}Is`F$O78ix(XF1jXyB>f*)#wGVn`<k^B6(e#6)@}EQRCYp5XcDuA_3neCxZENja
bG5%{H2yqsHf%2cbJMH8mi%dstlC0Cj`SYu<w9Xx%6yV>tS(YLnGja%WNs1s<AdPU87yblhCwrC0DQ5QJO#Ks8cy=mYHwhp5p4AS{SK)~=CxY5Yctp<E!k^!
WHNP|J^9bLH~K$MTrSL<wPIYSZ{(M)ldLF33{(sNy;mmIj`Jw}=oi0KdG}B}wz^uhAWwQ1eUdarx1Z7^UVw0$F;^S4vXs2bmb;U0uKXDj!*GDntdWY(EOu61
uh6fNII@I!S-hEhq7l5x)?RjT&`nxTYlTu2;CzuL;n-Z2s0>Ug5DxH6G#>PG)79`1yg9RSc8=$6RrpqA0>&!WH)C}IZc{t|A)5}(ghV@v&Fu)KWxBIq7vx@o
-?PqjKodZALa7~M^TJ+^woSWfl=dCUhl;ue3Hm)!&p(8uv8DD5-bhF!1t6|h>&YnS4C@6q71*HoAhR$Clbah(iudrkk?@Yvr<b|mCu9*0k_{2diJ4Pc4tL|q
W`ZSUPbh_1M$AU6qDIR|F5!?C8{BaGc|1tm<WP+`mwI|8lDIN_X`gGaIb!a=c$~VOo-NN|xK&YQLqmnw45nW|8htJOZy9qu5Zs4#`H3;i=AQ9)5J$$z8s`XU
ojVn5?5R`7W6SH`l@xm$fl}4(vq#-e$D7ZNDj+E;4zrUj50MNO$duUawuq%SUx67@28=!R*bbUFAM*u1YgO!H6CxtW)Cn(SWyGW9(pO_btn-K~BHsWp=oUzR
*lK8LL{59SpBrN?L$hP$3uo#kPL%~qhUr48;yFK=f*#gQ8Nx$FR+-lb+*W}(M6HzL@!Z4w;fw58t^kbA_!k@&-~9UmMVTMIOAB)q2%hiFzk=MQ&=8+2epG%)
tkb3*b}J+?p^9$jT3;FHtXW$8nt;RE(O>?MXF#H3%svq)Cfr+?*rz><pCw|9LJjes&{c^P%~YBK=_G?EHP+6bg^FaGrCTa5PI63kLQdR-E7FMgfoyo^{dp10
jgV-ulz;8Y(3>laZFjlME*kYjc;@@(nTxIr#4WTr6v<-fbHl{MV||D{Sv7_IOkcD+e#~A2l|@El(uzEfj^w|1AjNrv8YjVL0@$({0OoU#^+JLfx<LbXv)sJ8
;7a+k>f52oz$&lj#TW|c)(VXhA)6{^;^&4=8z1z(Xf=GSqZ;O2hF~p9XMNaE+3_~C1=En((px6X9}ul>kwHwIVPXd|e-CMf`Ab01vJi-Gl>c{gn%P<DMcsqC
G0r9y`0xXIVeT!v&5#de@}=#yGBqO6<^NML$lJxWn%_toDx_EM-dPaD8xukN@4OC)A%vC~DIGaKFHJvQ9pLm3`T%^o+hm;nCRi^6;Z%c05JNGUjtd^9Pm1&L
a*JnES)Pk6XW|7gdM+(3JNch{-^!eoe#|<PRd^q>(~&lFv+XGb2=1}s3NcUX0gqQ-siugR<rzOtkd=)sifF#S(CYunO{;6p9fM=EkFAvsiwYSF{e3&>YpC&Q
y!F$lhQJH;w~}O0p)HB0<<W3Dl1W5_v&H=HA~(@J%)grOGtM5Gq7L%PI<shmmm7SV-W!RL&0xL~m^^ufp|poy`}{{9n=4o>7HPSCN`KeU>Ibv{;84#2&~QT4
;JBRnn{rg49M<t??I%9$Cnikip1fl2I}aW`ZO)97T_G7u*#~GFaY=Xx>@E~uI?$;R${beI-7;7syKD*9&*OGfGL%EQ=D$xm+tAf(3)6NB4T2QxEoNuWg#N)@
5nGbc@`TP_gDukrhj&g&P7zf8C{J$r4{CgX?beW~Ab#Dc^sB7}{GDDfE!gc<<58e}aAI5X;R{Poo)Ry3S@3$qg{v+qJZSfx8*fnU7O(-ECi>zU-OyRVgm8R2
tLMcjD{Nh@bL`;<vKb<0o)39c&Fae}@x;jh3-FpJUIoqXJK)+E&&~%yEC;aS<0}*a|MYKvA_GIK&-y`#6PoC4Ln-wM_=WGWkF)_NmS<c1iEtyC9EMjiw`gfq
>{4}1tw4&cP0#6TV=3^RfvV~mM(wKu3Tka4^4g}>&j|(`Tfj|+A`6%I&rbc1KF}afR1xEr%yA?`Gd7uNRp#F&<*XfHyB`_o*BA`y?bPWUG685)gYu@k0veB%
+k$G0E+QqKn*QdMF&GNKd&}mOmMaihZeIi`mXd<w;u4AKNLJKn62p_RN)rJi5Bba|-sp82J~6qVe`5gpK{<g`Um8At5(_qMg=N;9+HmvH8k$uY8Z5IC9R=Fu
aSYGp3S-$mcp3xW-bb3*xQ2f%%l7G+T^zr%r;U1q_WClBx_ko<VE2bJrb*Q-p415q!~%I!#eWN64B)!*z!a(#wuyUnwI4{xEL)%Mv~5;cKcCAfsE-~Vi}1%$
DypbaV6OCal_eg}4C}k8Q|e2;E9l6%6eGOZK{IU!td{@=ES%Sw_{gE%@niD_&#Xa=YTuM;S~64|MME9zBN=NlC@@CPV}IHS1O}!4jLY%DHLDk&FJW02_*YU)
*Jv6CfS$`GICWLx_=8P|hRKb1)OxyoW}GMYLK|Nq*9M_pc_UN9==hYs{3E{a`NPoqDKacNH*j|GERq>Ql!MKs6kBPjol+Lq^~G=dPezrH@AsCZI1sn>$3~5H
f<LBn7`eu?i`xZ)SSf%cNECm(D;S8MWd<|sfg>HYVJQudC9v4GoL8I|xgyOuuJ93P<lHyUEZaq8!LLM9SG)p;BK{-!9$!<*{KlENG|#dba1~WdHven+J3jO{
J|^?V!s0JobGOTDVDBUjR2dnIU=$P^dampv9>;;tx$I2O-h3=0{m-krrSh}_loxQe-n15WEDv-IGj{Ijz-hdb*2)W55O1t8kQ$Uq;m#m%lK{xp8m6$(=F$Zm
!%5b|zE-9RTOD;$fhut-d}7wCex^hpUo~9BO@6~7@Oq}-Y~2?wwdmW|acgud+;P0PI1a7IZ_jPNOkmW)gPYtgKt5{NO`Xkgjr4VXdU|wyj<T9#2AZb0jfGUW
04WPp0^IGEMu*g8mR<NILpUEroq&qVs;?G9mi)5wg~Yv?Wm_7*TrrwLr{o;kc72w;j!;`3*{y5#4ZQfGiOWtNQe>j$Xyr~=L8u_hR}@vWkUac1tRCV;tkM4W
lh5H&g@mO7H=sjyY^jq5PUT{F=mO}HopVp!!i<@O(ckWeUbE&9f|f5?Tasum3^K;my=53_I$2Eu#fw}#K*F4ggVK?C?uZ+{6Efp(W=1J$YbMX8UlJN#!N<lO
9!$P{$coTe!~Kg2HRAed;T!_uL|tztN2pAczyf&nMaD(S{*_$<i%+0~ml`f0b14W_R|Di=vQ?rEZ0Cd<3_Q>wVKkD6yhfB9qD6oQ7fz*=&oq+l;E-%bvj7r3
kAQ(-9>#8clsCN(>AQf^CeE2aqp(R<Rg*d4txFFGhwFUEhE1H~9!PdsPr&jufyO-);GAIn+TdN+x1+8-uL`sAh&H<Xz=5~G&+1Mda6#|>^N@dne45d*gMx6f
PEjKWR9_oBQgB&j;$d`OEZP-|Nf!`8#*zPD3~@HCU_Dt5h*ex$_nB8uBlDtRsNC|Y&d^<*s_gaPvk->t3Z;zJ;;Wce*QED$n4D0%V}iZWlIF4dbllzm>OU#0
!<a7DxPs$V$2Z>222x%IhdrT{OK$x$2>zvb1N^-FB52BiGbkng!^>owIk!4VTdNhbf?3Z-qnxO|fdu1s0=EQ^Z9nXawMAPiWA<snVq_6wNag9d%(t@ILq!G_
;C(d1Hioz5mnMYt5mjBr!8PC;m$Z@kUt1C(p!=6Anvg9I(cfe;rPb{>kgb18>{a(?Bg7hXBrQHjCLs1)esgT`joA<0$SkvrKe0O8jPKwD0N7cN1B>jU_HYzm
@ev6j6}aBNxAxJw<`fm{MutmTrq{`Az_v00Vl}Kz3CEt$k~n%)>S#vkN~(6V8U1N42s1QP98CM0ZKN`P^Xn}Lc^!Ok6p;BfEo4j}ReSP<n4)+D-30{{e+GZ=
qg>pQs&f+@lO*e(JNA&M(-UZw7LXOMZu-UvrAXt)G(V%SX^{hN{>;?;<>zCkL@~Dur8>j?9Gv6I-EA?^bp8Q8C)I5v)A2ngK`}%1-lY9DqYXg=hu#nJ>GSG4
?EK&r?1u?n;U3HmT*RVn5^=r|+~6WAx(*yNmaMxaz`J2|+xvnv+5mP<{9wi+erxLjIV3reW@b<xf7TO&=lr)_l==yOwBxat1V0E%Pj|XC90()IzN@Q@u804c
KTgO^12p)vSw6#S%Z}}b<OI=cSu5U;0p{IYgCQNMI*Yxyac4T3{Cf9t^GuVoI87XYMpz=_M_Y>hh&VcN)#ma}Z4u6Bu9W_hov|~BsH9lJB$-3l6<>6!_G+(U
=Y@CgS|%FY-!Zk`qNQ_*Pnh!b=R}rB+1IW`A<ZPajH+=IsL5(U-e;xG)NrW2W1mS}5YwYkf3UjU##7+u(3!nzH^cW^;GK|U+NY!mI<TMt2GjutAD59_l#QhI
L2<v6VI9C0b6}PJA*Aq_aS)KXSs45OJNU{F72avkn!l|TIPvlC64enhP(*>fI9PG~YX}i-1}mrN*}m8Rd(^`sWpJQCUQU_EX|&wbq9I;Dh;XoS(tcJb1jfml
Y2x~2#4)E$x~uBh^kAFB&2fwdHId^5L-XU{%j=Uduoh4qJ-DAeUCVY4`C<7nMMnzQKcbat?E8RrFVC#Trhl{E!J(cUTCC5|Y%|uC*Zz><WFm-FxG=iDk$Q(@
`Kv#fT=;}%J9+i(1Ts+L&Ons6^Ucu^vhc3*&owKa7Ixe?Ey#n(AL}@L842*_$)B8*0*9*{Vv+~w?^Wp4G}^;T<xG<91^522IpKzxY&uKk(8$RA8I7pgABhJ>
!f#+n$|18G>$&wERVXkrDSH2@SY%p+-rVJhF0Mc9*8gYq2RfUU*L5~!v}_$;=%E<E)#_F9>q!#kgAyPXq{Z1S-K+~2vOF>-&mo_hyp;IJ-aG)rv)*_TR)nX!
_~a=ZPTvUxfFVhJlkVuLRj~fORJw`J2jLy~T)!nP_%X89!xoKeAnH`qrHo*Ps1AhtGIP4q>34#my<^MSw86m?QEqlr4+GdD_lY-<tu=IhX}?3zoK%BMDjj=~
T!cr?Jnr-fT)NNBp1|!+1NR$BS&7lxssH@O30d^Wb?dc4!QIL7xx2?DpXHvr@xK18fS{D{DO3@B`=>xgfuZSKfT)#GZy{aWnv*ewbH|3JZ1LfScrDL?8~tQZ
$U4Vj33jt{w^UJ=SZ0+a+JSzL&|hk?nmU(QX<p)TxXZPWMf{NH^{*s1ErwaZ1g(|s;GL~Y(Qd8YNYSyOj2GPqXQ3DU<(^CrUCm+%BbNN`u|d~T!<PrCV@DL%
2V0DihozhDU-C-RI7{fX7R;shQTqF=e32W!cAZLSTwF7=V7OEsW{bq37LgHdoB)-hr*FF_)S=3Jqsk!U#5zZ19V!%ro)HqFh~F)WZG^~*8&1mK^r-nZDo5YQ
KN(r8GfvdOLxr1A;q6Tf5MH~W6!s5+G2KyAH$l30r;20EL_hh3)i(0Ef)Kk?47$kKV#c5)7S5GREj9&H>@7!@cT&2rL}kDjP^&v~R<d&==(5j|L&Z7Pk3QHk
%F`2u#7AVf9y@mEs~zmQE|;*$g?W{X{Wkz1RedH(F71GUBc7g>@{!QV5h@PBU)D-41Md>`dD55%3UXXp#iOqDv$AvQ$>tEzcG0)00C?tYOL<&gHa||s%;K?l
{%Qs^$XsapqL?ni+d!in=asI$-1XNpPLW7D;jGx)+rvRKx~e;0L-{rq2F^3yHW{ZzK*h`{tAOLL4dLmlZW|O0k0HM{`^6ayB^dUJ*75<X?(xiuZ!}pEz*9`X
S((ec`+1`fBzUKcI%k=>9OlwQ@OLQM6@2mSu9qT)>|VH{$XGa^)$bTZIn<xFJ$rfTQO!OOcw!Q5$K#~0y3q$>i{U20^ZPxKdK?x8FTN1?%rU+{T?=@drGHCz
k{&M=-I#({WCv_&C<LFl$}XmctMttebH_lpWOdj0sZATGGw@$n;0c!VIhLCxtLjq3VgBbtA3)!Lh;erHCAL?KCLd`%#MXF4i!Yd@C)0!6mguw30K{Roe`4uT
X2-*^P*{|UVwzKVFOQ7os-!z3KN41Sq8nIRI&UORK-n_4s)I(g(98|KI@GXu%+~HRzT+M)hKa6x`7fno!~mz>^;+gY(8>4LIz7lwGG6C+Ff(_v><@*O;Zr<l
6E6i0Dn89r0rX-^v@e;LSrrH*p7Ta$3Wt4X9M|OBkr;LVH*`&Zk*0oTYp2F8$Jw~Fzv1hd&@m(WHFHyOAqH9s#^*c2F*y&B<^}X1K%e&D*rE!FMh*hA&^yM}
)EW<c>ZlZFoXmJ|1d6XGM74q_mJ)&iTRA$&UGd-|yo4JutzAOT@gcDSgj1O#gkHXsj9@V@2RhtiGZSq5&ZZNO>tQH~Fn@SQr)B4WoP-c-WxYr<Gy=<F1K~YR
DmY2d(iSb@{D(nY5&EosbFzBJBBSA7u_c?m7|(i4A7r>UI}u@sVemyQTno0#e)_~LKez6Owsxn5mOy#W6f*6HsJjFY>gS3R=nvw8v>8_x+rVl-z#2`{*{n8@
ZjrQl5;lMH3a|qr&+3;VC-#Qf;-S!rxyRu3sM5fw_CsE>N1-K5p$Q6tay$r#Pr!W$bbRsI;nadLC{T&yziN!sY&8Jx!-AyFMA<sbM&bkt+xY@rCN@(k<N}sH
2p!%)Kvf29;iz7(3Y%ju_7w{ODlEIX7(EJ2IBuP~h>3pqsQA&2(Apa~np{G`cZO7<XhiAA`9GHEJQN{*Y9MS7sN%=y%yC)1-n|UYvfXk*SI!m&Uk!uh;PAK2
XrUd1|Gw`B4sjx3B}JMHhBP%JD#k0@1jft<y$!o7C?m{dzA{SsF|pQqLC4$Ywp~Da7y(nHxCJVU0MAreH@TuTv@-~tmn*lrsQEyW(00q~$dDkS=FOe{Uw0VG
3Gpdq(}zMF(S;zp%xV3Td)ZP5+N-L~AvhMgsMjp%?EK%dPIifCd-ZoKloR)3Rwr&p2npO>ISmFf3tZEABsHb*XGbzPAO~fQ%S&)eQa67c!d)}~mMUD=cp9EI
0A$bU7M4{=(vuw{v8K;WF%vAcJLm!E#xrIq56yhwW%#Gi<e?&3E5;DG@f1Bh<+$@Zp+ewLu0yLwfQoRp(64Td@Zse|h)xKmr;B`t<(sAxP8hTdoI#-T8Qh?@
%<gM7{A-h0Bq2(;`UYMaG?7ONRFcKqmr+7;O59KVZzAwvzcn_>?ImQRuEV^l%WAOZSIDh4t<2%{B&4};r;8&(etZ2x8o$ngXfF=%Gdt8HIc}r(z*s(LdcA1v
xyRj^ECJ;9mQ~l{JkpQiGb}U{r4=kMvn;V(y@Ys`Ow4Uzi?E2v(FEi)3bVWkRZ^NjURQ2<#ND{8Ft<Dq@gtLJhHSG)w`R!(8E~VD-4Vjgw=4YVe5I1goYmR?
xAe!l+(9J=37=p?08D-pZBV^heiW{9n?H=x2-h)CfQY(T)|w>PNe&tWfadUuupmj^@zEpzUK$emTmf$H6G_=49R`AdoK=Lf#07qpLFMbJmp#3<)-p%}ohO!}
9$0L#N+QPOhsH$%c5m&hoWr>1@WGWJE`iN-oKEIz9nXK&EEbyU%u-m6<!Qz`l<SRH;|>F^ZJo`-t~G;vJ-H7|#s2LwnRKP7&SXjK?S7_7iPLmKE?FDlC?U72
dgeCYQoNUf%*1JP6?C105xclr4`^f<s_Xs0mF|qc5`!iD>(`g&2earl^ejAe^x|-#g)7yvQY!3Pa386PEF0ClQJRb{jXQ=AE<4*pPSo|e$y#>N${<Q~0qbX|
IZoqOZ7=|kjD0Nh`#c{}JtS{YG)KP7+q*A*d1USjsJITsn)T?A5#A1Py>ZzTkIpW9J@$~cz$!JBESK+3K1N2A=|dd8MNJRKQ1-mrH#?Kbh?rkQ|MC>*Zb+Y0
^(&f8_kjt3G{313T{U@(h}I_3C3N?XqwC~1W1NhlSXbP6z$3De(*XFhNZ`J~7PS;3E3e?k-hZsT!5NFx{=&Hzy`;gs*7_brP5k$8+}}%Usw*GUJ66|8uHQ~5
y*y5GPlS-LA&=V~XI0~PE6m106-)~~_WbdKd#;+}l&UGLiqLq{Av&|eIq%FuvGV+ZFI3nm!t=*fDjoGqvemUBkeduxd1;Erx=X7Tf3YN~z-5`#rpvP%1hEL(
?x}{Uf{ZY(OgHW>cLbyN)BvO4vou`?WEx5?;0%zcB;;`1=F&bXw{-xHoHRSN6m-Ll?2Q5QrUQ^L@+p`?A&~wDcn$@XPVv!}M{9?A97-wG^F>OFhUu_*badTT
7C|0!kxwKb@jE>etg<n`ULtUj)oqhtvLBpZsXhTU00=r|<drmhAYDRFZxm-kfx|;F14hy~n1iz7hfc+gj<*f7q&P>e&m}Sq^GQ`QR8kVzNWk-^vv#|$RN)ot
=PJsz62!1Pi6~ywu%FUuMX~@&`HP^vLM%&SwUgO|tQrF0lg4C#{_G>`ySnu`GvCqD36e9E@RlZBU4|t@aPC?&VbH0pj*1h3B%_*S(g-OooTjPLOPdP+ZL|(4
y%j1_lCD*z2d&(jUu7ar7>c5pgL5?qHEF+^TiW{vZQCO<8Z2bc+3OsB@GtnUJQQv)G$V6FAsV*_(sdirnvta3R(bq(E+m_s6u-rTZUCRS!**n#ltc0_FH1Wl
@NwBhN8AjyCwmvvUjEFr<r{uP{PYeSz1QIj;0H1M7A|Yh`cxxmqcW@7-#Qy@j&AE4@h@XrEx{VS$tlG#sgDne6(O)ys~bjcf>15aXy}U1dNzCh7vD#fb2LjH
GM)oo{-?1A3(rY(KXK2U2&Ge}TL9Ydw(dJb2ezO;o?V{ROq~yr+0{lMnf7*Kz^o1O)&aJn(9=p2I%PeiK@QIPde%b^We;As&#hv3H)U{B$Z9%dG+GuSoLM>T
eH|<^(zXyv?Yrhb4}yh$>kiKAnaHPW9a=6w^Z+oS@|~5d7OSv>Ub_=!26W_+6eSp|FyjLggG8gq#@%ntxlx_rnHZ<5L`kUS9R(uh0qwvg*)X+;u#YsqXp80p
V~IN26JqV1xhjur9kp7ovya{ZaIDTW0o$a@?o0VJ)Mf)D1oonJL&Jx|^WHVy-eG4zw!av;2(iu)EyPmMi8ea1&a(JUw=VDTKq6Qq34qfI;$R*Z>1IB^q?Wxt
b8LW=Z}y^)fdarij>tT9QWf^?Ro8;QUmr0UA$*1bm7M`WF|l^GEQ1N8nu>AEol*s6Zm)f|;#sADILNxT`|8NZ5wJ4^&ehcm#J)EpT|12*uFc$~`))cM*XrH6
(=U8WusikP(oEO{HR4KiNcIAkn78qEA?zG1jpCC0P-RTdOysnk=>v8NjP7hMh247%=`{hJfwVIzJ%}f_=dB53!T!-*wnA^VBKM^DRrn5=Bo~YG1I8oQ^e^?X
)Yq<o?8jaj3Bo-^?I2X^*MPRo5Vt-WX$D#-eeP6|PD`2J<NP67YE!D6KF3Ry?db6WuC!t<x#gu#j&hwhDSl1Yre|45S#jiwN9W+dPXRI2%<VP4p5+YG^F!n~
h-iEC%hY<EqNN#-f>vBBBZcaMn)Z+>aj9}Zb(}^iUUC=T(rXaD^-vUsKNwaX8KhIx^r`)_+DUD3RNsSBfI!`dgG|b0<c%#a$w@t*N9$MgA-rYo9e)biL&?dQ
^rSj6%l`mFDjE>3a-W}$aR-+US|iQ%-^IJy;HC2@1s=dWK8ft2TQqE;)99#aG|}DKEH5a#Ocd<JBJhSZg9&UnoF^oK4{5_*lcy(i`^~|u0sKbG00B8b&SmNE
Wj&QvSzP~EkB+!36m#d~0q`MERut0hAPMPyPX}tlD9ZCALIBP9=MKlrmUnsu6i=XaK=U;{m2jGzs*tXtgQ24B9k*csgg)FWB5d8gDkZ!?_Fo8|p0&e|bKpp*
|L>};hQzn%l6Yl>!MyzcTbijcOMKIEA_==#A@CX`oDU}L2oJ(7CczuJ&2+XpF#za$W|0rYGzC{L2frZe%LfH(Y4H8oHQHU*K$`c3%hkusBAEpR><|H}bG;F5
ZbuC=?>blZA=RnP>1S|I;{zO~-MZHv-~xK{4I3ovFe-9l`1$59R+5-s#$NCzgZNFO(e=u>@lu(AOMW-vv2~FW#I+w}^_mdiC14+bM&AXHAx0!01k1`ahkzZ5
`iOSJ2_F;k>87%L7@MHA?mkich>H!qv6_K)Zp{auFm-1A=uEHY1{6mb*A|IwFaD10&Eglj{llrmCK$<_P9=d|(0jU$8=4vuOvu8N+Rqh1CHuc)ml!20uZ2K3
GcS28tdS=@bkF%`M;5ZVpZ$gk#$<aRm}Bof6X?A^i;uAgIyqNCZQ7lHE4l9<sVFfe4N~z@Q+MGEZ(Zr@T|r<1boD!uO0?(oO_=|Ko%%G;(-^m8zu<I)oR*3h
_|y$}>*w$)yUbEx7SK+M25JIu|DW>tG-#C5JZNYHMB!~+wCwOx3Dy6q4ooF~nDz*M$lqJIiBj0;6%P}xh_xZdjaF4|m5xgO6khz8$2q>E*(PYoK0cOnmidz|
Jn6v11-&4@-z@gvv7NMfFHMyaCRAS~ho9xc>EWr^RM4P~QUWVd)<?w&0><6SRoF($^uA^Ax0A?$t;jgFUCOc1Xxpe-p4?J#lc!^|FG06m)Qx}Breg}5RM2F!
!T#Kib*bk)Y|7Q2eJShCBZ9H%#$~5xBIjCuZGhRexn^R}S7BtNp1t&lTQUpLjMYw`adv~~L-UJ{Sq(I;b*GD&iFv~=2SB5_ay-<AfA|m0Mqc^V_QOgV1dV19
pkcp)G(DW$m06Y8u9~CIR?FBm_@{q#cC0=y{|m}Y&?cBT8ZS0H4jxo~FFi}KEEzzVc<J6IEyxKTLX_MDqQEHKv=T+NMTII~sn;@=v0jcJGIWFw{8B^*G=#t@
d~G-|&j`FM%c0DMME^yqXvq#&vD<tuLtoDO7GBW~XTXhDtu{3NNM5g7p05_2&mKO{KC32Y9lifo-TCq~fpmbmyw<Gl!H5g_O)%+Do*x7*0@{jGMhg&E+<LKS
#^D5ww3T*pOs#&q@9g(KDkHy}9?@MeAv*>@ya!q`!oI(#aVTzesrC>6-W_`g=KMnu{EO|clFh6x1*Bk<+q4%~JIE5*CF3koRE~(9t31v+n(Q8QHx-V{8@UZ?
d&l*smAe(h=8jbG(<9-gjzS&26l4J;o~e#g0Ca+{Hmaqm*B?Q>anXsmvK~aHL!rO33~-(dt!|Tvz|pR#vc}@tR>>T9#KAf*PXll6ihaZiV!~}8d+JOrx@i8a
UZOTEBc4e`yAauXwUmvRI^4mz;o4v(e=$5X{cqJ0KVJXXfDuAq-8>nAK0lrs+#C1_W5Kx5CaP7eqS|#AuZk_9bYgo8Itax|FW&o*iUyb>D5KO`5d}$*>f<7%
aiIm>Z^qrUx1GEg7xN$|Z}7l68dMInR01_)lntR}N}+<Py=niqQ+=Dd#YK<!<Ns}-EvE`)cWVA?d2}PyWD`m%d5<zeaCiP~ONH)R$B3&7P(p4RU#<Fi7ja3o
0VpKPy=QS%|Hzu+F*(rqCz<=q0!Opi0$5O0mp<)LYq0S|rpGEa|Ls9Se-A~#;fe!ARRjp_Q77=-W^vF@Aj2*h{YEG&^WCFkvXwUlu&y%f#SIND`^Y;RQhVSx
4DfC+3_}R!y6mu{r<Q_xRS*AGz`;9+gJIzk$JA<C9G~Dz3pwsE46?(ZNF8K>?o^L}#$C3d)`QDUeP^&Yt6^1nt9yEDn*`%mzeH#~r7b;r|AmYv)V?^`v<t%W
b+{2Jh%%ZmmUZ*AJXXxUBY92-a4oE0$+OXYoW3+LPBA)kk8L%FarBOz0Y9imm*ERfdr<|`;u85cHZtM=NmZr0doRfyanb8y?;f0l50YhFwG_j8&#|d7#D(8W
W55gx<{6$Ug#2TBd!(*?ZUAzJv3_2unTm6=t(Ur6#y^uc=jLhk!wys*fBF1a($4S3JuR_*qFb-M%39L6K{iwrZ%XD@)NHv0f|BsOOsq_2ev-larVVXsVuDEr
)&ik`SsH1Tmht<wa>ByY`Q~{JSxWegn{ZcQGVflbq5po^<Xk>NojIBwd3KTJIE<@iwku_W(WWs?H`dWHKF4?GODH_3DQSPS<Z1SHO|>54l7?}{xi=JL$7=*z
(e6*$YTw4~vUx#p+W=|Q9BVZbqqIZIG0Q=Pkh*m79hmH-QsA%|3^&qAHMRS;;uF&e8JUpP-2-8OM16A82Ovi8C^A={A1;%;YpxUt95$?kCTR@?5zyIYaTH-Q
kev|yPC#zKA>&V^hNHV8L{JT*nS8cyW2C@dz~?$pbfk6r$$!>$HOScq7lF}(>#PCD(S}%+V9z!DLnmP43vs^qQ`&T@{z5v(V|Ho~<X6RbblV*D9U_KoUJcFX
Jc^6((L6xoy@45WWh@0SB!&%U;!zB)E0(bE#31-(4{8o&U<8w1AP>9G5q2Q1&yQf1F&%?^j0KcH=(XvM+j#6!tmmdRTfokb-w~!EGpLf@A-Eb|<c2ZC`0YE&
qA@jQUM6IWym$q{7VE@P?D40xbH}QG3P#Nb=YyV4=-CYJMYun@i*7=*1LuO8enTnY98utPudp&_OAcZZ9AP|5GtZnPl#x{B9qmcc4R;;!mfxgdI%`8opn<dm
Q>ivLMQ1P<iCs$E4DlVhe6t4do38(qIm9SF68Wz8CjS0g?sBaoYOKbZjy_lqSm+XlSIVxc@B-l+rAN~UE4J`!YacBibgp|2<-J*r)Wkzj%j6&LKaIUsQS2p4
_u8z_MS(a?Zimj~L12^Orf6Sx-$ir{|6!*=S{Xdt+d(nZtjR}P0j|H`UtGr}U)-%5W<Z;}!GW?BO;v%?<3EO;X9)d>EM6gfLD<Wnm8_k{cwXyaifKgevu(2P
%E&FdX;EUmVT>RT6F@MP)13}qw^H*+k%I2zpKqY%-NFvdxZWc{Rw1dgs~t+43@8Bapv!<a=L5ep%5tt2cvp$=`Gmr(_Hm41qN6@sJ8oLw20|)KojOMF!YNAr
75zI(=DYvfXZy;$lNM4}$uF}m*<|3;x*B33dP84XvxsH+H6=NG|Jlyzr0Ro3ZcLfJbXMo5w2hu?=Lg1>q=d<>zXfc5s}Jq_BEqfkG_qP)8!W$+rM41x3Q}3z
R51n3$^HJH*1F^9;#kQ=AIf1r+BJE4_C6j4JJck3@Tr8dH?ck*$)yA62!G|d5VgKV6!Ezi=lOu$V)Uc2cBf-r9#)k0yqb?0a(=a%DHz{GZ6K*g;L|JvYn&qI
q$?VI0EQWoNa!V6n;K-}=tKe(SIox-svM;OlO^(ZWW)QAjg1+rEM)CXeFNm+m7sNLqt^X5g=U&mC94l2iC}d~&U(OV)FlSIkX<9OUbQWa#N>gMt9G3=qLRfB
|Izm;7J#^Jac@jhUHm4XZoDQGi#<NX-Uum52{mTPk*RH>$w6w}3Cl{Ru6x^~$DBH&-wn^!_L*cgq~=G4|9BFnmqQ<vrmBBAp34j>Q3+PwHN%o|RX2T-u^-as
Tmi%0B!BC{x-dB6lDK<cRhPn%!K#|Hh$+O0SjC#DR?FQGJNR*y=07xRsA^JHgUNqp%{ky1i}DDNW)zeVt_s~F;&h+Dv?+^k{67^!{iHmqtMjlf1QQAUzT+yG
e(^(+UdyA2wfZ)0RQg8RrBrn8IcIV}zD277DOL2Img~Mk;8zrk!hiTe4^f*6VGVXC0=SE#?e|z8p4Nok=ZC};anC09Eq@w|sWq)vG)pDNYABaP8t$uN)&kev
sb$B4IA_ltXahX&qUzoH>*5}Hs#{+6hDiyxceqpHXv%(3?4p$*X9$q@$~?HiP6S$x?xh50@Z;kB;uXE^olY*65=U_r)Et}Sg8_9wlyIPgyms%*gjq986))+j
#+;pAxZ1b03Iy;sj0fai9hirSr<|_(Z24}xx8Nb7y3la4!^yhtj!B`hrndB1@--o600000edQ?f{6*|(00Hs20>J4W313-GvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_flavoria.csv')

In [ ]:
# listas/externo_lefood.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_lefood.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<JXN`bzJ~58cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppcnnU{7bdih4?B9z!pNF=Kdc1PQ{j
khJT|E4KRKjFNUmimR`l^{%Nzi4CIj3Bjhg_*Tv2VeY<iML9W`cFNRMLkE<>-IR9vIAMD3jO(u34+3mEFoodfu6C14HvnZSzk+>!CDj!t2ML*@Xb3giH&9~m
QoV0RcX^R)4=3q(+3DD5k!dT@q?J2MM>>8k!Lh)J0LHf8;&3!@Z>LVT95$Lv?tYU#t9ZaC;hC@F%i;S&K9VDE+ffB-b`ohyc+)XrUb&zc^u;O?)yJs3H9oo{
4O$IJAuvi4|Dm`U|MBtlStsl@HCa4Eo~$yrN$W@UX--%mAW`jj9huojooOA#*g4Il&o=+?^LB^42}?M$21KtL)+5^ynxR*-yzLFg8rSZ0$BUs1-y6t&dfOB;
r-`QjV*qKirPpI!1paV$6Rdx@xIXRtn3??3GDvj53tHF+&D$hMGg7&C-npAgC}36Mlwz=_g4Byq%6~Q^b~BxGf6qb5Qh6e*YgSvOABn*ENP%{lveOz|6NLuR
CR^o)`pi*<%RIwf{c#;H!UOASD3_f_Zn0~lEz*+l=u9z}%|-T0x6SYZ$p$=<8n_)kW`&N_P!GY%81owTLlHt@Kjrh<kSP^7vUUaToC+<F!^Y^SZd35RoJ4D#
@@8t+3V5sCO0|y}pNQQ%8DgHA7T9=4(%7RbgCW9%R#6k6kPEC`h9+ZIgG$q2kJaP5(YzU7vz!nf7xHLd1CwyVJE98BaXJ6-airQz0&x_Nn+bYc10=<@MIuNq
to39|2!CY$ub*Xk@kDokNqE*8LbcU*HV4u8C7W=pS`s0!!If3TvMhHSQ=RrUn$9}Ku5|2sQze9($X6%<EV=6LGOQiLKJqGR{yFKj{6uZcvk5R;b4z8jOSaUh
7C73^Q5^Q_wc9Gk_4_mrAZpVEd-|N(;<s7lHa`xIO4?gRs~5gyEjYj}GFdrv**0|l5gNFE?Sc`=F*YtbAkPRB<)$lZpk5v@SLMx&iL*#TARKGh+bgAE!6c)$
AN=IOh5}w+-(}c=OsuwCiVoKHfX0_gB|syeAWm90LRXY9jhfaT$tQDU@^#3Umt<zwcxopeB0*tw>f6*dNGNt^_)KDpy*1w}ejSwEh!fytKw;a^GOgV-Bg-Kd
@4H`p_m#yhsuAaDPflpv!};@`dPb3fA$fF#`J_^6SD2H^kCA7a@VmNFW3t0Uwz#Hm4$iYIFZK179QZDS`>;B9I2k}qE)H5v(4b2O-g%E%<>V8lboGobDl_=b
YN24o0+xne^OfB6Tv*$UZ7$8;scSi0O*s{*8`cxyvu_a%ya;KuhwX@lq5P(L@6A^?k^))(;Tc6;v{<K?KZ5vw3fQHC%n{uK(IR|wR_rap<oh?Q>$$2uFpLr~
WxIN0<YF4FOzdX{;9?)Y>E?QpbMY)0qvrbmY3KWWKFi;|ylglW);UeCL1<3Dah=ylYH!=u@fo72^9wYWt$L-vzv!q<P+HQ|nkaD%kb6FAxpQLrW^oMX&nP01
J+<5(vm!cDUJPxxV-?lIoEmls0KaL-UqTv6#QLmpwbTSx_M|TUQ<;x#|Hsv^Vtv)GN`3pG@{C$5uYQJmo@+&1fS>gk+M%i~(In6713BqR3i-aE@tE<u2XQa?
q^!wuusQahgj`Z!@vej{bz4ovGnF3f>Juvp(B}x}K5LE#CD}#+?%$<4eR}J2*1RwzJPl-*+E`1Rp1{&~5<&hgnOZ+_{>Jk(0V6N{g^b?cu%&RSzUwPazXcU}
S@W4CaG%PEint)r&}Lpw^eYnSSo6@KUi(n2($k{hHQQtDI>J|r5|*S*peiry{B=A8?ht8qAx+rM9RQI`ukYQdnJsO4a}Wqa6oNqdHeq#|l_Xx~P^MtRO5-@=
*QxLbfdB4=&@gWqN(KHZo!`YQXVhUecSA}<&q?cufthj_U=0zkFquzmOyO`wc;~ZXy<(Hew^CRR67dR@k&}gf26jc>cR7D$c*|ki3{7FK+dnGO@=Dg@wUS~J
m*f&GH7EEANE-@%uInodA35}2eQeshn=)p6j}(6tCMdSPV7(*vHLPVIcK#m4ryDj31fWHd)U~qAxs9le-zS^ZZmhH)kFr9+7_}=u((bE4XJ;G15C3wS6}6@g
yIjBb+*Z?e{Iia$wU3ACH0Al(L-%;)tZ|a>cAE9kYvLW_3PkG-TM|c<GWy5~V2RmNe<qGvYFgaa{PvWh-lDCMeDvb8kmfsrDr4P7yGs3f?x7ubxGIgqR2`YP
zXkC&Pcs$<(Kj4i;FcgsOtm(5MVh6-XZo2f(*;L1$XWb-tg?U|5lIEoL-`D8bSoJNf=A;Q)@ia8eKR4)t&KQ2$3$}SSDetuJZC+3q$LrcHr;qD{`x#-PwkER
l#uXed<j)aKMpX-o5c?7p7@>*6erT?@s-giZR7_vP_87<Yo6rxB9vx#^Bm)w+F~f}E!@J<d`s4QuVm%O7&rE5eCF9;5Ym-j>33h1x$g8WC_p6I{CUhD%ZU&e
NGGbhS7{k$C`{rru3;sv`!=8_RmfAdL~lk6ga&iygg`b1yO2=0qL`|FkrSNtwxId!P`)g!aMgC`!JprJjVha*EF|@PLscO!JXBVw;j1F$<u;ajsAvuYfa|+n
@&~}t82EDV#R=YWo$|kt9Jc+OWhT5H4Gk9kjAMA5Kf&FEO=2?HQz)Q!ORl#A#ub3>654H!6eCpoq`5v$GqX3LrP8zNd5!k<5Z0FghiGPd0C)D~z<VxYIu3F{
qQ2dZG*ABZdt9NHOGw8kC4jg*Yf;sUU2qUQ$|Bc8wc<KvuIGidlE&N(b7=d)WlzmpEB_<BuXxKD0L`OMemOf&PgAVDQQw)acyJvU#HGu@vWLA%BU~w^BN}}=
&xbaH0-g#SCS=>pWP?l^pY##16B&MYzv>o;Sl7=O8}h9Z0e?L2!4yah`IjVVtGdg2kdfM)y?k0n>O=eHbOx8rlnNeDk5y;rj*DD=_}W+_8hB#ZqFqWQ5^*+`
;0CG>%6AntaEK8%6*UyL@9BQw_-dMA(8VaU?~E|o)~3lX9P*7^7Y>;><HY&-%henmc4bJ=DCpsuKq3!NovqKo_W`(L8;^yC*YbF@CWFSmJX$L(?V##N(`9kt
tFKv-)9W|?zo!OF<BUNPi2GEGG&hdYARhQGp5o}(W07(!3H%X0b8Q~~0T6mkHq}2kz%7x`w;?*djehvB@g=styuEC~i&)*nJs2HuYGJB(3)71rcbarS8{Z;w
h<SdhPiMEgvnA5n3JzFv56k#(d^c?&REC{KOZv^vA=-x9B&Pq>i!>~}d0QH_Dlu?s^Rf0`(bzJ(pRqHnw~*k5$i3{QEx>%VuVP{-EBwM}D??q+LBT|xfC=O|
LQC8hBJkAiJT&m{iS&9TCLBDtg-M9G_Ux<R7k<Ib@n1A0C_qr3zMTtE0)OabTj}@LStEJG-gWw1q^{{D%E2tzR+}`06$yB2XX@dS686LAiX@=jKBNc%eg#1T
NdeRrl_`3ZmkAPu76N@`qVito)rSs9p<{}9N_9&H2YmID7t*I@9?O{BsPTttu}YgO#<!mPluh1w$&$j9%gp7oa_Xq#cs+hI2x$&zGQ;6mYv-e4#ghkK=$&3&
Dx_a|Ey;Rw`V!EGH=l|jl}j7+%RFqN51U!4^&I_593S3*<Z{c_#)D~>HeJOs3Bh5K&yD(lXbVN3rQ)&vr?sK3a&r7Z-)EMV_G5JM(x|))u@A4FZy@{A8)6m5
g*VTZ`v)0)*KTTuLB8x+{O_y;SjZ_vlo~m^jkKCDl#aQ&6x@Fg)qUcJ>Osx8)ZLCfYr6{jX~!97NtA4Sj5N0fm9g;F+p}#Ml>5h3y}9K+G3NG^Hm}=9raDm1
`8q^nz#lk#H(S83Hnhof_cI<SAZo!%a>yqj&e4#D)MMpMSMzi27V33&sBRC|Q+_b78%zR%IuXWtSblMJuBt~WD@LHkP(50Ci+Dr2D_!$!0i^Jy3UKeBC{A(B
Qe^cGI=70eY0eePQ>Ze@29LXgpfPW2Mcnn)+mgn5zO~<|X?$aq<~4v7DHxpR_aj?mM@}N8g$DNq%~xLVDRv6FLie`}G)mW&u>2ud!vd7T47QKnqG5a2f5f1H
E;Fda>MK!CC`J-c;(&RUxs}Vb%S4!Uhyu*!Yqx`{K~91se71Cv<Yi^KY$2!kt(AI&Dbim)$!QC1rI#z|gc{+f#aesOtXZU5J;+!%yFeA;he6IahBfZMvM^Sy
KTZH&3w*iTBY=1mAQ_jMFu}Gw4-NeFjXFBtH@E~g&T=N7*EH+*f`-=gX_EgLUq*ZE`hCPIZZ&<76;ICxFYXL-l@EBjgTFEbttBBsw{SuA#{y^f^x>b$Un7W*
xH?G$FSE&JXuLq81hb~aJ*)5h+1HK0o!;#GH=GVlBw=&a%D{IjG-&TYjE9fk@*dJO27`%ryV^6|qhYx?tC-U=d<`b%lbBzHi2W7(-?Ks%oZt@TS&u%y2i4VU
f2jzz_*3lgp!nE$<%qs;gm7qXJ?X+I##du;8-AK$rt{2{rNcElpT^p0aBT^unp#|(;ZJ6{NfM@F3(YWNS-NGl?2!gSU#0b|M{#1yo%qO`$4&BrZxA`0Z6|X2
KB1j)?yWDuNE8@oqi(q!p5e_;=kqwa79(3{m<UUAo~6*M!o=n^j|U9Z`X$=lY`Y%ls-ikdV<P55d9f_l&N(8-p==jRb0kIm-VQT%?5Q{@4QUBivmiy;*X%@}
_SyCOF8qRMQ|_qGRRHuNaTxhWJM|s#z7{F}9Pw+B{-zJJ9a)_)l5o}t$>L8qL~ALk6pOBpyWYgLP)Bh2^*zXb#svkz;qjz08O}L*;TW^LSx5<Nz({R)OBoAV
#Ti!A)+jGcgX~OLu9NOE{*uUCS1M2US}=2<{62QbiDj6tQdIA$!OrTX^==j9`|g$#{_HhKx32rRxGwVGCNN&67-Z5MPV5(|<pWM4;8#K)XoSb9xYbk$48I$*
&$E|v?A{Q}fpZ|DYB<49EVIvhtrcpMB|0uF)WY=`7pSU4mP>@45?{rt`b>O;_|*Kmm5YI}g%okbsH=jW|H1pbQ1n~@1R&mT9qFQ`L8lS7!>HVVii|+@b>I@&
fjAwEeN3sh3ZwAoun;e_iq|H)E$w9?pSu}NPeV|^N)q_U;>H8urx1<sW1b(QbWNpCRl3kgLGER#tRY`+%P|H&9t}BXA!YcAt`cE5kX)q_rpX;GZ!upB5>Q$c
c3g&d>&%lThatY~TyoFFg!!XCzkG-{F!GMvqxWz4W0Et?eb6h}dS_w^>#G|dBuInBb0_9ZMW@0<PrzFlu+ZEU%p<xS8^kRWyc`Gvq{QxVA9-XS-Va>)4Y&fm
<~obrcVkk!3Ct&iURu*Q3JR;&ns}$)IB`}`2X{K1TLqO?Ay1=bkto!1*7l*GQp0zXcVz634#C4}AHe41sw?;)ySzu<^rf^evRaYk($sGJ#g$-K^kxn$tc@Zv
aVZ!`Z;eV&RVu-JxL;=Ut5BR2c}6+=aja|4CVXPKd+N<tHR9mI=>g~^NpXOmo1+P#Y+YyvZi()kzA@NCMYW<0v6%!191lOeyk}uNRT;QHte<YtrxLDBV>>k1
jjdQw?v?D*MuSEh$4%YLcj2AO*L#Ob1O8igSRSQiLL?`Zq>LldZeAbQ@EextE;%sfJw!?B?+mr2b1g}Y6bWwcNxjS88_5Yv7BssT@yoSXm+X==Lc7^XbVWxA
priYyzT>RC4R9}b4^cjj$}OcM{hbZ~wx2i<!Xr$wuZo*d=YbBR4i-t!s#fHi(}GyXBM-D1`o_DC;~MKvp0@((RWjY_FB#u9Ud0{JfiGQYw?9vXOPzAcxNQF~
CbKWjf!s{1Nb^5?u2l1$yM0+xTBsUo3obw?vfh+=vrhu_GN~$pq9oUY_)DtFxy1cuiiPvClh<ub>)`>zsY$dhzRIhu<3_N~;e;}plN~MeAtPCccxqH2w)g8+
uCycP`VL$~O<>!b2^N^oho*>As6-nolv=5ii5PB!5d6LI)1E6hqTUaM1^oApw5H7Pk8xgp_?wz|7-Ppe(ddj^-$^Wr(W<1%X3=ZQrm*fWTTp=@A~pndQ+#n1
MoL9=L;*L3UtoHys<B8-1lvTM1ry-Vin_^hw<3o-xA%pHq+&3VXugvu+TrPDM?L_S;0R+BFgJSsbaLFAr57i;x)V8RJC%sH-K0}l2*z1dlZ3&1Llvm8Td}#{
E+}gOm!MWx2&h<HHz86rkd^D%@2#A<d{|Z2w%k29nPBIZcU|^skr0c{+BuxXm(tDp(4^KP%DNfj`;<;XIF^8zUt5C_)A}QxbvS=U@;Y?gXkrl=*Zm6x6l!9U
)<K1Cjj`(CBXm*u@(p-eqo!+eD)kQoDs3R`e-+F~UV9T$=N}3w(Npe`9{pel%mj2sk6JESRQLego~D8qpeMGK0sUPqP0tdErfFSaQKkvnO4q45GpVUcf3TYT
b|)!Alvsl?FO-<_8vUiXLrOWhhr3+#NO!a?$Q~8nC@9Z3qyJz@lUHDVauu}e5$2yiY6$C3ht5|<t9@S?KfD{sTzFTq_{n_DUhPT0GS7c!NVbd&ID@7i8Y!i3
WH!yRiYa>=G4WH;Db;Uc?Gg(-dQ(@}76L0APBuN-N>t;ljuK@k0ZdA~KkPZIc`r(uDR!_lCP76TyAdU&ou$glw&aJ#Ko0jvA?+A{Qf6tBY;ab+JM|{kSu<Y6
aUi9cnI)QwrdQ%Js2zmgT{T*7k)_)q;#nXhE|Pb)#wh(}T~l?~MA=TMtrk307}7Hz(bO9xH8}z)mbA1yk8?=78@^^)RevwFUG4H$UOO*;1>^nB`M*r<VPJDo
d$6C#bh2^*q=ea=9@tQH^C0Ab8PtjC&(kWnfv}k5#b>Ph_tdotdI%0i-b1>;Sbhm7FL}nA@DPcF)7qHJ&w0!D%IP3~6w~z&@Xwc3Wj_Nu0P4C3E}TA9R~jNS
&<|&))kKp+@TOxHDNXBlN;>I(Gfw|tzs=>RdN;?Yc{=-4v9O1ZFF*B+z*8<G_M#Exvo)~X)5t0pI5#PKIu<BU`WSzdX^^V$taV}u+Pq9?=8839U$u*kBr2#x
+Bvv>z5N=3j%%FFZeVa9)G=m{x{WrDZH0f1%IR(k5!a#a-iU{yJ%9Jc(qV^KOMj~&=H5Cc3iFAK6AC~{aPCg2hzaRaG`;SlD>A)q8%UhIVWwOm9AQtx=NFrg
+(z~zP~4uk+3-r0`T~j8(cmA-NVC%gUAXINd$~vR<&YPy>k#`7{jPP^s(~b^#sy7i0dnlTlu@14gk{akp;2p?NupbJ;Qa3O{YZ%edYPR!b>LH?#<@&RxpxH@
ZN=;&uc#T!AwcpQy{6g2gRa{pjvCqw*0*yS>{iY)lId$sH&mB|%a$Z-A)P;RR=n{g2!_TL#}DYTLLRFDw17nlCW>D?caqdP+SSGrxHBfB@qzQ6IM-XY6Q?`i
cG`-*NiSV6NSrVWwEaMTPV3tv<?CN4LI2lMro2}P{BMnG0<40?@FaL1vIhq3Ro9}|Fe{qzT>i>Ex);m$UtRVR0-l@-jS|7m0#UkiFxh|vT!R}&NDEH`UsIOl
kh2wr5&1j-{3ug&Rl;$2a&M|dZ!_wCP)*p!r7W%aKYDpPUG@7u;Y4j~au}b;5H^qanCUZFsKv$p<ZH-NL6}crTMn&?P-LCY;O|FUVoJ5>uWo>U?XMz2?tLCh
T%uhb@fY^qV%@b!ro$k^EO~Zb?~7^Q?9FuFsE*yk+<fU(3-kg2BBngHxTA)o+qNO(Q`7}ZKsXIb$)4+X7cG&FpjU2*NaB$7j$mbsTQTcH<q7_G7+xzWTvWlr
@uLa7r4<nU9|mGR0y7Xyc+AP$tKS(7+kOE53q7_#lC@A`iz_IM#O68MG8<(==XQ?A=adcFWU#UmsQil;4EE3Du(}5O;3v+&m^_-!aV<$gXP|ZC6sAZ_!0t@X
eQOI8sT<H0D5wAZ_5*b=a-=UHXW@K%a190e(BtDino2i_wo@6(2r`auZ8nE$u5T{XH?2~%YmJO%l>K%0HQs~Y?a};;DfUGRZZ#G$e;L0Sl2H1CGN`md-nu?h
p8KLgi4+I-1m(*7fV)g0uz|h)2Y`nasWxL>^z~#%ien}Sx|=DzELsQ7>yVxXkmS29AscqBxtO){2g>oAG14JKQ*H{<_~fp;hM9N@RQf2|lgX{NRo`#aftSAT
pC(uEWcJ_gk6XU>*Z57mf#}CdxI(0da{a}zNj#Kbs<x#wbA2h=#mS`vN9=m({jRP@hGv#PWUaMWS8VL#a=MY_PV~DNJ6-@*by?WJm{PsvLR61%0l4|4yB2qY
5<aA_=3XbiaAfEhl`(pwhDV%Fh~9cs@#TX-L1?kSv!2vZjRuqIWX`*I+sTq!aebm>Uhtz~;A%2lbEih4rcl>JivSe_Sx)Yvo3Q<t2ZVH0H-oBR5K6=CgQ$=l
1hr|{U+URi-@Z(}b+ZHdXU2cvK*3a33`6BsSq9}kS@qFCO-G2U8OP8B)C&!piQ?Glra|CO9y3>LPNjq_J|)SrExx7vDS#;58zoXTPL@FqMxCQaeBMmbe$G1J
e7gJV%~LibgKv&y{`a7^#*8r}aiV>J!{B_I!I^}_WU390$|8qOHfAM7PEl9eTRe;097@-*f(FH26PO!K)a?aXx#ftz=&34mJGKZ??S7Luuo9uHy?(ci=-;dj
0aGplvQe+P_NWjqG3<nWr2y9T7Cdz_F4gphY_=b*KAeUKvCLOVX@%S_iKF)a_B!>v)0BAfMFa%0fb%-?7kxOt4_`2oo@4o@7Hw+>dy_1|VNw_hv-z*zP*&>_
LWuAj@SYIQT~~BlVLyO56Akm7ce<QQ^%ph!ED89U4T}>b5+DEYAq+dlpxOexi@mOG>M-kX!>xdEqg3TAsufTB^(%d&*DCGE0nc`DjKluIm*LjJ3CX9ab`^AR
{h7Z#BDtC3D7v-Elq&uy^w8_mN#L^(oo69vV_B<-k@f+g&<7oF!UQUltAv_yQ;V&sdZfsW{e;(enfRJCV;G8s8T#f7i$|H-l>y1EoCxK~n~Qm1jx|dGBUt+C
=)c@#E-|Ha23lzHUI6^T1@MLo!aQVeoB8gMY;zKGG{$JD6onmVZtQU{qEO-zk5%3WD*1_(g$HudGs|)va$3kE2@QXaiOT>ON}hN4amS~}zi~_#?NQI}?#ce0
MY4w31O8k{*mrj7cR&p<btpO&0%=6421YS4l!%_t{Js!askF9k`dgTXi*8Q7fHH5MAEcT`Ed3bykre8T#M2bk{tqyfhxUH-4E8kQoe|f1X0EYmm^=@cDxYmL
gYEI{QV1u)9m<28VNl*QWm~m8{i888v14FNMVk;<<PDhxxFdfZv53&=Y5$dK4)KV+0o>bQ%D_C8HbnAF9U_(jM!f56%H<Ql&Vd`pAUMom$nWFrJ*Pp`*G?)a
RqY;ZV6R{Q0fxA)k{zrCdZA}vtt21lLqRoH0k&(T1Pi*WcQ(d@$-4Z(SYaIlW=Wt8Vwf2-+8^RxKpOS%=vMKl@Z&_S$`Lq%9GC0Ls%$)=d>rrblN1qBkHr;o
Kya85wFyU?+p#87p(Na%g7gVGfJT^^LqSU@SUmUC8p!N=RXfo%nlvukyy#w)Zj-xn6*K6Y8YMBm$>|7zCOG|ntpx}=S0O-*7kBtRn%f$!UC6q8L>goAQPuLh
!HFc`n}xjvn0|S@zt9oW9eowARoF_9VZ8k79IW{#{lhxD`b3ASvS6qBfIm^Hr1FPYNUU=-=7yCL_f4%)x^oViN`M%EmB?fQJ!adqNC^%_BAk`_Fhl*__4fuu
X)jk8hq#Ob@b&v%zjPXN0`T#LwMosmi=xUE8iM*aa}lwzk7Ddq+N&W!yIfB7(!a&EFTXEC-tlo9eZ-Wl%iKhO@co4+M8p>VIQ7R8!LVtVLehxU`@G5sHwfE9
?ocU~>h3WAg29oc-xIK_dQZ=ea)Q(LzPCaMPUsj2E_UVk{LiCU%E~WjPS|GKU?J8zo;;oKjsB#jn^bR2&13$Ug7V*<M6SzQr_N-Yhvm`up)7aFYgW0X0F6wt
bInp;_h+m53^WxA7<-3o6g8YzfMdd=_USLqG##XZ#ZEJ^Q)WMdSEnT#{Oh3n$xQk+|39~KsY-5(%+*Tcg5t`1?|*8-Be5|4arYY4@U-PHWPS;dd;4f_v0LYN
4<5K@_&Q8PZ>M(nb*gjwXzkTBgXoJ;PgDO<#hJJ`tZfuht+>b(5xi(l*Gg@QZE;li2#uwmTEWF++gm~2)jy>(!6;UD=feyJif<xsS8+lKWZmqP3dvB?BmA?z
)^)~_;{HAgoU15|Fb^vacEWs#8n%B$WPKVo&MO0u9dE8&^L0=IcsJe+m*XCZ-G=6MP4;$L8UobPuJO9v5!U9dFuJz|bW^!RD=&Bf6+^hYY3h`<U;i9ah($|*
DRD<E>@|f6PW9_eY0T24?KHKP=DR7*F&X6NJJG5`2fPY%&#<tKk0PPdNFSStC(>d5wo5bs%Xm4`FSQYzp3FvSvha}3o5V8Yqsj#GC5(G<ckN+ercNgI0JT8I
7Y@MK!w0*+CjN*`7gV&OqYNl!_Xs$wn!dQlz7#4!BwWj6+BralrJDuCQ}3o3Ny6UHepDvpW!2-&`F*=R;3V6MZ^1|oAH(7-SRcF?f8zkmjL(0YmjA34-q1o{
OrUcx+qD?<$k@Mm*|-t63JI2pvahP?E+rwn!#7ill@o6%yw+}HAS!&zb?+NYv>o4fl4*`IL#Rbj;+CrhVtMXbiZzc*>Cal0KStwr)*Jr;(WBGzDfVNXniUjr
WD*_=paSwH7`o3bpWCq=qe7xo;<_PM)%Jk!DRxECNS_&}{1ykU<qx0Tzh6aJ$e6z!@5x8(%OvsQ=ys7UjCiVxD_j}v4NnrGz7`EBjlR=**<4xz3ucuv$Ns^?
LhByvGslFa$H%HtO9~^UD^5;m&T<4uVrd##Q9QAjQC2Prnc+x%DfRbzP$c^%%dQ@vyNPD?N?WX+qd~-j*ByEmr!R~cV~(Kl1Ae{upj=G?6)PU9e8#0aZ(34!
X8qNdG!)%)soeoz20!m~!Fma(W#pLdt8E|L26~!VHs*}$LVmvChF+8dRsR)oSj!kg6F-%!$aLn_O<%(qBivtE`<n0`(teh>D(tFQRqdJg4wIwE7tD&e_4;_y
4BQX+tQE1*I?m+oFEsit+5;+4Ufjl%W##k7f=IEq3O;`O^x5;vtpf}~pGCQ=J`T|?x*oVxjmyLETs~%|iDy}+0g0I<AqPFiGG!O&{Lz@7Iw)YGHIg;$)gjWh
066OLgCdQvXQu5R?W9d#OS^CkA!dIb8)MmtEL*D{;^%5{(IFZJo3oSd#daAvpCs#589|}!9;JZkBN=))vJJ_Xd!>-FrM|l9!57anB`2qf3bMq0K2*Pc-vO<a
W$7(1>sv#e^#Gl){sn(XmRY-MsL;H<u4BaA7}Z+WA@||zgY(UwpHhFh|3LItXWakPc$#Nm-na0@Py&RI854*F7j_f~vDgG8`-MAH=oD<1=SSRz?RM>xiZtx2
Eq`H(tib+0V1&!psXztVK%;N3#cS&4W`|B-!_IqwXZa#P4#yQDPvGU21k5eoSf`D$6g%Z?bBk8|h4Z-=(UrC2J|cwlq|Oj{&#{`Ql@3+BZY#3n<x1gxcL`nV
`|(#TQMBwKD+qZcP=F}W8!lEA5~q1_0~=93$DHP}z+KVzU3Az2$f8<l4@xeTZAUvGBRV<}x7(SeqJ^p!AoRn^1f_Y^$3vu5<8*PUC*(HentWkE0%1x0|2=}^
e1|z})z~j9@)9bpB5e#<IU&zDG=kACo?S8ivDGG$H(V)D{1<+kJchBiOZTztbVfA(Sz6f#$5=ERTb3kDG_jm9pl6<F*Jy4!F|xfQRb`4^doZRqC#5*-{WCEV
(%yqDr%7A{W&km;8Kc1@7FMRH(pvguaZtb_Rc+G#Gs+b}Lz?m-6kFscKEB4ZY+0+~({MSj1zury=Ey$@+0Cq}0rXF{n3T2~(uDeY1C<nj)y0;Z>`Vc4`J}U4
rse*ay0$Sm+^*Z~*SfA}HgX@VHX9V9ZB-1Zc0)oX9qg0wyH};KfiA-*p^g_Qy5}o}5DStwd!i#S*5vOuX$Zy*yE$Mj%+Omydvvb~xuj$;uZq#LFn=X4F=TJf
{PDUH4+wkUOX7{K8I6)i=X@8m{u(cWOOBZINFBbe^r%yQ<sF9Sl^*TC^ELEN*);`n6`u8nfyV9$p+d`u`}8T4V+QE%(0;9VsH*9CJ=IUV;%ssGh-yQZjK@$|
MlmC{%(UI#IgRe9-|tv`qIqyW8)0whTza)nGMP7&Vk$U97=y@qGEuu~9khO|Xitoc_PPY~sGDqxv>x!(DoZae07aJ>(*WlTc+I4XYbLOM@se7Alp((9zeQzl
C?B!}EXF~j<-OdDqx{ET40bPrrKZj?6>Hp*^<Z<0?hj#JT^j%so8gJ$84Eh6c*ic%T-KR;Hrre+LtFB0EJGOuaF7xR0Red+SG!d0$2`*FYZ5N^>4Js%&CGm!
$qhjUG~br<H5cFa6bTP`k1kJAzq(0yM2cITq?-`N0^^hB8*DX)7)KokQUhyBDI5S)&V<WE+6FyZr0+mN@91=7(^Q6n1=-P5D92FrLsSY2w@FDjG<xVQYC6GD
=KV&`)^Of^{T(Sy5)t140Yg@C{Wm))dqJ|j#IK>pDLs+2PhysqNOMzLcpTFAZRq+r+#6LVUnLhV@lz@uM{97Ddj93ppfa4$<%o$|k~Z59!~oYpXq-zsV)B|o
)lEcOfN6a(ZD&-;l;tlEjErYlbJ!F$z<o|)XX6!T5BT!4+8HX(xWHWGpq_%9ZALiJ0O{Lb`3$C%fndSRP8L)_4uY4)Go?Qp@(d}AJ@;oPJ3fOc{wc^QN6qd|
k)<~1SV9aPh^Z=oXaJAr@BzLiIUAu5<9s{5^eT-6v&n9?RL^{OR*p9Eax8x5cq>HT54<UG;G(Oq-Ez1ZBAqNB)>+>_*6@EDt_;SfJ~E49ZtOtUlPK5_=Mz@~
n6Zft6okgbYL=h?F<_OZcvk}*NB$0GVmJxb3icf&Ic0hTXx{X4(FkgchZaVtoyNw|rKY;VSPBKpvAWeX>YtEidPD2xJ0@Kp=aoHjrf1Rx0^pA*flX-pV84(9
XnSa^{oCo_BQTqYb#rJY9|_^RjDcD}#0$}F>cC))5p&?LPHu4Y5+<8v2QkeO>b%WpX8q0|-G9rtt-Hzn7jj*I=ZiXr^@u^jN(;K=JqPL*aJIar1=WKTq26Ir
ko4%$$eva8lTl)9@k>d?BQ6CmGYd!{HRm?$v?+3s@d>KMx?Xdw(?6u5aT>j<(68tk<h1cwO~fKHbJ?aMR?FtCkpT*<<-P-v?NZgY?=4but90Fr^k(B`AzP3w
_0W;vLFwJx7X@Xv6ssv@78a##GPJ$@E|F5O0yBQ&0Svr$(4%5BeHDl+82Hdw6jI`5-|q6v5zYVes0y|=v2X-gcJ6aWlf~>;GedK3R^dwcgXW6>;Q{!EMX}vp
$O}rW>RBIVIN*l0=#ukxN2fV7R6dK4IeB6DT@0a+E1fNNGx6jRgIosQFPf7n(K``XV1Itn1Wppzj?(!Ko2!iu9Nggo8@{pt;a7=C+F*29yZP>$P4@HCi6%=v
wE8f6E?<tH_JmiWXofr#cxkR!UFkP@O<**tgJuQ&kA+yCwGF?sv6togTdZBMAaf~S&$(rMk5yLtYISs|rF2*=vGuHx0<F4SjmiS?d40+Rd%?an3g+R3u*%Au
0B|@HDY;W3CH@Vs0m%2Yf5z04On1D7=H?|Nm(WtBA6r1yWjH^jhjl_mVlw4v5e7<X|7ws^!Cq9C$H5x`F>dPmIKpyw!P1Nv1F`E%+fAnbDS<R6A8TZA>oiX&
L{YrGX%dj#RAkrBowRf%ylS6|iT}bTaajiPpTrg7N!QF!BR&6CtuCmR*%{IQK!lYMYJ0dqZFe}XOsWbd8Q0^SuF*tw4B$^_bK1tkEtcHdpnF}_0PQ-XdPJ|e
ac_<4c$t9z$BSXUZ#JHWFt2r+xR2}yq3Ns=K)}?|w4Iizw%S7As{N>iU?`^acL>O*WWGSePs?E9m^qgGQ3@k4z14nCV72K%YBYN2#YtKs8`_Cwb2OOnCyrr;
{(0WyNuQ`rO+S05`b<*I@rst<d)^0CV_zmzi<%rk2fNUMj|0%(5V~qMtB;3}A>fCS3st|0m(td$OGs7qGQuE~=d0?7<hRZAVQ^=>GVSXNV;bsd`se9XFtn?l
*-RPMH@uv(1np6fu6Q0#9J#Xc&ZU0jPt@ph8kvBPPYMlDoqy1+Sb#GNB$+Btkd#4B$C~yVHtFtpXid4KMo9LrGo99-n<wd6=(Fw0cRcI{I2fC``W6Q!K>4s;
<$c7VOrQ`EjD~X<(9T?wsoDWu0~yQ@OmR%q>oVs_44!rcdt%2W2dPP!JX=>zJZ6$u%4$Gk><~gjuy5=bZNCvaO{wHU2!v8$F#DeudpvfpHl-S;KrS|SWV6Gp
Sv!em$0f#|Nt1;N!uI*~O>gH>a^<g?f`~}e)Lqx0DJRg^RaFy>6;3r*1<M9R*?v9<Nb4Ww7wkF83gK4STApu4^YBH#n+csXyX|>v7p{t<t*3He2?gJC9PoEe
i$(;=<QB^uZ5eJ6z{oh(@+gMcP!~TufahuXTN3$5uH^NAHo-!1I)BD5kP4mp;Fjv#Bx@wjDHC3t^D$RpeTJOPWOs3E%3pvr7J)g*POI?T25tsg)daXxVyEOf
`O_?pb`Jsj0iJd>O=$PJ%G2wU3pBr<iG}4ZN4PzI!3r{-q7)1J#?NFrS37-qB3N6jys{RF0ft}(=XZpOX1yY8{bj-bmI>u}&bny(DbS>3d5T)NUBy+S$}o}b
*Tin;^+U-OKnKutyl!_FPpWzo#tIbEr00DGOE=@{BPiq$C_Xc&=2=nu;DHfk>J)?DXuF-Zw4$bBK(X~xclm<h1FW}imj|Mpz@};G#L^$D4qK_@*fyp)J{y;x
kPGb9&ez6&uM}>|7-A=*OXlNJ9EU2YzzXZfj0=;N_5T$`p*~tUt<<z=>lnoSzWZp5kw-n%8h?zdoMOF0>2e9d?B~8N;J9N?%+_#^;8Xps>nj~G0FXmu_h6Gb
mUof1yE!oSV_LDu%SP-PAs+cCWQR>Pu(2m1;qF#R^IUZ)M%JrF{!f@38U!)d3M6yY&biDEN1I-n0Qp%3;Yn@BO)ebP@$z2;=S*4JzyQ2cjIy^cKheF{s1~8|
0Mm8IHubTQ<jj=F=-z&2;7UOBM}am<7T<D!&a8`Bz>qF2ujRqwj4^E6t>G!2b|bp*K+1lZv2&p*e<n250xo<Z3`I>nZ}qYP_RqZs1rrqnI;S{`5|+wNDq5{E
Zsh4L;1VxGQJh^b0A6m50sc{L)my{-In(F;*sN?bqJFg79KS6xeV?H=9uR<^mg|7KXla*qI#m(dRE|!2b11gjAI8~@N1}tCNCS7>qv5qOnu?!d8L&-Q0M^L+
8qz+uj^#u>_%w<9m5-+XR{pFIN^b0h?$@c(JdMGD*HPcGAd28WM98qn2QF(tHF{3BNK5+)S*%N7-!gFQO00-e8i=>o96jiMg2g=J!D#qJ9s<(B`0kxPmU==M
mhhutOE|lB)oUd?_R*z7<R>%DE_gb&3?s?F01^&PAOEpiY-AyK?!0o9>+z*{(k9V)@o(=hF}5xkjV{S#Dj26LkTa-YV5blF`=^hv(r|3xO&D|4G$3Xf7z66+
dbHGv?M>{+)+5F+6|(5xwwLqSvSh(vyl9xwgF(hz#9#FkSQA&`-sr-CQ)mqneJ;(N>A8ev`q3xR-P{<;_IyhYaC6DRV6SR*d!LHqy`7tv!gg|-s~Ir)`_B^-
R45G006tZGKXO)HjcVKj?$B;8Nje&QNulgXg*gr-rzuF62&1srYjd9jT{)2a2wVO=360cQT5CtezP|^1OpF6TPR$Vl5Yb$|S1sf<qO|RS3}N3yL60fhm0G%?
jM9RyPc8@UbssCBWOBlJvFMKlp8UxgnB`{jL+jqe>YXTa<_gT4BV9Vqp`I6zsrXh&Fi^Qw&v}#Le<2)Z7hrkg33RjzpEG;*HhYmiIj7EwZ(mLqJ81~Bc`r~;
@|EppBBmeCjaIENJOwn(Ky=gvWF)4PIu^)Ni%NrO_;%URH3p=htO6!~!5%rwrN_nH-UvfHKT8O59b;YLrz0w{TTX3-TS-2ZMe+dNsFx?C94V5+;)I?YK7d+2
ft9`#t>@|5TeOSGZ(7^=kLa*i=UX!_1ylY==D}gPI9`CBTW1QQ%4h#b&EEuKoFkQ$LEUhfs4h_20xd{*o>|H1qz;)T<5{0ltHtO?@((21ZMq7hRab%Rbe~5I
WgZglOY%GdLTVK`pED7HbrAxu#?f`b_isPkp5mtbKKblDDe1=P|5)}*?I+LhJPN>~o?h&BHc%%~4V1R}wl^YHJ(|^U`JH?~ycSUD>h0A4)LTuf<OhY6akKsF
1kog6-hqtjN!G&1T0Uv915!&GnmXO2nz|XD%=@{6)`_GtoA=xpj_4E<u*r_x_^9AO9e;A1unCbfP@vBTtA7QK@zvSsj#jkd3D#4|ok}Bho1W*KW!%=;Zu?rD
cU?vyyh4o6C5@QXcamGW-<uYU&yPr;xC8H{>J3bFUSLoo^X>?p{=&v6Ct##l<$#Wv?BENN<@D$@2~taELqlhn0V~)vhAA?GfLzA=f&k7S))}17tC0`>Sa)}>
mfYl4u^N$rW)@1FHvlOQ`^!~N8IS6LB7{^I`Y9dX`{!Mohf@Z4A~oRntV1IS5436ZifVM{5$U_80@n4Q0(;dfn~!aN98q7VFvs)U4zHVIz_)GBi<#H?^PA=6
OgN1UbS?z{D320lO6Nm^3E+a%uSIv^dfvYxk4|ycEi3RO(C2=#j%Tl{*tsmLYCqF>c0-zu_nIF4IU&sH)o&%V9RnXy(7AC|D=MiD={&zJ@Z%g0vJ$QJ)gObe
_aKZc0{?%+t`TgfP$&B|^)$~gkQ*HO_eM^UFF{BacnStVSjGT9p(C!&LN9J#o2djcT?<nXMw}%eT86-UQM1JjcGp@4=$-sow5_^e?%6(z_pWdd*^~63B9E<j
P11Py{m{7Nj>~tR@7~s>R!C@m``*mq5;WJau_4K$ivT^WT_qMY58;|%7|y%102Mu$(XM!9<I+*O`Y?dylhl?-T?l_>1%&{jrZvLRFNqdL_02k`665`rmdau7
nKG3%QKWM^HA_?rn5-?TFe2)2<eU5iHZvQSBaz#Z@C1`!jshD$E{cjnK8p*JC7@|pBlI&DupE@eg@B%OUE7TtKkMCy*0hKMBhBl0JVkz<QxXvdaHDy#CNIx~
bs*h&DJyQIn1zvmDM*m2{!Uz=TlRu;@ctaKKysI%-n<hf$N($q?#U><@-%zt>g`=I+AZGqKPz~_hpR3xv&ww%uG|b|<xYo<!2*mY@g#5pzxbRluf@<oaujZO
z7q{qFey2j?nQ%y(@N15D6nP2XgFnF^w7SeC<kGt(JIHCW-FN@gJnqm*2W|wxXhoG529*7K=bGtmj{roFxg~hTU#6gTB+xC^xhLTV=7R^IJQen<!pJW<rA`a
av~@xlRnizNQf0?vDUjcPo1}73RcR4w(2DO`^sf1d)-mozoQ)q2B+Fb@;yAJx-d7m-z?evVO%dd+w_lXY#xde-e8oZ9?kw5eF`f3<6H0}c)<k#ptNY&ruEa+
AIkdY7;u&$rRU;F>1=<WssPv0&Pv8*Q<FZhmaKDLrT0)+*FT6A`M#x@1hpq!;gRh-6OPRUEM29P8P^}h*n7V{2<8SbT}X|pt!sul5y$V=E>%l!sbbC4l-S^Z
C3l`fK$kDE6^kp*`R;x=4dq51bHK<NW#Cp+gKIB(Wi)aP&odNbn#OiUm$CLP0X7oADw37Vx|TX4g#W4~_KT7KXJv(9+(t<cgYHyfRDBh}x}}>oCbn|cp^~Wa
BSSApnSy!#%xf*^!SZK#$~wAP=8}@9G9qIiA~snOXuwbu9K31lwK_q(c%J7IQ;jEPiYON>ct6gfE5Ol_?)nFrda9xo=eC9&Ahh>7(v(G@B8ezv5y=bkcNafd
5#pssfM@K}w%gzxog7wfzj%!W*PS%>Rq$@AUYL`uD{qV9Y#d}2JDhIKb_%?-T@Dw$PNtDog*wBRR{FQ8ByTnI+aCvvx3-;Qq*vFM@&{;Y`p!W>N!HY$hJaB=
K@#G#i6Gvj*C5(fswAm~A8h(KWE5a2scLv1RFHTJOeKqj2L6r<vuO@Bvc*~Z1=EBL8UPW0d3m<j9lRbKoJO>C(DM<*+OA?MC&b8vjys@1Mt&P!w!a)qhTuIy
vw5^KK)j~h)o9TX8)5QYKcfZ5Mf%2>%**B*CTJ;D2Kx2tQ7HR@S_6JhzK2PGBvXyh&prytX=@HKcE4E%3SM5cA`x`}cNxw=axD1Hl+vXp92nGaVBX?y;_nXS
Nqk|kIE`&2%zY^(4^Le-1&Z<BEPj`kyun0t*ktjT-1k@zOV<RQK>9IL#_5}N$N?OjS6B}ZuZ+)Cv0B$zuWkf_!DV^J#xC3cR{8UpEH5^RhnJoR;Wi$p#ljX1
=~g$@(vF1E&&S+7*u|XgH*JZbKAU_0)FpBjG~sc@qym#aHQ9zfy*yCDYTzQ~LZ311weQa=;I9Xwr1G*__>5?YYis$nNDkzc{V5X37@4ZRPIO0Y@67yXkR$+y
%K*6P%)XTPNV;~(xhFK5xf(t6dRnsv-~b(`3e}OQ_AQhVt(xJlr1$-zKcN_f;x(LJ>br`(pmn-ZEIy%muW7e`tBmBiq(D`pmx{?UA1;_))*OF@wTWd(g<nqQ
jWRkfL>_5T+_c0S++%T(IXFOAR830LX^k`$usOzMy9G;so}I4ce_nXHYi^){i4m@ADPM3(J#tqaC}+l>aaX{a7o#xj#Uth!>O*->iEg`57gbf9Pc*#u;CI5?
n{sKcz2lpCqeOb;zsF=SRmr>Jp1N_4x+`_l1u`D0MXd?dy~X+%m7FYJF53p|fMaS0lVj`yX5AT|pyiarxh(ErFZdoj=mBqe%e9_#ZJv~dadQy3brY-c!9)8p
X{h*F1QXvZv-OAosx2^%%hlw`B%;**JFVXwJFI%w{!?)>_vbP_uUf6~QX^T&5cmt&a5*NEx2j;ro&E%Xyd2TF(nN^M-WaKVKPH6>XPkQ$?tXG{HVT@Nb)5wt
z}Ypxgwt^i3G}HQ&iqh_x2CYP_1t6!u@4S46_YQ1Cn{}_&}+ZUJG~G?Y77sG?ft5j0{3_Bgg^mdWW}6a32)dKuV%Y8ca9*SR^F20hIuFSfpV=XA`?0*7qz^3
TJI=DU87Po=bN{`_5kI>)o^(w*(p_=)bebSxDwgis-mA9z3S}0h=NM`kV$V#h<%FbszJmKTq*~H(X>gF{Su$>0KPV}ADTf-K3RkTvo+95%NY{CMJc`9jnmad
O`4wlEUr%>Xu;1N0<z)n822>VXI+%eVlPmn!R?Da0^<JwVWS0Hb1`A^=`!%c`-!(4G@&dGNw;$4ErFd0_Ke<8ivoofY`1Gkj%Qo7ey`M`O*Pxs`1}RJF;^M}
sE(Jrt>Bs<N%T;5&H3Y4a%Awz10*<ZG!M^qK2E`;M$5<{)?(E|*Nl{@tfa*ZCG?wL$T_~hwyH`*aw383rpVVlI*~dy$L1^li{NryV=FCQySI!ZR?&4gbs;m|
ICv{lzsC~-y^xbRCF1-el7f(5VTvyN8;$tunOLy@%k(UUTP{Kdy%G@>Q{99jfF}v^LjS(tjH-ASE~t#~QO58Phzg2MhX81Q65o)gk;~8|M6SWQ%ijG|(kZNt
F^(rHyp4oF4DuV;qdC;2z-B8rVV8AMWJI9z!j)^&E6@~fasXrVjq4h^J3Ua_m5;(Me%hKgGqhDyf@2gq)SN#mPr!ss87)DR6Bk{Y?NsctpghRXB~aTni(DBi
U<vaQGXq(-n_C8_&a174e>XY{+S~hV)8H}0p(|Py_r}A51ZGMoP+<6ep<lAPTTIovR8%<)mxuhq!K4Nk`Hj<U>DrCQlkd&a*(OZx#A>FK$x3Iyvq#i`j{Sm-
-91$JG~S{;v3}*qOQB!%9Fy{zo|R3D%%O`yMc+r)LBNt>fVfSVYmD?HN9q^q&<1#R0s)?Kh78Oq)$KvY@4W!}Y?HWr-y;4__^Gt2%s)FY2u8iTdz$}^;4-?r
6UwNZ$7#^^4R%9+)B0t+fHO}!5*p)vAG?CM&WSwy!K|Kj`H1tL$rcRA%fe6b87FWUAhu<BqmlcOK08<QgE>(G%qtlo1TmL(?<xUlVn3cFj;0$vjhnm@@g}`Y
%v*#yENRsyggIni)(Csz4JA-y>O(mdYQCgG0aAc!=fW$fHGAaiJNU!&DDo4gq`i`yAW*LMMINE+@{75^#5S3_lpC^DC$kR@BPTu;f};?-#rT4FQcGmmi%meu
bO)-%ToG*eblDH~p$?Or+09frntcS%IIlWxKUyGxFYw&N5g8mXA5bmHP(iueZDbdi0)RC{G5;DNP{L_gk&K{jV}>~llVkD*L^HeSkg1wkY}3t8PJu4(GX|bF
E~ed9{?b?Fxp$T6FYRA;D_>-#GgbNT+|$rT8_sWc4X(Kz1`Lqjqm}^Ms5BqiPHf^Q<%o`rJf;?FdG7^Gx`2b*)38L8K_<)Hj+);kFE;FII<8<8rBP{Z3mWGX
J9ivE2?n}aR6`t!8w>U3?@8xVdOAj5!QGN?wL<6yRR+$RstathD}f|`1@T~Zn-^$K9qCM4h)x3IC)s9~c;&t)Tukp0kE%U+U$d?RS8RdAi;+Z#uGXdGV?MN<
!dJS3ugoq_Q2px_bV5NP6<(CSGvzop8_bV}%VDnB$fxm%T1lT}EUs8wb>D2@m#w~+@BcaR4Ai-^e{N<PLh|1q2zq^cY#u3r)DemEbRWHm2mA|4s2_N6nOWuK
xTO5vsfzM^6xh8g730{u0>+WVO@e0Wp@`c0I@6L<H%!iL>ccKyoB#(tFg<o9x%#em`?s(x&*hpYyCjEw&#s~WugziO_Tt-xo*?<&=Lr`T`94!6w-gw<J=S06
8ws0C!!93E^mCgqf9+g;ytlQU|1JXsPm53xxYEHh{fo4+S3|7$`6&<x<IUzpEj+a&QGudt;`Ni8JMXJmrQxEwDP<d~*T`u^T62C172_IuofYA2x^iYQR<LcX
w36B9wuTG<^>`ODpbt6H<_q;uJ%a?%rdpwzY61(Y*ASajEzFMaBvQM`JGUSucPm7@68M*<(<sC@+|(Hi7bj4|mq{X`xVzgvm}ljOJ58`3GVLs;irN)C9|c58
4e6L0K--2jh#ui6k#Ny!b>V`GAc}N&^-9F5OB?9RZi>nQ6vHEiZyU7{y$pecS^@YOPU$4y3;S752z@J3kai``vnd=8%V)@SH*O~2AF-H~ZaMJn8v~8;j3k+Y
UKZC~-J6v#Q*O}USarVgm4#7U8`_q&pjYbb6m37Wmlxv<imV;1pC7E6E{I0cNpvn8$R5Df(Ek^Zjy<e55`h^*TrN|LzT+Uot+HD)FIv$z%H7wA5PTT$kXfRH
^aLVmS0cZOn*c7KjPYxi%8LJcOoDwdpXu;`E*Prh&$zWm<ph&~E#?VCC$*-h!fEPja1v_pMS26!&#?1K?<xHw%7E-Xt8V^vZ!m5spsCBK&p3Ej7cfn>!STRp
vZPC`g#6nK-}A)fOXBat^96F*mXxmv|NJ<}%vV8^#Zbb}Mrqh=GZTj?%))(7*I>wQYc-i5HJ?LX$w#VN%KEngMUa$cANTPT=iE#NR<MxKg*KAfInX2C`-pmJ
Iu-L)$!`3A@zK{`tI!YC7i?CX+^YXJ4HXL&@21<gL+A1fOE1Qj>r`Nd(8#QFOtJrH*!>=KQ;S)F;x0AVb>}Z-_eM6AWxVT7Hh8{qC%*ZevhcI;9vERJ2wm`B
Ms`{6%Nk7lRy{xPx(hboffnsFG$@u0=@QOY$fxb;R*@amN`=807bRQ$s*W_j+AL#JUh4bGtQ=N0x}}z7>&?@AFpAoi^LO0jS#F1_a@GymKMb6ioAu`WCq7$B
y(>hA?yZ$Fp9)CF9G4joXRpwTN1AY&vuZd=$5MdP?OnHcYhGQ)GdDUX__d5eYo%nMcT)4n9t?5YT#F_#g~~5Vlj4VUADk6;=IGhfeK1xVj|vDK7N6r(^AkhR
xCARAODxSpbDV-z@wlsj{x9{3qt=jAn-s}l;C}k5xU5cNV-GRkvz?(tjZ=vDeBlWw*8E1L(nXLZ!xUJgC^2nMqIZVT;C%g0%=+Eyo`{;_gPUB?YrLB@oLM1*
5)oG5Rd>aJz_V?!?!1BnB4rjGLm{QwpcLx{*~@&B%RS$0Mlt?Rv+=mr5W_F*NJ>2W**5MOMO!!NKzN}?#dzDn3*!71ROj+qKch^^JeN@f19+`vKb=t?NeTxo
+SJb86E8^FXpE&t?+C*P6~Khzg)Rwme(;8_T&-ltbO(#}17LQ{eW%F+af<k+&maT!u17fCC|p7*o&@dMJz9uJ$QNy*F~ODVB&XzZcI4Nxd)9b4I9zu3tc$lm
QZ;%-HRV)uM6~pMtmClbjiXjD3tF5!Ov9C>haI!e(RzwMyYpq)C(Wips7&tsCk1U7m5&J9$%j^d{zTDh5ktkAV&b{1%=h|Jw|5s+Gd`No75W&-@JLj&v{cAM
vF@Y;y}-e@Rt}(;6A;zaX$CSErySYSLHcK7DNO>bcIAZ30JfaR61+~fsJxs5Q|Bg0j7JWUXksW)W<0x!<EgK{yN?~1r}DT85q0b&);2l#`#>YcR?xa?y<dM7
ZH+q2Z9HCc9%3+KggB(DN*h6><r2}vINiV%*7+)$ji~$RF7<HG9qB<9(Xm&tBod27u!3_|21*&*Q>No+1@AHzixx~~7-S&j#F+QhRha^@Qzjkr>yI+ksO{;V
U-|+VB=60yOYAHIkbRV}@O3-1lCLeZm@93lHs(*DsQ5(*lLuh-FB|qpz)WtySS>Fq1C(8iAw-Uv5z~}GRY&4r8>nR>72yH7wL!-n5&yxdYBuFwec!j?3sy>0
4)12QJsp;Y=1PaZAHMJIG0-K&Xt;1P0ztPV7posMx*j{tU`JcPSrIa!o0svy1gO2uF1W!XwS(&f5nk1G^#+q%kb|57n(G&|j(N*b$&QyaS?gaVXKzvUy>+jL
BKl$e%7R;I_ic5H8=tXViL8Ygu;TkPky%2$$iuos>hB78aX;VzSbkdh9>8<$0q5p?YbJM{I<AW7LrQR5C!5RZ<neM35T1S?d(tyQ5;WHbDgDGK!rNXhAW3Gg
aLe`}PKO`mzZkDMCY^e53H?;xbYhfyCUAcqeWnK&Jh;8cn4*&f#+dD;)#Jm-xLy&<p?d*Y4}#(Hcen-a0*zeh60j((2y+n|E5w_3u=m0`7h<upBHcAnM$7{q
-@+Z5zQ50?$uSw*@SIJet;UvN<P7h|dn+=r`93EY%wD@wmc11Pl4Ixgh8hlql$754ctC#)4*55|+4TqJd>>^9lTQVLWg5<>RPM5pce_1~1`^L%=qnqwNf98%
5mqyQnYH$Iumum8_8*m`k;Srrg+xIrgRKE&r2`c(I+Slu@a{DMyf3D;ET5E^^<Q&eefwX(!Hc`>4|92YB_3{rrcN3|u(uiGkgfzCV}yGXi59~)4y-$$MwXwW
x)2>ltje0q(|--yUOZk_`NQu%WOuY&*HYr@P8CM2&kjaj)qW9Pb@k*DZL(teFT9cc((Eb7<-JN8xg+e|5VS=8NZ4pxBjbGmDse7wFI+3;_r#etkF(?}4w^nT
K2F2fThmGuOtd*3OcX;u_y-r-#PAv^2RUjY)Nvr{-ZA|eV(p!u2S1Vdb0I^^I|rmlSO)}<gp(h~DDpaKmJA}vUvElm$9q4^g>>mDlC4Tu1M2o>8}98g(J`IB
2VME*ZlD$n@RyNpZun~<n_zz-zlE(JW(oG?z7g7WI>EfTPo6(7jlM)YS2u)73l~s%YlafB+DYH$!-EY^KA*q7vxbIu%os2nJgTWGv^Uqv>YVDBy{G2av;CE0
{ya6ic;iITP-<<pnk<YNfzGBpr}2e^PWI)-+5hJ~w?QEtoPBX!40)PEp|X4O!QY<OOMC--vRrtQ1RjMl!9zrR8E~SdU4&UdToJ%yjm~h&(M&N7F?9a<gd01Y
jy^X0oqBX)*7K4s9kh!vu-BzYqVB~vX(Od5ak<7gMn8~aFZ5Qe@=dVT$f1T#{c0u<KtT#=X^4QPs`(l##73|Y0vjFcLKHZm(DC4ae^2xBI;4+aY4+ZCs7k`2
12|Bm=(ulJUtaSay+=Av8mp&V8%dUb<N9_pp53%scZswDni8}B0g=scIf+H;O>!#+OGT%b4k4dUmwGZ`B$wU(@c^2$h}nM@hEzGPb;uF{AaQri<j`$eQ3fAO
c7ECC%p)}gzlk#;;Br0uMYk18tz(Hbq-~Z5Db-4Qr->Zk@uV3)w+Uj0TMMGB(<n?ezZGwBYu)d+&eH=l!1SZ8sI7pz)szebBfqWaNO(b4IM)UJhkchWIkC|p
W>$yTHetBucyVNY&ydg86fmpwnQHq*ppo@@8wh<im`6QPBBFtzqWFOn>-y2DzZ0mK1K{oyd;rj^<oxd(pfUI51ZuQujDA-A>CUnHsa`ix3uDkIS(F<vQ3)r2
IYAGv47ml(Mj!3ANK?<htee68iyej|st;C)VQ7117JW1-4?MQ;r7`R#5T(di$zVR4=xz;|?QYuAZMeV&OOz9-<Q+gF+NAph*p6-6=bhv(v!`W62GuIbc92Hc
Gda;w^7i9NW8eE<XK206npiEE8%pLOy$nKJznJ7J<%ttbj5=;)SJGOsh+~1fp0S!{&Iu_iE!KD|bHHN{S$kbe<!hz^A)qx4DD1;FPf`HMScr&K9(SgorDT0S
t7(T%FA$?;o(R9wf6awSG<Sv)Kx`vG!H=UWepxk9`H6{%2HU(Xu#&g21`LM(Ju5=!7leK8IRthR<Xb~v5dxle;UlcM5f_PNRX6?1ryW<EFvBvBV8RhebGY+p
hV;9SI;XwiEuSNUtfd`4Mb~|+rfEz3_?Z^XxJYw>?92K~|7#<bX%a*8c6W*K|H@{RnnVn1$_B!3UA8qclmW!l;3=QJ5@~<ok&_Wwu|k$%Lc%1hp8NT1y#W7l
#M0uHR$)^fd!Y|Orx}vHag<V3nx#XYvuYz8OJv>+2D<L&Fcaf+&eaCtm-pgB-Vyi!vsq@IX}lc-IrV1CUr$`vLR}TkOy25P&J|eYuR(Db(dA@%+{tfH^|K^~
p&eFva4;^Qtlx?Q=0Gnu@pKVaExRD;9tcFtDq~iq^qVA*KP63mFl!>Sc85-^Y~SY*8)6;6`y`H`5rB9pR%e<`w*(<ufe!CNfn}!4C%167^v5I0AdfME^%Ox@
Yivxyb|aAG`X4s2cPp*%>D;m6#FOP!)_}@N-t^iu+F!kUeJ`Clfw9h&vNBw{_3k>x!)33=Hlgec@B<%p3*R|W$$ly=Nw4Ol|1h*0;6eXCb7*Zi8`rQYX~9|!
wpHYnw}^==`l9j=^M1{CM@chU9g^VC;V7{H$e$Aa_C+3%2vuGglDv~CgZF~85^t9-j+>ZA#mfZmpNDGL7i<^Qf_B)RmCyv!;Mrc8;||bX1fqJ>5vjxR&GQ4Q
v3I;25gfGW9E}(lWhpK)u?(rk+f1WMD4_M;P8^zFG7m#Pg~%{S#f;tnQ&9Ex&3ip!xD(d1E*jztL}I?MoLuab_jC-EagSsOvspa`VJ9{(u}v!4;_iM?n_!MD
{;+LHUrDl%_xc2qVbram*A<PEFrsXfJ+8@7KsY2~?U+gsv0&eU5hsX5rdx?5?q0n^#BhprK}^A`WenCUsKg~ct9XJ-5p`DrltFjCrNct>Gfr-hr9vjBmvi@{
JVN$O`q@w3&@)CN>srTpB{MJ-hGQkl^991JpKRv^{*vNJRzP|Wr}ak<3D9DIVrd7eWLR@#O_PBb!cX*I%-YM>rLM-;`^Id>>rgs$X?j4&S~4xkpY1V%`*wUC
A*1tecxBWCrO72`E8GdJmmStK3M<T1u=N0I2YXa8Jc?PI)IlsDXfcj{Ox54Ln}@Bn(bKCrx-?_wjCR^qw_-P!jyyqj^oQ+s#q-B2Bg?M}b-}D&=q#=B5ToF1
Oz45XTlXnnLUb3pk_%$QdtW@eeQW_bnd#@aNUS(kfTw#98rQ2_Zd6zk`!(+FSI|mhvD-ROLRu|vrTKsYB%3wmQ98@kFP_x?%12N2()H`{nOC>tUM3^1wMGQs
*9($q0{R9;U{=o$@bo0s+@dJ_(3J{NL*(ViNTS^%v6LUo%Q>#g<T4B`{_5$1R0PL%fCtxXp<nQ4!QM=E<lv!7tTR(*0KPqB5se*o?r;`B+g%5}6ELhljymzL
J<f9={xg^#s^;u7pa%++ilc{G#5{Tq=tE8PxJOQ@(64#_Z{LJEdg;|gdv5?~oD%QT2J;S5AgxtZ>L^P>`GS!m@W8)=_`Vmfb9c>F^v4*qZ2;OinK@}Q5G?5&
G8!ci<@+dsPokG)BDCZUB|oZ8sNM(j3I1c=EFSiQey@1vb=s6)+#{Duxexa}Yi4g=(Sk3368Jzi?(!Qu%2OV@f=4tPhc8~&zdYn}j-c#i!Atlc%96htDHQ#$
)Bq@n?$ZUO$pmy~9&JLCU!bG@d>Lun;0xrd#WEnmgJClQ0luria}osA%Y}oW7Xz%jx{Ft0jr5m4Y!?8FY6rqTP$OLG0X$7%xvAKY@c@#tS?YK0Us~uobxmV0
aO+TQeWm^jw|3a{5l;ihQxfMO;?RdPqVzVPwUoF^(R}yg=S=(K?;JUwPbzC0U)#~8rBG<e?xU#Q=Sn+S`w}t+k{nhr&`=w-m{h@bAUNMuFaao2j%3B>MC?R0
rPUWj5_6xBAw?2@7ueqE(YGWp#EV%RQq~=LtgD!|Y^VtxQ_)U?1ocM6JQa^LLAR^_HP~fIRNuT5xFf+<?2&A}(Kpu#6bXoJeQ=%dI7IMp8d7jr2iZC(RLr3B
x)%fgB`{C~Oy9!@C%ujz^EFtnj{1Fi=fVA-hZxe-ZHu<3;xhVf7$%S97ICl|LN@7IIu6_FYWr+IOYG^lLS8u$#yX1pQ4b&m2dhqcfb*4T*w*7-)PxiLw-p=w
@0e$>9wJm>;75N(ihquD?~dkt<-54YiH)}K+Yq7#Xf&SF?6ze+wWor<*|7Z#XrjYl&9h8+v&sT}3C$5X2an(EWJ+@xnhBN1$i+ft*RaTMcibKPcmm;%tk5%v
z<bl1!34#BDsUJl0+XAy!~m%1@>wx1Jf;lFF34l;u&{436sS)O@f4=S@4TL_H?E1i*dT!UU7nN#*o|lq>u~b>E#OPIhkLVTBjbZEF=}LssK2*E6?uyjxznr*
!mU71Z%Pxfa@|&Am*2|$f5l?w-_4F1t4UbuKD_an=Z#@5Hu+P#DKc(D7tnNOE%IoUw=d7eq#u4nkt{`87j84n(jx-M$`1{dFeWiSZt0RJD+1T0-u#2*OOG3-
*GIaWc|2n2Icc6E2m@9%{ol9k_4JNKm*AC@Y!7U5$4Xj~74(}aCfTR3gxq7G5JH#kRl%=`{4Ge$*^$eOI})S`Q~G672^>Rf0Kyqa#P$;<KR_|Nz(b33faF%+
adFwt=^@=04OA7)!j&}e0<cI}{&$}4jhNcC^lU|&i&7#2I778S@~X2L=Xq_xPMvL@h3KY`Jw3Z{WJ<!3DUkw?BEF2ibsNO^k9VX0ctRgLX^adyS-Hpvmi=Lg
1V)3MifC5X_>yR_ATugL{MER7WdScw%W|q~+k~z-lRehX>e>nsx&wR$0I4hL8FFyJ&%Gj)jQvSweU0z?I{i}m8TzZ*poTB#c-|*op5dcn3n$v{$O+)a!b@;c
1e3Kp?@IZm_(r$Hzb!&8h$INV`7lNjO-2`UpMd)_`qp~tCLXP-T*t4%zcVRsvuGp3BYs?4Qno87UAJN=f*j22SNHj=Y{!1bt(%cA?GM$)p$y~a-}fY<&DLJj
jr#|+_HLcZb4Nk4lpBDQjXB`9(FoR`&aG1|k#?V?#9u^(sjt=w5_1$%<UZ!>|5^&RTCsA)NpDfNZp#aE@s|1XB;(lFcTDcPivho7xXJ{0XF6%d)t24GpIk^y
cHcXNm^C7ZDWbdw->EFp#)kaUDg6`-^j7kc?e*Iml$YnHp1a@BCM9ISS-Z~CFgf~vcaYc>P)JEei}jO!L>^ToYmYVBkakP1G_k);@`8+@p@Qtqzlk)M>^SBo
^QBZr?w!a?_#EF!v|cD7S|@lw&P)_K?ptmvcDT-~w>JX#mj%MQ)7;+wgo}J$WnXSyG6q-`>aT$3eQ$bSV~p3yfLwxIo8{f|;&P1`<sfdEYrNkn(VB<NB{wiS
pHXAiT3xewtr>+s%p19cKDhGBlOC9vc<A9Y@V0|BHQ&#-0c;+?TCPyaQ9JNV5B9vvD-9XcL#Q9Mnq+Xpx99r<IRUMhQ8_GPiWVc|MgN+}`50k~ZA{P+Zio+W
U_yOI1`>9J$en2=MEER+>mBeh_cA2@;am-uM`9ESv<bZK9Bc+W_xdF-TY3eyvxb{XukNBPmq04mu44xiQpW8@#h2(BzC!NZ&nRoWlwfv#QPcVils~3Kf&0Bf
w8*c7eaZAyGS*qVVIOrWZ!BP+B*Dfj&|Rzpa93>(by$1dCZoIwkQ6`^Y&yJv!s`_;H-dqh-~IJL%3Ho*U<T1@7CTeKaT>^Y5qQwMEN~=M-T9|{*p*$Ac1Opj
;Ud>>B2Kk#PoO=mN|TZd*IacJi`lWEeP+0rw#Dfv#Hh8XXdn8Y=6N}ShOn&NzfVA4qFH^gASi0j#@pKg?<2sjD4@c;03q{(FR*1LGi_!(lSit=CW+T)t<#{K
lo>R7!-P{(->0*#exuL6%NF-V?ckbPtdVuXucT9c`Vn(CuIW7856lH9x8Ox=XJAJO6zM-p+<qm_p#fzf^PemU$ADSv%Cwp4BDxa~Qv-z08HCeA{gngWtQ^Hb
uBX*9u?Cv}hUf@3M_`=VXV;b>jms7$xWnBDvXY&6!jXv5_P87@q1%>3(jK6UVMBwMaT*2AwOUJDni2)JGftuJ9Lha2P`ES#1AS)~pTe4Y1R|#`@PthtJ^yF%
bM7x`pP}iIY|NVIQRJG%!p-NDnyX4mwUd2={&nzlNbg;J&jUPMf~pV!xX%^);8F9xU9F2rI8o{#ldmyl5n+=5MiW=m@iLz_ac^q7iKV%QoY0EHrpV>0+iINh
0-aN%A!?MpI;a<{_t)x34!{z03*L7#kQGr+gGQmQSu3bRLI<n_`*d)GuMXVHl~dLB?XxhBq;GJfW7KA&>TG?d!H^6a2m+B-tqwUm`j@sp1G%Y!-jImjsO%ql
8*hc;^{|lZZJj-y#+PKd8z+ql<Gz-b048F2dK*Cs2GUWq8m0mlDtMvqv+ND&^OQ^ax3>a^pzP!YE7;{P#1NEu$wt0k7bnG@uR*?9`U>lVM==PsEJT8^fRCxZ
i4v%u%B@~apQXFe)+UEpTfT~gsr|o$GK~k-<6&B4%f*W1iE0L6QPAY*k4>!KL<a<SV(Mh3jX~ipt`bz6XDihyqbDybcpHZ$b@;M#=1l%lN(89}_-7=jKH@V-
@7eoaeT1Uo*xBE*DAh5@@-=u3gep^#xv_zymTjUvtPnkok;qL3RLes~@Kvhwb(ywZVpyV4yeH-0ffjKUoZn`m!CpRGI{u8;cVPp!r?Si5yXBnb;F?L~t#LYz
`sBw#GzZ6y+3S(dkjOs_jTPp*Z%y4bO`n-Ri0gd^5?qV?W0qkGack|45#hL7VFY6`{Na%@hF#4(>w=KT(cH+)fMQ4pY~|DQYk`|kWVNs;C+K1u`l?7*g|HTZ
jjY=-e)*4bnj})1BX7E{m0O+{hkP&LiJe{%g?>;|d)Ub?)3UXP4LD5!Im0|tX%FMK4E8er5oyf8d6HyFrh1<}K)xBYlLR|%$d_stI=>6}&x);`?TzUU%51X&
CJo*hT^(4s<kS3yh`TBb<!&S1mGFrRViSV@6+`a~lXM7_26)9iReLyXIsE-1-C|`K&47r~$Tr@tbLfqj13#)1Zg#nxN}}n8DCKTlcKoYXIr7vJj8G<sN^IsG
kRMzaG-1P%IbN@C?NDeh8XStSTNsKWES{ZvxckQvlMxHYunOc9t#vReD_d@UGlDNWXcA^Fuc^9MWdhM4!kU9#1nl!S9((#@WhjqBG+Sr@00000^oY%;v5SW4
00EJ?0gkT^2A7ZGvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_lefood.csv')

In [ ]:
# listas/externo_mensa.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/externo_mensa.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5;Skdi9$f%38cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTlHukOURZ3Nef+WAJgTA@Xc$ppcnnU{7bdih4?B9z!pNF=Kdc1PQ|U
QJ#v&wd0IhSXHo?ycwa@(Ot2zx7U?<NkXM5n9#DM+4D4$p&uYl1ZgtI>*ILLUdZ&}+!n*kJ6Nso+Gnx73$TxhAy?mVijvJp?=iXBe|JZH=@;2sB$~B3FBRCz
LN}+vs;NJb(%ip%S$_;G7Yzp#Zm%HtN_l>GkbI+RmNcC!+DGNl9dTX50#MrR#YW=bqpX5L(#C(DKe~`ugB>%)(Z_s2ZoAmlMwPDqHWHM#$ol>=f+FnbcwNJ7
3R_B=)oZdRZTyRt%Ti9mJP^|LGmBBJOjbM>pduOEa(qaeLaG=LN?}~XE;CFGD0*R6^!j|ORSBD9r@m{dq+8(zrg_!fmZnM&KG}iQFE=kk>~AJZ_?#0VKbrq=
MRK%VC%rE_tgIYC|EZ#A*J-K#wTjlke4FRv`954dfX_L43mGwNngRq>u~c>vu>e|Qe;rSM!XR7P3)mvx6Bd_b<<*%e0$61A?&3Bzt%@)nwXb^+J*El8mydR3
B(Tfh01MHc>JJ2xYuMb5Tm{}i)fzD?kBh+$9dl2Maeyiji6o(tH&UYbINnUZp=RoX8Bs9IoTuB#$<CNVhFmerzf=7g4aWd&+Pw&dPg5Ag*g!zq+cS*U1*NWF
_}nN#6+l8ETVRHuaE?aaXcr=;Vc7_<nAW6rL>oT1_&TUb5~l$RUz!Kjv-40}ncaJJ*Ln61JRa(dhcm^eI--J~;ZERcF(NBvd-}N@6f90#bb)MdW1;?sjSEF3
IRxu6FiZ-;isfGvIPDw|T@JHA2bjd&2M5XgN*2MAzgF1~KHd&F;UP`_6J&_RJ1yW^Eoh!E&q7vT_Eo=`e^>)m;d!RJ84LE&a|!3$G#IHzY_<vo4y8#oz-(>g
@*La~TAhK5FqeZ(ag?Z*wA_5hB(0(>i+8+>pMOJN3#7py-(%=-!qfZH^#w-C)wd>|3W5ZQX+L4Hv*gpc$}n;?<>6%8j|sZIJq_DPyBd4#HiRPk!MNa8qnuY@
&6wb$Rk`UFmyFIff->#4L>HSwNdw}TDR5`;?O`b++|ujkXs#m_pzXK1(n^{fKW!&%FS0V+u2B)bcM-$pYQlc%&$S<uytTtOdIS}f=cf%y)oQ-b;W$7>=?={4
flry&%vun00~MHwaFMG8{~McjfX7~18aR<bs-~DthOl4Vdosc-C@X*?fmp0gA4ADILi>Vk*+8&zxV_Qkbs3ctM(f++HQ9cC6;65;fC#Do<BNh%@0QFk<~1fi
M~z2oN3s4KInS;C%&H|ydx}->$wZN=ld0c!*&HAu|BI0^2!<(87V+bGT2rc1eLfV4M)p-`?`Ul@0s}6^S{Xsr4rW$osM=Ji!}aXy$`ki;*WBC9sHvmQ%`UgF
DaMblxDZ>*h_ZQ7q`;g?^^o=8x4%J5?^gm_Q_1g38<P$9mMM+!fviVVWndJL<DC062xll{(J7$LIu=+5b`fZozgJNZrVn;BMzT;=UWkOB5wIk`R+0l0uC#`(
sL=B{>j);k%M0?a?nm%2k~6+<j+UB<)SBhOw{^&_C7b9fYdH1uwAF?x*OS&2S+#X9{Oyz3>z^h+(b@XQplU?#3ycF5s}=am+TA6%Zq^h;ODq*-(k&JN0zA4Z
xD_|)lazvmS4?on8<rf|p9=KO<F=aUzZ$R@!SUQaiZlWQ3B@v|gvr`nCyR-p5s5!Uu;ll@;@odR#zh!-Px`ANxA08UX}JKRo6@~!v#tO_o50uNBug4I%m=0(
P?$u1y9)`jW7}s)Y;a6)#fR0UMHJw1ODmqap@$mEDm0fUWr&@L5u$I>RZ4ytf?&@q(buPv_g+Hq7u!RLP2$mn>vu^nPjo<m(sjNSUaR8L&!|6=Ts4(WsZ>&h
+0)^Z^+E`wbC>^T!odx##RJ?bo9>|&!AEdM>Q?SA!4S{OdZo0t0;AmRf~v_Zk;afn3ujHL=wjv^!AM8JmBIYOC<#@%93+8_zh{2qaq<r3p>gVYUUXHhk+_W#
q)GuzQxU}WJn;T*2Ld#oKlk!gBtd;DwX&U_ss5L?U%=;E4QiUJSMy-EPXT`$*{LYo^_}-uj4e&-6XEJR0bkfSyBXEv-Hc5!R*CQ*;$AS4K<#mHnOSW9XFW`<
ia8jdJ>3q}Kb1FW?EAkL(yfP{Bd*CK-H3PB0o=TUv<={yEnO%B7jvF*Ljz-B>xi&#sqLvUhXVQjDp!?@zvouHYVH3kX1ez?I)6OxGcF+Dkq|Apm02yvRSWM6
uKbEren2M;%u(#s^No8|tx2(6yI+a$^%=U?o}-M6UgS%|a^{}5%ibR^h2l+~3?Ea?(9(qeFjK=Z%(VM=`HkM3EOj@^R3wtCMD2!;K9*4*I(ouBmelRSpuvY-
6?&E%@dw#7+}oaE>V-ypnTiy_gNCObq13N>T|^ujZZu^*Q0TW7W)4o;_?*#@hFp!qs&m4Cd2mK-2KB;Tkbp>NU1H)1cCV6m0L`8K$#)sf{wBhNZ)>%Oqhjl#
zFmlUT9DuoKTj?)<s!vM_~YY&VUl}gGN`UidUuf2yQmx_8y@Y}I|;MJBN=|eQ5hC*3?5}#tV-E{MgQBt2QAAVnR&1XaG=W_K*ZABKWsoRnk!~TNDOqpce14I
%AG^m>&IVx3#Em|Ttdtmu{o^p+ldzj2=)J@I2l`qa5IL$wMHq|PxZt)N4d`cpVC6Md92)(`SbU37v9{EF0Sl?V*_6tBWcW;Pm}4XNg3cCu{IlhY*3t%9W){;
4IPL3WZ}X|ilYf45Rw3WN6CGRMadqZBPoO^Gi;#54?s8;f>$6Jcha-ZCJm3$eYK&rMr@d*VJ{hZ(;7K{QkmQzH$-uF16VUv#Gye@dW|9usw4jZG7|ZQqL?X$
qe5RLJ3&^+pa;0Tr;3&EHDF!2@u@WeN4R7>df+gK(Ru+@3VPT*ks}A(2p6l8jy#1*qndUwpjDWu|E@8Ds%rYZ%v$H`ebcX!d<~pIop$LY<%O~HL!Xxx1W*a_
fM)1~{*P<?I^#1|ZABLg=?b6CNhWu>43Dp?p4&_*gcq33POo3y)h6X4JbPu|W~Yu(d}cG8c8G>;u~&nU#W3A_?h;fGM_f2MtJzAJK3Yj8YO(f#*%473{H^=t
r5woN6bw@?<Aa42AJ2JwKUT))CQM7@ZizcO`+-DB&&W5jPiU@FNZ!Be@S53NMTQ-JlL151vcYf)$XoFCKd6iGJP*d!zK4b&n%YRub&ru>^g|k-R>NA~(Q*Bc
m&^kYRxC93)%<K!I{an}!usxy>8;JW%5h);ILK6wEnTLGUD2Al!HghL?I6nRwrcF<ooG%qR+9m}wR}OXAgRnM{?+baJH*bc83+)m8C^m6FJ|c$0tIWGC?Oc4
yKS`4lUIjAu|SmwKPtOw;a_=gvunFqk&6tJZo%uhmg`rhK5cnn;p7(+^7+7Z{S=V+QLMKL&JGaejh(cPQ3A=}@rz$R2RN8<d8c1@iCQUd9l=N-;gQ=)T<RR_
go_s$SAn(-{g)Yp2)m!Ih2t|fEM#Z`>2pc>Bm6`*5k$?9p3V>U_OZv{V*rWv*oMJyQ56z{0Po)Q7xB^%E>Mzabz{{iO`x@^w3Z_t_<%!}!^x_R8=(8iTOC`6
YI{sd8>O9~zNDpPo#t<=pvmNRm)kqtb9*F^U5%>?+k@~FZ}6NMCJasNeZoPK-?0F9fXJu{lhS$%J!8pC*?#VqL)Nl~m0!-F^*CFRoKvdq6au}t!2Nc%PwhA(
REUHhGvAAd*lnVaso_F>27oD{>ytDkScyEZBwrXMSv;M}P)^2i6aIdhhk1WJKg%xdmW{8sBZJ<39$yI~MO4piyZ)a^VT0A(CC*-(OF!Rvopz!?`wmIf^jx|e
q&bB1UCg$_!&Uqs8+?0?2`-FbI4Isb`9eT}h7*}@ay&gPRe7|+;v$DR6(7?nC@|V_xhDyc;Ol|QA94&HE>uC}ODOYWR(ggNfGBQ@SQlV+9HsrwIWivXF(1*(
d9~h(e~%Eq8x-rH45&xc(t6gG<Cu7%l%#FhAzl_CiEQV*MC)uIgL-NCYlU(hO|v0IKTvOUqbGQjJH#oBtujL@%6&ZMYI1A52C}J!6YRQmvK+N5z#4g{AM$2S
Y16E#*pXyV#k4{o4ecag&#W>`i*Wuw&dy7q3!!!>nr;1*DSB8keIaRKr>`(*rOwC>n3|>TT`Q`P_3%H3X&`2ksv#hWn%JFMMK0H@M(#a#Jqp-B;n5#>k}+l;
dfaifBo>5$7Ba<;vLQoOnNCRqg0iZlN7JaZ6|5-?+89k3+MySHC;Tj~`hj9dTuo)U`h!y;x52IbtTtK*i)wZj4$G-mQbvzMCg{lo%<x7HzOdZ<m;}v$##Cid
=6m@1qq(rn<4|cjzE?CwqP9T?0n@<JY_YnS`JVK^@o<u@nGfN-@Xv+&URs4z7~8+sTa#s=oT9<%Qdj_pc*XHjpdFg3oXLR;P<svoth1@A^sx?^CKD{$B^9$N
doI+Q89cXBR=E)jb`G~{3k-j(<*lCEqH5&X0=dU4-H*Sh5ja<f8;e%3@Qw>%B#0puFdukPKD-h3aLEyD#>oMk0uaG^i=o88hse;SRz#9<a^apeaA~0QVyY?)
a%US!m$riw`Qm7&bs#6l8I$ES=>nlqgx|XfJ<Qanq<nTGTan&){4@MdoSELRW>|abagVr}^Q*U}Az|4A8_)~`ArwKhfqE)u$ql(!@8qp@!Jzf}2eqyk#{;ap
D#u4ft<D(&5CTWrN!}G5gvbI~7#GNhMsTV3Xq9EHXhP=T;23hR`~JQfML|NO;KO;!u%a-y-yo6#<SR#3OQTTCW{2!F%<6sBiXhzex25|^%5HhcXNfomUO(^1
6wI2VliIEGWrEwjf>UN(eY`xha_C=6X)`dBnTl1$e>(j4dEUV{`Zj}%tjmCpBC3dStls2WDWFk*a6z$;2ZjXCe0gRVj>w)ISM2sNHg>Mu94K>+!(7ceP2GYR
n(Q1KRfF;157C=ZZa!*_G99-4qufgn)aL`=cJpj-)kvC@_iP>8@JnBdjX=#}fav*~7K~U?lI6${yB<Iqdg3;&&M&r=9bi_RB@GgjeL<KjjdPRq2Ya6{QOsMz
oI}3ATNEl|r92M$T^sQ=GSN$J;ei#y`}mH>e*O7(<=kWVPJJ3k-0#cxnLrLtYlYipX)i+WlHo15$v#;C{6Cg^ik;CY+?<%$<dcwMx#z}vTM}%TeGDajQiMsi
z`_#xzAhUCn4%3NC)3ZpI7~0X^wDO-(}j>6{&b#>q8HQS6aCFnF0<SM|K&S}AZMh3hXdAOy1XgKpC{|oz3>7NfhAFb8Q_m!dqW}aY(ZA4?xfJ@Bln{h5dZ)H
YP>1$o|liW00Fup)S(0bH$>&uvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/externo_mensa.csv')

In [ ]:
# listas/procedencia_r4.json (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/procedencia_r4.json', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5-~)^Sxm^G~f`9=c<Fp)p>rCx{$66ocxA48)4nzoP9DCX6`0NDs?Tl6E^p9u1wQHc;BXPHKLy7}Jp4u&1v%#^Kb0k%6
iO)dymlR5q(EsYY=0)wNVX~Vb@n?nW_<@N64aEOUwp2o&e+v=6+06BaN6_2tvludfaq-ou^Ed`I<)DW&4MGymJ-_tGt1maNv<L&y_3Bq$NbS<nG&})9M4RlJ
%fYTsg=wAs+xxi!Tx7Sgcc3s&i$^#Ed8*7+Qu#D$Zj|Vl0&Y2o^Cj#!71p+F<t^&HN0U~MpbuobjBA_td5^CBEQXa3CQgr07nXB~%uQ>m@<;5(Ajlm+y?Pac
o60f6=M!P6#iQXRxgmFv|0k2`nBma|h9#J3K~q#gM$_G@3}iLveme_aI6~{R1=tPH0z8A-g=3{j9jiEh8+#56LBc?V2eIswGx|Ibj*6)xL|iZ5G`||r=n=LM
bGHo@)54WlP<P-%bXAm6^G$fox?XQ$kD={t{)*FV0@?<%wBX;uDEle^b5oM7^>F*FSUjc#T3n1hcjPSq%xx40__i4aXLjXK$c)MM>eUJcAzFV|00000<dRtn
=3KVW00GqljRyb#@>_b`vBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/procedencia_r4.json')

In [ ]:
# listas/treino_v2.csv (lzma + base85; o arquivo sai idêntico ao montado por monta_pacote.py)
import base64, lzma
open('/tmp/nutrelle_codigo/listas/treino_v2.csv', 'wb').write(lzma.decompress(base64.b85decode('''
{Wp48S^xk9=GL@E0stWa8~^|S5YJf5<twuA0bKwx8cNL}xSP$f!ssd@4s5WWOrlr3gU;hVGuN%4>_a%;*@>!T!hp&~KS2RcVXXifz|pU?X<ej=RYsNHgC9gZ
^Yyj_@w3l^xQPPIw;tS7j7!DT;i*-LD=CgW>TmISg+LT`At;aA;E16tHJvpXoQElAO~A0P6>#q#QpshLvAfmNV?M4YwQp)!i9($3hrjK1d1i?CP^}K<x?@~Q
_U<}k|FCG>1i?@oUNb!>Z=HSr2pGCZ^&s1&j_XWHr(1>F>$hcl2?XsUDVg;L7w8rvMTl9(JVVGirv%q4sY9B44hig?`BE%r#o^Zq29?%ejt>DGIwE3W{!9Ii
^T8pa9s<-_nLk_t<B_+dejJAjh(+J+N8U?`f2xn<Go0Y8oPTpiTaK?%UG$Sjy(y^*#+tYyjUVk;J>~er0Lq#aW|rHOsotdtZQnaJZQK;e*L~nvL)e~YHZgq%
y|t{-ugR7Dcw@u4+RaA>@wR_)+(1(IvWUB%z5xJsyi@GBgq(^zep`iXdO5Te_WgU(g`i{+aPb>`3Kw-Pg`9|tJMZRWSk!W-!3HvP6$Ci@hZaj>UmZ{C1<}s<
)Ly&9Z13IhYP8cLI@P#*j_IgP${=Sh;pikjg<k4cBq6&X$GVF_^D|*MX<BDM)h=C=H9d!6;tm)sCXTboA$7A?zjnLNiJIEA3P7TGg|;U2bs0wGE+Uz;X9IH!
z#=ArZLUb*-j6G3B)i01JvZ4BkS;O|$sEE2SZah?`|w~RW$Z_}XumaOhKS~2h+4%)6CS1S?>%30l%EgXd)kR#fk@aDKlYnECfq>5pcqHj1xUcnweASV3pvbs
V_4;m!Z{ku(YLn3?*c!?MzW4^ms)>!>1Z)*1|a!?>ZYyi7|-e<5nj}PdxEP#Zk}QYrppJ=L>nijHKG{J0XMrqwF|kEAL-CbpFvkBmv!um%ihllIi6d>y~V<V
lcBE>!N=VU^b2cbk%pDGw1lBU?^(CZ(2o7QaD_VbF~`yoe8em}0kc8)@!GCcWxT+^x(TzSyg2W%ytCvV#}`Irb-G=hDQ<dGzf?LGr;!;iCQEMAX&bXvTh61$
X6nj?R&JLrzm<+e@;)a~-39}3-$K~i4<MhzJ>k@@JMg+GxHM;c9iN30DZ%XBFl({LhYno_)4I5=jz5Rz5X3hRwJ}azCik9{2hUNZkNws?t&}YscR*;$iwL~-
a>(d3AuR~T*Paq~M>fv~b^q8TuZ?M$?ncqB#{s{805GY5RG+G=tD|do66(z<e^F6s$yJEx!QaRzRny<;q_6af)m@!me_x=cK_75!%%l=3i;0$m_g^XX$XA2+
K!=A#1gL>$PKtCQ^>w~TR~o#M$szRRs|fk+$hDY`AswT4vYH;cKej<dW#=_Y<5#p0cxgN>tRC5}Qm*iHOcJv!E9_=|lfV6R!mk1+J{DLKc2uD9uE8GphIqf2
-~H2k^;}CaHT6%3#P3{5@|E}FK(M7ML+0#I&>OF3fzyWT)(-n7=rB>@Y<G54e}WIpV(Bu37Pw2#D&}Zt{vhS20IICED}Ljh083RWl&I$(^}Vh9_w4uV>yr6f
Dvr&*UT&(#;A~@Q?F5I`p!01E4~+%cLs0O{*cMDIy^_q-Ka3=3=&U}NuM6tO3}}+kaXy;Oio0M6=~4nJQi4%{Yo)DJ4xLjN%>~|thmqNY)8*@R_&UGhsI2aC
>g2)d(KxuFwI};W=n9ZO$`Y(+bCBCdu4!JYP`m}P_5~<QP)hA5govSa;^T(nQ@lK#yhB>CwiTiB^Gg=(YBQomCkmTFe|XkEqLc#y53K#u=LF~#7jhBx!@^^C
(1bWyi@gyVo+rXOpQX_-57*~(*q%7f91M~{-*g3_fq*I_njldYTe!t-kwrp<GDTVqjCw)yEtHhOD-0-$(Z^nwL#71~5%i<qzLc^I?KE$<y+3Jqe$iXE)qT?`
o#hLy<#!%$rv&i1E*Z^(`vdP_tWPO>sGp>lqQwreqOPdh_8`vz!R5L1FeqbRj|6uoR+4f#D&!*i!ZW`?fW`XyM=uSfTBdvBPL=7clnx)L))ekjY<$lC1=>g|
Ed`_*1~<W2u*+kh0c2>d4AjrKEb!V?>Xr5XQf(C{RJsyM<pL`5pzlhHi{~8WjFs2O)7b3stNagtbxvW38}XS+^q2U<SS6MTHWF3nGlW%>aW_Sde&q=C+z$;j
z3p+udJkf)`8?0;#k5UYn(z)4yU+LrRL3M^6CgFWizp}XabLn8Z@?Z0qBzaF`b#)fYt;g$qf-II<twwO^?r<{MbM66MKFdyOUmsXD-()WS8C=!)vr0zTilyN
m#ofrqoSo8G01Q`z}Q?{Y?pykTl$z$D3AvHVE2*7%DcwVgVrN+f6g<T4f~lW`l<H8Ema>$ezPa?{-)wJ>F13W@iU>ous&=lQc7U8djcC;r|V35u7$NB9+-@U
`eOllO(`8rs1Nv5d^vP4+!w`AF6S~_oTWlkW)_6*09a$0DG>YHPbL{W6m%s6!y`q6_(QfZq*V%R?N|t2{SrsV0KcVupym5?iIg9G-{lK)tR_lBg2|992cam>
1k*@+S|2iA<Vg!zIP5|&yG?7KVcT5NQE(&08)CY5<TT-tVbTyi5qXKQ&xx7DJ#3*N*z}&7OIs7so7gYY-8T9N)wB4QD<mQH#05=%=PI~Laz<jefi`nD3Vk|%
#M@9-ZY9SP#An4XqQVUo=!Ua@Ul56(%}AfISfQ7yEML}qEe4(Pfoj#|Nq(Hq$zliN%~Z}oFW9@?Q`rvvTFsUkTEj?R(kVFg2&&b5UX?dhzhrq~4q}KrNn}6m
Y6(f$Wa~rCmGX>5ys3}_{hm^kbj!$eh8g%(r3<easce7Io4AL7pX!_^uZ1%)>qSo4d7l^y*QfJ<e&H5P2<mpPm<3dPh?`+XuOHx)zBRP!em=g@)uqX_4Y%iV
s1yz+h-Y+&SsNF4#>?c9BcV(v>lLRdGv1_&7kc)^AW*%vP?ZBfb5sEp8cvmM4^)D>7;7{9Fgj~7fCu1Ve=1^2itK*(k<sj(!m!jY$m9W}c@OxV6`GVB&R0^0
gYuGDu@q*_l(nV#VIORsUzO|V{;5&1&y3I+g_H<H>Kb=<{g|*9g#K2%_S{3}|4VF6Ead^k0GkjPM&xADaw~x2LyHgE&pLzzOX=-jU!h2~o$08Co;cWQHS<k4
m?$&sysl%q`M`E%F8+-V!UHec|B5Czuj*C50MnX7gxs2;3-i`8+|Xi0{J7A3y<C|NPM!8b;-~fMOr#y(4}tEVfQbCKaiM(5G;}?NsNb`vRCz3Owsqk>#@0O6
#LKCgI5(JyODpXZDOj%k-x4hnM7ijF8VZkRX)FW~#SV(&*FZ|{Xyvzuly+mN>hBo|PM&)n?eiU&DHC*dPXT#<OI)(qR_rn=;@@z6EdAvzbce3!y6ba!LUqn{
qN?SOvUEB_P#IynlvZmVmJdG}XXXrR`V_L|H;qKT<d*%%&`vvnGc;GJ&#l<XC~D{Llj{>6Ya&<`oU!OQ1}s6vKVy7TF%C^P#*AAM8qg>_m&p71*f_OSgV~J%
jI|LL6tQX@jQ2b3(C;qwwb(!^13LLg=QS3*+34$+8Ke;t<(zt`)6ud29Gz%yOi#1f<$g9GBgWmg_ul8pnq{LXkM>;B_EMa!oG?5O6q86!)Q-wP@#CsE+o-II
=k4>((<}FO4tFY?z%|Be{yIpB%>!qM$YFei&$9-$DJ~|~WbbZiu(cJw4iZqxW0<1ksf`R&zgrdJ(okFlsOY+gAhRhk9k+~(Sb~~NyPBT$-MG$(fwz$8K5Vs_
*tVZSMqa+dcy>08XQ;VInZ!AbXlI#w<04e=Q$h|jc-FYFG#GOKt*%gLIziC@mXSprW1TbOM}!Fy#_)ziveKn3$xI~;yfp33d#E<VAO&i|kdh7W7(+!Axb_o`
-s&Sh8f4|w<*M6_&T#QhkJ*o&i@EIG7JZ&#EItN4)7X@x4B}N8c%OL49~XClys9ljPe!S3?xtuqU@<}2Dv@cPTDl1ivS2qbaZp7ZFT;u#78x`rIkCI!Hk7wO
;M3I{w}E%}knRHwEgAKQ1D<zid2NMZkgnVuFt+Sx?ZJry6L&vZ#W2yb=O;9LXp&G2W5WoKjGEJ!KF@gmp!>pMQ-o?!HF%Mu&&uPr^5vm?G;+QMh!rZZr(cP-
keG31)=*XX+oIKV-xa5(dxleKD$$o#ST}%^8cxAsPkAvFA_&jc(*gPB2!QTCzfWRdV7%<&ga*lgB}U$xqI=%QsS4T9!5&|lH}ni6E$*Q73@%v*zfdMtWg2jt
f%F27UdS|XBJDmcBXPrEd=?4km%B6^vC|sU%W<xAz1kdFT4Fb;B*!XDsrH(O4?lKR7ke9JOSZ%)V*;*g+bsbW;k`;?m+4Ht>>iuw`jtu`c(pUSLf>QNi6OJl
hOch_vfp`7^7MD?qvh$ED@`;}6zybmCoTt8*24$N*UYaVe1kR&==IPTYYsDdLsAl=_Il2qU9xsX#S(tHh~f(%Xy1153y~+o^O2qB_+n%8K_`?|R`YlMBwoPM
TGoI{)w9RHap?PQus9;Qp2}5$!hcWp&mPY&tR)?1zLN7K<!xSXEYa>1U8nBX#PrWmn(@vr{$1v~UhI!xyvK8rDpk7^VT<Fzwtg_HE`x-|hch;sRVnxAcrR5_
vJUdZi}a*#LDFd>-<us<k97HQtju2FU!Tm@!VlwkyJNgdt~p#KBgES$*(P>ok)R^*jqBDjg@LJUAK?#F&gOy!9Zik)8t$EwXI|Nxv4_81HzqBC;(+{L+|1cV
fn^CfWd>gB4bm`MY84Mh{NsFuyd+gyMg~IajJ$$HR?3ye5qs=I^x{4goPt@12zy;ydYs=cVq*Q<oY>@U)Zz`u`DrgHGo2}QN8F!_cq^|cGZA9JS`{jkcSd*9
V%Ta@UlvXv)U%f!yYo~<*1VH?=cj=^EzonbXo*y*C`N_7Uh_?)R|Jh(tN~1Y;)POWt%Y8G>YAL%@FmhHo8XIa-<P*8;6u~rLTVb0F`(zqn&AC1rv2j7;bxCJ
6wXd;*;48|de8PJbbah3uPd+r(pfJsG6?7d9q9Sq)|8w}@<i0ewkFB3lfdn@zg6dPvyKG2V`)0K42DlPHz`n_vozw&9=BFf^o~tP5%5&^ZWfxQlEX;SJ_-+o
(3KAC%pRz)CZHvu@)TQNm>M2C8|LM~P*%}#RO)OA4Q#LMLm+>JCNK>C#1t!IQ=1%X@Nw@CQpiX7@XHhke#=S+w&D|`7hJ%n(s`&$hj+u7nC5XV(Hq^Eowhrm
V1<sMz>)s6#QDXS7RAVm{5FlFoyx#E5OS#E7nzI=u=-`)a4$`MyiEC6d|$Q(NRn~Zn6T(GA<Dn-nxn^pHcc|=^?FSX!qJ)_8xO$KhJZt#BX_y&qE|IV+l55I
q4rEU&m&nWS(Mm~5P!nQL{4|pq!I|^^Z!X`Pt(<?gB<$@@`*1rg3va1Fa8xA@;@-Z+^WLqfpRc193N?K_py8$PI=A~FIpk6KYm>+(4gRF!r6B~uXvv0bdV)(
s_@thH()jZf#VuQ%SQ78D^NkARjT%79Z^h(7J?St(VG>Bv;sZYg^9-bRC@(+1>E8+#Pn7$W$q~0`t9qy)Ei^0tW+4<9QwEDamyrKQsiUO1YB#_S$cb0n{><Y
$)Z3rEvHS#eX9AJBv=C{HPcObBCL7$WP8SbkWG=v3h_Gl9sV45_Pxo>5OJ^jwxmRm&7i(E=RxcNr*=<4UP<K?1KrIB@7WR@i<~ofS~qp|PgBDzf;!&A5O(-F
`O2b|{2Ymy&YgAfy8ix<J8yTigg0mBAaAxu$)!6C>-)EZiMBGHjVT?@<svl}elgg3$%YV$&U?mI$t<!E(C#fE$qM@(3yv<^mE&`lrK@lS)&9jhjDIln@qi+e
U#w}?*BQNVo?VXkrRW|I4N2K#PR~85E)mXhqJ?a)(8)~7gGMy1imTT;8yn?VdeBvsc+xps1XT9_Dkc{|;^8yzQz)iz6yD_G?B<afX|{rZbB1&BvuQA??>8H-
4n<~Y<v!4xSU6a2QX}N*lC-@}m;dyCU1xh-VA12ZI$JLwQYUohA}@QkBP(?T<zT(}1<sFB0;eUvU3-v_#-e;%r;G3Xr6cBwy7EV=9sANSCgAUJzrCsT&w=Bj
(-kiSb%O1Z2^utzIhL6lS5O>!<NFIbNE$PpiTYllj=nT_2Run7b&gA7{_2MhP^SXhHx)G3>W=EJY*iAkYa=b_&q39o0;Qz!ht2-7-kRY%S;wZP$F1(i3#}+V
m8Ir0Ysy$p4W_P6DOETyxnl;to3>QkyIH!!F_Ge0h!oSs|EUBMOi1SS$TOnqK6}YD|0Ko7j)B=4O6|~rX}bxsn44go)M5`eX~^lL_|NP_0V7p)aPV`Ai8s;-
EnA~-1(!sfMf0K=8q<b+Z*kBX@V2e<ea7EgtS&bT1{z0{++G2Z!LAHB-QSAtc{#)iWEEqF^zw0#G0;5vmL#}^&$9|}<+I+13Wv5{YBEbmE7A#BS@<|@K?!sM
%Q!@y?TXXe)8q|sdz{V)dn|dX$Z;b;AHjE4N<CJ%GBMa7C!!LuHLgp>hVSt(hHLIR$#^+NB#W8`L#Xd5MzZjbie1@{JXb!NqdTE4dtdpFvyz>EOg`F$FHL*^
hMH%KAnx=WLol2ReSGLargPn$oD*};LtVD+d>~?v41B1T{_N4q+AAt>e*8hKeCb@hxXUygRA;?cHMr&GEa9fN&*`4idrs{IRkXx;9ELyBH&;BAj1j1|P5Y^<
co=KbBp|JmA97g!8uF;v59!DMSMPx>(DzPSo(_e+$L12#Yj!ct7gup2mcd5>*M-SvSBx!R40g%fC|~_3QiW0AvYX+o&->AJozbN7w8X<&l?CoM!IqK&rJ^-C
#63P>&9|D^!u$wyNoP2@Wb!W*9TVC>|HA#32TKE2wAZ^w=WE!JqZVoB-VxmEP_eqaI=^_wXxWyzrs?cI_PJ*62nAoRjS5o-mS4(e3X|tdyV*G}7?;2H{AWX)
tzk@cP0qCk4NgqSY0!MKcfXkvsn1xF)`EW8TeM9;mn=DY&!S>GUcYuXCXgz|+K^1#%xRFcta%EzE>EI@JyB(%D)OCvP8(zSB>GhPz?SR*Bl!TAlDUJuY=PYi
0K?Pirk5A}FVYFpp=z9|*c5Yo#V2a|nkmp>Z4DJ_U&i#jG=5VOJY%_0^SxJ&YR>LubksJ6s&2TPleyQ3(-hwp53!Ai`wNgAjVL#0LwwkNryiK<6c?jmG;T8G
5jJ8eraXwD^!hyiGY=<Z(88&{`@WXoSZMV>m9%@-mr7%t<H}Lk(;gC)<!oCm7P{WM)oQ%x13F^I1*klXB8n(CzVwLMKawihkg+FnCUN4VOBxI1@KIM^@$N`R
cEw@={fulQ7?-Q}zcwDI%3iYc#}>_yg3ix|TrP8oD1@d!Ow-iEdGaPX;VjQHXuh{79Ksb-mb)pyb&&k82I7B95*rxj3NGftdB0o2Df~DK!>D-}_8)#RLjp#C
WVTg@ug&3qOJQt?@YhtoV7&p@F`APZ6RCwwFxL_PGLaFxS@rkvG*nhwC~3k!AXYcjneb2)R<`O-t?NR&t-~&lck@czz&zEei$hn~>ZfXphm|&4L+!p+*6fBb
syK00T7Gx8<j&aOd%z8L0E_t}6)Nb=r;f2kd(P#qBRQru8zdTR@w#M7M>5B}HG-(rm&0sT%imrdZkGe%7u&c`9#8ir_hZl&^H7V<G^6qwM92;yZ(!8oZ6RPb
2edtA4?DH9T5(5dzCBpYG?9H%!!*sFrgR(I=&kZ*4QJWY^#lc<kthw7=im65eMH4Q!r}*A2U)3uIrBk*%P^FY|6w!17a%$TB*Pwny#hUISKHauWwp=icEl!f
xsVJFgS=j*g)!#!+>o7XJgBdozAjzEv}Oy-9rZh9_Qx+bw&eMa*S?MxMSl4FZ)UVp(Y%sOJOck+ErgHgMJ9#umQB@nkwsdhP9cDR0gflTHou2QL`9JiX#1+k
bqE5v2Z0i#Rq=!>*}MyI_PBEtb&HE1bJiQp&1_EC{{Zjad0ZTZRQN~b-cVm$l;B{J$TY<`v-095NE{KhGHsldr=#xd4A5)I0+0ClHhs2Ngax%|A=wo$_M|j|
Zo+0psP&mjXp$}b;Af&QXNMG1jr%<G5Q<qYfM^RNJQUr#X>YT4`_OHPh56RR7Ad4(G{TKJEH~1kV(9Ku2T0|=y-#+VR~lBEO@_(yPt$VkbSq($eyPP^^0wvG
Q#$!J1KKKYzm4ZcliV{+T9kLFDLk{A#|>Fk07k$>Lmd?Kj9R9d$m2_j_cJSd95Ss;#RZH~r8a$8YU<Zhq4sAZqu?;IE&?l-3$jPV@rTn;L~eshKWH-z-d7tg
ULVxYBg#6wjM_-D{9gAb$c*Y(LrJ@rTV5eiRG#44G0>oaejM6lKQOh4g?jMwSeiU__}GJKWwc!qLLJ{RU0-<R>-N2B6>4}PIoxUJRnpA~tBM<0!||lLaXvTv
oeBQolKfmEb3J5A_#&PLQ4hM?C(?aQ8mBmLs|rqflcGKcKDGH%L{~4W$9v6R3#K7mXAf)%>ipN6$MJV)FBE`9c10hoCg*G~zdjeud5Sjv&0OQguUim|T@R(~
<M~s)ZV|n`VYm`Ia9&gk2XPOMY(zQzE3ho3(P0SFG}$weCl7UrD%0C}@u8<;(CHpzXPYV}*UH(5KbvntR}!hpr;_l8QKW(u7T06}l%9S!OlRNb2#7s6h?I;p
KZ+Rxj#bP9mF$wI4z`U{8xDdhwtM&NMTX&j|CcSF;}H<K2%zv?B0(`&cHmN*vH3PvFNDbhXF!ph;S-=EfTFNDqPA0BQT(ly3mEklTs-q#xd(F~;pBMv1&s9J
31S9@x9474?^)kBNjmGD*1R=B3*#!!a%<`PdIHMbLg#hV21fuCd-F=)SB0(uSmS}C=Km2QVPDhJu+K>#sWH3tJ2wcHJ<QFI)A?r+s{=8(_GdT)so*NgGh>Vl
d?(ri8^)VAkN7@ope?X%!{0nU8&E=sB>YU(D;nC;N2pPs?kl(m(!d2(c_+T!mM<oL<_QQq7cQL&8Xp(G^~I!?P7-d)!Y2t|Cy{Ecj8Qj{YQHlt9i_nH3YqT$
@^$MWqOm_on~FLj6*mEs3-jw}v-H;X-HPTQE!FC;7o=i+p8Z_|)BUAk8nctt*hcTUQhXYECRC3rj5v4;D}nF-khCannOky@onT2A%pHKGT4%bC<Dfnrc|GcI
7J5i2<F!U$sFt_X?15vj<+}7h{M8;HndlIl7u(^xi;|2E8peF-E!^!2Pm!Cr9NZswztlfEe%*{B%Wl-{{EXfgZ)Y;XRt%x@;9?^FOvv(nY^0$il+&vPSOd8C
Zzir$hR!y*CM0Y<3K-Lx*jS*@J9XHtUeM-T+-_;f%WCB%Mi}YwHqBqeISgRKyJ5_z{@(XN5z`esCTP^#48op(X>eOn#|1FB?|hlU4sW1^>wHHRTq@O8vem<5
QGBcwTIi3>D%8hwt}a!z<JxYpdQC1J36Aq`>e@@3@&m&g4B}iw)}>XuqUlSrV9<~o!n&2qOx)&<!oF(TmezRwRFOVSj|7$}^`fbTrHNyanBau28@e%uM-{NO
4<QTJ57%XhsJbXkMSv<`X8ttPVbg=2M7bf^qiVc~QjIZ0+&wWayotma!p<9N!Gga9O-TZWWJ3`w*~)C^G}Lkt1-&;63?ZN9(ik7edK*wOvq*kHT*AZef~i};
A4Enp9pCqM%KL`(GpK^a0|3$Q<oqExy_vP4wTs=1u?L_)lAofTqWBvTJnp+rU!uwBPCww<mL?4iWmvf7?BoH1?$CdhS?u*4Ohk?rn#)x4jMwkeXm2)AjdIyE
x|AGtOXsF&ws_NmY_J=wM!8Ibn+3wQCGtbuOougwJhUrKgQNVQ4EZSD!8xeR%ujvx`@g*tOzbD|Up-KuYlm0_K(qO9)d?OeT<UzB27J@v-0--r1XW0Wp}!L=
Jn!xQ7Bv~jWXE!v3RZZ7nE!yLAD5v`0qu(8<jk}ufcScAqm;UP<?5s6^Jve3{~Oh_ipND9#WHA_f#?e5ANA51kFKI4#50q#5}x?2pKm;yDvA?GV}c}}u!Y`I
CiSDpWlYKT5OKa7gRQo8-fiUdr(VaNmc-|OBIp5-yDb{*vom=2(hxT#obqlD#znhjI~RUo7rlPF5lyFFQ}k~#U14Qn0tPSbN&z!2@#1zxRaJEj$P{^;Wyvbh
Bt|1r3)kn0<LCcNr@p}eCOcI616Rwqe!3lQgem!vQ^3X~=5o@MI%2V-8qmlCLH&>V%g--x-=A|_9RVtocTPe#3v}iI2M_444}TpM;<+c>))#i-GGkm7n~hU~
tR((QZb9A{rToEfd=xL|2wz5}Ev0f8!v%Dnu6{{Zz7{3>A$mYX(G{7j1_)&5Yo<B*sA{prCAYR*oRm%{moj)H_L>Z{|CY0H8h56eBfZ|jhT`~BVGgFNQhtfU
A>aGsS#m2C7Y1g!#V``&m-!B(M@LwPTJ|j!Gd8Z#Z7HI_DAmKRxoy*@@XmYB^U0|sqJJ7`ELK;~#4IY>Q7>j#0x-NLo48!cF0i(+$dT+ZbLH9><?T(E9%dQQ
Ajg5(z&i?yUPwjoC!DLO(|%XtYOb&g3=Fr80+91_VtTSThMZrcBRZt3GHFl>J4#rg739dSh5uAR$UuMk=A4@}@zmR6OlnohPO90~6cn6VwnS1#w5zypqbnp=
2?*6qBH~qCj#`IuM!}(Mj}>V?TK{)XpoXPl3-bgI7d=0QR}7|DeXGtf$+n|Vcv~f`6%=>ZhOZ*Rbu2hD#@RH_h-R?_5NjmWP-x^005E_J&^~g{^9!bsV*0sl
-)S)N1<0vPTvn639eU<b1pIdt&EX$^yp%|dme$`x{Rwj!<OOlLKJBTf1qx|Mtn;U$dL8F$Yg0cE#5YwePJ}W^n&Sb7@cRlxa(!MuOEh$I{Lzxz*MqAlUvAf_
2@jG01B7Sb56hd4;Dls$I3Se#e@Yey3UmW8SK;K?w^O=kL~@-^hvyutzi>CR5Kp)lM}UaD{4rA6?xt}a+1d4a{8`hdOa9d2JeZQil)S||{I;~JkQyiKO~(@i
K;-G*yLGcl*$zI@t}x#sG{)tE1-`Tv4g5{bT<qf*X`b_vGo4HWp1%GsOs8N7;k3{WWq+pQF6hxMO)}rMupA=q*RlO)55AnJX=5lRR(9BnWDEBB8<NkpY7>*M
Xh_=q+08$a^SB|jnrV?F_A~^OBP+)NmCmG?3!a@$+Bw&5b-e9jnyv?#n6(%kw_lxTzBAj$9`{i>18ZPVZtCL=*O)nKiG6K*p)cspkc-d{UR8M}$tTW86vf}D
LmqIo8yl1=t?)@bYM)<@4~h~@N#DdAHjoAq@M4-JL2&=lz0#rUGxBtniysYzyLpvYF-f*k)`A2~sJOf?tf4mFL}b1DtF2FG611y@MEqu(s#*ynZw4w{lP10I
2{`~<XjPjNhhS#XNEz2K#a+Ru$hc-t*z(=gV{N<4r3uAbLB6mgKbVr>CBv{{x*1mS0;*{c+x!3iq9{f{N8sS?2q{M<8ke;h|E?li5x(|>3gLds4)TQ?YGxb?
vsK;)8C0i8UV<DLvBGUW9SVK@eRdRn-NVbh{r?8Ec%-{MJ9Y2G;|-QA3gf^q6V$=4VfUwx4T|6B2NAaJ_Ju63j&id8{el~Lb;GTvEAP@CRc2`UkX$~QhG#}w
@t^_*NS&2X58)#4F=?laiZZ<%Wo?VEPZS07>B$dElY?P({WgTIoyeBRo4kQAvM|O@gCvj(h40u*VOHrmlViy{2?f6G3fudzk2I&VW`=V6%NSFw>ZxaT%3TPo
ARne^5qpVGKEk05F=-vpEDbXzaRmfm4h1$d!2xQy9Xz<+O-ij0!44#iebeWWQ|yd&b%9LyUNo=;-5l7abKZK23mc9sHme>!@M^i=TE+Ll_NLncsrOM+o*kpc
R3UTzsVArTujqKtvU@*dBV+A@Lr04vl%WM*=w0<Kfrx5|kaP2sADG~9ao_E7;7h>`=h?CvgqP;Q<5dZp?QcnIDhgW{Ga~dkSp)tewn1uC+K0&xS|uWHbq%lC
$F`BTcyPajxGgS!`X%i2h4(!v`4Ur#H8`z`TvU{dnheU`$2VRISJB^Jp&P4eqG{Cx?qT<^5z0E5-w23eng2?GHE3$cIMLEFb=9j70Js*2+Ec4A#J#2D@$8t{
rCS25&6N<A5WD^QDmm+hdKO5h#fvqF9_8pClB;<9%qzpiDXo{0sTO1k2HhfEYYcw#Fq@M6?S5N;^r(Z=)K#_9{9cu4-O#M45XZ@^i3(1TIV>=dilv~U-5SFc
&loLM#Ins}O;FPUGraR^RBh2Us68dx-~e%d5*J}5tm~8X5lVnjc8Pj^#GIIq@YY%dPbD3<0NvPKcTBRy2r)-A=LmP?6rs>_odf(50CxRkK*Lm2hO+|b(#jtf
Ho;(2O`LcUKlVM=0#L*561mPUT2HT|v;dIRCf?ypiY<DmE^pQ&kAzvbD+G^zV)O3z#i0nA(JYSCQ6vCNFKq=(pgv;^W}lGe_f4J|Vgp@9pRZ$kXNCG~nrMU4
ug;A5Q}Q}*yWdTZ%9?&jdmKw=Y1$JNaZ_gBkSq|;3Sj_gZy->|wkVdc>$|NlkirnmBO>FPQo?~cSv@F9Rid$mO<oPJ+P!C+H51^^QIi$Ifdn_FTpdtlQ?9u2
!RSJGn62WsfIp%Q0ZfVx^UgFVU}NCmWPdC^j~&`IR<XSaa6XkEUF5b6jG!t`!l$3u27>-t0n;R?QlDuG0A+s9X70C{6=K+Np-)$UH5<?{3TZ$x1^+@zPw##N
VH)luWsZj*JF~~*g|ZO76)CFNKFX^IYO*H3UQ7FHT9#0snjzuBP>a93<?e~TeztDn8Y+%z+HV|>gV|Ci`@tln5H6nsNQWD!FkL(D)r{|;$H%W9r4|{ho$$80
Rc{D2(w^vR`1<eGE0ioy)?uw5{**VbuBGh(Q8i>zTp!J}l03pbSZ8BNLQ=AzfB)k#LpD~*up)?>Bpz2-zU-^g-`aP^!RQ%Hb(FeB&)^_Y5#Xt*q1(%`CQSQQ
>T{93<CrVev(6d?Q#lW03{pOT46Bv1HDqniKmqLR`-rMuyqTIP-ww>_{VKLnZ$YKD)b%RGK-k`%*&D&7T7LeBB&{Hc_a(oe1zf2(Cvl|bzYOA1{)`O~HLpz7
UOc)A!&?PM`v2OBUs8f{6QoGnKdDi~bFCVVXVXub?JAcJytGOl3EN-qcu*YoZM@w~PLtLnq4_O}PMBDr5YmSMbHs9!&w^8eQ4Uhdlv$=bJ^VQKf*I<*7<^T1
xe_8UrJ-@sR57*?Y$+e<#><C6L*by_gH?W&UBu<ea2QwNN3ro;#5mh_q8GY<68+Mey5qw;0tm1Wcm2F0yY@RU>LLNDsYSi!p5bb78X_+`ouOPY+nDzl$??6Q
D?{W~XuWz5S}A`HcDTHgb7SzVN@@U&Iy@@hVc?P<v-WJemQIV{d8rA@Ud!w4D@_w?S}Z^{W-B433*!}sD0)&`xib0C8xzQ&I1qhUa01|2sRyN5IqgaW9Eq^%
%YB%0Ino&xR{P`*i!r%|ZFAgXmnpemW(7$|Gr-Kzlmv#B7p_*b!6e<>#Vx`@C=2m7--aevdfv^7tW8sgXDyDt5SJK-umJ{I38B4Wn;-kXH9+CN(q~IyY__C?
&e&G;Hy!1Kg~WqbarLG=!wZgI2COUo0;9zopzS-?@I+prDz|3Kt(+820EIC(gD_5Sdp8MltW3#i)RA>s&S63@l{@asOTNr~i?dtTVoOUi*!!+GUq-rLxL{{a
c09t#pRlCyNV}5f6WXSrqM&x)@aJfmI_-RXKZ-zc6}Q?X_2s;mW3f5PK-%@qtjE7gHd&||kd2%%YpJ{`i%J4aki{>!()SsIj9^^`$!<z%K&w_;E@%;hq+^XH
RVH<4D*lwDv<+I$z!j-&c^yDbdU?J7UlVvciA<|gJDY0c1CaijP+6F@Fr!DC0CrH($tLkOoJyKqXLCt>pGF;GvEQN{5rpMtdXQDHKiXEUaARZ!drQAMNhoOM
1jaMGP7u5c1ta<$mbP7CBl)k<LGUxm>R*xdy>d}q*Jof-8z-ZesWN0}`*vIhc3G45nxUjI_34ehW(+rcrPjOS!OH15u&JgF$}T`Z5t)ggJ8;S)9QANoF2jzU
H(vCJMNkPH01qPTGo)MrJf<-d*m%HNC1_pi;X6MzFM`;h=gkA*G5-t=jY4vszTB_{;%c+LcPs_-h}NHlC_FQ~qP_M*sLNN9mKVeM>6!nlf$Fv^>ywUE>B!rO
#b|pZ%0bHRqW;?-bjD*jvaf>gKQ2E|)ezcN;pGlmz*6!`@XNyTYcJ$=wW<jWln5&~LmW|&;&@5fyOT-nc5@Cgd36;Y>b$4&zv-%~*{R96F^Qufdg=x#<IHjI
xjoPGR2i}kaJ(to8%ywQ@6DIGbkE38{m4_z=!cP_7Spns!`9qIR5iJHxJ;Qy6P>*Of2fk@$my=ylYPI?GhsV6(iY1VnQMHY>2VE)=v!V4Uwwdc*5AT&x7?{{
dg8q2-AKmVvhh4vFssZ*b>EJYB=H34eT_w2TiXVum#MtFLS){*UQrVZqsz@rRzIND#Ia5m9%ftJUjHq55bY)Li!CWS&0i11fPoIQd{1Krm(H<#WVBVxIJti6
;BlBtDdb*@UD=uFaA5^2w1MAFEqW$vLhnB-N8GDONV0o>0<8vZ9!XIzE50=a8Ul|TQ*s+?j)bxdS}^(8@|#Q6jwb_;b5uhdOts9^;D@wJmza)zeAYmiQQdla
y|Q)M7VdZgV+r<dt7$3NeRH^~)kutZf-+kIR0RIbT_tt8+p;PiG7jR4ecuvec*3W%xNcMqfFWEh=DP~$sF#w%zXDfcQTIq*y6gj-+qqEb&Zup+a*U5rQ)vVo
*tLS+(!S3_jE7tB2=YhM-%nB71@#dTCtpx?e%Z{HyGs;)B;aHf*NjpUPs}4G3lv{H5`S@0Nm+qe<Yy8L%WHg)ohPY;PIrJA5!|+;8zn-#4=AV;m-*Zu5Dau_
ll;Z5a1C$KA2>y~d%WQ}&&g`1^=cf91I~ikjxMmnPvYdO?=^FN12KxcOLc?gZ*Hj8Bf)nHoM9?J6URfIzy_`NV=2jEc3lEw4Yjylm^K2aB3DgqVU?-%O#>qX
BwofhYCJ|O%>)#-(C+Hk0|WRWpz$lw;QUp3{#_&h-c?&kT@O;1T;+)bX_`V}$*jh5ekTEhxeGUGM!Yb;T;fB-<>B`6*G--lZ-$c~V|}1$dFrw}uOmo2#D~gV
lH0m%gizW_CStw>*#$1Hw}Xl)>h*;w)tkugcBtA_tx(rs7<WjCypGU74PY$UV>k66osjEB=a`=XLa_rX3(jt390*||d?Eh;zS07Y);Z0f#PhpvA}|L7jw>tz
RRRwKEqsISXd)F_3b~6KRtZ~hxam!X4}!9bqlQX8Z1-|49srk=1Ui*yZgDb5a*nZ1FWlHgDsIEdjfllaKhq(vy$mZNg#Db3I2HDqD7L_=qn?0`hVD&S9_X`l
)+B#3%xSEVhB&Qx`d4fyH&ru>*a=vgnR+?ImMS1>S3s|~-fH9^ttpJjxP@(Vs{eF&P(96e+v1T9Yk5SqM(<&xAhNrA1hG8mmtIs$3fJd@TvPiIQ1vLRLnj@A
ZwVe7$L3rB^`b8DrLPY;Vsrh5^iu?uado#PC-qvZUnP}aS7StK2UDd1a(d+K)&BgN*@0ZhBiCyFUqBt~0lj~)1enTAH!nP5S@zWtmx=*wgK+d!%2n&fIzQVZ
cHkn`%{u6jch97Fkd#tt6*!25IAT0S$$Wrm=NM1XnYWxeAnGUlrIU6U7onitcEN6`iJ{i~sva@c=&(5M;~00HN#-86D9JSoYHXNhq^lx#O_55f&R&8nIvHb{
qQ&>%C8L=3)Gh*0c#gJ|3FQSn;_lM?d1n?&e~>XYz3hM?jh(@iPhX*hhVWC0N(M;aVfSS<);)lM>$q(fEdKx4|Gt!EmC*%dR@w`P2ij%?zjmx_P+U6A*M<ge
qv|Uk^3`f4EN760k_a>A0@o0I<J(B1F+HXi5)HGu3d(Wd<z{zivXv8M?SjR`WBVUGx<*BbW0R@}NJs0?c06ItDD=S8ge)BQvNl9JKpP^d2@I;jCnGi@twndz
?ULGwS_pQs07t4R()p>8-*C?4o3X<g8IbnoY%Pv}*<8oKfa1oprbX@mp;%8vVl9qali+*ySGV`c)d#6?YQzoI*D&zbV|heB{%{%kpAw{Y6gFUwkSR`=-O%bA
bX#sdApzr6wCyrpv*LwP0Km`IOUDqv*HqNth^3n~j0W4Cc}vTR<;-zV%(*C~w-Ix_qO|#c9j_+PTmYBMzw5NUC2R4)sboX6RyeCY>_p)j8_J!Vnw)Z$<vTYz
aDs|h2)=(PKYn6db<s_56(7$<3xl9LRR>pm$kcy~Fj+XgS8nZz-mRvYSM})cbA@y^?9F$rk9{`C{4lE&JM}l}O1ryuzcJ~d<RH&kv0w~xag5o`IyFiakhF-2
pPy0gCV{-YK(oplMlcc4NRX%F*Lb!eF`&Nq;}CNwnu?lN`2*|d5U9TX^1IaTPh#-34O1=a8L3cc+Cy-ojC0y)vh}SeUEylwr0u*Q^Q{Dhpv*ijOX1*WrC-3|
2?lHA-Mk{!ej>GMvK|sgDq3lOqY})$^k|?n-{XmCw=zK8SyWlN(q}v)JQK`vK~ok-Vz|B$l@x?C!s0n>WJZtNxP<TExkTrHjB#XWqj%G6yBNr6!LGc-L6Uq2
;mDetbERVAb@*)mRe=D!Csj5HjSs{?B~!SepvSaxyt0BuP-aZeCWu#*hOuSQ;@;-~+2>LL`WELOL$eCDqM~zt26jd8j<jmwi*<S`pbeK}%i?NnZOV3;e{C?&
(Aqk7lwS^ERA@M6ElsBbqriOEg#y{LIMUGGBzAWRUAt-Z23jIJhqMCCTQtL-Ld4>q7U}97Q6l&ShWG`uN!W6n5yRkW!b1=N%y6Hfh>i60lsS1n^3UHMJ1|Lg
AG>5!ZJGWNIvZ8;`9dMxy3JNPPG`IMZvcH{M{M3Da7fpz-C2unZgpb|b;B$dN5?yJZTSyn7{%m60Zz|pIB~}<XJ52nX!&jh39CD>sw%D^>FwF>Q5<vWa3-;u
hb%R-!$kqqM9w0F5l`%myF&ioGF4((O40(K(DFBwj-D}9@Od}oP~RgsmN(L**;do*_4>-qEziG-_4Ol?2PYQTFHjNdHdbHjBNZlwL8Z)Y9LfVo*-1gzEJ_>|
qNYy@hx6QWUXE6II>&t8=>}y@{zZDNG)sqnWDM~cv4}+R6K$2n@=<P$d0{LCR};>GXsOEI(8+Hah_MuaUSGb&`1;XO+^L6!t`N6p1v!NVExWE_qP6m_kq>ls
zL;%w{zvZTDiS{q7}CyyS<I#Zz`vr=Z-H>PlS65W{!M-JMBfjd%9_<TKXi{<%FCL$U6Dq`$4tT8^G^RT!mlYU6~{-pf|_wdq`Dgf@+&fF=3)0-YsRTBD2bzZ
MLU}uoy4#=)u^`JTUJzC4dxxW{&JEKI$~U_Zk}5I7G=PlxMTZmSF5{}wm+DSRW<+kko&rD|5KtR_Oo}$%9Z@M%`fspX?MZo526E@Vk0Lq4~MpPQ4L1RKtCZw
w7}y}?t+5YsjCNb;&>fJer4GvZOo6&wUznmq&$K~Ngj5wLGU9(E;ZyQB#<LzFQeNC7(T0|1-BHH*>5hcOXII)l0QAMUaLusA|aX;sTo9Tn9Do!5TX_~6t9$m
iODC4*V5Yt`pkeXNF#GcoG@Ph)^Ds^p}b)bcv=r0NIHOTID|%<$I*pZsolFbHP5vZKL_ledp~vf$5g^|yh!1mV64IpUpV;i15y>f(A8y$q%!EOFr>K<siX3W
7I|0q7h{qIt8=rgNM;^aUH5u<$vZzjsqu$*8>4uXx>|s=_dm4F^4AqIQuUeN6P|?G_XH_<CksTgA6RoNq9$tnl31TGgza4{Fet-UOL58RdTA<3LRu|tD=NLg
r$LooF^%XTyD__<=mnxjXlvN6mWj2RFu%McXcOmmCbGrnoV2%zs5(?OPcH^%AJPFv%o`78!eA3_mx~nL<qr(HPBSf^{4gDeyx}OA@ZMndppK55a|)CG>O43m
yDulBLvPRlQ#y%=D0Esigy~BwL!6XhM~DF<wv{&QU+vr0J)eKNK{U1!Q~DVJ0Or02h_G{lh>JXG+UwScZh5e>;*{J*a1%!2;%PRlap01F@~r%rUSraS@H;}m
y7G#~IPBo~POIN@hVom$)YxKnMn&OsBP^v*3D?mQK>d3JdSQAU8y>KV!BjA$rg~sZr=I>g2x}FcEzJ`I`pc}Do+O>3E}2Q!G^T_fmMIMySb+ty>KwhM*!P<{
BTv#h36`FLXC*dZa!5Q>xS$`>*Pht@&4LmQqQnauabJ_pIxvs1!G!tMH3V_ocEDK4y>3T}g#Z5);-FWQ&{}h=w0uN^xqo&hf=u`+)ruadOMYXCy@11wYm?`S
)TmP_#NEr~jOlK_y=gg8siOPhD&=#z@L}P@mgBd2+`I0^Z1f*S{TX9o#l%~Jc6ar8p+sC^kCU>vbA1~++u{_vl(lHy4Or~3%MRHKQvUBMhszGDH1aElw~@2h
RbP3(YH$!a8rZinyzJoW;dD&$LeE|V&b8>gPf9^*0`cWW(IcSiH&1_wt1%yNBY6_Qakm(u7M<-PzKO>m=}wj%*HJWmTucKeL>7DPfL70%6LEqBvQ7KKX#3y{
g<RdzXe)_y5u0a3WW9g$oLUajmuu_?=_2|fKSjCoCK!t}70Vc29#NuGRFpE8`eY?|or6xxlBgg3dqG0y56@TAf_PgxEcShjoXP*F+Udp%T79+WtZsHAVZ3Ek
osC~z#o!GY0VWwu14lN2_&5gDRv4ziP{kzxWWyXw(2$1LG7wJ|z&IR+@4l}I(K<zw4-Bkhk6K5ealQU2J|YBKzI>W#)E84EXx@O$mp$J28kRsD{Cq|(0_)uL
LL@u>5v1gbUA3A9MT2$XxK<g9Ku;N;B1RUG-3jSD`yNrj*IthtwssmP>UU_5ytU4QgM_iBjL;wkZEIa27X`DRRwTvsZ9D)AU=Y7xG&{j6k;sOvGP`w9b=1eV
?mKQ17^tmZ72D&G4pzHA<p~CSe!5!z1W!LZqvbaJZx{IP`2b0tVdBQ)2C~|ySu5v4q0Lx)zq94TP`nDP&Z|bPX9<qQ)eSIXC1b9WpjkKd{S7$5=}($s+inm8
v6mVu9?)tl2|umJnud)Ia2W3_l3y;+9%3fvG5ulZ`BfXaf?c2Ocsxu>DN;|MFg6Ki(?hiip9IIhv49DOL!8*qDQZmiD+n^|Ko38x&N&aXb!V(ye?{6Y0zp<Q
!+&sbBqjVe#(?LMiD?T=wX_{yrB^7gJgYR7-o{6^SUw<Ea#9q<ntGc9f;n}kuooLA@v2f=gwKsw9LXd&=1N|ud9zXeR^NaSqOA1fF(7vKGP+q@2})uC`@tXt
?@4`lefmANO60Z*Ey$0wZSq0P;<rNfHb(12j9bqpH5uvC%Wj$WOE`bU4xbEjW1VYmoIGd%Zd#N(cR?D4g3F#_x2lgsKjEkv%ks#-Tow-7WPp52oo_BO>o495
H{eaq+aaxD2R^}FoNCa9^#)B1ZKHE1x~u|eQ48UkW_RT*LH|>{V!*O-6-v@_*=yF82*BDYRtgU3_2-S$@dr#|MxG1uclP-ym+1X7zLOe$S^LZD5C1Vu(@)8m
EG>0d;Gbi|O6FtZ?lLLiI$yCV;ZK7;p&aCVB3R;@*4FdBwv?V?DHtFblY?n=&iS|qchB*5&m=}dTmM1hvm~6Wv}&h}1KjSjr1&^F$2k^kY-pzXSRYn4oxfp7
1G;>xwFi%Eu(Hw5Cq6L|R(4`pSu}lyH_Af2*aZ{BHcP4K7#DL~IL)G1c|+U%`u;Ycv9FMWoZd1kND@MW5FATlLSlB3x^A6jk-Ik}h8OaET(pV{LaUfv#YzNo
z~2;@hf>+ZTH9Y72PpckA}EkS1IGZxCbla@a6&GSyC&_@jJMAonYcv&^+wTP8C=xd9gZP9)c#V3;S!y`KBRxAB8+e^0505!o$;2|i+u1Y?KRfWU7ey20-3a}
4_{pDUA-+M4>Xg1NXKaY6a@OGdli{SORXXb4Vr+xa7){0HZ>G1(D03eYJJx%FYQ=hw)eQTSTgr6#<t%%TQC*q#&+HwI{Dte=3W&hz)<R6=QTPen1y+kX$ggv
hmucfE~D-u3fG)v4MuH8^&1LG=&X$FzWPEG>4IqVpyRxpQx(AbcJN(<pGq!(vXC?XSr;{Il_{8Cs|uCs!bp-H)1#_YZg#V$W!o~?K(yb097u|{`_2v9TlWvR
eX*c0NPka&YxPh&sO0u<M@Mj0J|Y}CL!WDXJ(nTZ$Fg*A6y0wmbb<=sk}2Ur%1*urK*9Niadhf3A*iH^xcj~O1{m1boVS%i-Gzq0WELeaJmR9=fHebA7)#!v
Jm+UZC})qa5CHaz_W@Y9ccs?CA`pwEUfbI_bw}>wH5Tmk%L1q`y+w{jp>(VsKw2Y)`mSp_-jk7jb_b}nQ2+W;^tax!OZ(($+sD~LO+wl6UB21fA>~@H_zXg!
vtTOJEue>b`Hv24sjoqyav9r#JHqW;|DQh4{0gzwL{v5>#i~)bEWG)7clPn8Gbb0&c=1~%hMj;o^a;*XK*WWST2ym-SQx4r6V!9M#{`{$eM?rG8(r4>r=%Q8
W_k3LCW%k`a6>Bh|4g2q<mGwWA&p$u3E>~axLI5OU@b(BuhKI$C=0Q{^#Gda%2M3%jIv{bUXu{PAN4(a@s^|>SCAJvu|hh2TFXX1N-g54Qoau$*a)dcdSoS{
QMGFc|A4CKEfmU$w;B?LyC!w}KCd<rb^N;yo@wi_q_(k7$C``wwsdAFj%Ya>fgUSiY_GK9MG}K<R2vmI&W-T$ynF}>r9!_rS5&}ZQl$a6k^(syojD}}XhnJy
BP0<dULOe&2(a;2NrtQqs82B?gh6OnN)FU^daS+i$TZVpK{P5BJ;HrYM=5m}(mziqRgxk?fxV7NV&mSpGm*;Y=I69`sHbUFj#N}NM2pu=KX<@4RFP0VvA`ec
NWa>fp;(OjCQL99(HS@y(IxMCFZmpWiE)ZaA<3?MM|gC{2c#cF16|@)S65njdFu6)G?BnidrB6aC15=TXY80iw{!3vaE~S3oNWbMlfQerGzVS1<N5*Rd=$@!
pl~2moyE6n8u?dY>qM~Z$f2ThUd~(V*sR1IRFG=hiSY&3Um0iF(It2!6j<%<N?&F&Sd=ELSHS0;*o9N`I~k6}T|!RpVNVOVbA*SQMQw6ELj$pPzQR7S_Sonf
M{8DuFLgoYm;wBKGt*min7q*!ETO9(-gPKk`c11`D<-S5B0j;py6y#hfqszk9VzaFnC>cIE1K-DkGL(CsdjTRhgD$Y5dHaGt9z{cc3oEHaK`L`K_*KLbD`cv
0|S>Sq9O&NN0_5v=bL$Q^#@KFcPckJiN9GjBm6F7N<pp)|2Q+>@e3JiR{~JUZV?4&0Ts??OmxfTc7O`Pn{7l#=u(pLF6^YXFf$tn@M`bFcUJoNp%2~9E;6|7
q-x?lHo}O}2?MH@nNhKaPMz1#vzjZVKhydafq5uYxK9-xFo>2v_)jExNL<I+u2)nKIsg8>*$hIi2oE?=K7}RtBE;Z9KRRmVmudnN>V?Nv2zY-zbL1Ej_*5k?
lTI2(BxbH@#fkj0NYLc;>%*C<nILHv@Sfh8m7c6>oKT?$+^F786qi@(_ezMk=f<>&zzcScdJoWLJ`zM5KJcCw?v*h9-qT_2T<>TssF-2U6OX-LD8O)z%{*<P
bae_C5Mk0+PahQ3F>>i^jH7y91+=T=mJ>HxaBU8)P@>#0>^bmfyI*~ty>hL(x+V9M$mB>>D+Jvap3eYk4eX%c<r?t4!K8U%Hp?r%-1V=Yh|F*f!Gk&PLYnNZ
A8eph-RCht0_#)O!}znoG+-YJ+uJq!HR!hAf&)IUi${N@J8pz4QW^Oj?w<F&qI})z6<N7C`PEz<lug>oJfH7if&U6GwdE>mag*`egzoW6^KTE1T5!HZ%ShYD
)kQDO-X8&MEZE@QodY9!>g?d_>i&IET>6@fywKG8B!cV4qSja-QFA6RZ3P^R&c$3NB0aLCa4nQUouVK;Hf^?Pqt<(E`9u7G8FkVMNRIRCtLOG{4}8f;{APe*
4gyYHxUKO}M>(C+SVG7bZlN6k%VGmP=DHPeJh9^%*Ipl=?&GB3FtWj08?WnNEAl@~`F2~YmEJXuE#^Gk9A~MvaIHL=EYO|}H36X8g9{j#7b|FECy0}6UR)+8
9;8>94gG(9q?op?|8PME;6rT%zQD1z->oeIIrJlkexRxgUT|R-w)qz3gEuaF`7D3Hp4V)OptMcQE0GA&GyA%re$1fYs5muh0HwZXK7R2xP-_^y$O!iIG0et@
h>KUcXcf&yo||7SKTWbDHNbU=d1pSRTKYlH(Duxd<#1M4D)90T1sF8RoDwICIgtMnn<q(2nd0oTMXsn~yT}||3^0|-p5nOPKZNl8^xJ0mdZ`s2c=lJ)wHVe?
Nk4MQ(8tFMN?7f<eR>tyAK8R$;G=3`dKB$t%A%g|O)lNWZ>(z5N6?}OD+E5r0u>6>v^`L(9P8q0K9Hs{S<zH98M2lwa`Rwh@YusSP=P}+gyVyGT;FmTj4Wwx
q!J}ieVXm_^<wrblX?ok?LKvRJ1v>-;A&kVBBh>6q1d&c#?8Ac-+6(z+QW)Yh5aL4Jf8;dAT~?N(C4`pRrz`-@n1c#?di)BbGs!c!|^b}&JY-jfQ!yK_r3fE
-0~!;Wv3vWlRvtM`>s{p!-XeR&VG~~2ijhgLi8)1-f3x=kL9Q_W&c*Ords=w^NItqTJ{#DpjJ%8+}0IU>&H#YRNDy4qm52o)BP`WTwm&UO7>7$3vILt75PIy
TkQogPw#6=8NZvJY*oT0tYT)O=3tYC%fD5Z+5S#oC5Q4!F8Y;vh4Da%?Im|c)9KI~#%g(nOq6W>HvoiF{A;;+*X1f(_9^65rh%ti7b+^??Sum{DzScbXpu7l
2nT}G8*+q+*ka^5<%oYCUll&J2JfTOX$gQ!9eJ&X?Cha87f{LW{^je<$U-1g3LFtEcSopqkX1+2*+lb1(L%YMfxme5ze^;0$OQbm(^U}@?a(t`n}3~|CC${h
xhloX1#oxeVTej&sohP+rzUnYnOyY_C4G|pl_M^NjY3rMD+6Dq>Q)ZcrotKB3gI47X~E=H-ZlAnMwHC<I&w4$GYsO~<dT<cG6$d&b9DFGd(Lw}D1WK!KJGsK
_7$!+3wp|5BGHwt^pOOj6mq6CDOQ>K7gGmoJ!39C1I;phPutbmp&%vn48LgTK_@X?SR;02d)d=Tzu&)L;U4~iCUkg$bSDg3GUJm|*rW<#nggXSna^Gg|M#XA
U9<^ShNJKtYhyRA-pDEZ;ru}g%BI|g2q1Lizf1cp0hNH$3-H2-om1|}a6IrVtapv<55KXn2wmFV*mmW_XM~?j=@#12$x(bxExo%EKBXfHfs1i?kBSok<t8Ou
zbr}-@KIE(JOG%Dgn-DD>V(7J(m%&#0cE%@cU}Q&{yi-$WRwm31<Mo~sAJ;P(i}pg{oN_Qq5j5`0)@eQI#FzF_104<V3#`i?`Q$Nr+o=)-v96DdR7WFACE@R
g#=qvO;ZaIgW$$9E?VTJcrBNoK#O-<@lluev3C$&?|LU=cNB?uT3cV?fXtnDoQ$hFp<4=WWxhm?*rKBmzdO3~AV$^TKtf-t!`FP(&17h4t2@(xh)oJS0~)l^
N5~gzOW&yJUcK%O#rmh^wZIFzfa2Qgkb!KUcyEkeDLi82LmM8x==YVJ=c%w&_21l{ko*T80y<$&lL=Y0A}gzat+}9)aNw1^S&q8M<^5R_?Vrzfke*0l#a{@3
GwnO6ziyubaRM<oBWT7<Ik^`6^0<dV0i&6O)0m!@gIKB@TKV;Kl%(Lw(pv;0!R(gG%=klZ%BV{Ez-E^MhAPOqarrFn(3~7JJe@desV8{OKLJSekQv74OAZxw
bA7~9?}*#NmIDlCI6*F);E9*Uh|X6_5Q>SbywZDmV7>oTy{6-_7qG{kzDt)RaT?)LqM&ortpuV%C6UGoqs<=6lyZ}u)LHPx969FaL6Vh~%HBS(!zTyDFQJtM
59J=!1c=LA?3v(OKfIe$U+6nSuK(Hh>*us6GrN8<L<0F0M<86DOpL-m&$@5y8XnpUqXRP~y4hsrgeo)LIYVy&RZ$O01o9B62j{8EmF8gmxK<cnN~*i5Q&}m1
c>>AC$)_N&WP9}Y@QEL+BkKTsa5J*gYq!#U-6~t+$2>0E2r3c3Y^c>aKYlpWA+&yBSy8&WfDfZZ8%v>86iub9v;#x73LP<a$H7FyJxM+_WR*m;&F!_HJYjfc
I7>drg{EGfIrGZiV~3vA1vDASD^0CQu|3XR$O-|KZ^NTubZP#GDb%WiZ@D{R+$#Td81@R#+Rs&Z;@n3JO^jE5(4)yx@zZM%)m}$8H(Ns)jSDeLG`%DK;maa?
SBf(L%cF$ymO!&<9KyyKJufi$PTchftz($B;N6n}L)Mbw*(xICU%9GU|BXiheO|8XSYg4FU)7qH3QyFL3s6z$+qp&Wd;uh-icB&LU6ennB#qv9v{E%Nr8p3}
>i=044u$JXEm>@U2Dod@{Yp62X$b~|tXsgrOBMhI;Y&*+>CAnOhNslh=mcWCxIJFR?u;ZBZIxnAG#Nf{r~O<cC8d#?-TckVJa${bVsUie*=PM){zq(mcL-_9
1K>OSZByTC9rRp^D@~bIie`h#f_9f6vVkHlVcM_RyGYMX#0w<Up}eC^ye1kvED}{YhjcF$J=a9#u>4Bm@(cec&<L)6+a?=14#~TvQ{ZzZJ(8)G0acTMz>&JF
kH<!t`yna~wL6~l7;?9!J`CRYcsIcczHVXX=;Lp%GCvvs%X29-g0(%vy7dGv4f)=GxW^&oRX8vhzW(*|fP<X(Em&+^0FCBpQ9bcr+|LTt?`MYzU_gBe?vScg
u|mQQWq7tV8q5v4^Z2ZT<c*w)l;$X^5+MwEZYi0Qt(Zr76t0HvvI!^ZVHvT~lUm)gJpAwjO7c@^g*hJaSWlr|8H(`{E@^HESSRWfK$`dn*CS|7lmoqIja}pK
3b$5&QCwU7($2_sn9b{a`baqr<tlh8<;|<}Fp$bI6qu!|I_aoH$^}Fd%KaNI?ge=Xgk!5h8hSIY((bGK$MM5JYS!F%Khk9-bPMVkIS?Jhb@hZP9bvEROhlS5
u0&?)8BO~O`bVg;toRn32+ii(K8Va)8z}zN29&?V2ow*Uc46Uo8K<ggJ4=9@ZxAvPa|0t=e-&>36Vz!0N3XSC%7g7BE#1_P<kKjYfkbC}_(a`@7C}Iw$6)nQ
X7~go0pjv3b%<BQPG7}5bM0gk$B#W54o*BX*|U<H+J@)4enayzXIIjt(o<mBjFhQrWV}h5IWm7$Um1%XfWQY#B4?w>^aoK`{WV4K=zmpQ``|!k%68^dzLI2w
9Kdub4s{5o@eqCF8qTBd5YUWqOUjz6uh8%aB4$*7x?<J>mcj>^`(dQ+nP^+O(Q>^oy0P8XAHA`~dqdS{nG0*NKpZoIpf^ZsINVj9*DICBn1&y^nV;I7YCyyq
1x;QeBX}{19-(W4N76!yj&Doyw<%@3l5C&VE&LA$h5X#vtO`Z1TL>fgW!432ikft^x(hF5&M(whxp^Ib1OjYj@IzDpL9xzRssW`BW%qa<SYmS7b1NJ6F%l^r
GqqM@9r|a6*Iu<ea@XOvnbbH}+nGB1uPnXACj~Sx%7v)<M1sK^2G|5vuK0ay+ye-S2IEabn7di~?(ZWB-(m2)yS8%uflw_Ttier99DP*(5}d{au%%mVC%^1#
l@nBZN>;Zq_oxAQ>gbfr^9pjKy7U4(cW5<|wmJreYlgOK9<a>d;IWSASkG!j5@DgFq9V%n%G)C8GR15Y{L$}mj}VZFOHHuRsJdS{_nVzxI~IxM{9h9zE<eBE
#^=<Mo)9It?C{Y-B5=#r7a&$G^#k43m!6OPT`a*omA(4O?&qN!?|pG`+sO)mVd@)^%8LRUt)3H$4%5=TK!6KdG^Xrre2E}Q=Dw`6<NM_X{;jZo6&-{%I!XWW
^O}Wb?h)+%j7hz&bCse$mvAvW2KNqAFGs0C!s^YrW{#@lDa+!lQ##%;<3R7=)fk(<#!rjc{3Pzu6{$fbkRMjgp0Ch@0g-RBB`}%J=1SE(^tT8OL#C0wcbECh
CJ1Da=pE!|2noNjQd+H@18W2*fp}nX0EEp#Cins1Vz?=+WY>Z3(P&4vzJ@DMhqAnS=nSMnOy|Dq@d=<a)?%<6{mG+qGl$8xjAfr&FDjZyXyDddR&}9R9s&mR
sA#_*cWE=^jH-4MpuApR)M6jiG?Cj&NrSCBAm|joNZJ#|oZ(NXs(~+`-lHfWFC>2|-$g~n0_i>d`?`!N%S)#=^S{(|!#}_M&8-}FE7Q6{2fh%l*=;HijI_vc
J`2vd{dqYb68OA@Z?dPhJYO(FBs?&No66ZTjR(p&qj6v?t~|8#Ym3xs$2@J&GxUCrrJ@<(78zriqQY8Ay?U6!1Qr6Y{@vh>rn?eFq%NwVpMqu!^Q^JE0w$Tf
QkqCP{v`7$$4?lB&IOl{)Xs(_)K1kH)lN(U+FkHSu`<DS<!*>K)_mi$b#fn?%fL$<F*Qy8@hcIGl{(EvP;LjHMi}Ho#fNh{ThM*I+&*}s5$>*WefW8eypk2Q
Nx61#PC%d%EXsrhR-c!F2r*1j?w55IVX*V$#jymH@@M#qz!OCA<mjQm<Bjaat{*&$OFIyHhT?KbC%OT9X`_CHH^(RquP_ow%$qc%2lzJdoA1pa_^}>@waPZO
!oh74ibQmL_2Sg1Iudf*Ak6d<Mo`O)VM0IscB0>r8LVClZ_3z2XSlR9i6)_Om2G_y*}X1A_HM!P2t_c`Oa&?#Xi{#@+`;>65N1)xs4v#r<ym&+F$Ad2mTH{@
(zoV^L>a+JcAQEQ*eA~CRU|n~^YVPuFA2;0w0I1C|CCfA3pfB@Sg2)v9Z0(-ql6Atkj+d-H0+)aApTvG8b6_1MNLIxk8bGO0X@7f^<f|YL<%ZBCEc?JTzT0{
HxeL3KfnEa<|1&r(t8uY32wNG7l!7>)(W#OmI@>ugE}gHE#HPS0i|*_7~M*uzaO2DMJ5|X>_@bs%li0{e;6pBcT>A*(#MwPv+)wBe`mcx26UM=o~AQe_e|}<
vf%T0SAu_4<fS>^I*6Vt=eq*uu2T*eg1wPtp_a$DW|mX*ZjT25I|55eP|A=nEaFb$pUGuBban@y#C6Z1v)5Et?=|9X+G*v>4(D@4^KRCI#mQ3i2sZ?;$pIkH
8u6YwoaNkCLzc>0Iq$s*`xIxEB?|qH@Oe{;3YvXyQsMR&Xokq?%Ix9|0>PP+9LM-0n)>V%>6O2=)h=o^^fWjN1OSq^Q?qxrrm&m|nJ#$(QWK?ID=3>L?F0bP
7btpci4iNKbpQ~{Yv-3j2vtbjd>r3(KBJWvO}p^u+YZli@5dt#IX4sEcmo8AW1}T9n<61Ir@X#qyo!xHbSohm>OBtD?!SiApAY#XSz8VqARaC~{TTrlfJAVe
c&Bll26$ViImu~KGz8{`f1<IW1Ks2hxN4KAPLpVl;G{OJdkt_PP2;MLodeXP2B%_@T92$cR07gh-j*-~j;zD!9F<>w`M||n`>&kg0fi&f#(W^R?YgYqj>bc{
QlPW3k274qz)ZlpWf#?cz}CdFbj4`2K`I5cht=y7(!Lw(&6P+90St2jII=n04Em}un&doW>3WV}g`aLloPnXJu*a*O%P3K!gB9vr#Vuv=jBBa?PscDlRE1cQ
Wgqc5JP&kGQ2@`pr|z-lt<3~=fH-g?&E*x{vwa_7&XZaWo_4HZdz1!nimKUUs9&U#NcelNL>6!bq!{14xMV6RD7m5(gd=a|Tz(3&MttMffauxg6zIVizC+M8
pEJjltrTeH2a2b>mzPQ?3$p~=t%|e}#*&!ANh#Ws>ilEBW`%V`{=Z5!G9Ee~<Edl)>k9xNVU_EQ=p~D-hfTuh#vcU&4Pgs58(6u3gaNp40@{hLg0XM84|X-{
M|Lh@L2V<OTLM*=dOZpt@Jey0l<2#-PoW_)LK{cjHKC)tJI!8Mr>Twx#K7rKrl@Rox`(ONz39^lH`C*02&bR(pzQ`t{ii_-j#<sy&D#g%!G-V)-*iG&_k?0O
U~C$Xp?w3W*q~X=No9<+qAh%nK7D$EOxQBS@XV60BJUh~%C2}^WTYWLpB1*+v}GX3Uq(QAvW^*lbIMl`*_5G7+Q=rY)ziq!|GAgSH`Y3(PE<c1hWC4TuoLnQ
J6PDcR2clq@2j(z$m_Mvp68O7Gt4+i%elh-Pd{sc)U*Oz354Q%OAL$a0iZR$s@YmY)*5VZCBhw!BLoSj)DFflm9r8=eAw?Iz<b!dUt)9i+FM#t^OIr&Jp_7F
X4Pce7e{g@D)aY*82C1l!T-XvT;sIz{-Ed(nO$J4`~AVKN-$Jp{1t_4(m*DOw6j>6ThXKr0vp@8Wy5V|r=Ja~E;!05=1zrd7vWWu5&yyOOGJUXjRalm+|2rd
Q^Dvf^VlVk(DKt=ep?~*va`s8ce|>BKuGPuE8fEr*`mmnGVQZ<YGP0v7^3s>7pc9xG;)x<>Ae(hU7TfQZHi&e2~u9AQ}!Xd#T{ep*~yhBWHC!#u0XEKYyIRU
AAuYg<g)e<@;1~*kV(=B<wM!i<cy;N;Od)DZ`OxMj40qU@Fwq3s)ie|vv=4!)zbr4bKlW5le~D$s|_3PQI`3LjJqnI7j(Z9j<|9TlA2%<kX7e0#{Aft-R=u;
EZ7@l9L~HnQcUs*r3_jv1mkP<npH!D3DHjz(q%l5hk&dVRE98BS9{6F<%W8ZpgK^2Bz$K%(7Lm?GxL{5uveFjCE}U!AC|Iu@n-4*e#$i)qSmX_K=c?G_f@Ub
o0m-F;D(X;(D~o>$-*$O%Up~pvt0|HkF2$G;b8oh&0%Y|)J}YFJzBQw;(oBw*;?g=roT5&1LhIhwufvVj8azSwGfL^#?GNHy0`9L|913H+9#Dfn=y@eN4U-I
XyzNXUHIc1`fB@V9;m~(VHwI~syMC;!Df*({$k{Ve13xAA?GF(h&}Eec$<3N&{C{sJXCW!lq?_Gl6QMOX&J*Kqa<-4eY>N8r7zL6_JBoSlDQphTNQ<)&#~wQ
!S-8QU+qC;q^p-TVzgw!e<zni>S{!=iuuT41YG%e;WVd3nrVzwNAElz;P>A}$hLrqu=9>;YKX)B9F*B7mtFo4nVFodz8_$<!|N_RtQ32dF7*q$Z>!zK;F4~L
^icxf<8Dk@_Oe$X-2o-u_R#}zErF(_Xe`|;)kpy`q(3qYx+z&Ew|uxq9M!wN^IZNFm3j%qfl}xW<R90<40^$Se4FZ^0jbcB1#}&w+<?*bVt*NNr!SY5<#=pH
Y)4+4Zpv&XgUA^#Io-BOm!-Q1B#-E`O)VRo*zo~$i<|7xpICo;<K5<T0~Db$a|;5=9@cBz^z`hSMoi&)w$kgFpyMAMn-Z94XA}>}aHrFnc+tdH!m3}a;E_q`
C^Z_TbMn-ba5}s>Y|Ii6DXWet4IbaeP{&y)*7bEckkJOB?*_`k*q7YkJQtCL4n)Xo$q1j(PE6MdwlL^B8=1cRar9$}jwf(>D(;uVM_t}8$t0T?5uTPZ!D-6p
f20kRVj}I|22n|<uB;%BLp<>zwkry`^LC1WrFh2Hp=G`qMTh%x{cdkTQw}`V7U~_SBs>-CLR&hZ5GSjGi16mdKHUdxmFCR-;*xMix<D$2L`$z7GYU%59em)!
>lsy3M5HS>**QeChVV|SgyUyNiP|h|E4;BJ#jG85*pd^ObPQnT%WOb$u%u5H3Yun>;|s;Zc!z(NYxa4S)^_8Iym~b+-y(&`V*@wLIs(OidO3xs>OYovH)Z|N
)wf{Wg4dmrWs!DU7!t}ws=oZk=?|B_3cb$I=(s+C6Y_|>cnSWobI3$tpZ+r*3rorg(+SkfQVm{St1Bj)R-``Da%>f|d_2+Ldx;$IjI+@&i{0z4mZ=tZ%AhE&
8=U$!R4`xPDI;?)I^5@5e14TD9d#-r4Bh6zyUacU|B1gAy(27!Dg7@6%_Z%XQ)?KOX(O3sDLHwtpzYYAeeE6nSYs7**)mL9e2gSFxT`xwgl+jap7`&7NtiG1
^>2H(8U3I3s(&h+P-FY4F!f&(kp_1!`UquqBg=ZNPHoHt+oo|F3BLMVw(2)`wtqxddoJFx20dG7)9E;HkrCGu<7Et{uYcu1>K8*p;;OEZA-%2jYX+E2IlBaE
M3lHgaz$rnaOH@8M7^BEu%TO-L^(CXIkGaA+a8nq>{1Rzc$3+;paR$jUv<Y}@Is{LRNlRZ$l^)eq=BAf4A0@!2Fo`^0t(y!{+{F$r7)-<oXeq3y6<%|8%RX2
_<Kq&VukOf9Wjruy~cN1uwB+wrI!Mv(ERDNhRJ)qldScdb*;mUaN4Xrf%r1hk4<9BwXhikeeQN|%Qr?P#T#1A?)p_<JH$#Kwzx&G>v}o;f78ga6*!pQR4nPK
L8J)Hb;cJ<cHzFWGF+}ElJSP(PP}+JS>S0T`-V^xIaixg>(1jon^pW1Iy;ai;r&7aKQ2WhClO+MNJ#$s^a$U;AdB=()jz^_JdIdmkHz9*o5je#+csZ)T{G5`
q$BQfV6#xh;v=y3mGvXt`0jHR9xHQV`3asRLu~#j=@9bN`YLwJx%keMlQ-`*CIAQ#O_H1nPd6$aLVP|6!DlSX$ss(4y~1^<C`rpFOESxim2>07_Ctn?hiSsk
2SejaT6JBVAm7;J+|$4L$}oFdN_q$%|7h~^%x?tK!YX1BeX@69S3R`C^ou;mm$DY1E4AF6tLycML4G)&UIreSppmmaC(MpDdQq@zb*sC26Mli@Qx9eEJ#x6)
`!j4KfhlI7D|Yf&Oa_ws1vB}0eoQ&rt7eribFGP~y)xa;d+F7D>M7P=!yxZxafN*jOIP2y_;e^S(4+}wimQdi)g``l&&~8BiLVv4`-P2YY5F~Vfj}hTmXNJP
Q;k$jV!{z&7_<M5U)HL*n*TgcBh~V&#vgTx>T030n;7gc=m~nA|1FNcg64YCsu0B*srIEn@r^kb-PpWq_L4Sd#|s1Qk-{Gb7Uay<e=_TH4FY1wk+bj=e^6-}
yKY>`c`k~bzH+HIPV?3k(`ul?BWn>2<M{YZm*+tR?$C~GBRw$-y`FBYpbAOUGtl3BDoC+Vs6yV-4tefony+JcaBo-?AF{jmsmfKN72?URBAMur`}Z1y&SFqw
OxdjJxGL<p1o8NL2~o+9j*5X&bvV}Wj^}LijQDU}b#@n+I(3#a>Oh3SJ(ufCBebh|kIjLv%Pior@)fmrdWAd->O~r)c=-#@gQRF-X4I^hxALbl!B5ONJ+bEL
zG3Gx*!DQIJYMq1#cft&hSsU4NfBDbI2`duQ@?GQ<Diy|z-m+X<#}UjgOtJXX(aYT{VS+;z68S*C4xRHsB_K%@K+jZ1`U;%F0Kvs<>K;deyJImuw;=YwtU#3
8G#;WG6LxW2MrDFzd2U*(Vke!#av%_8}f^f=@ys{^XBt~YXg-}Bhcmg7qXiIgP)F2KZP5~L0UI6TorHpvbBMY5?K;v9Je$kJ3I`MD<wT)kE^7Qaw3&o6-%p*
qh8es<5Kje019|+1M54e0`vc##`U7!?purjI#bsec?5BOsa2nM7GZKH?Cr{_xZ>=!aL{ti^ABtLOD6g=*l*%h4k-{CUUrO*gzeXxWkOp`<-u5;F>SJi>-(o&
BG$wGa_p(lx=6*auaT|^@SRi0;9?DU?3zwwZn2bM7aLwtkmc9P2K{_0zNj*ch;3w7AhA<0iG1`T05zb-!vLc4j1jM_@(XiGgh4$v4ak9pw^4WEP^OL$P7D^#
T0CcKP9X@SaPg{vv$}m*o+X!zIuc+L8Ue&gylNk5%MO8kVt#!!OtJso!>+*L3%XKM4)UK}DSQpa<)AB@rF?W{$|&wxBpQelp2Ffq9%LT7Mu2c%0oa`9L-n0|
Bm+*23<h&Baf9s>|7x7F1Fr!h?zvv(1LDp~_%M^we@&B|Wl5_?%x)Pylin>@SLCs5g9kMZK*je(a#nEUO>o?Yr;TK(pDt<^xx2C_Ycp6aemKIQp{8#pY4)!(
HeYcN<~Gfob{YP3bwTTKqE}{`{*tAzuc!hjv7qp96@$X&n$y)j(Z?EkeB2QFn!6wN8qu54Yyd2hWtQe5f0DUwe_X@j2rbD`V~*pej!n*^x_FPOW5C@C%5aI^
yv+Ea_H2NHfmB$*K2;qLFv`n{4o+Bb?xvE@jJ)b9mi9P@7nVk1bNJgzi}5cec2KA_ZtC3$v|4*#*b_MkBYo*>(R#T{9aDdFA=Ku<6W?6M%5CiewrALLEhP4G
zghh%TK&E#A09suL@hv!(^wFJ;GgGXMl&ZS@<X${3*FWUTiU7%eipnuz*x6taCZC3#a|Q|AF3;3H<oICSwruZEFGt<7=2!fP<~;%<-R0XKk!<J*wu(ELd7~e
Sh9`0roL7K+$?dXgN3q9&AKQc136wJv##9ct6g)LL$i!^nWYi(Vt^`Kv1Nl^QlqJX!)(=sAf8p+JEQy$>uBD2$R)*@3N5(iM*ob9t`CHm@sBS&0`Com7+&U1
2CRKFh2T)h73sjiUnVS5nwdE4311_({|m3Vm1wo`U(26YNSUVn$v$dqL1N#@r#R{kZc3?0E@%c_VpOGGvbpN<{gBncHd#+j<*{$g*H}U%pW{Ra@&^|MdFUST
+Pp#Buw7Y<9IH0f`E!-?wkS$KsMe9~CJma&lPLl}1mx~qa%QLpO&JbK&<l?hWg>p)K$%zUo$F_&>o)vAtWp|i4bl@g4S#0#<r+(946<3EL6wakTc>0pfCSA$
){V$$NW|J*zC&}Wn8dl}#m`D>rWnu!3#C^{cFAA>vV$af6V|VW_;)<7V0?E!akIu$fW*m*y$EA$zq`GYPSj348S5_$Lh0~mMyl4I*lXaZV;9`=gHdQ>N48b4
kh9_;ARz_hhG9=dx*{AUt7>%$?{u>B#|Xw6w8G;XjIjT1|5pc4Y8e9j82``;L^Iy#s$s&X^ZRYV->GVeTHEMYAmn%TXK0LtUFPoMr5#0u?vkLmTF?-<SR{31
_-@e4|4EE7jD@hIN1sqCv<T3ASDGLm%4*%RApgPap+uPDimn`e9M98-QpE#25o1b^s(XQmby~w4se>6do5Wz3>d;<cr01068#sPMd&0xLpxyT5%Yx0PS4VXJ
TC?hgD86))4xj`rqRM+WZ2xP07qa%#lf~Ie!`kT`62}Nqpqi@BkJHth)?!seWDbU8%<nf)Ml~~>l+=67xk<l5vA^5I4H2Li?3*%c<Rw()`$2`-X4P}SY8ASD
xIl5b69#mDL30NzFh-{}n6PX3Ts411m=ZRfz$rJ!Ch`Uo)a-~VGZo!Z*_WAvOpxZW$zn;c3@SCHt*jywRdFLQ_RK1=sDg)yC`X>HYd>QhGSq2%tH6=e(z6)Y
mYpJ6-Bd0|yI6!g)j`5az+XvA34crd3c;-D7&yRbxf#j$KMS7y#K^t<WYMA==GGRm4}ZXUTq>!@7s2WempE@qJOx55LLp<&8TbHK=%H-995NWNR$&m=+d@0d
B?d9)B{Hj-FhCHP$QlO`#|z8df~rPnb#Fi5$-B>U1G7^M4e1ny|I~sKxySSo=$@6z=P<Shk5Hqn#A0Hu1G?}Wi5&LrWc$&A_VG{_-svYNG(W2X$J+zWRVHta
gTP)+66EH<xp&OLNe@X+6-9Lh({n?*i^T#~)O7RlYa^nF@AA}kDPRC|JpHZtsDUc%WIZk+j-FdTVv4&8yP3a=^QtetV?b$`HNoD3)ZyQy0x{I`5M=QA_=2B-
FhCkv(idEH@+5LW0i@X`LY{id5Ej-K;sOtTIR4RB87A#WBI*m-@4T54Y|Cs;<6jx4`~YWQ0I~-)NOU{{Nn2Q@`7AJlc~?ZwW5f_(2@c-6u*!6eGT;LnJGaSO
fc^DaI==Nrl4b0XGpYYl5b>ktzUWp-xr;5|bs}Evgt;a`cYeVr#70$RbO*Ua{OP@u$A%kNz1uDy^)#Z#PM-6|)dV*{p`<&CE&GU=6V*mw@wbbXXzgDSD9%QC
!NyijdDAIeX2N^)gS{G78aPOQlUSu#^jbyDyEfa=YBo7ei%YK6t^{N)>7qZLmz?{MhkC@$KF_7+tw>6S%)qzVu>aVf2Rma~+JUc_owz7M+^$7+os1}H62JY=
kcoZ$nD(>1j?wOtV?;+m$GKFZQHy(-Kdo*09{Nhg6(lF$Yc;=qpYpHZL2Os>idq<(8)LO9mL#TQJ))~TP_DvgtOj@5D@;=+nG@F>rVNHVyG`%m=)4TR>!<nf
M3e|70$GZA%;*qViwy@)>c_J@I>%bAV}7PSL;1iHXg(bSDx0wqyy5Xc)9f#jyO9%LklWqgN;TOZ5L16r0AFw!p6=t0oSfQ16%I4q^8~$E?BPl@>*P?4Niw+A
Tg1Z+WM=7;12}lBC%BSbwae&*6G?WgN@!GES}%2f7fOY7Q+1(c40%M;_O~lVR#6RV!~7eS$Q3=AqzSP}=Y--9)V83cSG}l8JG%=5?bs>&McD97uGq_$J5_z)
uVB3q0LtFU=w3^F1AurF*rz@0HH7o^hN)gU3Fq!!{?NDo1e^7StY3Yxb&6_K>)`~RmeSa3lV1WVhr;-|+~<NB`IIxVGjC-&xlDMyH6|n^p{@}smna73oBbT{
Em>-=AqKK}BAT6|%XTe3stmR|Qu%F9G=*YPIu3YTXK1?L7U?mlsFN4n4)S?o^)j)vZ2}JOd0p96n#s|NfDC-mZlE?sH|bFBkIx6HO;rGz1^EIaQ0K>$+d#5e
fh(_*+#r%4<3c`qu@o!nLFg1SajoysZ9@37)fls^WK@T*0&S;`D_s@6Ta24ZMm4~WKwj0Rs$8|9m=EQ~0ngwOCM#j)9ik}RP{k<KalhOOWCz@m7<-JrVC$^5
yh>cKGpvlRr!r!>c;~ve4X9semk@;tPOr|U+DXy9G7R-*oPdgprTg~nHSYuLCIr#Vyy{{moEm02a(Tos>?D1=J)Yu&opm0*GeoXC=9L;G29uZu`TF1zCHN|8
Ypj9$D=n<riJ>tW=YaW~>M9H=00JyAO*6|vX_mB{kFQ^?8<DzllDk>4ZBK$>Yej=X37h{zXNnrTm+%IPvgx*x`q}DXcNA28DvZ6K*2+DGUx`_-j(Qg4Eu3l!
;#^xJ=Gpqu6GrHl2(9u^T(*ovQ^D}4xUZh5EV*$^;EbeLlSylZHsK-s8?(E4XvmWaGHX#Tf~Utg8s7#_l<ikXnZ!!5t#>Ny`o?(D^L7=0ugGYokVC_FkU_pi
(jj5(i00|?RQ3efn}v_p<IfzCr)JSFs>zA-UcO_ESmg}pb=wthV>l+ZhFC1X-c>vGwbHT<|82-)EDekC!|NpftT$D+djnz!g$ay~UdVoS+4W3q+-$bKl)VX9
c`4y-h769Q`)l#ZVev}-&XJ8qH;8Y1LL^iponM^OK8t9?0gSc1B)~i2Ld-$lW?PJCX*polBGG*PeTn(WZE|8pc?13yl94a!e}{iw`+@G=pQ>+b1@hvoLbz)8
E2-ThfnYr;XQPM{6mD~H4r7`y5Tt~))9kenf_Iyli)>64{hr}c#Ypx)!aJikkfgDsQrQ7!Etm*IB)mt{LA(l7T@R}VlnHIRZqkuI2ua5J?JZ$V`LOagNdiK8
^KQ_fhCEKH!E$N_If^tDH{c~CUe;>LA*PUkgGhWL>fN}sk3FVk3DHGylxe<Gk!!<r+qm=B?)2CKEQ`(XA_MFx>t=dpHY^f>=Z4}GYJgsC;~id$FB7lD1k#Rm
9<s~5=-PEKeDoqRMfqjxO>aYMSp%Wk@%vuAXYFaMz-Q})NHem-x!CfEv)HF`=RK&a<VqZKM2saJT{LbHjws$;e%PKRPz0PPLw^oX*i{6gs<X7+tG!0iaumAz
evNQE<|r{aa(p+{7olaQ;YOwD3L=1AM*;>|77&&sHDvcI{6l)*ei7$#)q|yoBk69fYX=2y$<ff$F?L)qF<e@Y1dJ)5ZQAC5PLgogV6f~N%F_@|>hl-X$$0Lu
e&}RcIdU4Qq%u^&a&Vq?B+e92s9e`q)6DPR6!&ppc_o{gFLVzub5l5heYt}C0iGDWV?h?9d^wzUTgw|X1aN3vt}vrVilDdC3g?G`6A>%^QCw1x?2p_mV=pmC
M<=KQhn;D-e&t>?*_%Xfz8Ma#aG%0~;A&~;Ywn4uS-Kz+c=8{N^Z>xvWks=$C|0&#6Ox@j4H@hTRL(+#cn@h-`&qPf`9-z++bkzMFiZCdpbA{@*j?$<MGVHA
;8+FZFX01v+4CPpi83e2X^1U+veYuhQ#4xk9_ib6^84ThLWA6EMf^$4$k$XI52QAQw`D!mn57Yg)q(IJBzC|92{u;apX=mYfFaVpPN|aH9PudtlH&v{Dp-H2
lp@d05Z>-BCLgML%e?I)isSJ_!YuGMaD1}`Xr%!fI<CF&D!9r{aVrnG)?uIMSaPGXV#)(7AYb3JRJyJx{Ro_akiCDc)9~ZQ>6b*TJ|RZ2oj6$Sa}|nJc%*LE
-SQ*a)vIWi0Cdumr|0(RZ`?`E?|Q$lfeGU_&Q<x};_wMj@Eq2O`43<tP5&L!{9;mYC3fLv<T&}U|1(rbnuaein2Aw(8+pZe8jRzT_q@O`!`RG2UEhS&2}a1|
eg%kB0M}->y*e%6dISlC*LJU$hl6mhQY)gqPw2IIIhF^nS7VD4lV?o?CyP=OF8&O85sS5a;^9Ie2C`Xfpy|!HIh@}{N~+hTAq}KPC6XZ2EY46b@H`6LQIUA~
5YiW1zwYaSZO(gn2jNL@H9KI6RsCJ;vY{au^^-NHj8RlU5CLkelY6}E5K)NHbAaq%M#*9DS<z7qswC>b`N9Jga4L^&v7AjMt!Q!~Cqw_yXJx}DC%}$Whnj~-
{{w9N$4rkyyl{}lEghy3#?fu-i6mx&0Oqu~muk6cr`Gk*_9kOS5qW8w)qSJQZ$1;FWl`dIogvwXNv_Sp(~?2DY(U#UC6FW(@jf_q3FqP6V+Y~GAz3EI5Nsub
UmqCZ9&md!yxU+cN;~uckHN$%_YNEIDB(U|PyZf}vr_M|CF!u=W)+Bqw-g_F5nbw%UkCyRO5Y{zNFJDx1|qz0LoSdbtqftr<fYk4J)~>5iY8!7im`&W9-_UC
eN$p}*z|2aQ6usk9a-Q&5bC(32glXp_bVSF8K`>-;I`_E45OCgb=w=UA_>^(^$I4KL~gjhyIt<tW-RvCAJ}3V9CF;f7VY|HoWmyq>Vr<9I3)tYqfWzj5lMaA
W)YRKj&yydeL5~dKZW>pqNcZSE0$B7z?7CON0y}2ZbQn2O@Ui%|GeqvJEV*{a2kS~WEd84G?oIsbp2QMQ*=L=949e*^iWVme;+8x;q|;bd}ylkm<<khDLaAT
ZaX$$srHSz6EQ0!@3x2rJs&o&4l>;y1&rVpBJx-yRdzO5y#F|(&kyg@3F<;wvW&VcN|Z42fT+CO2pcDh>`{l*EzeVwSL01@xeQwMQv0>FCC1$f3qF>H5D`Kj
WD>;yU&u!|`GNqe&Fl)_BQ)&+bhXKxqB2e^1_9+Yyna-u0{0Pl<<ONv%}r}V1usA{KPe1QFra*DDu*niFZvtMaeQk;vful5zZR?o6cmIyTe|NSZWf?1NG@ZF
oG`R!Gm`q_HJ|k^|94YwI-<$JBjs{ANWQY|Ad*{9nV8mkUIwE*o2Xx`=V}h$UEJFq&spN}ei85I@sYCv0-~x5MiSJ$(?v^i<1hnUhjvG$Nql1#)n^FNhPEmS
<kF<yrw-Vy*w&6T&CRx7xbk#iSrmtUKDud@2JAx(7j|X~&(Sm19w(mH{daJNn!r^@m-931xv&m7X~tITgVdM9dsQB0JzW`#K;-QsjA-$xn=p(ecD(0H%*@3~
8N*r|T!0#0zC{3ASx{B?0%`3_J1(`T03>*IOyrZ?WJ8>SI?aIot^G{jFDHUCy|oGcUzN`5pl3Ke*AyOI8!}pQKHx%+i{<E*Eiy1y+va_8q%eMgi?~=2^N*F?
hDTXp8JDs@jbZ4(m{jVynLk&qm&J%`EZ9PS%EIW^y_|oENlxSY_ptC!6K~xeLPv@bsf;K!XzA9m$qa_JuK=?UHWiZ=5?108JAU!<E)aThk~4X!1?c$)N9D}t
b9FGU(mQx8b3#A01+`1%leU{Q7z#tEOea7?`H7XY&`7oh7_y9LV|}@nly08Q^aKN9mN^>qgK72$K0Xw&C)2J(8S3x*_<HYo&2mH44T0Kuy?#U!f0L>sE9S;9
{dxLfe))H^tLP{t89;^Z;HpAS_J{2L$J)<@*DLL;91-u_JWXZaKu!gY&&XVZXFR<&VanK9N~-*P`9Uzxkp$vpL8t0&3R54W;#Y><m30CxdE8GRo3SRdWvsaj
%!Ji7ot?`UefJ|sb@m$0fKsfez?ThNDnj7MpjtkWjA<sp{|cE5Pko?CTwby+1wK-eCw_-)^cvyMKHGH#UOdbjDDMAa6Ey5~{)GG&hxwMVoWNRHyFD1P!ef~)
I;w<3ZJ%OBgF=baGB>nr%T{ittM5O&r;_vkR|;q?H}bx*Adp01UV_nA67;_>oZ?enu7(O_fn(=CJwQeY4D~V7XI&{2Get$`(jfrRlH`o|46<2pyHo8ES3DEu
sqLR$?YnBGfOjK+ACRa<0Q#(y^L2jRpkohkI_Sd=vtSXnK@JE^dd<uf`p(OgYMXtogin~+s0f07u2XVYHP@iO_ok!EZ&Gnxn^x=9%fM1t3HRt3ywtTXV0p#s
YH#i?UfW7DG_wx%G)=j|Pc{hC&WBDQYuFU-v-=^@E)eT7LTOA*$UmqsNea+<Z2x4UqvYlWY|ERlsArTrq|nPc$?gMTvIJZ!*Crh}N6y0IyK)NNhh&Y=otR@W
lgCzHGsx^lP92Oo#_=XsI%<fq6^Qx5=k2bolTnjqg;LjAqD>x`I0X*0h-3o_lb^nmsABi7(E5Eu;*V^f)jkv@;UeiXu13Ok!qSq(1q)>Df-|BtY25nm0;@n%
QJ~x(Z`KJQ=1XP!*!xdzN{gFi+4_v7SO7Nb@+~@)YDU2lV(zx-SLKY};~zR|**)(b0JrcW+4o4hRB3-Sl_~Gw&)^s*RX3IgFkd!&7k&$m`Dg;Ph)5=pULBDe
x<$;plr9=F_-6q_46cZQ7-;V9ADKbd5ya#{rj22&*;~JGdC?(!dhI7vv28)S-hEK$<xVM~V1pU+%ETG2G=V!>-UwUm#iosM75hIknMgMtgma}i>!BfFnvI9t
#2P6ou(S^!|E$WRO~G4Vg22*<Tvzvkb(oN@_DKoAq+3fb?{?Zck)*;A$B5PjW(P(UXPwZBe(g5{z7nCGHcvL`Qg3bOWcOFzmLkZl)>bx;NvQ1YW*Ye$Yl#I!
QCNI$F;ON@J*Yjp(Oh#z$Nv~TapEnN8Wgr0;!sm4Bw}wja!T;FRM8BI0JFb8WL2><8H-uH_O9(EqAJyirym>fc*~-sLFrd8XWtyP+}ep5nD1_%mwM`hF*l#5
d5m0H_SZJ0re^b=Pa8ReRVoSy3MVshXaT^+w9F!z`{k==v-#N&$nRZY*)bbt-*{SrI@Y6VtJN0B%Llhp2I|Q%&|&RnUGUIu6*9+@@Cyb_1gD{aDYF?hyO4v=
#W#uJYe6$!<$RReXfZEz6aB!z)Fj{4mp&7*5azNNSwhQJ$c2bC+-U%tCpsmEaHfDHB!BjaVv$!YX0i`#|6<~M^Hk;QxGxF;-Xl?s|3I20!|}Tr2toyGmlr+I
7!dmR&;N~83<row`pmOHIDwH0C<0Xl=_^MnK=dE$h?i^T>Z&+1Zc1(}m14RRtkeGui)(1&wb?*Gjl#jQ7XCX3sFvtxF)>$bF_iEtT%BpRXc~<#rT@_^eL8vl
G9hS0s9P@rFFJv}4=1dOgDdj>=jj%94JDcpV$`#t?VYqsVMg?y%0XWTYeR|8#VqophU&BQU&<~UqUDgSd+7uM*5sLE+C80^+}Rq3<*%{hzD9c4bG<}3(FFWA
!ks@!bFXv_?RCh^V?~MJ>r+yBo+-aXYJ@(FB09RF*rrxO`iB*%E~LZdjJ5NI!Cc+J4cFDUKYKTM4iPy?nEZ$geF*$xP{5Ns@^6o98A_^h9`T$1+tLRYdkc`g
SwQdn2YC%*4$<08m6$nT;G6v$mA4_6;S}E7qu!&~jdk0}I|+JP5eo`$1kaZMUXcPO8m>TJ6HM**XInJCCNCn8BC4Eup#?4i765FB!CsAT=d)_Cqb0J(y$JvG
S^YH2erM%QZ2!tk!6=*`T?x1gPR`WDZMOEt&bC(bnvwQ!`gHFsDQei9;&Rz-Hn7pQ7DDzOnTw({ALv1Vf@AtI$sXHV48O$y;qkcUHnmVZzuW$98nZBOGEofR
K~DL@>fm1uL%2Lg_D^z#5Ei?#_frySj1qcq^hd<}zxH=5o-&t^DY<f|UB5%=_d<j+L%2U_jJ)Wb6GlqbS({D^hg#=?HXwrE@8&vbiA8=Ss4VLLcBR4zD#CUP
gw}XsMP?rX<lru~iON>x8pP5cJ+v;MYyHBimXWPVQ*{?acvm=%i7D#)uta=LJ<3X{Xv+P&G;b-0IvP9#3;l(r;+>1aLE?{0l&2P(@{rvG6O}8b%1K6WM4`n^
ZFDWq2ZhfT^Z?%|)<(&aK_v&_^X#nULpvKa6vxdRdXAC1R`sB(W+*^51)cq9mlP|#K7B}xJD0R^clJl{8$VkcQ)x)Q5*82qF{FDX;)i%rS&FXAQ0S*(y0Bqs
N^BMOv8V~YMQ~jwW|5kRg8P$(sfM7mOGd4+c|g=5$#ftsuZ#+&t<A3vh?tTr={AR%dtt6jz*8J+T2$=*g&b%M7+jihX=TAP2jr#V^XVi-VEW)n`c;skCS5%V
Oq_uy{UgG(3CK4opu&e-{^@ZMk{u6-?W{Ey=DdUW8#Hw`s_XmQ2L`u!L10*e6I(CXnWm2PzYhQ!Bts_L7&;lQn`JBEh)6w(Yh88776#~>^BS%frGfr*^f{Lp
IhCck&hieVRY$f14^yTp=M0$#C~-^I1@DYY`sxDQ8=X18sM|+fkDHXo(cxF(nlr&p6P2f<2xv{gw;P-@D8iHvj=l;4W=TSLpU;`nl6*<fATZ^Jr>o0!g%wZ@
Il9`GRmmL59Q!HhP17zupH&v76ZGC@5m%>Ng6si3w|q{@aVF#R+eB_Eh=f|W=HGA!VVZoBxc%$B*WJ^`GbKx$1~$0|SIgc)5Hu(K+SYahN*3_vX5A3*@Oa&v
5GFPr4g#YlFwQf_C{6nk!!3#K0H5++WNu)cIP;W=hJMA1-=ck4Nk|_e9&;I^WS$0}M)%NNdpoST)l)|zY>*gsN;8FFG1<9>f*ymEYij<w1F^^Q`J5k7f=??W
^?EsxJ9Djg&k%L1znn(g)N;bqV9!;Vgfu?*7K%hF7U*j@GaOK(*-GhaSjET0wcDWaYPvjEiE7d1Ej2#|t5+7rY*H{}^*{oK4BWlu<iIjb!S2bq%_HtRX%w3e
)OAEG6R;OGKQro^=5SlK$dA$Im|0uwnLoWws3zh&!;RQkIm}T^0?Pr{oQ`cX7VM>I|Eg?-BG<t2Tp}!u<-(HG_4+Rqt{ihBjlq9&{eZv_UChx3zk8+^VQ;@6
VPU7Jxd};J+M;wwpdTx_gW<<55Y_BbR}H_xiG|oEDJ>Ow7ecKF4UcHJil`%UW>Kbk8;tY-@M+xf(vXS*paC?=J0EW!s7BBKnZesGcH$}OtP4@Yea{&GW{_~B
$IZ$bSPdlRh}Q<Ej(iz!hHchJ+rg*EL2pKavO`>OD(^`D(At~$uwQgC@y>S>@tn9LX~U`Iv_Pwt+(qs6HJl}`IVuXl$jgfo=jr?&15>}<q~CbVHusa|LbiKF
`be(jTd`>NQw?AseL41>c9WDFIOF_GZETq6DLJb;f;ddn?gE}WV^ZtCXji}jRAzt_mt!A?_?@26FQ}W;{8ibVLYoIa+xN8zA4Ibce-+3otKV1&^*npivG=*>
cA}^nt|ilPxkdyLm`%#0om@nd*IET(RI*2OS<`jO1?RHDZr+6=D$~EL0LY5^_bm<9j@NP0Jc=h0AncI4MblkCP{AUnF1KqOaQXBG890A+7Wv0X*d|S3sGZkE
M2R1sF?RRNiujRFD<U1#Op~+}_qolaxS_U@hI$2P<pJ;9zC*s>T0u*o2nmS!7BGSSxvm$gn<Jci+H^}R{Z$`G*pws7?p`CCTd)>z@g_=QpBY{TDTlZF&h~4A
(K-ignU|+YU<@aL;|H8WhiRFBggoS4eldCs<<>+UgP>7SS3C@pA#t~j!f+EM|9KOAyO!69AduQQ&vAem;^X=B3)Un?<YT1(i8vvFt|V&6vm_(>HX{`7imRp4
`QB;J%!Q)daJ74py8yt$a{qm8@O?QCM&fH9@_nbwnoKqoIuQ88_8_;q+H>$^5q1qBiD0*a^iSO*S~!e$s|R9gAt8zwGz2e)MItLr+(lE;JTf0eIN(Z)00+$Q
O&v!D<#tb2E3eRf{|6V|u=T)=b>*@c^3sapPpzL;$K-yqKyl;ts{i1?ndDD1>X}SfE&&}d)Z@Yi4^?9b7}4qPzUvShxLM&EHD?miRRpDka#AU2`S||;B`zDW
;)+z_u1tCllZtfI$^G;(gP|Zlk;T(QsGf~a1~TiS(9~S6zsrI94-aF;&T@#tmO~q$>P22dHI!r#TL0zN*3d|1>K2z<h%|rU@uy2bl;jT)HN}Xen^0OC!Ibpt
lYoW<RfX5%Zm|k1U4T(Q2X_Jc9vV*uD=d@><BQu{Z*aCG<<v5IQFRPN@g<<S$Yf0KjFPaE`xr5`qEc9?37cLiocw~XI})9I1NkwnZQ=cP3s2*3cW<DqJX{$z
e>X%<d3xU|Q2;27nR|cWPf@+^kJd#!Uk7P%9u}>nrqz)k=m;h8YfbKhDwliakGj@3>|(P(L<4!pCi%zP?D;NKz!(}XP!n)$`=#+AJ$s%1ylX$^u9EfsT*xIm
kW5uFxU?9Hy8<p6N}^7wv6g8&NixMUHV_P?G~NQX<>I|qrTI%o%Xr`Q>~bSnr&4jXV|I4UV*o05qdpo2=!U4o?;%_@>*g!f!_AwuM*D9`yTU1kRifcGa+lPG
@*8qIdf@oluWrFXP;W1?A{;_@S!St{V)RDig?c=V$=@QgS#%a9ZD1Hxn!5cr`9m~380a}jl<m<kzGPb0#PYTSaeJ`WpB{*4x@OcUO4Z^He=o#gveujaZa?xw
7`gm)AwnFs$+!9<z2w?!I`2drnx?pm@$oB^sz1g*yi$OCamOoU^#HF^9IZob0+{p3*lz5aUlg#e^focE&Ty8oBEl2%^09*eogvKg`rqHDz&IO=%Hu5x1?$4n
W8zKb{eaRp0cL>F=WMR<iYa7q8ug<kgqt2Zdlr%|6Vmc$m+n<20uGbua|oe=B_RZK6vw58%BtvGcZ#9XubUQVg_N(yPRqs86X(O$c5t;3J(wh#&s2&_yw={=
Ra_PSWUoGtC)x&VYR;1}7l<Z(*BjVUNJ<JE%2X_r#pMsbg$4whFXCrF5n<BJVckf}9c*Y>7yFSp+eVSb$AD#I_Z=W0P|dxz=xCvLqh=NCt!#pWiH*~<27p~?
r>~ka*Il!aqksSHglDfxLtHPw6jp8w$d5(>p^{z1N<Cecv=V|9yYJ84_nw7PNub@(kWULoykU7=K9Pp3D^C7&_<D0x^TD5RcHaCggyQtm@#yXAdtRGEj!Xdy
4J@b0&idMyi#9FDIKn?bnRF3t^LXPtf}tAF-Oa4AL17A{eH?h1CstA``rt+I)2+IO*{eximi7(w&I!Eg4Zn-5c$FC5p)0p^3$D#*_!n%6YeEos#)_=;i?!5`
#jHXGyM|$%t^7^ekVK6&>~;p^PGG>D5WpN|2<Pt9O}rlrx?d{5Fx*UkA-+m^WUe)GLLl6X&i~3lFr;f^enlLgkP#FvceL8<rMY?$J4IHo8bcPPw&93F`5(XQ
JdgQva0Vw!rxABL!*|Wb@S*R7kyC(-A!9k-S8@YWY`yj<><>N}ax`?6=2u@xfeTLJH_Q<M48#db<ieE6X2eo3)|zqPV%#jYkWCoJ?H6Hn>!-Xz{zGn>zJdtp
;Z-9}E_VVwW^SEE0J3>`#R%!E;x2(y3t3%{(Nri0TK>)YR4sq%{&wz;mOk}cWXo-r*?C-m<f&%+d4S%%L3N6-whp6kr(I{yo9=JP)}q16q&td@1kkF#AW3oH
#z%Qn-_Mpc3vqz^5|WP8r9gxfrY&;Y8>ErB?%$93e%*b2S`t;A;YSJqaOPs>x2``TGmI~%Xa9rgupj=RJgXZ<=z&w!p43av@CdL@tMOgfMs?Gd6ma1VBWyJl
eJF#){P-B~)kr&N$P@NhleK^bq#SO_NN+c3`GmPwO{PkdnS>pt03K+9!M+=r;&&VF7D9|mRT@`qwyQq;l(}^L!;n@6!$2-9Xk{sZdE{v71!to?(r2*rC%x$F
S+h=fz{8d~sH@Bo#h9NLbJ+D#>(&?RQMjMk4}>K&4@6Ni-Vy}}WnS!*5H(r^;6l+ke(4i1McMlS0v(EI`zsxTlItRYt7b=x#n_!@^Wd6dLFy-#BcAyTXJ!3?
+CboA1V7O{&x)02J(@?FBGoeWv9en>P1`(8^96>gl?*<H7a5AI`E#T<#y1IQZ!_v0BeivQvXfQI^yu$7zruGXtPd;6*-SDZlW(u>D~K*ukl?m|=nFAEfF{=Z
OtHH>rJE>}@3wavFC*P~Jc4Tqb9nFSpNhMm!>Rlo_iEcUYzOUb-v;pRp^q9=n2*^)mz+AE7(r%3a>hDBM&$Sx7*&r2#4FiO9pou)B?|*-z*r7uZxb*`YI#!i
!U69d17rJJ$g#}0Y`<3RH+Vnm4~OEsWM`-Fb{g3eG=Ff8i5IpY=Fr9W^su?g6Uth}0KO9k5f;vxu@74pgXec|0n`skfkW*PaJiVJR=?y^Nr|^_(x++Owg>m9
I&_GEIm;hj5YzNxf0YpJ9+vXMZ!*2YI0U5Ss>xz%nK~*1=(-q(cisN5(NA=B*)`L9gptY39}ac7rOiGfU=lJ{i8JSZZx4SZi=8+5K0Hf%wYHcH408y<Tfr!`
5)zk=NjWMqcxobL&j1@MJG0*6<kaCJBeWfHBa-A%miF9;gxr;$b;nk!zhQ8jP7@mRsGf9mK|tLewi7^)n8$7GIz#}b)chJ%VH-T!n?eG;<MR70`2;O^v+Um{
Vc_|N8zzUvgGy2l#CY7rUOo4+y39Ce7x9jv-l7G3Q{u}9mu!b24y4x=v;jo97-JSwkgR&gI4AfdKU`PCBQ_Z=rOMe{(Uredq?&MH9PYb0yq&s`U7cEajPmmw
tjCu^vn_&-{PC=tTSa8mon=8zjRSnkJ3Rmo*Cbsz!5}=3zlR3NKQihfR%G+JA`QwuEoq!X17iHP7JGPE_U<d(-GO3^Y(oS2VDcpuXln;AyDQ;K3=|v8-&?wv
mU$4jdTh7MzIINb{)dS&rtnGJvxRbRklh08+Edq2i4{a@t#S=!0w5{ijgsaW4W_fKFg_d#>6>hMD*yb?xu3dqPIiRyEVmXx1VZmPSs@x72+E~YZgL_RtR0#3
nuZGCnbR^nvojzVaR%KR`2V^kxnq{^*l@$XiJ9t&-^E<A81EV|-I^rY62#rCwI%S)$=b}z;88B7AR>RMV8C%>hlU+++XBaiAeU~vdtIm9)0<fyKzs^&NzgG4
+>eoS2KF?+*L)7$k)dWmfL@^QI7c{U7ZxrFcaB2@aK7h0KJz*sZHPZWgtDq@$E==ToVW>*TcOYvTH_I_?k*?6ok=<6l!2|t_+Ytbdl^cS!fZpi#RDVcSZAlW
nAHdGGjHPPU+{$shyUdDu`BLaa4kV?xVr0}$oprpEBtf`FGxl$HxXWlo_KfXpjuE21?qTb{`RmzenZDG9smw)9pK`o_3VoMA~~E@8{HK=J11%>&ASEemZR!z
k)HGe^ubs_TG=Q@Qfm9p-NkTp7%)ZV#UiaG-5_d0@%ZA+nh5_4_Q~f%CKVo!BbLyBcdEr&qyTF~4djLg;l{+={vLMBQu^Vte$}3I`6BXDr|wPd-?w*4MOOPX
Yc+P^`npsd{5bTG)h!`*w2b!1SbOr4*p%skh~L$$oQ657frFXgWCFusoI^msr~K7qtl(1jZGX8_m0_~d>(NsXbG#RTY9%##ECG*55Hwi@o#imjO|k%#7@a*p
k;Vo@<E?1ep8%vh+g&5xDQx&vCu7THlO@2wl%|<?5myp2)U*g!oKd`jvJT{ed$qtFi+;U3Wn3RlY5S(-b^E&3sEP+V03h#3R=+Cq<A!UU!s*_-OKb2>Ivl4}
g+sS$;-aUwws5g{elyu`kl7G@V`Up2h9YA0=`}qq_6|v_#uKE&tzO14Ldn=$YblMSh+!o~GRMP#vVL-`!J{VVDz4pl%OcE;U<kd8k@IA_CQgIr>AXE`v8DC(
N0}1>4bWqA+m|6%OlVM{<22G<kg7EUc~XHsn?g@qImO6X#*?2Rp4G4QE`vDMy67`!nRkQ+^G^cvu#tvv;AK&v6DBDTKGRkw9ls<aG*zQ1m45SkYW!YL$fFy7
_6xDfR$RZPJ<a?wdJIHRu6jrnDI+6KmfMR8)<yU%d)W1rd<{JVgrT4BasvBlf?yj?n5R1YLqrq0pK4#lm?BV&)SxHx*YFDXQq4%8O;YcqiQJdKudsVcv<O!k
wfyYaQ}E*E(&O(&_yPAaEJ@lilAHM!x>kb_rIhQSOCFSpBP-02KgcqqHtiswRjb7t_ybC8HmSHE6_*rLVYacgG))lG&O^Lzr(ge7k`%9KObo%OZc{mfcogGt
9NrWYaPS??N_rnWVaWN)dBf<WT?%V01ZyeE%kqJ3B_)x{q^ol0xURbLIz8|QotjJ030ZFmvud<7_JQ~|H8RJk;j9^6LwKS$%h<N1fDJ&bITZzyhGBLRm*Zo$
XPy0c7-mV_-%o|ZE%Xj@oJ;_di^j3bFl<5yH%G7=1{H;p3&aPYehVb9NS!FU5VvXsb35_-vP#1a6)@ijJ+;R$bQk{#LGEh+*=v;Wt|i0*OVB`(XI3l*?!}*Z
hUcG<GrTY?>m-`VS7_G7_xQGAAwS^sNAoDnfjV`2$59g!zvs{E4VA`@Tt@zY0-S_}t}a1UrN-Rq6YQU;Ni6G{PY~rj9A;u6l~}XM7QIUAtg_qyT~uM8XforX
f!tQkLBs<N)I`x<RB!6jX33>qrXWnk;?mfLMS(JTB>_9OhQ;I<r@&2a+jVQHI6)$j(Igh8Iuh6RfWwg8LqnuVMXgjVNUZG^fFKAr(*33Z(6Q}&%7L1O-PcFq
9#Q_-2Q_~}Gd|7cQ^2+tKqhq~L<v&2Lc&xsEU~<u%4J2o314Tx1JPHh`q?H`S=Y}`LasHt(m^^O_3yTZxIS#6#jfq>Dg?Q%utIian|8-<u(te<iC>c~d`Icr
O4+@y<zi>w_r*_hh>mMfvg{}(^c@JK7gmCZbG>!}Wt-ix&@h~tnP*2gj4%@sFs&lJok~F@Ld;;L#<uSz2xvf)?mk5)b1l|ZbkY}qS}^>5zoUIbgmy>^3p4A)
fL3WRAihVZv*%;Xp%$D=@iL~Q^KC9=QnTQqYLR-4O{!BC3*V%{|LvjD1p!FfHEsEQQj8+$G9$~sgBx6l_D&Uflm<SPg+n)Q@5?AjWy3btm7kyZ@g+C9^V+uO
@j>2NRDcQTM#-q8pfrWgGo%e~U8x|wYwMNHgn~eCTnch$5a3-}-el<S@rpe($!N3Zc^?pRkEi%|^}sDnp1*B(o(v(j<`_?qd_21&^CJ0SkN>by`z|%qS_Y~-
snIQImBCn1IT}L}H}+H6`0%n5h*oS>elTm(nV>khfqoX1C84c)3y?Fv&jcCUiVCky6%$@03<K(rZd%4d)lukHb!F0aM)B;8(5gq92RUxEbUllX$tqls6TncE
(=w!+JOC<#-t>A9(TueQY1hREvAJgm{LSOa+m*S1mIxM#l`Le%(6MY}-Ohs{Y>mrU-B(j?{@cN*@E{kI4)1WZZ^rz^xT9>K^s%I&hKl0}zIF}#F;X5fZ<vZ@
0V7VgV97^#y`=nH#d+z)JX1(t`9X_}LgCsZrhV5(JDEy5%qXbF6p6NB%{cRKbTp$}lQ)>!i2{Di-HA~lCMI(;f#cVK>WMY<0A~3mk@>*>?<|b-a8g4InwSOH
QT>yPvaqQmO}Uad4zqVvyo*U$(xIy>8s?f1&{>nP5J*<@oeV*K*f{d@J)DW>NgSAAT@2YXAbuS1j>_wovzFqfZok$rqM*8WW<9b)cYbN*cvM3Q%Fd|xqFx11
#Buej@czh%PLlTaPsVJ&w~~{&f!m}IG!3?{=F08bkYTWcL6E#Y)&DwI+(XDTppR~fa~UAAcjnv<T?Cn}^U_PV5fiD$H&)-%Jo2ut)kATg-j@POZ4r}&vu`9c
1uF55f{K48GCjOl)xWQJKIib_CGe7S;&m>C5AUceG-_*Fc)5CEd|R1k?e-&m|8d`SQ*X~SnoZ3B9ngm~81rT^+_MntlP9$0xi8>qM;i&=+l5K=&Qvzf*zELC
ZWkvs2buqdECSR&0ZtLTR0tby;y&j+Mu2=n+Iint86QLCmGuLa=sf%IbYC*mDU+Yxb`=D^73EPt&}$a{k&~}qmd(qfAFUzMy*f`B$Dqm3u%9OT7AJw1EgtZx
#hA>;Pw}GQRg&_Ofy(hOj}iNy<3q=r2Pt9+$QC7+2MBe_o80TQcWY9fQ}Kq6(64t1qYK2VvF2clou%2-V6V6vc&k>m0Cb5~8|km@)?^Ica|01^k(WS}*u5%y
60A8%3I-B)BWEGX3sP%!C+H6Yue*~2U)!;2!KSb5=MaryM*B>Ut0v!)L-xO|_FNFA8!H^iJy3WEf4o`Ssh$-k=>3wue`2*|c^*hq`Q3uc;w1T&<F1RWBjHS_
up%XbZOz~ydl>VWp80dcG)5Q}^2bCmM^mY36Oi!;o$b_#(44h(F#Y-N2G5I=*(q3i0>i||72OeAydxVShZC)UnERj<KpWog1iDlVgd8~309!|4T#+>mH*e$;
;68~;w_%L|T}~K)s$kGLIC1g;4J%762=hX+>x@AA1*>>q=ds)O>|;k-8#6=!iMmJM92bxwDOSu9{{q+5pTY_shmDf~%o~MTYgNW!`khnTtgp(S9nqcyXxCXQ
`wwfE&)|LbO+O8-uPdfSsRYJjSbIq#_*Rq+pjFoo)oFtsFWt>{J>exhyr%7@P!$=CB45aF^Z6+4;WAZrbEE0wwVNdURzDeHQq*|$?mUoK-%6x}c_RafJ1z8O
|8aOMO);23%W%oCPtBNGH5k9K_M>U2zf2V635uKHnb;-FXmDvdeq;AhN=Clr8)#JM-n2y8w{_4eBTIiR8_SCqa!BiH*~(dim7LNVDZP7842yCuWK~`Dzecr<
A`9$r8beD9HcD@e$$LwmEe1M#-4j_6MW0T8kP<>RSZA+}nH#ZI^ukrAxiKIr`+ZYp3OlcAc~Ym?Fg;aw&du&lqWQJ(lrOMMdvyqbg$YoY5wFMx<bWY8BQ7>K
0zgySP${A{fTzoyVI?c+n`%uCgnGa<ovf!--Ds9~im?zFQ1J|{8UgpYzw9^kR)h24K~FhKjwl~mO<GBa9J#?q-PB7kwi#;+up5YzXJf@-=@{4!k5;O;69&PO
i;8uvEu4}lefP$bCb6XrU&9?V!DnChy1!yxjz6Vt()0spkoA(XVCFSsH`GNMi&daceqoH8WiaKA@i=ni=^J;$jCc^%)|GM6(g?4XMFjcJFXc8I;Pl1@<Q&!-
M}XdS=ZV|ZXzd5%u;>OZ{=m4|JBgUI%yW6AW|y_9llzV*sR6H@1TjlJ%0NZr_;W(U?tcmcr|H2lE9<`wTonBx-c^?2#@y&c#((dl$#KjbFK8We$Yp7vUcL3=
VeIqx;MHae1)6!U%J3Z~RflYy5Jj0Hs!!u)ky+V$Xtf=E-N&{Q{~4pS`n=_u(PQFMQjQm{&IX5*(36G%ZximG`h-waBF_O9B*+9GDj=`QDtBOO4-R|jU`#Wz
IP*!{o_^M@32>nb$Q7=kr_^x&>6<3_gynJqv`HzptTGf7y(rXY+D%jx1$HL12D@NdE=f+UBbY9#GtC~(Y=iVixC6WjRP{||sTchgtPF(uh?ah(kIZL7F`)cn
g(Ii=5|$noK<WdII!>PF*w29iC#W8CE>r;GUr_G0g|x<n)$S|Eu3gl9Pt{BoAO0aV55U$2QUwsxOmnVEW~?w)ZmrT)K@Q*n;`nX3A|z)4nSXH`di1*{Jwa58
Nwfg+HhAM=!6vtfeds5nl2`{6%W>yNP(0A1j6G2CyKs9%MEI=M%;2;rIsTR1zH5RRC=#JFja&q;Fq5|xYv2;H>h>jm*ud2r^C|VCysI2nU#X-<OY-!GD1(BG
9{{kM%zF>VZOwh91-lK5H5!YmuC87Z8_J{ZSpXTfvlaoU?>j3=G}D%t3I-*358&*ogzVE%<XQq4t1Hgg7c(r!b`g)|Gv*AF5p2ixM%+}trfgtBazqDGncvSV
%+8rwqlifm0v6guPt;kocWNjc7J%J++d^}LU>ew1-!u(hH^vtXiMG}9fX4EJK$FvbD5s_log&*skf$fNVx(C%AEmsVwL)-5J^KGQd3^JaPA1}FJcG;BD2^ZG
6KTzwXo~i+x5?mOf0GZ7v3cL`^&_n~4hUPVK=-5LFzrQ?+i+)=WqTo#-TklWZ>9ozJ}SB{DGN_pEVfGe*$VK?TJ53W!HPZp&66c#5qpZE15JV>;EIyH!GJlU
>_pI3-fgx-OP8g)xCmm}B{WNe114m_wrAsN6d3qXW}Ie|>-qGYat^tzZrs=e%TB1Jq4znvi~<b@>@q217_P~8e?A_*m3QHiYr8}z=&||IN0~;d;#Ouj)lq;w
bYCOI#kuGD0=fjsBzD(s6VQi>0fzS#>=~@t8d_;28FgL%rBFvI8~e|Uo3y8KxeRNaf*W>!4doj*bwcsdi}#YIMB!!Z(sK~$6Yb#z6SZ-It!e1BvY-_q4|W{7
>+^?JCr{ej#xu~Ftp6+!yA)EWo+yVCqG22Rk|X984uLzbZRlvtH?Bv)k(}A=rWamp7u=6B{>~cP?N=4|3#)mu7V?cg1k#3RWREw%9yJK>^Hnfy2f}3il_wGD
do6O`tY%??>jIZC98AUUVOd{xElq%}M)8Fq1F#mjQv#NgrysC;-NjY$Bu9~p?^jBIPn>uaJvqJFN_kE_u%Z4?^99a;gfy=PM_nF-oJxG;oY1{rSt=%R2}qg|
M1&y<zzVHENyc<X?8F1$icDCjy${|>-J>HWMNeY&JpoG+&U;coaXjr@t{|xds5>GGlBNhi04Px9=|1;TUr?o&Gpwo?fHqy?7kVM*KOBrUAu1t6U2-ZG0s?Se
$5Hr>Oa%2JnJqD3QK=!ZOd#>c$a0;C0X?>H9;ZV)en>>i;3odOZJSlK#6Px+qj<m$4;_J@C7sDKJ^dBRBci;A+ekHcu9)6o>Ie`HdH%k;mf$XNlVl4I?$Y#S
)OE?q>t)K=2w_M5QInGH#}<QSr~Tb!&@Ymg#P~~!@s)u-{KMun8=SL%^*?!mx3>#0VTKr(X$s3D(vzbd=gtfC!2NqLQ46xXOYj%jP7@4JOfEb5{k@}qA=h86
%wvac=<`JvGj+1gVe>fDz>f56sVoQPOa!qU%Sc}iV+)yyY66F%C|B|2aqVj-2~*Ui`~LYVl0e)~__IprkJJDUuN9?J$amEyoT`P!dO@_DPwm7iR^&PBJ2X7T
h-@1aJ={&{M{AWi^|CylB>m@z6qb2=^PSty)1ds~>K?pYg?-7_oIWnS7285lxG)uLg9XJQrjvL&7f0`AI3>=-ckQ({qCV8V7llW~X;KgUluc4P&|dv;`-^SW
Ab{y-xU`1$)zL(X`iOv8<Z%Otkep-gr?XVmK#>M;JD7eafpfC@AIjm0v3i)i)X*MDZ_2;1!mwC@uSe2S)QBYtF*QIL%f6H0g}^K=K<e#YL2H1aBu>mi5LZGd
;i;HHT=_SLZ-sK?wLnKM?WDmthHE>|@H6if?4YIK#=Qc>7MT%+5UX5cuuDGEUgmb}3+i6$skdiH36^xL&|Vfia_im>hC%iGi@^B#!IZOcep=NsK%In{S+ml^
et%wsV-YzBbmtN3wGsT<37MAIVc{={z5M_RxyFsyxRqk*R_JMpR6dU<Iwk#pZ#kC4n%(}y4{8n)`#Apk%I&JB(pBrpPX@1xqM`P!rw->80=Dydl$xRBOhy_Y
vAaiuy?q8*+8c`9!Cx8ED%x!0m2}z-M6TlEG`&J8GVUiOMBFn##us2ord$59r@X2xSH4?&+81D!?wJg+3a~IiGbS)O5M}*|v|>O!<;Jr0`2nHMsG)#lm`T_X
UpOWgcydxu*@D-4t9?@c9g?2u7Jo?MRP&=QRsI`%4g0)2kVCA2k$E6EmT;h_sPe4!;Vug?1Z1ids*P#(@$Q-Zt~qAtm1u10xw%WyIN{$*UY8jOE_65t#MDA~
CWCw-0}l>}0pA;e^F{=4u@w+4LfvhNNwwTZ%^o<o>QU!~yJI=qHdGAhV`CXp)Qn-h!2Ua7<9|2rJr`RcKC`1_?_KrZsmDRD+`C1Qxv!EvxKc4s2=Q!PWL!0B
Z9_yV+bMQd=yxl8D#DUx9IqzsnG+Ff3(n0-wO5X>11gTypfc|T$P5@Z|D`T0elL+e!~$V!%xcn!R3eb>JxTl7{K$yMLcv38A{wX8Isi?AR^<XAt-+jovIK<l
prcR;#`j~sQu`iERheeq{Jdv=_YEJNv&N3oVD8f_fG7*jWIwsvn;obqI+sVz-aOxPJ)ar3uRug8#orFzM-(SZ@LvuY3>&2G_xLXuHkYjDMMOKKB2)2&w;n=c
4vG5vduJEhUkL12A6A9NKl1V&hWW_JCwM7di=7;tSP;z%IE2;(UrF(ufOW%44ptX~#a%-g$YzU;Xz@ZQ-q^((dFc&d)k55IfpotS-E_J;aD)53;;!Ni{f8$`
*+WsBM2H6`xZNG(Ml17~n}tSR26Og)04_9n05KVg$~I(oAfW>rr^@C59RUQlnb4<UzVV*XpCJ5LcXB|n9F#v3A~qeaqYUmUya9VvIqAj|@LE3kL&jq4GUF<c
x<jVcRn4s4&G_WTQiKtK$lx-t(8TNai#_oQEK}N(pL|XcMEWsYZKw3LUsF+4;r7kckq1}VLw_-I;*;=^p~ik)UvHdWN+@C0xQ^)U;X^$-C2)@XJPQL&|71uY
ymZl8ckC+vK0FX+?x#NLq9rsIod=D2s*;sFxpmvO^dD_Y(K9(q@YH%gOg;srXY-V!t>V*`$Y=9?HJKyqxn0mvIPzCE_T@)eeSe)K063wkaMi2dDCG%pDaKXW
kSb<(<~Q~H$Ya243%zjw_qlJ*F)MzvNS>21!oC~4aM1DKMHhBP#fED32UN@o0VIf#{W+4$-6u%qHolK@k3aOStD(*`RTljh&Ydn%^XvV;5~R+P$i`O`@yd;1
d4beL-5HH|+(Z`^u#tI)*LG{}9}tRjr^8q}LDbDFypTYdJI4jHg97DKwgH|8H3y%L>*P=%A?BeQM7qqe`jASOv87(<opa0~8(6MCl4uBPQrSIPfKB~uk9Q%*
%YDJOPrjtCyPt4HSFm^TPu!dfhg$B&oM2{^O)f-nj6q~1)uAqGM*)!&PKAUftj#@q*CBk%ed;fg+@2ksC$;+qBg7`>7D06o@ROO@B~y8)Do(4dIH^Qae|t}|
h_})#<Xhb{)!XXOc*#(g8=ChrKD!BoaCG&RBuKBRNB<EEK)=(Eh*&*af_py1Mq{Q7Zm*TtwmQ(G#LpY>(0Fq6-hZQT-L2G1o_2fjrRIpB9u*vKsC1n%P0lR(
Sos@xpitgt?LP0R-kWQibOqX%BH{U~X7S9KJ22Dm2s<+Qb`EvkXc~7qFAL@kzU|${VnxOC1}pP%$KV`^R-4QxUTuhA!p+&;^ec>eu}?0`KHmT>mWiQ6ez<hS
v-ehRQWL3Vx}nxJqu>$*%~+iTu}|rBZ~;M-K2O~DoUfD5qIcHBmJ#M`%8WZ|-3i3?Q2^gX!u8H=%t{TDYVLKozo1fKs{~zRmCuK(HDUKnZMTCvl)75nlNIUg
*c7I729JT5gW&}1^TCYt=4OJy)p>al^t(hj_o5M9H;B23H&+mdU$r`&=cxt#cq#ivDW1F5`6af>E4T~~Vb9=rEXkjW?KB%=84sPTuwx&<W?f*Zo1<fp$cx|u
x__>vk|_#-;4{=}JRUM+aRe5aaeQE_seEDKJi^+7rbO{e1tBz@+|UAV#t&@IW0CF$4o(JqyaSw+My?FQ)A&ynz`%><*1zczNRL*&Cgn&_czpGU=zmdl2$W+`
aPt}I=J+4vW{!>BAIbc{9(XrF-j85k#dn)Jy1gEIDrP@dl@5xlRT_*}@(nn7G5*OIUP+u$L$p&e_MN8~aAOe2iBjk4d2<F+3adOygA<>k39G{)5Ub=@3O>jr
T@$=RmP8^71>CQMXN<5#!6u&$fT^lE{aEy60C$pOHnYfm^O>V{gM%7+8QAR9@hmZx7dr2CjxJpYVIFWwi7no;S$!u>i&DR1j<V&Vl8F4tR^xGPgnsou<0Vg*
sL|fovVUC&s`Bo;=d#T=Hv=0P1Jrr^1pA-4dt-xUuFw%0TL0klH8NXdikiKB?qgDH&c9!v195CQDLgTaGE;&INvDTp;g@q)p}iN%mL-CPt_Vwbifkved#&vC
-DAIklS;yHADsw~Pc2m**fZ<zzI+-5JxGXW6Atc0S~kQqp=Li8&&&w$Y%qC1SGSK`rY|l?gjJidwr8WJ3VhiP&TwM;!ty{M7{!Dk^MJ)reUnvVeQ>)WA#yNA
_Yug~r1<Djo)F2=c~kv+D@3WFss<UB3HqUhpOLY-1pwd2JsYixpS!6A`EAqDIK%5oVsn&;SYk)TY!HV~ej5|9!(tX<3xs>F9Aac~*{Va39E&zKWP3LJIj2O>
o$-qJEYa%JOMqylzPt5BE{a+wf(3ZA(&iZ7T@B1w&pqtI+18J5VGMUKMDtON6O9n48k6t$|Nk4!f$H%h*W{ZZe@4I)yE++7{2Tb5ffz)$ELtbjWb!2boba}H
lw}zsOz1wO+V4Lw;3MV+p&{xRr#!;EbSJ%P%zeB<y|~H@eyD#BoNi@^80KL-1kjOy*gymkE6-<(td92(L*e~jky?OAW3^G1dc(xJ2JMV+92>jw%*pb>4ascl
9NCsouJ_Wh#Fx-9YL`i;_I%nm0fJH&kyW4vfNA0(A~Qp~^`G_90p<5^3n&kI=HmI;mbtX*ltJ%)h2pUQGLnH23{sfRfVs&OpLT~frESCHRaP^t4BfT*JZxKZ
iiPtP3opFg*`inGCk089+;Q1-<#$ef{~xW*s*K~t;k=0!W1tjA+K+EEmUZAo({wd0)WCb~$r{<qwtr{UoN+=d+7fkyPq#T#q(@4E0_;nQU#CN_^fZjb071Ij
^pdf6n%jUx%x5<7ViI3guqX>B#L5y!(n<Q}6R!hq^WBj=$%>EqdWyF$&W=>?Xzt=VZNj%!Ma3({NnH{9YElp&A(PAsJTuReNEDfxybGMVHw{Im{+e76NioFA
4Z56~rbA&iq2W1%Nr?((<S-g<T*zdQsNZcdbz71jKQ@hBAuDq;=J?z5%Dy%;#Qz+Z#UWowLk_T(&GZi0u1f|gG)lS@lsIz{f#Z1)T4bBs20nLbiXli4{sT7c
j`}y_iU>uGF2iDr{IiS5)h=T*kf3YiO(q5AP@gc)k{<F>EG0(VehU$KK%_3Ap*^zVww53U3b8rytR+W7!CM-DcyT|(py3Q1MVY~4;Yfe5P@wsVYFAW|bea0`
rOGq>Gp(dt^dAK)-tR5jCfAUj=|()Ixe9zE<ImJaq*TpcpbYDkJT$tb66U7cg_9s`1WBlp9tTe?6h+pPvjXbcPGKbK_-GTSC<|kRD_94^$_$ubq7?{fio}rL
U;2BfOjmraofTMah*b#?WKTYdIy*vyss*sPc2_x42QCayUg5TKRAg6LJyClie{gEES7@~I$`EGVyA9jf)IC3q5iFfCQ94y@uwMWE($f`k+u_8eRnb(dj7jzc
@Cv=whbqY7pD(s;f?%@Z=DXg!Pr4-gh8<^fwzU{3@zkM{fpfbGkgMilVQ~i&U4g7cwFF#~s={;15}W+?_+J;Iwm>Iu{M~XIh62ccdxMS~4S%4N?;3Y_N+kxb
;4Z{H$p_|h#S5rO^6%W3LkBO)^%;K1nyA0)9D`j|D0|DKyQiuo-+6Ro+n<}z)E#yYq4%O;;dWXgsNq^6F9MUvUF$T`TEool#t=plIP$2cJ@ed)t-2|6ks^-n
dlu_>-=yzf1n-)KhI{PVq|r)};U`>FqPw(mZZTLU7F!4vm4;S-gjDA0J5>zOSp)f6ygsy(<07Q^3lD~a4m=}=C)0XPA@@p3ugl1mc-*&CB&5h(xm==Of}dyf
?o42+IIImAtHP}N;~jUJ?<JbIJMBkP?+h_yMtPhLwT!mY0Oa!sYpX+|vZEd%ek0u^!^s^|!xh4;++Jz>2HZR3)Z5CBWeo9GP-dOpl{9nRE?r(S=O)GV0_VtS
{ti0v@~C!a^|}IL+*oO3D%<5K4LBA=n1pKPOTA~l|A6wu0kIRudWU+X+KAOGP*44J*#gsr%#{mjR40nhcY5DgChes&QnAQIqwS|{)&38gLCC9n9!ej3!E`V$
P^E#4s6)@wRL7bX`vC1r7a`+2DiepEtny&t;}zomVQ8q2;aPCfAR=)j2xJKlWm7G0>+ULyL#7vXAUUt%a(7$^7TvU*6us?fS^V3-r5j@tLpUhkiel9`d&?C`
I>2*B>#-sP*Z*^9DdH7Ry(`jIWAr=g&ddRH7GbK`_am)bP*q_jVcXKehLN4P*33$VB;-R|h<s(PtiDz1q}JSMms%3Co5Ce$K&`>g9R?`(Lxr>b{+`BMFl0||
5w3CkS$#hS|LZ5FlhteHZ^h>dYvn`K7a~Le<I1powzOs7tE#;0vJoV8DYzm*b}pqJSO}!*j)l-jx4(|%Rq(<{gestPdd*?(2L!)v<-wYsm?~+FFyzm)(wn#_
B~0n@e-Eavz`~Ovi%REufN&jHZhtb&bdbz}Wi2>t)Mf{x6%y~fgX%XIC}+mO=|r@Z*G*GhQu7jt2~g?w;4W*%Z4)|BrBj*&RF&Lk@(!lwbg`Pfn_9{9rzp~;
fj>yjWBq1P1ZN~Pclx-c?YyQnUUnne_<I>jPTMyfS2J^%hw)#}6cA$mlM<&AxBg}&J4m(n>O)7itehq12FUIkG^c-@qJpTh=H(xBH+BxJD7+_5jR_|@q%7$d
_|y&}C5h3-1cdsTN>17j4kUiH3w+)0{iKSni$&&gi{Fl0_{^iM_-SFwk{2BHIA@BsKQ47~(sqifO!NEFF2H%JWux0o2+KNnBRuC~snk)~Tf`<8ote*`o*y#K
EH{UB`gbYYN+m?x5Dou76|9x9d;5EEM(PT5Dtt(W?2CAdZW?d5s-9X1g{ZdTgtA>(fp;Go%pD9at#29kkwX2n3x~%jq-=r%8T{If3MmKFu$Ck)W$W$N%}2X+
I&1DHAnCnzp9`ZaxqTkio#Q0F6RSW<F4>3mo1Z%(aDfxI#$*skL^#Mu4)jbVE+wgA;_OUmI+n$TQ)*m8soW{~*F`inT%$1ffF{s7)>(SvHE<!qaJ;0(=GZvM
5)&wiR$S-6)h_ombp}ht$s*RZ!X@6jrPDUEglxX0SE7)xfsCPJniv0M52gFgVFj4?5Ct;6H!q6fc8cbuqupl##kx-G6N?dFP|!4?!C%^nN<HOBZ$h6BS5ep^
IilkCjw4&7#i1VVhG3)Z-%7qv51*RD>`c|_4*Zsb4_y-k<t=LS|8O1GPt2d?hy}R2&^k?3uOcQA^qVGPmcDc}EpHe)_;RSNH0FUJw3KkQXIQ0$XZYlVK`r>X
tpsOUohjv9+jC9G2h0n4S=IU`8}x+rMI7$FU;YO39x>&<|JVnd00D4+f@D=y;zf1$f#%4#zBseSbY5H=qPeUSr(i@#cY~yPriZ|zFF(g0ZAS^6>JLe^%U~<+
bqH;yrVgAKOw!-KSp%@es_Na9<W0lRX-k;gqa8~{1BfnZCb~<y%ntCW&6fWtmTZudLkH;SYh98~pPM?Gra}6bE2Cl64m=yFbH;0JV*ho%6)SJ;YM($WH>&Dp
p>9QkG5zBGN7Qj7Sgro&yCc^EfBWF~*ES(@17d^jPEB)fyaHF~A{crZEXM_tawL8O0Z2-LI>Jp?EJRXU63i*yRSD{u^!Lf+7CGUGa!<7jtt}IPYmd=MhK>+0
I+?(VdVXoj_^Gb|4kekOIb5A~zlFWMLW3zeMwfQWR&OO`pl9mKm&}7^2V6n^{YHr*MY=w;IdIgZBXicZK7iVvV;yR;*vpGrN(bp64+K!sBOO@ebm~>tc1na%
#|JP0UM%bbGq_JC(%<+dX0TOQT1?^xVLlIES3zFVQy^fLG*lJ8-eusp`&Ej&bG9>pFBYRV`jy%+7b;S)@`Yz{Y?N~;s(LM_8=<}XTYAnkh0o6dZ>srM!vOm|
ucXeB!z7kOpf}v4{20*2TjH@^<e)!u8S}WFQpwTt1FO4)AL?W=D@8RWVQXl8M~YSNqctuzGQ82*41+}Sw>GwO4S39y{a~UK^}V^A{p5e-ks!Y-i;z*d#daaJ
2(L0_dnyrw$`1L-QBGS<q~DO-7bgl)`j}6@*MDiVV(^%OT+mPxHK}nh>=FFNXa42bCVF)(3u{zoa?H_nl<3&q|9&QvTWnxV35C)lvpqP+T)x<Fymi*bSE&hE
#fr4kGl)$I7XT|uN_*}$R-Wq{aEvm>a4;l}jlCZ*8Ai>V7J(YkneRIvE2blI;vEG<!seQfb)-6~Y-f3o5v#9hAi{8>)C;qC(tQe_{65ud{!Ypi6JO7Gt2s(!
@UR0Lp{S2sjD~c3xoIx;Q~$Dy^pIdnAfp2J?gwd9?SGY~WH%>t%bWRZaq@Zz_{e``YD*HtfEBObNr8niWeK|kV58$@C<MY)hwR1hH{%aEhz7CPa63oG9pS~E
VR5}Ya&_i?+F{57xwd0uq;xSYvz9mOf1SlE7N`j?Z~W%JA;E_ZS8ED>YeX}|MEShE4{`TiGR}o*2tWC7HE-E!WY$Th@m90`@SVSST)UqKw#Wu7>}kr_3y8A)
dyFGqpMWCg48~3TMkg^b&5e^V+^hM|^^LzHd~q<)8svmvk`b=F%Q|iy%A<lb4ne}P65Gvc-~fEM;`L-{`9P79P|Wq5jHw?30ZL0LZ%&Ds?2>u0D9@e=x^O&&
=fURf9};MaAQ2;=the4-*D64DM*?is!sxgZh+PX2zkpRGdEdI|efruPV5?Ffwp2{3a)_$$fTr9{2T^W+OK@O5cabF>I@AUI-O@*xKY5aiNMrQAksF!cmiNnB
Zr|i8t~jHH0N&=d3v||&j>oy(>BKUVkOl^ZmJ%GS#YFDU^CgL|1YyZ+_MG20E&1IuLaVvTS3=z_!y&~uJ*g6YM(<)0%GIN+gxyUFQLvMRW+(CcemanoAAbTj
oRs6_(tRr!A(ixdIqK-k4ma{WF*D~yUG~YPmD^O+;+o{9CeJ!^yQlB`m)5$owRy{R3vMU!X_U~lo-ak|g<T+r9yddHl|wBelF$oLS>A@+ip_&T_Aap=_8-VC
?R90@ClCASJyhD_H<TfwK5SB^k0C(rjD?{1jNjxgf+)K3RS8-ieZdZseVXW(if-I+hfy6uHD*?JKBdS4{g_Z0IsU;+PUBdrhF(x9ROM?%i|K-FVsPJy=jOZy
EP;IUGM86WFfj9ASchzzgbCmd$dS-9Qi$`5h@6n}4lLG4S}~MaFDvbkn_UI6Mqab?X7)-8SDPF-q^Nf$@+>R9?dT1I>F5(npZ+Trz)S;vg*GDGGIC2B-AEpQ
8C?1Tz4}j?#A&&v_XPO}bE$3Mk?3t$T8X@ju2iz;(HpvlW)PsX2beyd=<^Wy@nqK+apQ@8QqszLTh*;-+EJ!$6-wD?^&Svp#QP@01x0U(L5Io>rzo|l-3T%n
AML6^N=COV?>>}CLzlSdqel`*;#lxGsu=c)PoOCyWEr~Uw;UjWn~&Vv7Y4e_l;uB=niX00q+qebSpVpuq&BFY+jA9eDMDwRuUqri#mtdRkA=;jM*5L)A=_T5
EXR02cxoe2sQc-lY959Wn#I<%z5dYnSpPBAk7}YcRpIfoe}OY6K2#CN!gFUgU{n*qMQE3=?J#V(UJ)3TRWDI?GV7lPb;;hK3zr7;8sf=k3jCMiA(oW4;Hi<%
-)PXDry4joMtD4X5mi}ziAS$}L7<k$ND)IBkd2QuhN=dHHCeY$KKrboyGU_Bn<`LU@M_^cn}r^j!#E?zg*ecQ?+*d0HI~VEtIAvcG$7jN8%OT^G^<yw7@S-y
2{5~L;fvOp-Cth-EWXswho<td67YDnG{B7@o!&)=se^!!Rb2i+13A@$uWq;;2m9<FvOdI6K>-sVAvxn+UF*zhzG%9Lm$Ci}rQQG6ZXm@L42?{Fpr{S%3#bd$
MrHfF4|54KR5t>leS<hEb!KCNLiBEcor642Z1u&_G1G^Sx7d|(b)e&mZgV}Nc>6Na!O5Hekkg0lPfA%g1w@i3l1rt_D&vW;{8-!mzJjr2ms<g9|Dce6G6C_%
%wQa4BV?;@YrRq*s7&_XGKsC9g5c?&9z<IxltEY0wOuh?D)_hMAIS&BE#<z}G`3~b>9BcBh!=P&p!*AJ-}@an)zw3$H#mBdLGSYw<bQWTy3I=%fw#4SLc&7-
H+C|XAzNOA4%Yvwx1`Ok9iLxumNdA0!+QmOa)P;_*e`nWHM(UQpyx_Bg|23l0;gSo5Q6+l&?8FKuGgY#XJyOLy16xZ$D@64rcR;)l4Yy405MFXORboKNnAeS
0d4RtmDd!V8p<TDM-g&ta(;^RbQb7>QljS3yNW-KIJi^t*j`c~?_(n5!H$ekHUQ(Yb#T=o5Qfz6=(gxFRi{sbvj>D|am<a{Kd;z_X#aD~B;*CM8w97uvH{SP
_p*pS5B{RF@!e&AMvjdep9#y>Je!QtpsK$E&0Vu%%zjausTG4)&_&j4t<|@LEFzzcg#sM^o1R8Uu)Hknvf^PIR-Jn^PC=msb2R2zZF(5zlYVP}*hT%_n3Z6L
g<M5q>Hyj$XMM~S*+SPZ%ku;gUE!-|bQdr_v`E>|F?JuSuxo?H*)TZJ)+RDeQ(bNjH(yu;UlO3;jO3l995z(d*;)BDn;v0ID)v*3)LLY%?uMs%$sc1O;vI#U
OV<LIL$}ot{{d&#nUAK3Ujs3OE_qZg#*!eoMUu+zFWwM0`OBC^St=FAUTGqsLx05{9iZ#+`<c!g&KA3cx0iXDcMyP1rk`bXZg(_{+97{HeuRrqeGGpW9MVaM
)}vt}YF_}oZ(ubi?8iJFyf3BK>lSRnpygOvtD)@(p6!uO?1*cw6as##JHHC(Dngxz`>;djPzPQ}2>b(!pww)Ad*BUH%>`sz`IO+5zS)Y2+rzF=WxT(+!+JB&
KfOOvfBl&9%t?P+y{Eg3pqJ~{Jlg$b-OCD7z}9MSKYF`r*DkxiAfUt|qdT4Rq@sHjC^h`B$y2nN(Yf|qy_Pz|Ao-*7?^iracH^~$1G>nC?s@`iD75q=tzvgJ
ECrYU(8HDDt%QRsL@&8n!MS+ljezI$KgR5X(cHaz3#c?`@I2beUWa5sPX5YUUeQ9D@*e^=psaM|3Vh{8JRGGAw)#kcOj^$E;qv5r)F}`n?ZE`FW)GTkDE?`x
YTr%I0lp8Vsc(-@_m`B%rd5)K2^HzE#8aHx3)*U|-HT9K9k59;9y%2xgxBJFCO*I~;DZw_d=Y^E!5Ai)+36@EQtg}vyivnAm$!P!J&j#<x-U(*l9@cn6Fx<)
@}#iw1)1SGZyj>LZ#W3(fZpaP&+1I#o-S=YQ9ynW9G}Jm9?JE_94d-ixQl`nqQI0S>{ZMCoPxrnVZSbGe28mPixw(p?Pda+lL#P=PtL`pw-Ap*!)bAQ-EkHo
A?QfcYZvuDawy%i%z+!_0#8MXzowQSjlqT5laDi)#>Vg)lx-)toqTB4^)|U@6!!A_+v^OV+ko_%UBcKR2&5baQtcDg9pq0EB1r+k4our4`mE8HP;mS?*;=!(
_T4{s)KpqejjbjNN$@dx%-_EW+aAgYA?njP*$T;6gTIh{jBkkR&MwfDkaQ&)#!Q&k4FYKG{JYsM^{4fr{n-JLNuQ<0r;>1>)x+h9Ikb-rOoEY~PS4BCcyqA3
l<WOK)~cz&m)$$keG%Y<fGP!v!Vk(%i0`R~!}fDP=TwT9#!7;?Y|*;0p*ruzBG(PKOc|{k9DAH@p=*>Svmo^85K)te0_C$1^a?7>eIoXGXfGWIq#GD>3u^lf
H`uJ>wgx@%SW%;0NlfjF`Ls^PDsKe`t1L;mqtaRQ4ROvHuD74r(Y4)MF7D{n^AEO0PYF>GSR@q%y+iMCY@SJlnnp{-Gq1hL5<vNeLL6|zCBHL5PCaYaExch0
hi{-S<A|#`n98GrvOX+*qvV@R^8EClgD_$(&5NiQsLoB_k)buMMl+(gNV}Ro8zN8(RAbe*LHQw2@Rq!7oU_REGWEW4T8tOo+8ObWgrgtdSBw*pxG;sT-VjRP
!;z)@q9TcnD3DX<6EbL(6El%Agy2mtFry<4N+PPJ=VzPj#Tt@SSw#%Q(WV>y=6I@R24OvgQHnzj&&SA;G2cIvsYA;ake7cGhUC*2@h?x+tEgce{fzhfn37qE
v-IRs5-nH7!#=pY6<921441l50L$h9vy;>!wmSg}JUXMqsW3>|Exd%pF88t(ZWfgDB*Zgpz-^X21iAKU%iv5z?51^l9ZFqvJfk}c8t~RP<zr(4jZ^kdQryn(
NhRMn&nqK_AeCxJ_Yf{q4ZMvM;(DcL-Duqv&cq_Qj3k9Q$z<vs$#<02wWv&t)DY~E4F(%5K;d;*yS%bMT^sg%lcK$62M_FmrA^*NVd&k27tE7s{Cd0Gz@%6^
NA$+<h`9T&Ym2NTzdDUjHJqa<r18vXrbeLQ6(FBVd9YUN8~gZz^T7R9iM~QZuYJiCVw<d$z-}1xo?WPBfd2#0i>o?pEo5zz$4uz&BgL@+Dcx6WhY+{DDeFzq
8JH2)l|!PW>Jx#&=S3Uty(tp{ei?p|`7QgQe9s&RH-R8I8v2R9o2B;i&>5`OL-Vge+Txx#7obR4>tkh#nOIOlL%PFH?TvE=;26tXvnmQ;-->^B2vb-NY(A=o
KxKzYgi+ED?T;rcM7QzcH{9Jd1=a6!d068(%(sTEmf;w(Ss7FevmQGzGg;t~n_|G<T1eUZ;S^)zra+NdqhjQ{$(80aGDYOYy;Jx_&?3>y3*c_!mlyR0;)a*{
4X^J*0Sl{(_47$_*m+A{ji|fri<TMIR%I)*E3N0kRF%ag6Eek&H+6FXr@LQUtGFT$qrzq!P66J4ewN8SUXk1jC%?Ji9*3~*Bm(0b`X&gN$N-7G@dRGOUOaFf
l=a0;65F`{3IE@yy>{-&_gO>2(}sNjyT3JorrBFgS+X6nLFY*<w+xZ_H^TTRo9JbjdxM16j}q;M^|P>P+plS$n?FoQrh;>%@kA?jC+@zt+pj_S?zdfqz(;8a
7q2s=%<Bwq(lTnG(U)&CK=Dhj^+X3d38S_e!6sPP6_W+OmNRYhZboS)v0Biyvc^nk|8uB`8`g;8>0qANL;h}qv?W5`Ehv4`(-NCkZJcT(#xP%%UxG@dq?YZ>
U}48@bk)Mx{UR;nbH~bx8XcOmy&TiP39G+Y`-e<t!%%FtC!p%7sUST`67@zpfr_|DoW>d$f4KHk;&x1GM6woD>Akm8=2g+GHvh(oG05qn_Rg4UHywe$^3pY;
^EfKgJ1NNvVaKSWO7dnn%h!r)*K@<8HpRU9B|jn+T7E>LOsloaZqS@epKSkTDM+00HC<rMCP?o|;>)!v#QKD1Zn{yN*&F==4Sl!HVmEk}KZ33bgEv)Bes~-e
-0blhm3xJ^5cQ#=NDtyn>9ihFjpV7B{=f=fMil6L;sS&fJ22F$prt`xhchr?p{kYC`o{`Jmz(wm<GFeNd4g7w+a5OO(~sMQo;Hf2g1PKvDEyqCvXTE@Csulr
DZb-S*p&PxU{BrCN_aS_kfO>zvFNJkQTX%7O=}HTLWR!(Oa)#K@*slF(CBIDN;{vWg(Jv9?{`xrZN^%<gmmtnrBi-4sES#JK;0{@R{u+1ukvDVx1vR?gdS)V
N;wf*@1mV(e%!7XPkW9t-m(Ieg9pa+>Ot>;3!3LDncjSj{pNn1G-s8I_NgH4L%BDQR<OUnNGh~!o}0oz%9$y<J)=Yu&Qxi>NiYub=eTXsxP!m~&>2#e?Xng9
GA>k)6t`)`EdNAxa9;+)0J(Y0N6SmdS;cd<-g$+Ttc`w#BFAr`Bl0f(Yc-?M_<S9EvEGgpfV)e?X+U!eCbw?mBXbVc&NOom=bi8yqIrXK+=`R^Ao#5&{=4wi
N&ISh_*&@q=2{BE^qoaCv@>lgajQyu(_7M1Rdf1f(Bs)7Jk+VZYSkX`IhJKDZv0jwAzfSDzM~@vtWqJy{QWCN6m0e{uejJ-o+?VEw#|BZXD!&eBw%_|5X!Wn
9;!n2!3FLvTMMQhyP2XRazbyZIHsbn_ldTu(%fFq{!#YC?D$dHT^cUoIx<n3Zrm-3sDAUB7Q245UTF3o!E)WM__>~7!xz}K8G>bjV_p=dx2=PyYVF5}THq?z
6iX7Q<{Y=(Sv~6fPl3#8orH>~*r6y<M%RP=eG{S#{++O1YZ2~U*H|gl%Y!)?8n|<ze)EzS@D2DbZ8%s%egqnB*10C88eimyjDb%pgG*jox(n*;r9{&ns0Pn!
6O2z4naKFDHN6uJn`Feh;rkylqOV6ymLTH>f37j$Hoft+CJ2qZC{8dfQ~ROE4iqrVdd;fbq_T<&)RRFxVK~;*g5{R)m_qh9+AI6a!WI5Uesc;oyBA_15N4L^
;mf~*{CeI-ODm>%#A#s>zaa_y?d78Wtswx5362Aebep>qr3K|RG$cSv-;%%W9eg*L2n4}}bTk!o*scY56zjG+?k}DrpZr6_EH4N)H?pDw$Te0@Lz9H66xn21
gl4guZkVI2>Lc%BTMxke#5X=htNlALCI6I$iUU@w%5gn~X1<Z;j-F%?k|wE5=d9H2o$LS8dxvZiMmG&bpG8FNJ%#i|y+D2z$&x|@pISvNZf?~Pu^BLXqSF96
{#~1!NHXZVs1yt}BI1nuM^)lMoQEuyoRQ1$ND56@hWW`~UqwaRKHm$RvnAb__xONv%#SFgZl(GzTZh?uN|GLRQkmmCAgQyd-%uv_569li?qBf!s)3ly$TFtO
2Uq0xjmn5hmBQpDJKMA1UE}mt59^;UW5RT8ff>1_JRbp(74DE|aRGBDUinz&{OIkiAxPGUjYcq2kuj{1{ZE6DyrAf$l{WJIgM{3r3rAE)JJ`dzI{$DN@Twrs
J>bOQ(@n_C5-`)$8)eO#!QXt;QjEM7QFx-4r(S#v@qjT)pP1R5hb-+1Zd$E8)u(*zUCl(gB3UEIhi{ZYZbWgIc4Pdj@{n{}CU%p&^SDLxwbl(x#d=G!pdmS&
(2f;JvZMF$!zVj9lR19w0(Nb{2AAYbew@PSEn27W!wa8-T7knKp%Sux&Go6BP^b2X5EH9i!1<gcp@`ujV!S)a^H}qF1#ldC5fccPO7+rPhgtQq^m8o2(+CyN
<_PV2y@~#+u@)qsud!OjqL{ZBNf7D(lm(2pVUvpGfn^p^63u8;l{UoT7FZn)8dwpU)J;IC=&R5ABCvWFHd(kc0hdV$)O2`;TySbG=*1yjw&TMv_tfqJTbO)=
+-O{l78#SO+x0-bJQ7j?Hn!sAegAdb+SV!-*|hPXd2#8i{R+$XI60W=c7lcE^-7u!I9%EDDko3OP)pKi-q*)$toUl~VigUXK)17UJ5iNAsVn?Pz?BC9fAUi<
Z`>Xb9$cpWUADy^nT}VpztRQLp^DiYSTx-RAS=Gr?T0qDdxCVVO%Td4W(h0C2C3x{ywu9u<Sj3}PAc9K3s&#5xaP5eS9{4o&7eA)|FGL^Q@&ihK|n2OwlMpX
_i)@cEl@e~N)?HsU+{Hvsb^9>2|H6qa3EJL?m&61F~>8W`FBOPlwp05^y(<0-G<-@5ncXE9zaKo4TPs7G%O-${VXL?+y{OkK^0N>U{12ku9X@MoM=GQXGl0%
esb^ji-=Gf@2=^nBDFXdBu8TrfsX4BAV(*q|Hyuu`bmHn*qXY7DgYRu@3JN!5NIFnZuCX3N%1M2av+cz;29Ev*TVl1(5A@sziBwMI7Q8>gQMk0Edy4<q?x@!
rJ)%hZ|yW%P)G8A1@$#P*j4XGDidu)-g&ayHir)@n2Nj+aqfOI$I{8Ra8{K2eKjtuPVRRB>;uD5JB6=H2yph(jyVIG%d#htM1_JIU&^D@7x067Q8#|_QJb-p
Q2?($s(H=X+wz%Dz-L28*8vtY{nNGH3#6?Y0bWjeZgyijXGHA_0Qi<spi01myGFG1`si+P>;yQo2uwxZqj-jxORBsOBN{M|!oOC$XtCgMMwX+<DOAd{c&~MI
JcU3giyzd6@ae`&uxRETp)$c2&$isD+N9L1Xb_Cp)}7R%-h8*><`S_s_MKYv98Q=xx!&JyoTu*oQg&);;2QBXNt50E5+WBM@a`2+a*Zx1f0zTpM1{CN2_ww!
UBJ0*Pj$8wjgIgKwEKj$`CXZm<t;LjzWV@c1f3Ct9-5k0t|9_k&$30~zbm=Pttyoueezb6daOGjx#K$*gV|83P5BE`asC+v<?+Hc_6K@Uh`X--UT%j_Rv``=
uA++O-4B%OWDh%)DubXt$Uaa<5gUhELZf~&%wp)^4SP~SdN&Tf7L~gsRy^o5=y72*_pAyIyV7d^tEe7d7eEbpHK{Oti$)aQqMNGOg2F?w%;{3}_?2dgF%pUF
5gwSU1xSEhNVma}nj0`LxEshGMj{PIx2D3kP2YvO$O^SE#?{Yi0U15+*4(6;MkRef0I_>$!>SF<h!&jwIDY^~-bSaqr)tuF-&wv+|H$2!;uB+X9~!>cW5`a~
jB)Ka-;o-7P42pVD(JA#Vx%d46gNE1JEC9quNIbJ(~>RQJ&Xv-4EG42q#(hP<f6eN<}q~W3Cae&adP0+5++8Szt<d(rL09I$yd=$Mj5)&(6I=B`l%~XB`bCW
HfqV>eaKcJtdbE9xX@BTPGZfa1Ii=Gra4KVHavo81{=iIJj#6^+x#e1d7G!Vksj@XUtxw}#h0x=%zp=m!QAGeDSR<*YBK?$bTIXjc;pg>wROi$ZZlr+$dGGg
*$4UIDgcSINn<MB){iAB%6_5#1A&=cHJZtg(JwZ<8J#+jG`v%$h@%&*`o&}l@4rdi4nG&Hgj{7V2m6}(mVgA7Rx(bqM<*ZDbFMe?BN(JSO6CUXu)sUx%*-h-
{A76|pjCjhSrUZ8sfmty+=|7#K?w_7#Y)E->?(5+k!Qhp3>OIo$Zfn7@dAM-(x$L2a_d*P53C)0lsm%ZXj?pbQ+;C#h0HqUVU(Ai26$5^A~)NFzpfS95~PI5
mLCiMKovJ=1@Ou4On76yE;2Sq3dkD?xMqE1=CxCN<1#)fmhKHHL$GE@*&{}P*DSkdg-%Gi(h_46@}EKlF>sKqS-)q~9zUR;TZFPs1*e_kYb^+mRt6n=e*_BC
H*V2g?b4N@$sYf*oN}-;*~R>atRO+KF#Ke3o$1xA(qDof6{Quat?KKoJMBWf^BKJm2(n|E<oh8lt5wfH#|SVKEbEF<Vog*F((MP__10>-fRD$e?hR#HpC--}
W5}t}x2xbSRAr}u`U*^Dx=-I*z$|F{Hn`RZ--@<nw<?mllo|RTT}*yQ$dG*e@~Yv^N1Dj6SJE6f-mVwpcN}0jzR`V-_p88GtYJIOY#W|rtVQgIkI*KKlpY;Y
Zf<0eJy`VxZ+&}&;zN!Wy#|6zz4wWo)&vWQP9ho<`XW&An2V?B_1e-W@_W>gxL;z{7{V^%#Fh*M#(pM#lA0eiRgyIg4hgufd?MBNWjAeur*az9m{yempQmTO
NpPrj=LJRi5-B;E*EL&Mx^1x&jxF7A?`{7$inU61x{PU(E)%|%ArkzWf$${Eci;>Tfy@1;>Cl`0!QCzb0v<e@&0kqVMg{_~mmzje0=7%>b+W59Dh`c_!e`=7
_Q2fF*Bd^&9FNb<HHF#b>y!;rxRoy5u+5d^8}S4%N9^Z7EtPcIiLg;|7(!+)LWN*0&tru-u7q&uQh8=q{<obmd6@yfiGkd7^T&~#Cm*j;Yt?FaejTwoAlg+-
HqG*aRe)Is%1D)%B_;n~S8wZbJxy2X;=Z$->L_8~dwZOu8==D@Ia*shnHGgX3FEYLWG@YgoYYa49F1hxRbei!d$1fg_;>R(z&9Ee5Me7rwXIu>iNHQtk5_x-
e&V1*KaqXbKD1Oe2;al-rdf(f9XPQFWy(AFFC$RuuX9pB8sg}3&L<K6pS=l&b)}l?+x=f+mB|>F`mo2N$wjmCMsP>H7u^HA-j?<K$@J^N_vyLYGi)X`f&{);
P!4yB)X<(PlO*_Bt4V8JVu{nZ(@-0oh8^ZI3++ez6r?r#to$?tnQ*jtax;JAM>(mTDEoqPt0Hwni%l!RI!^?DF8H#CVgDi?YNC>DBs)S#>EEgM8c*2r#aZ=s
ZFIFiMVL2$P)9${`fwOHYsb5&%lg}v+!cqjZJs7>$~otWyyr{;`^j~b_JIwmk6yF$y06b=b;Jyts%s#m!a6>Dn{k=kfU0c@5n?6S@J<%$e_7_gs@rnYwbw}o
mh6#}p&Y>*QVghWKJFxnSd1hcnM8*BZLzMVs9M!A5$?M}MQQ{0i4KPs<M`7`nykLQii#vAI8iG_{EMlBa6L-|>s`SkS|ZwwpNHYcz#$vDs-0Asij>Ut8swwY
SC@(%hWN(p#Z@^cCdA;;bTABo(CDVK8FT`Cr0qI#e*rGU&Vb_d(-GUDEEf2%gIAl|L!es0hk`b3xV%|<=eI-uo3go_jZh{h-s|x>4BLg(&(<oRPbWog=yJR9
Ad1w^X&w*b3vN54ph4j4sL=Q!ELDygH}zo0prbr7ywV;=k^1KsGC;&zq9Pq;Zmhb`CKh*hx%OBvFYV#QOINp2EG-}VnId*E-g&HKt8KghqW4;9X`!36d~7Ww
k)V+bo<;TtWd#q;c`{e9K@mz(CJkT4l~ycYF%G;#h((RJhkRYq1w4#rC~|ZajYVWeI8IX`DPK`2uRjy9l%r1?RC`|Z5mEx_;-H+^lCFrepeClT3Z%aNV6HOP
j@`7QNe)}Be=o*&GxqM`Xinb%8PZve#lu^{oh;8~qPH3Ym+bH1LKRk!#r7WKsI?R<1qi+9_`^pQ>4)kdC-=4lz<{%jhGiE>u@F~Nj1#z}9Jx74&0xX2_5xJj
@g+2LVh;LQ<8jQ%&mGeMIkr0wz>ATE-YmjDf3LUIES=6laeAC21!(45b(4!&+K}|=zZeLE7(FJ~>`k&BlBjMKTvd^oI-g=sa{hzjl?11i^p;zSUXM1pKRcL7
^<o8>db5VrOn(3dO@^e1Qr$(8NWOz6qW(tNdJNqTK>vg9cU4BgNA#Kgl!jj1<kWE?)eq_TH@^n)ZU-gDZ48pIX}oKV{QN@jvn+ey&)&Rc3%d8>G}^Jcd0<k<
eM_PRsvVk#=^Su{Eu0wCOi3&)$AA;`Q1bgC)U7h;osBK57A`2V2LDYmfqm5`Y%dGO!;DsyNTO5odzmsfxSh6S=nqTLDH_MGl7*%I`Bjp?qhlL}g?kpXds9=H
WJG02gETiFJULjoE>*sp1bABzsR7zw>r*Uv76cf)!9k#hv|U56bjp6kwYMBzzjoBa(+c{a9fI5k0og23)f?02x9r4aSc~)Jm^!JvGRR?34Fn}atru&;0s{3@
x5yDzsHjB5X<5g2Pa)#7l_sO7MlB?_Nw1d|8~81Y7-U?=b*HjB_!V!t3hEGcm4^#<cUln4qx6kswms3GtT+=Y#<2U#F%_dYc{XvnMxV2A6lP`-by_<397&`y
Z$fW$2hz$3)G@Ms_9@fT{wA~}m36icUpzliSEIZqC1PTC0*S7uz+hYM3eu_PC-4dUOgw-_%T&jpw257GpJ+HhX5TRhGW0v*qkt98yFq~dP~ecwgpzP>K-0EM
T)&bNe-woB$}x*TY&(2}<!<z&s81xcA7W=mb%VOBKO`)?@mg`v6~9m9rO&%28d<?O2Htk!3{YrK!gqe6`t*u^;WfbH`^nA8z6rllqz}MJ*Em=>Ia^B;0i8=s
NhJRO(Jo-?B&5yL$%1A3{q_$a?-1!iKTT~LZ=Lx1rp!~vCkCjG0|Ef#NF+vq%%=oxL%@3_-s2yJKM#2p;*H1;S`qtB_uWRkJ43`_=n(3iOQ-<brsM)_;xoSt
H7PE~)EXUx7jIBKsILE=xiQ@Q6;wQxgz`Q${UkopA|V-A&UlwffKei5{21$b8t8?@RSAWpx$}zZVS7Qne$!~xy~g3Rq=K5%32P{LAilQ(I~g^3@Fbo|O(kEp
-lJ9IKZtPtC{y_jy^h@J=*~<IgjI_t+3Y;JuXE)@4YQ=Zo#?_5-mZ}5a<kr_s=$J)rLmGmgJxiFDkzK|emA45E@akBua)lE@FAp?{j{ezK&9Brf(fg9qK&{~
LHadfYsN}NhU4+pRVCo8n96_MOxdsA((gs&y7W@DU`%6bq!K|Gh&Z=}bhWu6Q32m|b0L$PApe3>xL1{k?|yb+#nYatkI6t-$oY5$7Wuq9K`oQ(0JcdO=Jc-;
w{cZ7_kredKBECMxakx-1Dqf|GAM6@N()lnNAj(pssFmFcaZ&&v=htD5VgxX4iOWaMZnKU5PQ!23Lzsv(Tj$Stlwl`5NQ3X5IGpfVFj}ECA`^9vF3F_pQSTn
V?AM(f&UURjZpDvpvUiUyCIejhv&sC)x-M&e_2%xm!OtoU4GwpJqBMfO-*}C_I>74rL^?iS!iee315|ExAc6W`40F#qspGo%-1XBph}pCxLQ$m9&P`?xCG7G
+0dfkUVs0HjwCFq0!6qKWdo<B99;kVh}ORO-r|~(0{X?oP-59g#15Z6rDKT_?)$;FL2n>{BI>X5#rYlxAy6vKGYm(gYp+TjahlmzLxz>IQ*Aax>rUR@HWJGH
yFe=tX5EghoSR9Op)1s|amSXtj%QxkG2O;G(mS9QA%1>y(?V^Ja2xz@`>ZP#meJA^oJMQRrWEswwvJZc-`XK3Oc%bwikq@D!drHf;|Z=>!sf6xU`d4?PZYIu
E|rvmTc5N{(f<}xAh0#L5T_e~ldc)PWMnQXKnxxX*&_P6p{=@5A{V202~RMh1-Y85fSqouk@pxd*>heHVf54Bp7_>qlsg?zzqhl6eV@iT`gFyMdp%tZsQv1o
e2vxwM?31I@!w5wj0LPpJshbH=-#Az&$&HlI@hT*GWK1LkA3ReZX;9TOJw0Sbd#b#XvOSmb5jONdN)%x440ypJV`3#E{^tVQahk?k_d&LQ3`Oi%l%_BCZwLt
FO|4{)S4m|z78G6{-UMYxka<6x(P^zYCrK%-#Yq65@qif(lSBM#lCqaPFP3>Zn1FZvVSN)MkM_&TBY@udc?qg$66GKQw16pb}w9Bm~1vVB%4ymkb57;ZsrA+
COD!l>5(i^{(c1hCO}O2yb3#nk7%kAzAFg|a>u?g8<AxYSZ}GO;KSgBjxqn<i8X?2frn$W<xrt_9Jn|wI)k@zuN0a(f)x^vf5&u|eB}xbf`|QNNBsqGC1A|V
Rd{YTNs}&@Y>JE?IN}r48`jS2fGs?!0hEaWEvsv36B`Yb!NAQjl;i4anPoq|xVaW9H2JB^O`8@{<ju0|_q{~Za#GG)wI4>e-kLTiSUvufh+tVA%Y}J<7pNB5
#$497LM+gsxXG4uh@6|(UGRNr%(q6-At|!Zdk*xWFZIKs+olJnN23R$?lYWNEP33PaH)>XqNtEqe@)@gM#v36fzwiH&1@WfvMc%X1P|~)<ABw@JRtx9tb9|W
1yKgwW7%Hg#UqK>NUbNGwv`gaXlofZv?MlZ)b_$#vs2p6jeY(Vi$~MAlZwmxSc99jnlG@GLKXF-qhLmV_V$oqimIl55tZ=cf2pQ7{cvp=?gV`!;Fb$(v{>BS
KigtWRbd!JTW=h!Z@JiUvU}A3t{G}>?d`Mps?kFXg=u8ME0Bw4JrT*dzbf1xH~z4fWJ90=k|x|_!3NM@)#37~wj!jeZj=ucJX?1~;}8s;SqX^=kfeWFQOZ^Y
y`ssMRm2jNja&UQjd+^pF%|;qJrOkbz`*!z-PPg7cc_-+(8<@IMZZ&TW;AL%&_o|oVwp?Vs|E62)PJniKnW))WiA?y17rRJz|;p}?b}$)_#QSAbX=h;PSA!*
zW{3;tK#y#0p<otV3p>t!fdf8Zi1ylU(Kw{+V005vIfl4<vfUUzb)KaH&~4y(VlHA@3WKY^p~G&ir8b_eS0fTWTxu3=A^{Qw803(Sube$^=sv3c=ed$&!tY|
10LQtI&q{oeepYPM<YD=m+cc_rEjY;2<Q-&`XJT;65YTkzN4>0_%DW6i^l6`EjZNd57tb|B-wREkH^zNZj!gaJU8jd#tK7!&6ssoJ3L7AO;rB0Ci?#Pj{lGi
V^7C#h#?wUS1wd%24oULjzN6Ku4x#Mv+d_N22%%yB*5d8IG;>II$DLf?G0u|Z~|%;hKCZ7XN9JC=N{WS8u2oZ@CXcvn(ra`sHuwdA&(ABX-u2VQ+WKLS?v2B
9ijlYd&C`5xp79w8^PAjKiNK$TI(~~2|<==BhXK0bCWw_B^hFSpUd@ET4)pu-%;DLwkjKp%uq#Jf=)b}W9eUMfn401Pe9Gla6qGosO;b+yr$bV6Rlj}{1Cf`
DTq$i0ukZ0i$K4!UO7A!{>xw%$pX3?ov=lf{Jg~Z04auR_(~-5c>TQ5GBgMM|Ngh3J^%)(K+zv9@4>P9Fd3R_QcrX&M%gS8iEEogTh*(>V$XtQl1$i$sAiho
MFkv#?&O`5I~K?S6d%&P4sl9~%_;3{o^Dg&y8$u6wFH?9UxMfRiK*ju{RHAAj7zM%+?jteHRu>H{OJZ|`Zrx80E@bMqIr--|H#+l+uZ$D<{MH^!7;w_%lmDI
Mysr9VMA<a-D2(>)sxEi;wip0!xyTA|FR91PXomgF?$o73{%-=1y<{V@H}ssKL5lb18iL}*II_%32nSB#JNz@72pzmgUCBU=HTi4ingS<I*>|if_leZm1niS
Vtutz=ko=rX>TPDnPe>Wy~_HoADs-{-15IvONvn34%gDeSZpoGJ2)biZ{6{KMpzxa#p--~CxGs=m~55E9d1SPkVr8VQ7l;}vBk0e;@NZzC|GzCmIikC;aLlI
!V{*cu{cTQX9p=>9}>5miveNUP0VM_NU*RHoj~U;Zk!B?GY}2Aot5T&h!e|DM!2QVsoLQ#W)HCY2<MwPRgULZp}9$UTBS;=pY>BDUP!T%=F5~`c%QFhEkA-2
Ra>^gqy*h%-1Rf~tn7NvD(A@GS_P{JseqQ;UAHpO{PtD0f&k7{`w+>Cvm$OwK1_#_lrDr=&2#&uF8rLALwC>&c%Y{jj}R@j!LlS3D(0%ln#!^3f#=V^3as-m
#iyvKMe{>7=fi#<`zo?eeo8_ci?#$BMsa^pt4eev>_99w?k8OIg69Li=uuOruzB2Pn`-T4V66+P;`nqBUjyij70Ef(VOggMdm|-tD7oEl7$erTOYnXrH79UG
a_%54g~%ndiO1g0?2@`RZ7dgwtypN3**hkDLNb(aqY^zWJ7wo<n=Tj1EbU?_22lFHDMi{&gr13iGQyt_lhP>gV?C_~X*k8G;gCR@fdUaIA;+ToB*Urue@!86
Spv8qwII90nMS+5tYE1o8-{&0)v?-N5h}-JAj$aH>ucd#t*W0amKW9VP~Fs|(6dYPksX<-z(=PMEk`f16Sfb`0szB>?DGwVcFX1Nz8(wYx(5YGM9lU9UY*b<
8?k+3=2<(NI%*qiS!h;odx>Xr5ma)HPx?ZXEmtGDMC*=xwY3E+48bwjL@JJqc{wqS<|fHA9c)H*veR^~3&_3Z1l-AfVqp|z@A?G(?rhwTGFSw8U>VH)rKeLr
MoWc5@YwE`$}~mCe<ZJEKfIS+MVukQq%T(B;ht@D!|6WX`SMI)pb2!-eDuS?e7f_IJ1i;pa9Tm^XdoGlcWAPIE1kTFH&U@A342lHFl*rN85{qVVNu2y55IOn
Ef!&iLkj{KW;8=4kD5eH;KJ&QGFD9-p6k8GQ__*DJvdaU(8n*V<^4s{u!(9Rh;|O!ofWL>7}e2K5XQFepZ>^3zXb?yG1tTUGN+CL4h3`>tFd2B&#c0GtdKV!
4g`2-K!0UFf2U1KyxR%Ituh{!kk<m``J(dT>%P>QpsoYeC7lu_n!c3G^eovkkzw1$zY#+>sMA|XTaAa&vz_bDD)}zq1s0lBWs<EbLZvensU)OJst`rE@{V+W
UOZEyQ|^+1n-8RoP0sq2Csr;W#;QMB=cZAjdufEmS^(shp%H0asJIo{+c!kQBG-_>yr2wQ$H%`s@6FuteEU0W8vxtygXuv<R%BwvA7gjhFFJy>_&9^!!V++M
3R$StCv;%KNv_mL%su|T>M?9c<NNDp|5@<eoP(oXvCvOA;pd_sCnIK=MO~JH7ihjj4FjU)#idrUZLrtdLPJGA#|}G+9zJVGTg0#i=)#S>6KQWG#!H~{@Un}2
dFfU&(li;o94)FC3ArVWB2zi7KAUgX>fc636&%9&ON#*^aa=4SRGI(0P<!G(chOO;oz@UkJE9S98=x=8$A*!`%9s-7El)uN$|X55WQPr2zLr``++4{qLBTFG
CY2y?KH;LmB2tu)eLg8u>MoYC+W>r8I+pK2;2*}4N2$oQ7Q8XDY@^}??ZO$qG1yJLmM(h&X_07VN@l>nx89A$v956ju})2b$c2K;Iv!+aI7nm5)LhD8K|v-Y
+iUNxV$geUP-1frDxH_8fhfNv0lp38Q=P?VG+$<lr_vEi5~Tn9*#D(_^f*lkqiTf5<#av^N;5$d^B;tH?<O*FZbx6ucpMX!!_FhmQrMiSHbLHI)kU{v*r?wA
cxvjwW>xqL75~l7tAT$0^VLfL_Q_PeCR9hbeHn#E^ZapjAP44~CD#TFtA2*l7q`r9)QPEvOLJany!4Rdn=BJ$8a->8HP()>t4VzidOJ=+yTm>HT_(LGi4iaT
5vP7;UW_D(pBjVJ>W{M~4wjxQotdr*-}0QSa+Lcqtr*|ra|}rVGJgCVEbc=l3OLQ5p_IeLOWxg-n)tZoc&iVC4t<G?uJ3)Ki5D_tx9p#91vIz8*56(1A=atb
LsWl{Sd@20aq1R~)lJL~4k=9~kK9DQqMsg5xf*e3xq!N7QG>-?Q&(FFbDf@{cBLS!t;^36n6CJH=lV$`jnCrV5-e{wU)_Dj`qDjpd)wGh(Genh0fm4+&}_lq
^vbO&Pkzt7p%v9GhgM6?sn+2;n!(r120fsjZ(ENL^Z82TsX%$51tVeB2z4>%aDdi~j1P>-s1Vz<!iueCN=bWA5ab3b3M$KqVF&c<<9OFQMC;LDF=D+%&EIPv
TX!*+uch5USxIqbDquUSU#|g-MgY-avaq?uwX2@$2`O9u1PFy46g84};Zw5-tOjAP6+Y7#;6~bh#hTfsx-2ydN8=zYqZ8}5+{61hiKmj6O+9!yY)An6W~8%V
D4Z5-ec+BW&YkHhSc4ZZlYr*9imLALol=tjAfLf_TxnC-O505x(NlLu&wYk|caXY<r_56uz8BVHe1hg9k<%Rg^h$4F*J11kL2)|v6Jj^x`Ow0UlchXDVDK<)
sJU>OHLM0$k%rRoEmR`iBhYQnx?N!!k`EvoZRHy-6op@Lfx)Z`ZN?7jHD5CD=AIWk>o=RofhG7ueycgd+U!5ZhsGI4f`^DH4-rVZP-Z{fmIzNIDUZT!EpGSB
X=ipZrNu88eI!RyYam4~v0zxo2&T?CLMBB&>Ln?gP#IkH4G-;^5KEtmOB39KlHFRb=BE%^^YXCeE?DkAw>vqWJU!@08MgG?LBnF+dOF)(*>-KQ)Rye9-P7--
6%R;n<jsyDM;9+=ux!UxL-RJo9W=5?aHn9MD+Pzaj1ByTt4v&@sS!)9&9iSGS<aE8J?VLE#yKoPz=d*F3&~K1-ojo|Mg<2UDnm7X37aJ2RI*e$B4yhP$)V&T
#jcOK>cC?z(B(?tJv0`^@{2N)3x~=`hb3rYpRUB5pi9WwXyILYblYlMTVjCKsE7QaHb}-{4GKh-|2ID|$T?$rfUz+jU7rmgf#6la61rjxMjfU7x`kyKD~4ve
{W`&TTg0<Z7_BZ3iLnjWh9{q$$J<z*`+`lPPP%!YzQZvvjiH+~{`^aAeZb?sS>T?liQa{Ah2fLpg6tSIZq*1RF=lvk=@rdE#&(zTKWSdaAt*XJRO~LjV=e8{
T_i5&g1v`dSu84?eJ}&G@*XQ8*w!uXOn`<SL$nxhYB$#hWOjZCl#W8}Rz-y{pIXI_=&5=-{#U1{r^`2eE{KJo=(H7)#`-Xqn5&9N+lXnBK5v8ejnQB?y?)zY
j|m8O>qjHOs{@)`5XX=y3r1UsiCeVM4Y^RpEOgIRp{wsWPeH!>Ir)x1tj*7Cq=CJ7+KYvzuCO9w;WM%+_1uTOwsyQ<vm;qA7Z$O54!vM(>AI9N__J$RS5bc>
^&nlW(`k;TEkW?YcUcW;5-EIIDyDEg3h|<Ol0jghh;Z^o7D2_A^M-5uFXXa&=-nhY+ye)`Lr7x*A3PdKh|>ZgykWotPq=j*)icFMSN}tt=EvQUw(C9Jxo~TI
6AKfSi}((jVHv*X;>?a`5aC%wVC~>LAtW*g85Zns#3W<Bl-PY0K_EQh{bjp~WR3|h&Oom+gPYS21AzOj!-A%1I)P;1{^VI)=6jfEw7BSBM*(CZC_!(|kkNcc
YA*~7`htXZW9xmg3!nOvywgO8RMG7OtdSh$uRF-#r>Gn)X1i^s`UzIB6OLuN;86dHC_nonXYVSx{55EE!-zL!Lm&Bc2m1W~lOf3&<GdNl16~*E@MkYyu)8&o
p&=bI<OzIW)|_%<)>iJ<$<CA*9<<wmZ*@b<?ydaYg83%VwyeOp(u?w<TA>WgH?e9gfYph40YcB_{-EI-K*5_hpr?2s%hKXM>ac<;IH`S(fLIO!OTta^#~B(4
vMxM}I$lz{QUpuTBP?NP5C~VlN!7@wF4QOmuM|lW-tx`t_cXPth|meFy^4FK?au<Uk<4)4Y;69eHc$#x&7^}UX3Xqy4}&Hdm+Feo=;UR~xj65Kk7Q#5;RfhY
ArU!x0FZ=wlz%jjt4*mF?kp&ttNG^v@4|n&`~)1^sbHWiQ&bd=8Y^z#?VN~I3u;KjmhQ-0{&a8LFs`JC<v(;vdJwIQlE9W!QXBSZ1HyvcqTucL@FY-bVFYB6
e0EZ(3tMvp%qis>iEKCraY3w_&Fky+!xo4RMc4OIoqJ2;*qC|u$?jkC!wl|QWPDQRE@Tk_a+MvfH3i5jwR4o^q+qVkcF%Fu$jtrc2ycR5Ag~Lb#|#%(tD{;w
Ee;dkEW7*1%ZKmW1gdLy7oj#{!QCp$)sbAOZ5ml5>oOgj9VyKJxZj0YvaYHMFh$|`$w(yB2fl0(qh?NE)B^JIO1js>g`N_H4r;tdaK)a1V|jx&4#Pk$b)YV#
p{B+eJDdv$)iPU>9pj=N#ZVK54&GPSniE|$00R_~?kS{96x0yqFWKr;{V9xZu|Ac7#HUmiKr3C6nc*5*N{DXfjpL+ui>MXH4ax?eU>K*?t-UUeZ^0aqBNfe-
YqQ*PJ}Z33KF~M``G_fVQy(K)YW?_#Cv|;i@3~+EticVWSjFdk_vQqLh7QikyF#7`HXV?i1K$`U?F&|BMiO9Bvj!8HGFWPifaw6W4SNdhWM+h+AJuVQ`XE6@
R#E>A{8BjBZ1mC1c&x;L0I9jDINeSYTF)e$?*Zi(L91$a@3}q2Z_iq=*s3BkZHPnP{LtRY_E=_t$XDD$kP8eq#>S;-_l>oX2TRk=3b9izw#A?3L|xVW3r-Q|
U&MTSy60Z^ne*>HSaeR;u96Js7$rpc2QAW6^9vOKUXs53;Kb}MlW5EgB=Er!*&3Ey1A=$sM9T*d_5#^Gux)a_&FmLC2tN^mk9T*{)!PmC9E4S`;&PkbxSID8
(5I0qWC?$Q`uTPdT4wvI3l82A9#X{W3g3r|fhxd4{h2q=bGf=QRIs2fyxh_R><lrOv8?c}Sek(3dnM$K|Gp^|MFgIbA$~qA%=z2GXhvnLF6jhAlXwsuFhrpm
b~ZsJGiQS`uzwcy>C{6gHnejd$W;`jVy$PZ0{bwU!SaoW^9g1=ly@`)a2mVi2ungbfh)BD`{UuvG87jzozhCLWS{gx%z;0lR^OCLM4`wP#I+(DlbYk<6lu->
#qIDEK+j8g7X%?vsN{9w8GGse)B;SM%LcB#83nHBGR+b&CjH5q)Jo{`&MSxKlKA!mR*<YcgqEDn&TyxyTP@7ImoS~y!F-oKYo5&yIf5hd-Z(wL{{}3TdN#-L
x}Ii8l<Z{1WwLHuy};a<eNlphYyRx@msOrrBDOsPo}3-~>+5X6oWwpHfAhLSRxiS|IbD}gAoI@Aklh(}ibzwHm7(EWbpY_h-kECVG}*-fZa9AoRww7=HUV}5
fZPPqxai&H;m1xBc!xp{eY}L}tj7zzLVI`Z>ATB|`Ev!As@c*smt%IDaia~fp<i@z=I-}VJt%2)xLOrZX)URz#SvrC%I0MSA>KA1Wi;5&0t8n8Nsfm0yOb#R
G0s9?>5AgU<{vrSzCPH_r1YKG<?twRFSR>8pZ%i6;t*z(`8KfQG9X#0+9Zpl_p-biePt{JAZQpj?oO6~c2<vx*wFQMom4pbSERWfw!UPrESmJ?;4sTY#-T>>
4i315LzBzmexhtYC^u9Y=f`;w2%IVFmOalV7Cs5Tff5`vIRL67t+RmU=E*ijDhMkBFG!R`%Q3Rkurm|?=3c{yDOt^BT^~1;z~2?MPjUZRqmsd+qj&2rz8>Db
<bdqg%ytpE$<mw`Klw7b!e~by3RRqhnvd3@%<wOB!(IgGhrppjU*w%^JkR&x{+@q1Hp7jrv-AyxuTPRft)d0ia%5&3%cmmWtp4RQ#t$YGOrs~qtU$V5Vjn(q
x!9|}Cx+Y5Ixi~>wXytLuS^H7my?ef-|z*tWAT8CA{}?i?y5QP&-{aCM(X1C{9hw@E}Nd!J2eHNW8V_eCgCKU>7PRt3Jz~tKpda*Ouh+spdCmOh>A`W<UGDf
`;|(^*DZA1DH3*}oCsfQ$2H#X%X|n*81ea!H@a5Kr;xL)$3#k2+j>q6Qeh`VQb0BM*-EFvLu$)9<FcP0_%)4$)pA{BM~8lG_9(Fad$=pKxuPT^LQCe6A&r%A
CAmS67WwpdJkbX)UrZZ&&NnRQ8{L&Rcu~&nehEW<c!5+F$~penJlkX*!~#UAnitZlq!CUi`lYg~<&>#FnRH;m_OHbmyM|9q`N7{B)~d8-K<`2S;?Vch$fUoH
q7IvK+GB$qZvl*=9j~NyO1t09A(lGUb#GzjkAgC4u;wY`7a}Z$8DX#*v>$s%>pPr2watrM7IXr@&@VViCjuPdG9nq0RQaj5C@kv}22wu#EZULr+B@f6?1ENt
6{ua<m8*iDqas@TIxt9y`y5m)U|?|^ntG<Z{+7ZKYL&+7BS1M_^=!+7lWn{DRZ`(DyeqGfI~V1J9D#Ql?iYwXEsdX_B*wX(YBWLLj{szW?{{Z)=T^7+UAndc
`h%y+Uem~&b<#!S_Th`Lf`M?#F`mRzUb53)=LRF_9O`bWZo6KI?-ZY^pan-qd(dw8x;P2Z!y)6~^=lCfAduQJ3hAP1m7QA#2>Squ!@*RB3-7&@wvCi$OJa%?
EwUD;Gg=_`UgU*}P7(++y-Vt)?*f{g`QxZMNJ?AmYR`3QC=L9j!Zw%(=V%e`(F4?(qJTwfc(5Yw*{8?I#?Q9^(r^O3DD140pCKE!A1=V;&22ntD9)I$5CxiJ
b0X$gLwfC(1O0j#C|nkNj9};5yBpOcdA@wj6^ejUm_3A^Pu5lq{L%~0aCudP<=FS@LAECe4(X}u+!ZO_;$aujz%{yX_A-l!SamkNLM90d_phkCClZS=)khnE
P;xLUByDQ{Ed>C^@0Qe7Pm0^T{YBuvEGEJ(0&`Dw=4iuzCek$uDNppQ94h(2fWq=zGiUFeSoC=M=>CwAhm9gmb%$LZOc;6XC5o)AiW{&>-TYD{dG062_2BS!
9RCD?8%MNrC&F^P+ZAEXF66t|d)o3{Dfd6MBmdt)r)}J`bUfT#Zx1-+EE|we6a*-Lcn&`5-!v1JikeHiP7T}#fULie*I{*)OuNBE$%~lqcnaz{u+ZN}Gd!Gc
5|m-7o4;`^xmVH@?{Nb7d{Hw2H~mqi@1nuiLO(9*pB8qZpbT1XCMIF6AfFQaxA_h3tBg&k`HOT{c8q0P`5rE}{$srlch;QPoHHv^k?n0N8@xQX5dI}{N-464
kJ}7;U`p*RnrCVulk~7m6X-)V+c-3sGoh+Ty~UC`uSgv$*#9Jk2`ldM!j>fiE?eWHdnCGgT<0s?qjWexfySPPSd$L#*|yquV&OxS`o-x{d)UR&Qb)J#bxjO{
<vw%AmaG92zQcXl3bCG`3H2*xg$f0lmf>*w0*(dU!k(&qpAm~JBS5Q~XB5;;61~!p?Za<2R;P#y6dEMtjF_Xwt|(-_BXE;_0(NhBKCSfMWi2ACvn0l!rB3j1
^veQpdK4va*z93bpV1O0Vv&oO@+WmU+<3PmPw=dBICK(g(S~TGWCN~Iq@B}Mw3D)e)hcIS^jq;$J`>rsoi1<$s=k3BXbC1>&}o;f_>l}d?{><W{qq@4b`c8x
Ki*)pH?&@w$GFX4HB74d?-XjPa9mzP8H865$&xeSt$$;}U;8Brk@8!K8lavg@`1V`!Fug-JbHDdxZ%U2$?mtA!hGK$7(n4X9Yoq{){+>mL;M=@7@JxszA{zN
QTP)#dl2XAAFpuFTL)MWYAU<kyxw63<Z^2Yo&c^c`nZh}PvRBT4Fh79gN~n@DNq4C=I9JHmFDixZdq-2F^-9oyidLU4~A3Yzj3*qEu4AS&*H8Kt6uKb!QOa}
OwJZ`YK06Vj2VuO{?v?Xh|vBE56lj4S(6@&i!|k*L)2t=ZVusU#Ax~=QKu#3UMZ(mAW3aA0pPlH7#~~}x##ox_344z5>X38&MIsTpSMnpAHe?)+ssZy@M?zw
uj@6fP>}XW9WZ^$#}#U={#&)4ixU$|6G$sf5<E{S_!`QOo8xD#qi_4Y6%`By=~{?uJH$|QoGG<uC6Stzyt|*ZDlhSTQ(Q`+d7)R5U=Tg!5V=cfXo)UK2q0)3
>KfF+ES0^9BWiQhwb>fKF;EQ&0JY|guYm#9ldN9d=!&3v9sXh{AkjNMu|G6tZ(_PK+@StAV;<L;!%&z#Z95#-yIVZ7n`@+0D?GFjOi?N69LEKA|H~J6^h=3X
sSjJ;6oVUMQG^0tT$c(*bf{GuiyTi~1qSxl>6vDVGH|W1?zaCG_zpTOef>6(y?~zjts462d{;!=8n~t|W(d@Ww;8}T1k&7NtP1E;(T&J{QDXBdhOb?)ISwUl
Hu7fXaKa|of=GEW*F$6%0(kkS%ybsf`%$IL)Li*_Ji^Uv`!_$TXExRUpeeKDQFm|6M*{3cLxz=MD4QplCUdROQ<^bYjml<wsYD~uw`2H;x2T#=3)!pg3nYaS
I*^6~?&x40BrXv8&~c7?4xJBIF<&@JDS6K!>m0O>Bss`1n{)C8?|SN`r1rFww!v4}83{0whDKyI_MV=@ZCj;+#qJ4NJt92{m6g4$HG4>)T<g-<Uc+-ahnkJG
D#gON?ulgm{Hh~I_k3hRVlke$P6riGB}wHUm?`33>@C`rOKgC1XO91WgjX{C+-YF5Cb1s7`4fy>5a3--;CV>V1gW3Cxqt#g`O$GwZTjM}Nx)0xc%8Rra&GY|
vf+cKZtg<tl95aAouax3v`xgd(vd%6DYf^HmVO8OP}(Q3TExigVg>>+X%0dU3CV&1lLL}kY3`-t;a?!rA;k9&0@3q~COwzSJbWT)jT@P2c{DF~Bu!tPV@@)p
5Lu|UdDWW>&#vK-ecL!fG{UhL2_FSub<?M<uV~8Hp!|4sam7}u0g56ZQ=l+TV2|i^BBLE+s}{1>6_K4Yh(<Etk*<5o`4SGGZ%QYuqqXIU$};F$GR1<>7Hnp4
YL&3oUh!w?hlyKmQi6Zb$<qL2999ey5?4?wJ%fRu|Djk-d$0Z8G}=kd$SjEiSMiOV3Q+qf^i^Ame3o{oVJgS5+G4`P0Vw1b6j<(4>v>7BN|V(oBXi<Wne6vb
VURAw1iUa*;gW|V95lFX8<LrmpDS<rTE+N!Le)^hG&6PY^}z(AJLpK5VSSrv$Oqkg{vo|FD>yAn_{TzL<uphWRn^yla1$>zipYq8zUoNv1lw!=mY2tsm)#BK
%VP~WfgcKC&=TvV&u+%a6P7{tJ8s4Q);sCua>K{c<>C_$4Nw1hq8waJEp5loj$4fY0mr7CF1tm}apLmr<WTy%*S#FZ!YH2iSRdnVk6PrRT$|_U<uT<K_EG)?
5DPj<T%EJhX*t*_aM>QPVgNA?_=Ur>5Bpm%gF>xiGo$W4GVkIPeB>tcRKV`Q^7$_o6@}?RDIfwJb}->z8%f4^XRUaDhu?Cz8@p8wxa=W_JI$0hHNBug_@`kg
;)uv1m0^5wtmqRHGH%AkmogRcsE&FjnzoQopc(S)#791}C>?Zi;Ym^-t=r!NesJTJBS3xY<=s0La;52$_6;#W(_#vvzPw79@sr-yWHu@jN<Ba#ig!UgyQm~1
BgM5b&?t*8f=~44i(}2)%DQi!jCsHIrV)rCY8Mz-5-VVhEa54^I(U#`?<OQV3r<!%&75%z7*_F{5+?kqdYy&Rs`7RU7Ryp*4#!>44n-*Cb3_XQ{XD1u62=!S
L@wAkHHW8jO(qshus^$1OQOoa&?N?~*P(p(^Ha|gs@nDq0#5pA*Na&aFUj6l+QNNkV(ri-f~_7?`%O_M>DT!WPFM`IA8y>DWLyOS*|}9t7TP7sr%&@zpPuLs
;NLt_#cVfUQ;IbAklEFLwCrF`+{5ltP?a7`;Tz$nM)!qAB>auC(#a<`1(fLRiotit$AiGqUUj553eGtVAQWn68W>+L!L_`mWc?YWuyC>nos?bQT8Br_dt-ww
5ehFcK6P8lg1l!vQ;fjhK}njU6)8U#>xe?&DOdu3Ju`}d96D{n4l}2%32n^%MEcGu4eb%<^D?#jfQ!ymB+8=t5X^KOZ?x<4XB++CBbg1|IYA%Hw<{<&a#VS#
=t#5LJ)*PZzn6FIC}esnk+Xn91<_x>KO-cH->GmNHLI46ZS_*<C7wMZmoHK@ZKvhFmd$kw1oa3jy9%=Vt|tdqLT#cqG&A(h4l_#spsEfZ6Obz}(oHxri!?i_
=e$d8m_23H;d4Fv<H#aR0M<UrgAKS*yHu&HAF52Tp}0uiTC~GIk(_x&?Wl4M-QxS;i#<!~oc`EIDv|8*9{9_{w6pqWxa*bJZA&c(J_^YN=t-B-!8vv8mJfz@
zk}7b^?rN)RSFjj52tU}^kwEfwUsHjQ#Hu&RV|*Jfl&FQI8#C(9W`aBLrtuo?!~-AQiscmMw7lz<}?$nad0G2L>Nc5a=UIr%34l5YXotBZ`56jpJBVu+%&gJ
y490^B&#v|!a)MRS>VNGe%|T#cSd1zsow?_3eySi6zE|--s%T;9s5zvV+Z#v;``x4-%Ft_Yo}yJJo&yaVXG0{;AekW=vN)*`h%!DhzroR<JUqdiLUTBLmEt|
og`f`z#Sb3>3NH!hkGQd5c%zd6mm9HJF!W5&$5SbAJ%{*6Ar&>1fx-OiQ>>h*_?C*pi+<2W!O@l4b$@Ca=zH?_Qll$uq^>rQ4^!e_1+pshi3>;goRWG&ES);
NwO%_b`8|9TTL@h-x7ab_T^OCTf0>(t~if$t$|p7s>ds7MfGWKB@W!@GAONbh>fn)mQyJ+&Kh%pTyb+uV{>9`f_;CniJ}-H1n5X@Cem@1n&zbml@@(OM&66q
eCV6l1PR&zs~NtX1DetoOGRB_%<k~}Zj9F$V#P}Dt2$0G<8*=JZL!k636@ckS@uR5Z4-EyrX>JOE&sg=vmI8;0Vd0e>Fte2C*h^d5i;FZ&Y#!yr_;1*=F&+^
XFUq6i5>xPF2W39cfB%>N_&xuE*>6Z&W22^mRZY!)Vrv8z5;X_v|I*nG+S8s1L=TC2sFi~@86j(!lnb;k}MDu&d%8eq!QTPc%3&vk!+<jp>oeJ6w~|gY7m+`
sakcv9^149Y712h;w{*kQbhKIqK_K2ovZCksl$f~aER)_95GPb2yk*_bEsG!7lhm)8_1c$1lSm;#S@5^4m@&Pva>S(Xw%z3oWw5Ji2k&0@G7Kc=0?Fb?J<-U
lmft5!6jA>n@cLlE<J)|`$Szdh>tqtxP_%G%JPv=<=Qc1=4^-WyV<F!OAvad!$xR+MDTY(?hVFI_T%3#LZLVHgV^C6^8*St+Vl?EC)J#$m+RmVX77A}QytqY
RQ~WTgxd{nny3}kM-~!OeQQnGP=?#SQ6~oLhj<fJYsUa$hawFixH4Nb5p&LU{5rGc+mck+&66kg(O8W2K|`bu!UIq%=Hbzjpo9vPt}>(N$<{0lZ)gX!DG=@@
F*sGG_lhPa_euKjtxf+c!qXsT-Zc$3%0sbnT61HF9|Q}=WnpEUeek(wIriI1O#D%sG()ucJ`Pm0mlfmlv31U&aR&(&m$+AAkc>0aBHWC;RU(M;Z)im8&MY$~
RqV0L@ezq|VOQnV(F!iR6mAb_ELCnE%23Je>In(e=ecQEv*$?)J)V6>p#*AsGL4NHPCR1~<tmIU(kSqYHA2jj#El0WYVov~bv=1e2O?o$5cLKbYUIF~!56}@
wc?;3Rgz%~W3iST#W~c_g7KccJ?mJKkKUg4lA}H--HSiv`l`v35^)l?KG@h}0J>FvGcSN7r7!`^>!9}iXAhx?W`xVUj_aZx>Hfi>74Q4+i007>A9S_3a+}Ky
vYiEAc=^ki@YFp!QKrxs0k)<nPjxM)59#_<E83@ZsY%C+cIsg^@}q*?x4g;7U-KKxIpM8ra0LqLKu_Aj6@Q<eN&8h$rRjC@r2@fkXl_o|bUE><kZ9i1!8W0h
pqeiF5KdXCA-wuKYO9-6H$**&kr@XSdi%Y`Bv+LmU|;f8erVVI1mRO(9dwiiM9DvcFnYaZCVjMeP28=q>WY86z1m3Sq<S5!Dt>8Ubso(oDoi6tLr9RSBr}0k
u@&*l@w?$APt(jv=lUNEwLLKH%qC}V^LH#TP40I_kYW0#sWb;{-h|AW%re|86Aq_D5<#!RSRMps4A=Joaj@ki*&ABX&fxFQ{aX=Fw3n&IE$nljcuTkZZK~44
2SBLmN+iO$Q7(5Lq{o~eS~M2F5d5wQIL<Z)J}Vq#i!eAFXSZ!U*TevvLinZd5@^&lO|4t5K2niMj8|mh=yLx_F!g}*YwykB=25*}@Ul>l5--E7QCW36#Topp
dlb_?AMIaIU#@=;l&-idy*qm3&-rdX=(X<7-Lwh&Bprd>jA1`eQANEQIA+-9q0<)xYBX>nf_5kmZdCbHhi>3pm!k|u6x_^^9H`6V^`q8_B@oqPG{v;%NbA*>
o}Nv={EV*?y5h~$!YV@lS<{%e>`zaO3(|T~idWDmp_y<S{QGWo=U`5z+A+wt<~ErRK%NWSORH$47XZXV0$IShtM`9>ovh{~Lk9cKpk>^5<l4onFse(;kce4U
(nYX_$xddWmLkX7lr*|eUl*s#?#HF28(fq{qFK%24o$*vPsOlm#NwIVSA9+;1@sqlxL*#{4jW6L&>Z=@R+US4wNb~Wu2l-0aCwbq9n-_iGhT>CD0mq!)O|r1
lA^J@A4I$`LW$o2oC)__Q<@ENFymxUkgyhG5?Ripo><mGJOgiuj4Ufvz&;Y`+%mTTpqr<TGT=0?-9d_fc9x2=rdu(g-}UD=Ac#@aDHX+0C`5hD{2zlc-ehID
dYt-bS-i9^CVUY~lkY3(_;(SN2b7u)*vY$MEslPQ^B%Wn(fBD4aoA^fF6Hi?KS!UW{_?>OFJ|P{R^yjTr{4tmv3APvylUh8o71$#eZ+hr%XB|M6V(Lz#%6t`
n5P~sA$Yfo0;Y)3!8A&tlW8SS?El(TvD(-}T@;Ie4|;aSOtI-6+^JG^(L@#~3*hltvTFh5c?*UAheFEe*pvMwqy;!VS?N~R865lIeIKl-OjPAvT~B-pcGn`0
B>VYNay^y7HV!pSFx=9;F9b~l&IOmhyxB`f=-H29&zpX@-xD;39QBK1nzgIYk(0^<Oob=!TpAJSeijYJ;+JD<b~gcF#{(z$GC;rh`MtYs({9>`{7dQG{cslu
gJIIu*SnjY-D71b&f$*h=n25hp%5<dLkBfy6(IF>B0$G#AT2MwIPKc6<7jTQ_hqsl;`cZShpw*$;F4R9b5pzxPOTg-q?Ubaj}+Ga)NU>7iT^MvwBy5ge@~+E
%9Tt4XQ8OTo<WtbUDCICdFa@;Z;v8YngV#9MD2Q12%WP;er48<3jZ@mptu7{qQPKMsk=baEjxkjKkNn$e7%W8?e7U0MzhbYDV5}GB#_6UV7yD$l8(WbZypib
X;7={47P9eBf|t%W`(OosSQzdz}*zrG`T5)JdMhR&YMi(haFj2AavDU9$e6^U&-^bU+O=0)>(OS@#4d~cpIQ=Ekh&Dan|-k;WVBbRIxP&|HFErTmn=2p!cQe
CoXSL!GKuetIfYSW$oj;=!RhFUPB>I%VH(=2Y_DdatQOgL^sQWd1*-Qt-(Y0E(^O1OxEc9PJdgjCp(@t9?e=qH=JoI@TZ&BI_wv+*!l9AyfPWrk@M>I{{_Bi
+70fLKnA=R!@{b0;U1B4Yf%>X4sPEc&t?)^-xPNXybc~cN*90Jt3ZJhr_DeZdZH(y(k_{p#UlItKMQA$rET~_#gC0B?54N`^rV^oXWrX#C8XyCYP|`kSk&wB
eWYc#rfgsg5Rjivc(m<N2AJ|EEMeAsAAA+7!Fpt=qS>!#TX1~QY+sN$7(NJMJB}6+qhYs_z4FbGn1AB+OZ#aMr8&g^MqZ?T_yjm}Vwpm07G`CzW*7lenF%o5
gCeHqTGg=QMDb<&cEeHBB>(jWj=EK{8?tUSCkGs<s{!%mr2I;d8NN~}hWU>W^mkEnxyjzT!;FH}TthP_2i+nqXj9waL;$gLU#?eN!I^4yI^d92X3K5{82vw9
xA-+y+neGnPHX_hXelHeIDnj}I{&fdX+s?(7}?VdlW#n{-j7SR*gaEed;mZcOM5Pbs5@E!1JWq<wn0AN)yvBNtF@k*m+K}ZpLVh0O#YcBh_o1|l68d`S7%uU
6wRSNWerIM{zLrp=QX9Vt<+(yOtW?Vh%%34S`K1S??iA7H3cl3uWr_rwj}h~LK?5bkG53*5rcqD{1}P!0y`QyJm!vU2a#cb)Ud~8snhm8f|T@z)?Z$`afs?p
Ze}|XzH;h`J_b@jv^%%~N6AQghu7^v05p6!?LCiv>J1}gt(f{>b&hA`9rE=Fk0Y6B6t9Bq?@}xuu2N)3UNqWC2nRYEoRP<`bEn<nGE8>mTdgqBi;fJiJqw5E
N8R}=X3XB9v_%KPl$5*}G1|@`uYKSNl$Z|!%st}GxKN{`+NYQ=NEyMR3K@S!;H7c3<T86BbtBOI?-7x`!WiF=%QCopN;`-xRuy-l#UiuS)-}Og_r7tAdpEc4
!>Wk(P=5Fez_5*TEaI0DyVk%qlUCNnFfIzeYUB_%4D@)C^Z6YIr_ryVbg3UUXIAOiV#(x*+k`VPO8p7sHpT|i>qM)t-rA0~0#6&nrdH*QID8btnLdO&N_I4+
Kjvpub@z9>g1Jm1?W_p&+gUv7M406)Z-rg1`>OW!Z-cmy5738yy6Sj;t^U;R+hDY+GbbX6zcnV@pmH^*1a=SCgL56TEPljd4?_fgBp}dTnXUqCH1Q3$SWByt
Dg&rU$@Q51iy=2-`XKc!ieF<_=+k_Mw((+vJV5;}6x#TdT=7xxuNGD84u#$OltA4pG^W=SblS8=&eIc(hM$EO(!)m}^PRl+cDHl@vgCNNHR&k;iP(}1eGB#5
s}=>0;fO~l>`<dRx!kDh)`ewHbMKIV${1*tu9pmYesWB9k0z~Rw)0>5MhlK4940E9m&k^3pRIa4q46O^{VY=w2^3RaWH2IhJl&Id+CiT7!du=LauXqkV)d$U
1=*Jv;Q{l6qU=)(tkAwexef?QzH+!I?jgm(=H3a3z$9OUxvc4^idonyGWdvRuXXXH!g>sj(EwXVNf2NY9y9-5P0?acgiu$KoiUr7T)+kKdNUwYTz(Z+)$cAw
0uR_>`#=m=@VJ%p%G2T7Sa^8SJQX1$)_gG$XXL3AkSyUx3Fpu$GsE0y>qN+DEKpndp?0X(+Ia|f?1+oS6Nj9sY!D6pi7gZ*G*p^U?MUI^XZr})xr;QGF$vi-
Vh%+M3ejbIEqYwMfG!~AxVBHO1GP*%HbrLE^IRceVD;XG`jc%=i2Q-1UDKYs7nMj;1-L{s2~?*-*M_akVQ)p})k)s4jzbs++yh84zQ)>XYVn!^pkKzT9WQjl
0*S_&fOOdBkhqMw`3`Z6X8km@6+9TNj6;l^@JQ165uEb0!USAg4LMA3nqis5qlSj!r9KXwoFME$T###-HJJw>VsS$D+VsRDfukvyWX9phUJ{Hn5I?zC)oDHg
h2oK#i-^2z8PI8gF>Ii80bQELKx!)VECFXIc=|s7HY&$0AZ+LW-?jzD?l?g#d)DMT(sBwn!-sy<PqQ|g-A1Q&8*|T5EAKoRD<lV*W|z-1n?T_&@NToDPJmB_
3{s-FOCZa7ha355J7db5VhGwL@BRjMrcR2z`X+BT0unVXTi2pN_7cGX)@!-=*kqcV|C4fhKNOtPL$G`uLI>nc`o9v?VEpo!$`&+mf$4(d(}Kg(>%C5|`$R*k
Xhwe~f$zccB5JXOyN3QofmHW$ThUq~c&kmdqF99zneXae5!X)d##D{FpwF>xcAy~*hU0*ydGnD`2db=FN$jPbNz{Jl6zf>e-<7SkTjZCLRCul%@)HH!X69`%
{6qy$J%yK21BEdV6uF9kxJb~C+w~OtK<c>pxdp}zz4CVz&DMY+LJotyxLF#c&dk7jXHMT@0M+0}YAt*h+Gf@Anx4fBQ8zgMk1OJuq740_dki;qa~KY4(zk6D
JZjplt>4B#R&4zueDLUT3tw>aOo(r8XkTb}zwLZJyZbY*`=p`~lr@jW%_yb|3*wgZGoh;!zaY%Sea0Z^q7Y<Uk=|d3_}~y<=hVPH<8}`SgW6CvAWU`stZMNv
xzh4y98EiKAYwu~9tkQ-R-%5{NgpD|&O2nC`61yY7mu5KE9jLZzNHA2Q5Q1x_3-T+&R;$xD9)hBv@-S0P_3Q@F)@N{I{6#^%jDu&GoGYrKYfj>@ywSdh%|qx
M`OXofmMJTv|v(do95)wVr)<_{>H^Jh^mN@AKqjY)%7_IwpSWOde$=I3^B*{ypJ;<L`C~z(}A(x6Fj0aVUUnCd}!rm`LDHx{TI_GRRpHOeJHOCUx<zRd-0yv
fAhrBL<q?i${UCYbFyTYW>kF$Ql_D^hGDnSG38jMY-wnFC!CU8GWyShKlIYm(a|*E5@%=`j^ZyeFpI->FY=$bn_G&`lj~Eu3|#q$2Zo2SxrZL1mzZli$(C8Z
0UWxDq}=fH=MDx?hD1uMN|L>;2CEDw79@|F&X`1zDK&(hjf?i{4*fuaE(#GK^-xSvADIK;-+AM?_REAVol?G_W}nF$qv=vsjgxvF$H^VX$~tKq!_0B{WOm0F
0wu;L95lxUnk};jG^uzNnJzBhAq^|&?t#nH*^Daw&(&23Q>gXofU^(fhdY8%rTy?25Ja1hUHZCMJ2yx}HmaKA9HcA>T^RrY7SR#rY`pNQo}k{F2SrylJ6=1C
m5=pNBMXC3HBuT?3DBrF?$<<cL`Mcn)zJX;VwP(Ko{mS5N*>~ni<qVK95%BHz~_3d=7fC(2(jd|H_5=O)nVoPZmgQX`PK^o=euc$-5L>6UM_A~33>Y{`-j9@
YJpA=GQ(txw08%l9jKmIjbcf^_l)_2%%s~>F81Js<whn>O9+X@Skg<<=#txfG8s~F!@PVYI<>~0&yb19ok;5ExWcAo1UKi=E5-~Ocq=m^ALTdaOK4)K`tn*j
!e}Q#M4;_e2eiicXXb*soq#FmiiZH&rNG$fja!5A*<>xcq+APy!qglcr4okE?06uBAXS9|Qwr8Wn^BXhSi76@)@Yu5Uj=q_jWZXwf$65*TC(secETwJ#dPIL
4RH`?o=ckKyvYjDMj*NlhfRL^K95N}3nyFtzDzrQyU7?vfSisFLzdV{6UHq)%D#>!{2M&^s5LYvqhFR*>-Vxra5qz+W^k4|2qdWmJr0_r{FDqnnFu_&EP6gg
&`xN>icPu8`>rLxU#QUTsh0Z$6yO!PNJSgTymvjy$<*1Y4z$1I-t|t*aB0dUZs63?L9!({1#O--jH&L9rC-!Ls5gor+skxfzh{pmrCSPm3XsrtWBO`M<XI7u
Rc$PF^~^|GPIC)bQht%mlytxvkWVm?Kna#|QLNF3+cs)YMXc+Iuf!<Wrtgsz*`$e2EW~Ey6j1zP)ix^+%q&zcSI4aN7}QI|jt)SO$!jlM8gW@y!tE@NHinZ&
IqpRDn=j#=ajxLEPAvwN20Y*BZxC2ziy(5TYY@i>A;f(FY>!+>F1+*MU}Y+dn*XTzi>g>#*!+)Gx0N;9GP;o-5?f?OMml3ia<$s}#B{6k1f_VK$->`XKAM5P
1Kg>YNPdr7Tk`dp0(b`pQGEcJ5~A~UbV`Qu-8MKWIUM1|bI9Fv0`)5K0{AQ@vziJMmdsOy07uE$r`sx`{4A)F*vFv*YGPgcR!F*xuG76b%+BryeFig025Co6
=1IR*b*+UK4-@k0iG(<uTF=iBpiU6|5<GfeEUESB6YHvdH&Ed;UbaYL3YR;5`3ibe)Oca|BRg-(_q-Nh{?}zf!!ZZyg4(2}^}4Lh#)}#*g;hx&&CE*(@MHoH
m!ex142nP9hYtaf<Sjo+Fy`e^-INo+!@EDx`ZVj^OmlZG|FoCVUuwLb^vTcE<YrqBz&*jjB;NLA<E~#(z7a>RS*cpypxL}sta>%Bntz@Pn{Q7L`2CI(d)efv
9HT^Uno$BRK?)x~=dML8fowL5*|x%3S;t(k3xsZdMt76&MKfXLwi4ekqat6P%;fU6Gz=7LtxE=**LXAg?4ITRid1H^NL9yE`~XrCc8Lp~)Nh*{ZU`9rlTd7I
kJc2VK19x<u?gABT=%k^A@d{7tyesTZ7#*)wW+IO__S6iR^)|>*DI9N-3MM%O8d}4R{!E@g$<6mkYVvKdwWqHakmA@9y!2$ln4rU5S5`foS}UAqutYghIV6<
gYjI79=RlPdD)82%lVkgjA9=>ACHhKtIuU2I%hki&Of)m3CftV$ETht*+xEIF$7Y$95NpeBm&5wgvz|mB;R@C&J1A((!3t5ZvkE?$5?p0p5J0cI`p4~Cz^8=
gucWk?a5f8#Sv+*+{wzhj~GZKCY51PW5G*_AcR=@+X;cWt{+Iee9u$fN(9X%nA7i_ff8%$`_fJ^tYzwjb&+V~6Pj+Fu>FtT2qJZ>ZF0_NnA4^Styy|6UiuV4
tk(i3Lp0!b`|4en9_|X6qc>i@*#$kb$5l|6F}GoNF}`F%2AP)48m~{>5B=av@~Zy~Edutc!h5(={{J^8H0P-`vD7S}75ucvZZZ4vx`%viZIR{<*?u?jlGVj%
+B=O-(B5~sk(Z6rRq9J_xk+Nc9szyAq}lIG$(6?!yK0Jop+uLq=Smi|J0bFamsaz4UNz=&(nb?ME^v!*YkO@5#GSq7v7e^;XSrO*b{k&YIVT<O{<PYV(=X-A
vh%V#hpluG5kr`4^ZmLj(7g-|_ufZ5ZYlLK{aSWV`m3#O+Ly@+D_Kra^U&z<OJ#ZeScMN<<+FN5XneYg00J}DWG-0CW>rZT(NIjqY+aDEa?qMcdf?x*D?3#Q
XJ49_<|gvXAxyDDDE6(3#Ud}l*GrI@FP6_<@F?&O*#tYJe_4BH3&Uel)N6*^px?t~IImy^2hO;sI5bPGUPB!uOM-!AX7U<wl*=6X-=f$gie$+}h`;lE2gtx%
N}Q2Z?{wvVb`4*86N<c`b}XN`$XdegpR~9*;x+(RrYY*$xfMdYU(q=ii@gqJ3m}m3tm9jQlr9E$`cUgh3KVYOjs&5T(;Vg=Y6=SW32h1`X5B+2n`m%O%c?v0
fQQaS76^Dm)A9oy3-K`1krRLsiOu-fv9N8c)w7Gi1=Skp2n+(a*>O0uIR=ey<3|_L;ty`sZI6Edj1;xM=Ua5KSxWI!kd(!+fPGa8@^E3pUu~(HYvZ?F4Xn_3
GMF&(%%wXJq<5WB<Av@QpmT$HT<>dEeIH?ku=~sEU<qB3kk$RPbSY9Z9HXaQM>)>ML|9<w>Fd-#rZ1by6}Q5j63tO%BinVGhhJTKUzJ*4N-NZ@rU-<t7rtBk
^wqmM2lK6=d=y$KaAD-(Eix7|xS+;GNT}X4fROD0=YFSfx)6ABZ!P_LuO@wMH^Y!Or(r44q^;*7M=!AU=*jk~w$u#&rZJI@qp6eIZC6^R<5~j7BfEPg`Q~Tz
QGsf@Ej<HkJX}<}kE+E#f*w~HcQKCgsvq$&qZ#Z^k8Bl`-z-ywjVX&{+ow>Jb`=j1l3UAyZH6(nc(Epu$vT}WWl|LR-`$}@Cq}fkM!A?SqBTG+zYF8O-5l?h
!3B$#D5XS&a(N;NF+5d%^|mfmTf30>y=+a#*0YGG%OXSVQhyGv55=*R?KO;A{DwZCI(LX7gy_|Bxp($<tx>T2&~*8sq1=anyjs&eFUH(6i8e=gxz1wC70hO+
knN!pqETlN@gVj0HZi=%gO)7>m@l0;GmZkUOwKFXy5jU%UN*&jml43yi{v5oN~1FgoX`My$J~+RJ$(P17;j9-F-+)oZ5}`EdMY?%*)8LHT`7Msi3NaV&jO>r
`ZKkX0~$Nye8-u<K{@oG3T@iH-Ypq08tLL?(yyt;*BV9S2=7o-Ar=KxCxe2ZIt#DE$;QaJPo19X!zm@{y|CLpd11vjIJS~h3FL)I$2ZVZgPJ(QFp9vTmfm0+
HwuJ2ckIT6#FKsz9wVrE;I>Dti02Q%yjMw9uCFjXxE=f!ZM%q?GjD=kK<FiU3%7cyyj~sm(2DLlckT0vr~}`ZIJ!AWvmjSgw$T#4JvHpzK41$`D=R>{JW%=h
{dh6`aw0M(?+XbCiSMnD{NHW0zd5NlO4CV`H;IBY`3%meaznV}tB$j!fvtJtEW)M75(h>cBUv4+^Cn~kGvkUzAB40rx&v9@qzB|{pKN-eS7M4q`xP|#=Di}c
20S@~%OJ`>v5eFSzSk9w{Z|Bl32YJ_BibIB7NSm-7AJ0Rdz~*3(~8`)<H!&<)F`+H^m3Mg)76$b3N-a{Q9-WJI5prV3IfJY<oJp~n4b0=-RnvyRXYnm_4Ll@
#ZXnjAgR4i7DHil<b_9RWX9L}Jj<YbSp2t8V>RU=ozcTA4b*-V!|!El8YyA&*DEjVXws`Tp*e69_H<H786ZmLK2CrsQJ#DGdy<q#cUuG}i=Vg<_~y#YRzt0j
&C(H3%#P9AXn!Gw)qUk!fOZ^pX~(TNGHWGanKs4-d#xtCLs+Zu`oC!wmDZI%+4&TEdgOcehcSju3&Z6Cb9aFlYOeB8-q0HtXXL8@OzG#m-jvV=;^GpuZX9>g
IBIB<Kx5Q?(gwJ>&-qiD%zmK6={qWS#UIkjHY{QIr3C1BBF#NtV~DkK9Ru2?mxZ*4CDG1es~la>P*fTrTl?Az2Dhgb5(ujR;L%7$Rn)I0T5?lTlUe_U;*S+h
eghppQ5%aO*h+l?)U*h}1i+N8W)wOD)Ca(z3-B_#koy(XDF42f`2#TliQ6)(vD|fNf!K!*7cIuBi&irkD~&Xal-wt?X7>+Ep_ZLQGUYWOOQ(drU(7(&7se;x
G29(QaPHT$uJ1cp7c<2Kd(|H9f@|jR<tS5^IXHy~CQ?HO<T&Zy8F-Szvh_*LOx+mqUHR-l;j%}eE#gK4?yo@gej);GpO(nXJt)vYZ6Yv_1J+Il-HjFyREr5l
r%q#AlC;eAKys06evY0;y*OoID=NmLK5?$1s~$nQc3hdQ`pVM56QTEo7KeA6A^!{FSrB@H5}NFE?!95F_6Wyty^~(AN7{j_vF9k!m}5K1Tjc?LP4NEKyYhS7
$Z0h$U<7igB{F)p>i=d5t4_DRejaySkGgE&4Vz{E(JQ7AoyQka0oI)VkI9EJn;$ceV8k3=v{px5^PRLf3c@T(&E%2}j!gxWA|RTHjg+w3EH%4tu4BbKIMjl}
l#G7AZ;;zJw}ipH8rJchi0K~9K^K7V-ml7m6h7_Vl!*xj-=mymM5X^{WBG0_fq^NR-IjG-_le+;r6fgaodhF+p!M4l<&ofpdn?Ll`nR#f&eBg+4H0X_Ch9g_
Pr&8#9klJt469;L;H&}x|G67uCPNiHWQsdOh3sVi>=)G99DE2VmpC8GUC^G3#&B|9Z~GUoRfC(`yES0Rufc=LyLQzIF+9l<0nj0vK;of)T@=O%f)rBsPA$Vk
D2v*$Awk-S(6?PoL{baf&9`78YIR27*77c>#evA1qiZVlA^<Lj)-_6ue}tR~m>SCyCZe=(_W999RAskd6%pSJti+L;u@+ErC*o+y6*tpq808e?N0~Pjs4iDR
59YQ%b$6#-X5cOK5u|h&JEfE%QZ`NKu3P_7?40)vcOuFKJ|mvucGhds<E)%&02xpYDRO*@lKr{kK1MPH$B<r<$7Ak#@2JlbJS3R(Ovr5v4RkYgFj3~tvd)Db
Nyh~n+cji_iWAEdW`Y98hMS=3pw=!c9mN_=0^&G3O{x}?KJ8j%{PJ4*XLHqfuNqTR*Y1nC$tHiMF7UqU>V(M=+^w#WrRI2h7MBLRR7$FO9EYKfF5?>>6fHE3
G;IatK}IV$4#0i%hj#Z-aiX=pkC}x%vgL|RLzK{SdtiU+L9a%{l>VoXSfGw=P<tCDXZ!i;oD=j>+#fGBpN`>6f}R-809TE*A32z$xW1kSo#hsI^#E$p&|0&V
<U$+(f=9O%I7PIVBX!1uAG=}KC(0gihpvEA?HTV2o8w=;{EpQT^3vQ$S$~^t3z6dM*v5IcY*tVcYQKvtMoJa>H(^)`N6@@iWO<fv^jP-d^wf_S16nxnvIS=X
`F2YEM#9<kQ0?2WTuWENq>A1>B0Z2Nr%Dm4LqsfkG~cahq!pZ5YFP8*_LE*g=2AfNrrgb3VcPmFJap`tqtbC>*TS~TB%0bKXtD+W)xg8x?7_<!lJaBTdnsAf
^A}c-Ocl;xZpLLPMH3txVbWr-_2!FI1+xErbJonbNNmsfQ*wh2e*XUNa0=OUdtc2n$X#oK6fpn?Z<4FmWba(>yxFiTux#Z?3VT_*i5^o8GR{lRbk*-n|Eyo_
_d%h>m7!z_LF*|k{UGoJE693WXRPLOaJ=@+R`<ae{aWCh4x7$^FTK8&3>r+U)qb53U7Gg5?I;m->8=hb>5}z$Da5g9{^F3mny6ah_FtGJ14L2<q|G5XEK@ZD
nWfcwtWt@~5QtSf%Wm26L4G1MIQq|MTB`t+keKkpSh{auskzHt2v+e}na|J^+5oQ*d^v(}QBOPb+H;Dr@onpN2l~ITxGi>or^BfZX*tHZ+z+q6G`75XHt6Ze
K^el|JtqYU^Z<1FfD?sT-HLa4bCtv7uQ?YFx^nwgM@{eU5SSqg`?`{<mecC|dLBK%VDKm*Gw*2J6Ys4TNoFpbN!dyq17u?)XbWLqLG7TfTRjnAnzBo3H@%&y
AZ0LrR_4t>QJ~dcMZC)^$LBY+b%Ai*{FHRss`y3Tn(UtXBqqzQyI4)49({k>l09$Qx)+8)jTLEe^>eL)(p3AnL5T26qEiy#Z+wdLvNX^YFU^1jyXlc}0hLrh
NSML~dOg?wfhb0sz5a*7T5M?7KCgVZgAWefud1&$KVFx3J&m3Yc%`cdV8BK^_Ql1$^>AI<%~N*G{=NVJg0cNJK{&TO00G>{1kn2!)TslRvBYQl0ssI200dcD
'''.replace('\n', ''))))
print('gravado: /tmp/nutrelle_codigo/listas/treino_v2.csv')

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-u', '/tmp/nutrelle_codigo/deteccao_r5.py', '--saida', SAIDA, '--orcamento-h', str(ORCAMENTO_H)] + EXTRA
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in p.stdout:
    print(linha, end='')
print('código de saída:', p.wait())

In [ ]:
import json, os
r = os.path.join(SAIDA, 'resumo_r5.json')
if os.path.exists(r):
    for b, v in json.load(open(r))['bases'].items():
        print(b, {k: v.get(k) for k in ('fotos', 'por_metodo', 'taxa_sem_prato', 'iou_profundidade_gravada', 'iou_profundidade_refeita', 'diagnostico_uniao_so_prato', 'situacao')})
print(sorted(os.listdir(SAIDA)))